# Dynamical Systems — Radiation Analysis, Part 1

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Purpose.</b> This notebook applies dynamical-systems methods directly to processed radiation datasets using physically ordered transport coordinates.</p>
<p><b>Current block.</b> Cells 0–4 compute Lyapunov-spectrum descriptors for the processed N003 ISIS RB2000164 barite-enriched-concrete dataset.</p>
<p><b>Interpretation rule.</b> Neutron energy is an ordered physical coordinate, not clock time. The fitted exponents are therefore interpreted as local expansion/contraction rates along the ordered energy representation rather than as temporal-chaos claims.</p>
</div>


<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 0;line-height:1.6;border-left:3px solid #FF2B2B;">
<b>Cross-dataset execution.</b> The validated N003 Cells 0–28 now write into <code>results/N003/</code>. The atlas section below executes the Object-Bank-approved methods on all other structurally usable datasets, preserves transformed-coordinate metadata, separates dependent observables where needed, applies method-specific quality gates, and writes only scientifically valid result tables. Features that are model-dominated, sampling-geometry-dominated, unstable, or under-supported are retained as diagnostics but explicitly excluded from the later predictive feature pool.
</div>


# Lyapunov Spectrum Heatmap Visualization for Radiation Embedding Trajectories

<div style="font-size:14px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:20px;border-radius:8px;margin:10px;display:flex;justify-content:space-between;gap:24px;">

<div style="width:50%;">
<h2 style="color:#FF4D4D;">Introduction</h2>
<p>
This block estimates and visualizes the Lyapunov spectrum for real processed radiation-response curves.
The present radiation object is the N003 ISIS RB2000164 barite-enriched-concrete dataset.
Each measured sample-response curve is ordered by neutron energy and delay-embedded at dimensions 2 through 10,
using embedding dimensions 2 through 10.
</p>

<h2 style="color:#FF4D4D;">Mathematical Foundation</h2>
<p>
For an embedding of dimension <b>D</b>, the reconstructed trajectory has a Lyapunov spectrum
<b>λ₁, …, λ<sub>D</sub></b>. The local-linear Jacobian and QR-accumulation formulation is retained.
</p>
<p>
The radiation coordinate is <b>s = ln(E / E<sub>min</sub>)</b>, where <b>E</b> is neutron energy.
The estimated exponents therefore describe expansion or contraction per unit of ordered log-energy coordinate, not per unit time.
</p>

<h2 style="color:#FF4D4D;">Radiation Data</h2>
<p>The notebook reads the processed project-repository file:</p>
<p><code>data/processed/RB2000164/rb2000164_physics_ready_strict.parquet</code></p>
<p>
The data are grouped by sample identity and the physically meaningful response curves are preserved,
including transmission <b>T(E)</b> and, where present, the grounded macroscopic removal cross section
<b>Σ<sub>R</sub>(E)</b>.
</p>
</div>

<div style="width:50%;">
<h2 style="color:#FF4D4D;">Analysis Procedure</h2>
<ol>
<li>Load the processed RB2000164 Parquet table.</li>
<li>Group by measured sample.</li>
<li>Order each response by neutron energy.</li>
<li>Average exact duplicate energy coordinates only.</li>
<li>Resample onto a uniform log-energy coordinate so the QR rate has a constant coordinate increment.</li>
<li>Robustly standardize each measured response for numerical conditioning.</li>
<li>Delay-embed at dimensions 2 through 10.</li>
<li>Estimate local Jacobians from nearest-neighbor state displacements.</li>
<li>Accumulate QR stretch factors to estimate the spectrum.</li>
<li>Render the diagnostics using the black/red plotting style.</li>
</ol>

<h2 style="color:#FF4D4D;">Physical Interpretation</h2>
<p>
Positive fitted exponents indicate local separation of nearby reconstructed radiation states as the ordered energy coordinate advances;
negative values indicate contraction. Because the independent coordinate is neutron energy, these values are not automatically evidence of temporal chaos.
</p>
</div>

</div>


<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.6;">

<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Local-Linear Lyapunov Spectrum with QR Accumulation</h3>

<p>
A delay-reconstructed radiation trajectory can be linearized locally by fitting a tangent map from nearby state displacements.
If neighboring offsets satisfy <b>ΔX(k+1) ≈ J(k)ΔX(k)</b>, least squares gives a finite-data estimate of the local Jacobian <b>J(k)</b>.
</p>

<p>
Repeated products of these local maps are numerically stabilized with QR orthogonalization.
If <b>J(k)Q(k) = Q(k+1)R(k)</b>, the logarithms of the absolute diagonal elements of <b>R(k)</b>
accumulate the Lyapunov exponents.
</p>

<p>
For this radiation analysis, the ordered coordinate is the uniformly sampled dimensionless log-energy coordinate
<b>s = ln(E / E<sub>min</sub>)</b>. The resulting rates therefore describe local divergence or contraction
along ordered neutron energy.
</p>

</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Core idea.</b> A positive fitted exponent means nearby reconstructed radiation states separate as the ordered log-energy coordinate advances; a negative exponent means contraction. Multiple exponents describe stretching and contraction along different directions of the delay-embedded response state.</p>
<p><b>Physical caution.</b> Neutron energy is treated as an ordered transport coordinate, not as clock time.</p>
</div>


In [ ]:
import os
import glob
import re
import time
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

try:
    from IPython.display import display
    _HAVE_IPYTHON_DISPLAY = True
except Exception:
    display = None
    _HAVE_IPYTHON_DISPLAY = False

try:
    from scipy.spatial import cKDTree as KDTree
    _HAVE_KDTREE = True
except Exception:
    KDTree = None
    _HAVE_KDTREE = False


RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
GRAY = "#445555"
BLACK = "#000000"

RED_DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "red_diverging",
    [RED_DIM, BLACK, RED_SOFT],
    N=256,
)

SHOW_PLOTS_IN_NOTEBOOK = True
SAVE_PLOTS_TO_DISK = True

plt.rcParams.update({
    "figure.facecolor": BLACK,
    "axes.facecolor": BLACK,
    "savefig.facecolor": BLACK,
    "axes.edgecolor": RED,
    "axes.labelcolor": RED,
    "xtick.color": RED,
    "ytick.color": RED,
    "text.color": RED,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "axes.titlecolor": RED,
    "legend.facecolor": BLACK,
    "legend.edgecolor": RED,
    "legend.labelcolor": RED,
    "font.size": 11,
})


def _delay_embed_1d(x: np.ndarray, emb_dim: int, tau: int) -> np.ndarray:
    x = np.asarray(x, dtype=float).reshape(-1)
    n = x.shape[0]
    n_eff = n - (emb_dim - 1) * tau

    if n_eff <= 1:
        raise ValueError("Time series too short for requested embedding.")

    return np.column_stack([x[i:i + n_eff] for i in range(0, emb_dim * tau, tau)])


def _maybe_fast_subsample(X, dt, max_points=None):
    if max_points is None or X.shape[0] <= max_points:
        return X, dt, 1

    step = int(np.ceil(X.shape[0] / max_points))
    return X[::step], dt * step, step


def lyapunov_spectrum_from_trajectory(
    X: np.ndarray,
    k_neighbors: int = 25,
    theiler: int = 10,
    stride: int = 1,
    dt: float = 1.0,
    rcond=None,
    max_steps=None,
    query_extra: int = 80,
    chunk_size: int = 1024,
    random_seed: int = 0,
):
    X = np.asarray(X, dtype=float)

    if X.ndim != 2:
        raise ValueError("X must be 2D: (N, D).")

    n, d = X.shape

    if n < 5:
        raise ValueError("Trajectory too short.")

    X0 = X[:-1]
    X1 = X[1:]
    n0 = X0.shape[0]

    if n0 < max(d + 2, k_neighbors + 2):
        raise ValueError(
            f"Trajectory too short for dimension={d} and k_neighbors={k_neighbors}."
        )

    anchor_idx = np.arange(0, n0 - 1, stride)

    if anchor_idx.size == 0:
        raise ValueError("No anchor points available. Try lowering stride.")

    if max_steps is not None and anchor_idx.size > max_steps:
        rng = np.random.default_rng(random_seed)
        anchor_idx = np.sort(rng.choice(anchor_idx, size=max_steps, replace=False))

    Q = np.eye(d)
    sums = np.zeros(d, dtype=float)
    used = 0

    tree = KDTree(X0) if _HAVE_KDTREE else None

    if tree is not None:
        qk = min(
            n0,
            max(
                k_neighbors + query_extra,
                k_neighbors * 4 + 2 * theiler + 5,
                d + query_extra,
            ),
        )

        for start in range(0, anchor_idx.size, chunk_size):
            inds = anchor_idx[start:start + chunk_size]
            points = X0[inds]

            try:
                _, all_nbrs = tree.query(points, k=qk, workers=-1)
            except TypeError:
                _, all_nbrs = tree.query(points, k=qk)

            all_nbrs = np.asarray(all_nbrs)

            if all_nbrs.ndim == 1:
                all_nbrs = all_nbrs[:, None]

            for row, i in zip(all_nbrs, inds):
                row = np.asarray(row)

                nbrs = row[
                    (row >= 0)
                    & (row < n0 - 1)
                    & (np.abs(row - i) > theiler)
                ]

                if nbrs.size < max(d + 1, k_neighbors):
                    continue

                nbrs = nbrs[:k_neighbors]

                A = X0[nbrs] - X0[i]
                B = X1[nbrs] - X1[i]

                try:
                    JT, *_ = np.linalg.lstsq(A, B, rcond=rcond)
                    J = JT.T
                except np.linalg.LinAlgError:
                    continue

                Z = J @ Q
                Q, R = np.linalg.qr(Z)

                diag = np.abs(np.diag(R))
                diag[diag == 0] = np.finfo(float).tiny

                sums += np.log(diag)
                used += 1

    else:
        print("WARNING: scipy cKDTree not available. Using slower numpy fallback.", flush=True)

        for i in anchor_idx:
            diffs = X0 - X0[i]
            d2 = np.einsum("ij,ij->i", diffs, diffs)

            lo = max(0, i - theiler)
            hi = min(n0, i + theiler + 1)
            d2[lo:hi] = np.inf
            d2[n0 - 1:] = np.inf

            kk = min(
                n0,
                max(
                    k_neighbors + query_extra,
                    k_neighbors * 4 + 2 * theiler + 5,
                    d + query_extra,
                ),
            )

            cand = np.argpartition(d2, kk - 1)[:kk]
            cand = cand[np.argsort(d2[cand])]

            nbrs = cand[np.isfinite(d2[cand])][:k_neighbors]

            if nbrs.size < max(d + 1, k_neighbors):
                continue

            A = X0[nbrs] - X0[i]
            B = X1[nbrs] - X1[i]

            try:
                JT, *_ = np.linalg.lstsq(A, B, rcond=rcond)
                J = JT.T
            except np.linalg.LinAlgError:
                continue

            Z = J @ Q
            Q, R = np.linalg.qr(Z)

            diag = np.abs(np.diag(R))
            diag[diag == 0] = np.finfo(float).tiny

            sums += np.log(diag)
            used += 1

    if used == 0:
        raise ValueError(
            "No usable steps. Try lowering theiler, lowering k_neighbors, "
            "lowering stride, increasing MAX_STEPS_PER_FILE, or inspecting the data."
        )

    exps = sums / (used * dt)
    return exps, used


def infer_embedding_dim_from_name(stem: str, X: np.ndarray) -> int:
    m = re.match(r"^(\d+)dembedded_", stem)

    if m:
        return int(m.group(1))

    if X.ndim == 2:
        return int(X.shape[1])

    return np.nan


def infer_channel_name_from_stem(stem: str) -> str:
    if "dembedded_" in stem:
        return stem.split("dembedded_", 1)[1]

    return stem


def gather_embedding_files(embedding_root: str):
    """
    Gather the transient Cell-4 embedding arrays from the actual embedding_root.

    Expected layout:
      embedding_root/
        2dembedding_data/
        3dembedding_data/
        embeddings_4to10/
    """
    search_dirs = [
        os.path.join(embedding_root, "2dembedding_data"),
        os.path.join(embedding_root, "3dembedding_data"),
        os.path.join(embedding_root, "embeddings_4to10"),
    ]

    all_files = []
    existing_dirs = []

    for d in search_dirs:
        if os.path.isdir(d):
            existing_dirs.append(d)
            all_files.extend(sorted(glob.glob(os.path.join(d, "*.npy"))))

    return existing_dirs, sorted(all_files)


def save_checkpoint(records, out_csv):
    base_columns = [
        "file",
        "stem",
        "path",
        "source_dir",
        "channel",
        "embedding_dimension",
        "status",
        "error",
        "n_samples_original",
        "n_samples_used",
        "state_dim",
        "subsample_step",
        "dt_used",
        "steps_used",
        "elapsed_sec",
    ]

    df = pd.DataFrame(records)

    for col in base_columns:
        if col not in df.columns:
            df[col] = np.nan

    other_cols = [c for c in df.columns if c not in base_columns]

    exp_cols = sorted(
        [c for c in other_cols if c.startswith("Exp")],
        key=lambda x: int(x.replace("Exp", "")) if x.replace("Exp", "").isdigit() else 10**9,
    )

    non_exp_other_cols = [c for c in other_cols if c not in exp_cols]

    df = df[base_columns + non_exp_other_cols + exp_cols]
    df.to_csv(out_csv, index=False)

    return df


def get_exp_cols(df):
    return sorted(
        [c for c in df.columns if c.startswith("Exp")],
        key=lambda x: int(x.replace("Exp", "")) if x.replace("Exp", "").isdigit() else 10**9,
    )


def kaplan_yorke_dimension(exps):
    exps = np.asarray(exps, dtype=float)
    exps = exps[np.isfinite(exps)]

    if exps.size == 0:
        return np.nan

    lam = np.sort(exps)[::-1]
    csum = np.cumsum(lam)

    if csum[0] < 0:
        return 0.0

    nonneg = np.where(csum >= 0)[0]

    if nonneg.size == 0:
        return np.nan

    j_idx = nonneg[-1]
    j = j_idx + 1

    if j >= lam.size:
        return float(lam.size)

    denom = abs(lam[j])

    if denom == 0 or not np.isfinite(denom):
        return float(j)

    return float(j + csum[j_idx] / denom)


def spectral_entropy_from_positive_exps(exps):
    exps = np.asarray(exps, dtype=float)
    pos = exps[np.isfinite(exps) & (exps > 0)]

    if pos.size <= 1:
        return 0.0

    p = pos / np.sum(pos)
    return float(-np.sum(p * np.log(p)) / np.log(pos.size))


def add_expert_metrics(df):
    df = df.copy()
    exp_cols = get_exp_cols(df)

    metric_rows = []

    for _, row in df.iterrows():
        exps = row[exp_cols].to_numpy(dtype=float) if len(exp_cols) else np.array([])
        exps = exps[np.isfinite(exps)]

        if row.get("status", "") != "ok" or exps.size == 0:
            metric_rows.append({
                "LLE": np.nan,
                "MLE": np.nan,
                "most_negative_exp": np.nan,
                "sum_exponents": np.nan,
                "mean_exponent": np.nan,
                "std_exponent": np.nan,
                "spectral_radius_abs": np.nan,
                "positive_count": np.nan,
                "negative_count": np.nan,
                "near_zero_count": np.nan,
                "KS_entropy_proxy": np.nan,
                "Kaplan_Yorke_dim": np.nan,
                "expansion_entropy": np.nan,
                "chaos_timescale": np.nan,
                "hyperchaos_flag": False,
                "dissipative_flag": False,
                "reliable_steps_flag": False,
                "quality_score": np.nan,
            })
            continue

        sorted_exps = np.sort(exps)[::-1]

        lle = float(np.max(exps))
        mle = float(np.mean(sorted_exps[:min(3, len(sorted_exps))]))
        most_negative = float(np.min(exps))
        sum_exps = float(np.sum(exps))
        mean_exp = float(np.mean(exps))
        std_exp = float(np.std(exps))
        spectral_radius_abs = float(np.max(np.abs(exps)))

        eps_zero = 1e-3
        positive_count = int(np.sum(exps > eps_zero))
        negative_count = int(np.sum(exps < -eps_zero))
        near_zero_count = int(np.sum(np.abs(exps) <= eps_zero))

        ks_entropy_proxy = float(np.sum(exps[exps > 0])) if np.any(exps > 0) else 0.0
        ky_dim = kaplan_yorke_dimension(exps)
        expansion_entropy = spectral_entropy_from_positive_exps(exps)

        chaos_timescale = float(1.0 / lle) if lle > 0 else np.nan

        hyperchaos_flag = positive_count >= 2
        dissipative_flag = sum_exps < 0

        steps_used = row.get("steps_used", np.nan)
        state_dim = row.get("state_dim", np.nan)

        reliable_steps_flag = (
            np.isfinite(steps_used)
            and np.isfinite(state_dim)
            and steps_used >= max(100, 20 * state_dim)
        )

        subsample_step = row.get("subsample_step", 1)

        if not np.isfinite(subsample_step) or subsample_step <= 0:
            subsample_step = 1

        if np.isfinite(steps_used) and np.isfinite(state_dim):
            step_component = min(1.0, steps_used / max(100, 20 * state_dim))
            subsample_component = 1.0 / np.sqrt(subsample_step)
            quality_score = float(100 * step_component * subsample_component)
        else:
            quality_score = np.nan

        metric_rows.append({
            "LLE": lle,
            "MLE": mle,
            "most_negative_exp": most_negative,
            "sum_exponents": sum_exps,
            "mean_exponent": mean_exp,
            "std_exponent": std_exp,
            "spectral_radius_abs": spectral_radius_abs,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "near_zero_count": near_zero_count,
            "KS_entropy_proxy": ks_entropy_proxy,
            "Kaplan_Yorke_dim": ky_dim,
            "expansion_entropy": expansion_entropy,
            "chaos_timescale": chaos_timescale,
            "hyperchaos_flag": hyperchaos_flag,
            "dissipative_flag": dissipative_flag,
            "reliable_steps_flag": reliable_steps_flag,
            "quality_score": quality_score,
        })

    metrics = pd.DataFrame(metric_rows, index=df.index)
    return pd.concat([df, metrics], axis=1)


def style_axis(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(BLACK)

    for spine in ax.spines.values():
        spine.set_color(RED)
        spine.set_linewidth(1.1)

    ax.tick_params(axis="x", colors=RED, labelcolor=RED)
    ax.tick_params(axis="y", colors=RED, labelcolor=RED)

    ax.xaxis.label.set_color(RED)
    ax.yaxis.label.set_color(RED)

    if title is not None:
        ax.set_title(title, color=RED, pad=12)

    if xlabel is not None:
        ax.set_xlabel(xlabel, color=RED)

    if ylabel is not None:
        ax.set_ylabel(ylabel, color=RED)

    ax.grid(True, color=RED_DIM, alpha=0.25, linewidth=0.7)


def save_fig(fig, path):
    fig.tight_layout()

    if SAVE_PLOTS_TO_DISK:
        fig.savefig(path, dpi=220, facecolor=BLACK, bbox_inches="tight")

    if SHOW_PLOTS_IN_NOTEBOOK and _HAVE_IPYTHON_DISPLAY:
        display(fig)

    plt.close(fig)


def _safe_filename(s):
    s = str(s)
    s = re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", s)
    return s[:180]


def _short_labels(series, max_len=22):
    labels = []
    for s in series.astype(str).to_numpy():
        labels.append(s if len(s) <= max_len else s[:max_len - 1] + "…")
    return np.array(labels)


def plot_lle_rank(df, out_dir, top_n=35):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["LLE"])]

    if ok.empty:
        return None

    ok = ok.sort_values("LLE", ascending=False).head(top_n)
    ok = ok.sort_values("LLE", ascending=True)

    reliable = (
        ok["reliable_steps_flag"].astype(bool).to_numpy()
        if "reliable_steps_flag" in ok.columns
        else np.ones(len(ok), dtype=bool)
    )
    labels = [
        f"{str(ch)[:18]} | d={int(dim)}" + ("" if rel else " [QC-low]")
        for ch, dim, rel in zip(ok["channel"], ok["embedding_dimension"], reliable)
    ]
    values = ok["LLE"].to_numpy()

    fig, ax = plt.subplots(figsize=(11, 8))

    y = np.arange(len(ok))
    colors = [RED if rel else RED_DIM for rel in reliable]
    ax.barh(y, values, color=colors, edgecolor=RED_SOFT, alpha=0.85)
    ax.axvline(0, color=WHITE, linewidth=1.0, alpha=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, color=RED, fontsize=8)

    style_axis(
        ax,
        title=f"Top {len(ok)} Curve × Embedding-Dimension Results by Largest Lyapunov Exponent",
        xlabel="Largest Lyapunov exponent",
        ylabel="Channel × embedding dimension",
    )

    ax.margins(y=0.01)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_01_top_lle_ranking_fixed_height_black_red.png")
    save_fig(fig, path)
    return path


def plot_ks_entropy_vs_dimension(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[
        np.isfinite(ok["KS_entropy_proxy"])
        & np.isfinite(ok["Kaplan_Yorke_dim"])
        & np.isfinite(ok["quality_score"])
    ]

    if ok.empty:
        return None

    fig, ax = plt.subplots(figsize=(10, 7))

    sizes = np.clip(ok["quality_score"].to_numpy(), 18, 180)

    ax.scatter(
        ok["Kaplan_Yorke_dim"],
        ok["KS_entropy_proxy"],
        s=sizes,
        c=RED,
        edgecolors=WHITE,
        linewidths=0.7,
        alpha=0.78,
    )

    for _, r in ok.sort_values("KS_entropy_proxy", ascending=False).head(8).iterrows():
        ax.annotate(
            str(r["channel"])[:22],
            (r["Kaplan_Yorke_dim"], r["KS_entropy_proxy"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            color=RED_SOFT,
        )

    style_axis(
        ax,
        title="Ordered-Coordinate Complexity Map",
        xlabel="Kaplan-Yorke dimension",
        ylabel="KS entropy proxy: sum of positive exponents",
    )

    # Clean conceptual legend:
    # - one entry for the red radiation-curve markers
    # - one entry explaining that marker area encodes quality score
    # This avoids an unreadable legend containing one item per curve.
    from matplotlib.lines import Line2D

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor=RED,
            markeredgecolor=WHITE,
            markersize=8,
            label="Processed radiation curve",
        ),
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor=RED_SOFT,
            markersize=10,
            label="Marker size = quality score",
        ),
    ]

    leg = ax.legend(
        handles=legend_handles,
        loc="best",
        frameon=True,
        facecolor=BLACK,
        edgecolor=RED,
    )
    for text in leg.get_texts():
        text.set_color(RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_02_complexity_map_ks_vs_ky_black_red.png")
    save_fig(fig, path)
    return path


def plot_spectrum_heatmap(df, out_dir, max_rows=45):
    ok = df[df["status"] == "ok"].copy()
    exp_cols = get_exp_cols(ok)

    if ok.empty or len(exp_cols) == 0:
        return None

    ok = ok[np.isfinite(ok["LLE"])].copy()

    if ok.empty:
        return None

    ok = ok.sort_values("LLE", ascending=False).head(max_rows)
    ok = ok.sort_values(["embedding_dimension", "LLE"], ascending=[True, False])

    M = ok[exp_cols].to_numpy(dtype=float)

    if M.size == 0 or np.all(~np.isfinite(M)):
        return None

    reliable = (
        ok["reliable_steps_flag"].astype(bool).to_numpy()
        if "reliable_steps_flag" in ok.columns
        else np.ones(len(ok), dtype=bool)
    )
    labels = [
        f"{str(ch)[:17]} | d={int(dim)}" + ("" if rel else " [QC-low]")
        for ch, dim, rel in zip(ok["channel"], ok["embedding_dimension"], reliable)
    ]

    finite_abs = np.abs(M[np.isfinite(M)])
    vmax = np.percentile(finite_abs, 95) if finite_abs.size else 1.0
    vmax = max(vmax, 1e-9)

    fig, ax = plt.subplots(figsize=(11, 8))

    im = ax.imshow(
        M,
        aspect="auto",
        interpolation="nearest",
        cmap=RED_DIVERGING_CMAP,
        vmin=-vmax,
        vmax=vmax,
    )

    ax.set_xticks(np.arange(len(exp_cols)))
    ax.set_xticklabels(exp_cols, rotation=45, ha="right", color=RED)

    ax.set_yticks(np.arange(len(ok)))
    ax.set_yticklabels(labels, color=RED, fontsize=7)

    style_axis(
        ax,
        title=f"Lyapunov Spectrum Heatmap: Top {len(ok)} Curve × Dimension Results by LLE",
        xlabel="Exponent index",
        ylabel="Channel × embedding dimension",
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=RED)
    plt.setp(cbar.ax.get_yticklabels(), color=RED)
    cbar.outline.set_edgecolor(RED)
    cbar.set_label("Lyapunov exponent", color=RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_03_spectrum_heatmap_fixed_height_black_red.png")
    save_fig(fig, path)
    return path


def plot_mean_spectrum_by_embedding_dim(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    exp_cols = get_exp_cols(ok)

    if ok.empty or len(exp_cols) == 0:
        return None

    dims = sorted([d for d in ok["embedding_dimension"].dropna().unique()])

    if len(dims) == 0:
        return None

    fig, ax = plt.subplots(figsize=(10, 7))

    max_dim = max(dims)

    for dim in dims:
        sub = ok[ok["embedding_dimension"] == dim]
        M = sub[exp_cols].to_numpy(dtype=float)

        if M.size == 0:
            continue

        mean = np.nanmean(M, axis=0)
        x = np.arange(1, len(mean) + 1)

        valid = np.isfinite(mean)
        if valid.sum() < 2:
            continue

        relfrac = (
            float(sub["reliable_steps_flag"].astype(bool).mean())
            if "reliable_steps_flag" in sub.columns
            else 1.0
        )
        qc_low = relfrac < 0.8

        ax.plot(
            x[valid],
            mean[valid],
            marker="o",
            linewidth=1.8,
            markersize=4,
            color=RED_DIM if qc_low else RED,
            linestyle=":" if qc_low else "-",
            alpha=0.55 if qc_low else (0.35 + 0.55 * (dim / max_dim)),
            label=f"{int(dim)}D" + (" [QC-low]" if qc_low else ""),
        )

    ax.axhline(0, color=WHITE, linewidth=1.0, alpha=0.8)

    style_axis(
        ax,
        title="Mean Lyapunov Spectrum by Embedding Dimension",
        xlabel="Exponent rank",
        ylabel="Mean exponent",
    )

    leg = ax.legend(loc="best", frameon=True)
    for text in leg.get_texts():
        text.set_color(RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_04_mean_spectrum_by_embedding_dim_black_red.png")
    save_fig(fig, path)
    return path


def plot_dissipation_vs_instability(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["sum_exponents"]) & np.isfinite(ok["LLE"])]

    if ok.empty:
        return None

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.scatter(
        ok["sum_exponents"],
        ok["LLE"],
        s=np.clip(ok["quality_score"].fillna(40).to_numpy(), 20, 180),
        c=RED,
        edgecolors=WHITE,
        linewidths=0.7,
        alpha=0.82,
    )

    ax.axhline(0, color=WHITE, linewidth=1.0, alpha=0.75)
    ax.axvline(0, color=WHITE, linewidth=1.0, alpha=0.75)

    for _, r in ok.sort_values("LLE", ascending=False).head(8).iterrows():
        ax.annotate(
            str(r["channel"])[:22],
            (r["sum_exponents"], r["LLE"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            color=RED_SOFT,
        )

    style_axis(
        ax,
        title="Dissipation vs Instability Phase Plot",
        xlabel="Sum of Lyapunov exponents: volume expansion/contraction",
        ylabel="Largest Lyapunov exponent",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_05_dissipation_vs_instability_black_red.png")
    save_fig(fig, path)
    return path


def plot_runtime_qc_sorted(df, out_dir, top_n=35):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["elapsed_sec"])]

    if ok.empty:
        return None

    ok = ok.sort_values("elapsed_sec", ascending=False).head(top_n)
    ok = ok.sort_values("elapsed_sec", ascending=True)

    labels = _short_labels(ok["channel"], max_len=24)
    y = np.arange(len(ok))

    fig, ax = plt.subplots(figsize=(11, 8))

    ax.barh(
        y,
        ok["elapsed_sec"].to_numpy(),
        color=RED,
        edgecolor=RED_SOFT,
        alpha=0.82,
    )

    ax.set_yticks(y)
    ax.set_yticklabels(labels, color=RED, fontsize=8)

    median_time = ok["elapsed_sec"].median()
    ax.axvline(median_time, color=WHITE, linewidth=1.0, alpha=0.75, linestyle="--")

    for yi, (_, r) in zip(y, ok.iterrows()):
        steps = r.get("steps_used", np.nan)
        if np.isfinite(steps):
            ax.text(
                r["elapsed_sec"],
                yi,
                f"  {int(steps)} steps",
                va="center",
                ha="left",
                color=RED_SOFT,
                fontsize=7,
            )

    style_axis(
        ax,
        title=f"Runtime QC: Slowest {len(ok)} Files",
        xlabel="Elapsed seconds per file",
        ylabel="Channel",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_06_runtime_qc_slowest_files_fixed_black_red.png")
    save_fig(fig, path)
    return path


def plot_runtime_distribution_by_dimension(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[
        np.isfinite(ok["elapsed_sec"])
        & np.isfinite(ok["embedding_dimension"])
    ]

    if ok.empty:
        return None

    dims = sorted(ok["embedding_dimension"].dropna().unique())
    fig, ax = plt.subplots(figsize=(10, 7))

    rng = np.random.default_rng(0)

    for dim in dims:
        sub = ok[ok["embedding_dimension"] == dim]
        x = np.full(len(sub), dim, dtype=float)
        jitter = rng.normal(0, 0.035, size=len(sub))
        ax.scatter(
            x + jitter,
            sub["elapsed_sec"],
            s=45,
            c=RED,
            edgecolors=WHITE,
            linewidths=0.5,
            alpha=0.68,
        )

        median_val = sub["elapsed_sec"].median()
        ax.plot(
            [dim - 0.18, dim + 0.18],
            [median_val, median_val],
            color=WHITE,
            linewidth=2.0,
            alpha=0.9,
        )

    style_axis(
        ax,
        title="Runtime Distribution by Embedding Dimension",
        xlabel="Embedding dimension",
        ylabel="Elapsed seconds per file",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_07_runtime_distribution_by_dimension_black_red.png")
    save_fig(fig, path)
    return path


def plot_channel_expert_summary(df, out_dir, top_n=20):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["LLE"])]

    if ok.empty:
        return None

    top = ok.sort_values("LLE", ascending=False).head(top_n).copy()
    reliable = (
        top["reliable_steps_flag"].astype(bool).to_numpy()
        if "reliable_steps_flag" in top.columns
        else np.ones(len(top), dtype=bool)
    )
    labels = [
        f"{str(ch)[:13]} | d={int(dim)}" + ("" if rel else " [QC-low]")
        for ch, dim, rel in zip(top["channel"], top["embedding_dimension"], reliable)
    ]

    x = np.arange(len(top))
    width = 0.28

    fig, ax = plt.subplots(figsize=(13, 7))

    ax.bar(
        x - width,
        top["LLE"],
        width,
        label="LLE",
        color=[RED if rel else RED_DIM for rel in reliable],
        edgecolor=RED_SOFT,
        alpha=0.85,
    )

    ax.bar(
        x,
        top["KS_entropy_proxy"],
        width,
        label="KS entropy proxy",
        color=RED_SOFT,
        edgecolor=RED,
        alpha=0.60,
    )

    ax.bar(
        x + width,
        top["Kaplan_Yorke_dim"],
        width,
        label="Kaplan-Yorke dim",
        color=RED_DIM,
        edgecolor=RED,
        alpha=0.75,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=65, ha="right", color=RED, fontsize=8)

    style_axis(
        ax,
        title=f"Top {top_n} Curve × Embedding-Dimension Results: Instability, Entropy Proxy, Dimension",
        xlabel="Channel × embedding dimension",
        ylabel="Metric value",
    )

    leg = ax.legend(loc="best", frameon=True)
    for text in leg.get_texts():
        text.set_color(RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_08_top_radiation curves_expert_summary_black_red.png")
    save_fig(fig, path)
    return path


def plot_embedding_dimension_summary(df, out_dir):
    ok = df[df["status"] == "ok"].copy()

    needed = ["embedding_dimension", "LLE", "KS_entropy_proxy", "Kaplan_Yorke_dim"]

    for col in needed:
        if col not in ok.columns:
            return None
        ok = ok[pd.to_numeric(ok[col], errors="coerce").notna()]

    if ok.empty:
        return None

    ok["_plot_reliable"] = (
        ok["reliable_steps_flag"].astype(bool)
        if "reliable_steps_flag" in ok.columns
        else True
    )

    grouped = (
        ok.groupby("embedding_dimension")
        .agg(
            LLE_mean=("LLE", "mean"),
            LLE_std=("LLE", "std"),
            KS_mean=("KS_entropy_proxy", "mean"),
            KY_mean=("Kaplan_Yorke_dim", "mean"),
            n=("file", "count"),
            reliable_fraction=("_plot_reliable", "mean"),
        )
        .reset_index()
        .sort_values("embedding_dimension")
    )

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.errorbar(
        grouped["embedding_dimension"],
        grouped["LLE_mean"],
        yerr=grouped["LLE_std"].fillna(0),
        marker="o",
        linewidth=2,
        markersize=6,
        color=RED,
        ecolor=RED_SOFT,
        capsize=4,
        label="Mean LLE ± SD",
    )

    ax.plot(
        grouped["embedding_dimension"],
        grouped["KS_mean"],
        marker="s",
        linewidth=1.8,
        markersize=5,
        color=RED_SOFT,
        alpha=0.8,
        label="Mean KS entropy proxy",
    )

    ax.plot(
        grouped["embedding_dimension"],
        grouped["KY_mean"],
        marker="^",
        linewidth=1.8,
        markersize=5,
        color=WHITE,
        alpha=0.75,
        label="Mean Kaplan-Yorke dim",
    )

    bad = grouped["reliable_fraction"] < 0.8
    if bad.any():
        for dim in grouped.loc[bad, "embedding_dimension"].to_numpy(float):
            ax.axvspan(dim - 0.42, dim + 0.42, color=RED_DIM, alpha=0.10)
        ax.scatter(
            grouped.loc[bad, "embedding_dimension"],
            grouped.loc[bad, "LLE_mean"],
            s=95,
            facecolors="none",
            edgecolors=WHITE,
            linewidths=1.2,
            label="QC-low embedding dimension",
            zorder=5,
        )

    ax.axhline(0, color=WHITE, linewidth=1.0, alpha=0.6)

    style_axis(
        ax,
        title="Embedding Dimension Sensitivity Summary",
        xlabel="Embedding dimension",
        ylabel="Aggregate metric",
    )

    leg = ax.legend(loc="best", frameon=True)
    for text in leg.get_texts():
        text.set_color(RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_09_embedding_dimension_summary_black_red.png")
    save_fig(fig, path)
    return path


def plot_metric_correlation_matrix(df, out_dir):
    ok = df[df["status"] == "ok"].copy()

    metric_cols = [
        "LLE",
        "MLE",
        "most_negative_exp",
        "sum_exponents",
        "mean_exponent",
        "std_exponent",
        "spectral_radius_abs",
        "positive_count",
        "negative_count",
        "near_zero_count",
        "KS_entropy_proxy",
        "Kaplan_Yorke_dim",
        "expansion_entropy",
        "chaos_timescale",
        "quality_score",
        "elapsed_sec",
        "steps_used",
    ]

    metric_cols = [c for c in metric_cols if c in ok.columns]

    if len(metric_cols) < 3:
        return None

    M = ok[metric_cols].apply(pd.to_numeric, errors="coerce")
    corr = M.corr()

    if corr.empty:
        return None

    fig, ax = plt.subplots(figsize=(11, 9))

    im = ax.imshow(
        corr.to_numpy(),
        aspect="auto",
        interpolation="nearest",
        cmap=RED_DIVERGING_CMAP,
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(np.arange(len(metric_cols)))
    ax.set_yticks(np.arange(len(metric_cols)))
    ax.set_xticklabels(metric_cols, rotation=65, ha="right", color=RED, fontsize=8)
    ax.set_yticklabels(metric_cols, color=RED, fontsize=8)

    style_axis(
        ax,
        title="Expert Metric Correlation Matrix",
        xlabel="Metric",
        ylabel="Metric",
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=RED)
    plt.setp(cbar.ax.get_yticklabels(), color=RED)
    cbar.outline.set_edgecolor(RED)
    cbar.set_label("Pearson correlation", color=RED)

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_10_metric_correlation_matrix_black_red.png")
    save_fig(fig, path)
    return path


def plot_lle_distribution(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["LLE"])]

    if ok.empty:
        return None

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.hist(
        ok["LLE"].to_numpy(),
        bins=min(40, max(8, int(np.sqrt(len(ok))))),
        color=RED,
        edgecolor=RED_SOFT,
        alpha=0.75,
    )

    ax.axvline(0, color=WHITE, linewidth=1.2, alpha=0.85)
    ax.axvline(ok["LLE"].median(), color=RED_SOFT, linewidth=1.5, linestyle="--", alpha=0.9)

    style_axis(
        ax,
        title="Distribution of Largest Lyapunov Exponents",
        xlabel="Largest Lyapunov exponent",
        ylabel="Count",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_11_lle_distribution_black_red.png")
    save_fig(fig, path)
    return path


def plot_top_spectra_overlay(df, out_dir, top_n=18, min_exponents=3):
    ok = df[df["status"] == "ok"].copy()
    exp_cols = get_exp_cols(ok)

    if ok.empty or len(exp_cols) == 0:
        return None

    ok = ok[np.isfinite(ok["LLE"])].sort_values("LLE", ascending=False)

    selected_rows = []

    for _, row in ok.iterrows():
        exps = row[exp_cols].to_numpy(dtype=float)
        exps = exps[np.isfinite(exps)]

        if exps.size >= min_exponents:
            selected_rows.append(row)

        if len(selected_rows) >= top_n:
            break

    if len(selected_rows) == 0:
        print(
            "Skipping top spectra overlay: all successful spectra have fewer than "
            f"{min_exponents} exponents, so individual line spectra would be uninformative.",
            flush=True,
        )
        return None

    selected = pd.DataFrame(selected_rows)

    fig, ax = plt.subplots(figsize=(11, 7))

    for rank, (_, row) in enumerate(selected.iterrows(), start=1):
        exps = row[exp_cols].to_numpy(dtype=float)
        exps = exps[np.isfinite(exps)]
        exps_sorted = np.sort(exps)[::-1]
        x = np.arange(1, len(exps_sorted) + 1)

        reliable = bool(row.get("reliable_steps_flag", True))
        alpha = max(0.25, 1.0 - 0.035 * rank)
        if not reliable:
            alpha = min(alpha, 0.45)
        lw = 2.4 if rank <= 5 else 1.3

        ax.plot(
            x,
            exps_sorted,
            marker="o" if rank <= 6 else None,
            linewidth=lw,
            markersize=4,
            color=RED if reliable else RED_DIM,
            linestyle="-" if reliable else ":",
            alpha=alpha,
        )

        if rank <= 8:
            label = f"{str(row['channel'])[:13]} d={int(row['embedding_dimension'])}"
            if not reliable:
                label += " [QC-low]"
            ax.text(
                x[-1] + 0.08,
                exps_sorted[-1],
                label,
                color=RED_SOFT if reliable else WHITE,
                fontsize=8,
                va="center",
            )

    ax.axhline(0, color=WHITE, linewidth=1.0, alpha=0.85)

    style_axis(
        ax,
        title=f"Overlay of Top {len(selected)} Informative Curve × Dimension Lyapunov Spectra",
        xlabel="Exponent rank, sorted descending",
        ylabel="Lyapunov exponent",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_12_top_spectra_overlay_no_duplicate_individual_lines_black_red.png")
    save_fig(fig, path)
    return path


def plot_positive_exponent_count(df, out_dir):
    ok = df[df["status"] == "ok"].copy()
    ok = ok[np.isfinite(ok["positive_count"])]

    if ok.empty:
        return None

    counts = ok["positive_count"].astype(int).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.bar(
        counts.index.astype(str),
        counts.values,
        color=RED,
        edgecolor=RED_SOFT,
        alpha=0.82,
    )

    style_axis(
        ax,
        title="Count of Positive Lyapunov Directions",
        xlabel="Number of positive exponents",
        ylabel="Number of radiation curves",
    )

    path = os.path.join(out_dir, f"{CELL4_RESULT_STEM}_plots_13_positive_exponent_count_black_red.png")
    save_fig(fig, path)
    return path


def make_expert_plots(df, analysis_dir):
    plots_dir = analysis_dir

    paths = []

    plot_functions = [
        plot_lle_rank,
        plot_ks_entropy_vs_dimension,
        plot_spectrum_heatmap,
        plot_mean_spectrum_by_embedding_dim,
        plot_dissipation_vs_instability,
        plot_runtime_qc_sorted,
        plot_runtime_distribution_by_dimension,
        plot_channel_expert_summary,
        plot_embedding_dimension_summary,
        plot_metric_correlation_matrix,
        plot_lle_distribution,
        plot_top_spectra_overlay,
        plot_positive_exponent_count,
    ]

    for fn in plot_functions:
        try:
            p = fn(df, plots_dir)
            if p is not None:
                paths.append(p)
                print(f"Saved plot: {p}", flush=True)
        except Exception as e:
            print(f"Plot failed: {fn.__name__} | {type(e).__name__}: {e}", flush=True)

    return paths


def save_expert_tables(df, analysis_dir):
    os.makedirs(analysis_dir, exist_ok=True)

    enriched_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_expert_metrics.csv")
    df.to_csv(enriched_csv, index=False)

    ok = df[df["status"] == "ok"].copy()

    top_lle_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_top_by_LLE.csv")
    top_complexity_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_top_by_KS_entropy_proxy.csv")
    hyperchaos_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_multiple_positive_directions.csv")
    qc_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_quality_control.csv")
    dim_summary_csv = os.path.join(analysis_dir, f"{CELL4_RESULT_STEM}_embedding_dimension_summary.csv")

    if not ok.empty:
        ok.sort_values("LLE", ascending=False).to_csv(top_lle_csv, index=False)
        ok.sort_values("KS_entropy_proxy", ascending=False).to_csv(top_complexity_csv, index=False)
        ok[ok["hyperchaos_flag"] == True].sort_values("positive_count", ascending=False).to_csv(
            hyperchaos_csv,
            index=False,
        )

        qc_cols = [
            "file",
            "channel",
            "embedding_dimension",
            "n_samples_original",
            "n_samples_used",
            "subsample_step",
            "steps_used",
            "elapsed_sec",
            "quality_score",
            "reliable_steps_flag",
            "status",
            "error",
        ]
        qc_cols = [c for c in qc_cols if c in ok.columns]
        ok[qc_cols].sort_values("quality_score", ascending=True).to_csv(qc_csv, index=False)

        summary = (
            ok.groupby("embedding_dimension")
            .agg(
                n_files=("file", "count"),
                LLE_mean=("LLE", "mean"),
                LLE_median=("LLE", "median"),
                LLE_std=("LLE", "std"),
                KS_entropy_mean=("KS_entropy_proxy", "mean"),
                KY_dim_mean=("Kaplan_Yorke_dim", "mean"),
                positive_count_mean=("positive_count", "mean"),
                sum_exponents_mean=("sum_exponents", "mean"),
                quality_score_mean=("quality_score", "mean"),
                elapsed_sec_mean=("elapsed_sec", "mean"),
            )
            .reset_index()
            .sort_values("embedding_dimension")
        )
        summary.to_csv(dim_summary_csv, index=False)

    return {
        "enriched_csv": enriched_csv,
        "top_lle_csv": top_lle_csv,
        "top_complexity_csv": top_complexity_csv,
        "hyperchaos_csv": hyperchaos_csv,
        "qc_csv": qc_csv,
        "dim_summary_csv": dim_summary_csv,
    }


def print_expert_console_summary(df):
    ok = df[df["status"] == "ok"].copy()

    print("\n" + "=" * 80, flush=True)
    print("EXPERT LYAPUNOV ANALYSIS SUMMARY", flush=True)
    print("=" * 80, flush=True)

    if ok.empty:
        print("No successful files to summarize.", flush=True)
        return

    print(f"Successful files: {len(ok)}", flush=True)
    print(f"Errored files: {int((df['status'] == 'error').sum())}", flush=True)

    print("\nTop 10 by Largest Lyapunov Exponent:", flush=True)
    cols = [
        "channel",
        "embedding_dimension",
        "LLE",
        "KS_entropy_proxy",
        "Kaplan_Yorke_dim",
        "positive_count",
        "sum_exponents",
        "quality_score",
    ]
    cols = [c for c in cols if c in ok.columns]
    print(ok.sort_values("LLE", ascending=False)[cols].head(10).to_string(index=False), flush=True)

    print("\nTop 10 by KS Entropy Proxy:", flush=True)
    print(ok.sort_values("KS_entropy_proxy", ascending=False)[cols].head(10).to_string(index=False), flush=True)

    n_hyper = int(ok["hyperchaos_flag"].sum()) if "hyperchaos_flag" in ok.columns else 0
    n_dissip = int(ok["dissipative_flag"].sum()) if "dissipative_flag" in ok.columns else 0
    n_unreliable = int((ok["reliable_steps_flag"] == False).sum()) if "reliable_steps_flag" in ok.columns else 0

    print(f"\nHyperchaotic candidates, positive_count >= 2: {n_hyper}", flush=True)
    print(f"Dissipative candidates, sum_exponents < 0: {n_dissip}", flush=True)
    print(f"Potentially low-reliability estimates: {n_unreliable}", flush=True)

    if "embedding_dimension" in ok.columns:
        print("\nEmbedding dimension summary:", flush=True)
        summary = (
            ok.groupby("embedding_dimension")
            .agg(
                n_files=("file", "count"),
                LLE_mean=("LLE", "mean"),
                LLE_std=("LLE", "std"),
                KS_mean=("KS_entropy_proxy", "mean"),
                KY_mean=("Kaplan_Yorke_dim", "mean"),
                quality_mean=("quality_score", "mean"),
            )
            .reset_index()
            .sort_values("embedding_dimension")
        )
        print(summary.to_string(index=False), flush=True)


# =====================================================================
# RADIATION DYNAMICAL-SYSTEMS ANALYSIS — CELL 4
# =====================================================================
#
# This section changes only the data source and physical interpretation.
# The Lyapunov mathematics above is the same local-linear + QR method used
# for this analysis.
#
# Radiation mapping selected from the supplied Radiation Mathematical
# Object Bank:
#   N003.SEQ_SAMPLE / N003.TRAJ_E
#   ISIS RB2000164 barite-enriched concrete
#   ordered coordinate: neutron energy / log-energy coordinate
#   scalar response curves: transmission T(E) and grounded Sigma_R(E)
#
# Each measured sample/response curve is converted to a uniformly spaced
# dimensionless log-energy coordinate and delay-embedded at dimensions
# 2...10, spanning the embedding-dimension sweep used here.
#
# IMPORTANT INTERPRETATION:
# The exponents are rates per unit of the ordered log-energy coordinate.
# They are NOT temporal Lyapunov exponents and are NOT, by themselves,
# evidence of temporal chaos.
# =====================================================================

from pathlib import Path
import shutil

import os
from pathlib import Path


def _resolve_project_root_portable() -> Path:
    env_root = os.environ.get("RADIATION_SHIELDING_REPO")
    candidates = []

    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()

    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            candidate = candidate.expanduser()

        key = str(candidate)

        if key in seen:
            continue

        seen.add(key)

        if (
            (candidate / "notebooks").is_dir()
            and
            (candidate / "datasets" / "github").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate Radiation-Shielding-Research. "
        "Run from inside the repository or set "
        "RADIATION_SHIELDING_REPO."
    )


PROJECT_ROOT = _resolve_project_root_portable()


def public_project_path(path) -> str:
    """Return repository-internal paths as portable relative POSIX paths."""
    if path is None:
        return ""

    try:
        p = Path(path).expanduser()
    except TypeError:
        return str(path)

    if not p.is_absolute():
        return p.as_posix()

    try:
        return p.resolve().relative_to(
            PROJECT_ROOT.resolve()
        ).as_posix()
    except ValueError:
        return p.as_posix()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    # Prefer the compressed CSV because it needs no optional parquet engine.
    PROJECT_ROOT / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",

    # Use Parquet when pyarrow/fastparquet is available.
    PROJECT_ROOT / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next((p for p in N003_SOURCE_CANDIDATES if p.exists()), None)

if N003_PATH is None:
    raise FileNotFoundError(
        "RB2000164 processed dataset not found in the project repository.\n"
        "Expected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

print(f"Using processed RB2000164 data: {N003_PATH}")

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL4_RESULT_STEM = "cell_04_lyapunov_spectrum_results"

# Everything produced by this computational cell is written directly into
# results/phase2/. No per-cell subdirectories are created.
base_dir = str(PHASE2_RESULTS_DIR)
results_dir = str(PHASE2_RESULTS_DIR)
analysis_dir = str(PHASE2_RESULTS_DIR)

# Delay-embedding arrays are transient working data, so keep them outside
# the project tree rather than creating another project folder.
import tempfile

embedding_root = str(
    Path(tempfile.gettempdir()) / f"{CELL4_RESULT_STEM}_embedding_data"
)

OUT_CSV = str(PHASE2_RESULTS_DIR / f"{CELL4_RESULT_STEM}.csv")
EMBEDDING_METADATA_CSV = str(
    PHASE2_RESULTS_DIR / f"{CELL4_RESULT_STEM}_embedding_metadata.csv"
)

print(f"Cell 4 outputs will be saved directly in: {PHASE2_RESULTS_DIR}")

# Use embedding dimensions 2 through 10.
EMBEDDING_DIMS = tuple(range(2, 11))
EMBEDDING_TAU = 1

# Use the fast mode as the default computational setting.
# Change FAST_OPTION = "full" before running if the complete anchor set
# is desired. Speed modes change sampling density, not the estimator.
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError("FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'.")

FAST_MODE = FAST_OPTION in {"fast", "ultra"}
ULTRA_FAST_MODE = FAST_OPTION == "ultra"

_LYAP_SPEED = {
    "full":     dict(k=25, stride=5,  max_steps=None, max_points=None),
    "balanced": dict(k=20, stride=10, max_steps=1200, max_points=12000),
    "fast":     dict(k=12, stride=20, max_steps=500,  max_points=5000),
    "ultra":    dict(k=10, stride=35, max_steps=240,  max_points=2800),
}[FAST_OPTION]

K_NEIGHBORS = _LYAP_SPEED["k"]
THEILER = 10
STRIDE = _LYAP_SPEED["stride"]
MAX_STEPS_PER_FILE = _LYAP_SPEED["max_steps"]
FAST_MAX_POINTS = _LYAP_SPEED["max_points"]
QUERY_EXTRA = 80
CHUNK_SIZE = 1024
MAX_FILES = None
CHECKPOINT_EVERY = 5

# Uniform log-energy grid used only so that the QR rate has a defined
# constant coordinate increment. It is a resampling of the measured
# processed curve, not synthetic radiation data.
MAX_UNIFORM_POINTS = 6000
MIN_UNIFORM_POINTS = 300


def _find_column(df, exact=(), contains=(), exclude=()):
    """Find one column by exact names first, then by substring rules."""
    lower = {str(c).lower(): c for c in df.columns}

    for name in exact:
        if name.lower() in lower:
            return lower[name.lower()]

    candidates = []
    for c in df.columns:
        lc = str(c).lower()
        if any(x.lower() in lc for x in contains) and not any(x.lower() in lc for x in exclude):
            candidates.append(c)

    if not candidates:
        return None

    # Prefer shorter, semantically tighter names.
    return sorted(candidates, key=lambda c: (len(str(c)), str(c)))[0]


def _robust_standardize(x):
    """
    Robust state scaling for numerical conditioning.
    This is a coordinate-preserving affine scaling of the processed response.
    """
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= 0:
        scale = np.nanstd(x)

    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Degenerate response curve: zero/invalid scale.")

    return (x - med) / scale, med, scale


def _prepare_uniform_log_energy_curve(part, energy_col, value_col):
    """
    Convert one real processed sample curve to a uniformly spaced log-energy
    coordinate. Exact duplicate energies are averaged before interpolation.
    """
    q = part[[energy_col, value_col]].copy()
    q[energy_col] = pd.to_numeric(q[energy_col], errors="coerce")
    q[value_col] = pd.to_numeric(q[value_col], errors="coerce")
    q = q.replace([np.inf, -np.inf], np.nan).dropna()

    # Positive energy is required for log(E/E_min).
    q = q[q[energy_col] > 0]

    if q.empty:
        raise ValueError("No finite positive-energy rows.")

    q = (
        q.groupby(energy_col, as_index=False, sort=True)[value_col]
        .mean()
        .sort_values(energy_col)
    )

    if len(q) < MIN_UNIFORM_POINTS:
        raise ValueError(
            f"Only {len(q)} unique energy points; need at least {MIN_UNIFORM_POINTS}."
        )

    E = q[energy_col].to_numpy(dtype=float)
    y = q[value_col].to_numpy(dtype=float)

    # Dimensionless ordered coordinate, increasing with energy.
    s = np.log(E / np.min(E))

    n_grid = min(len(s), MAX_UNIFORM_POINTS)
    s_uniform = np.linspace(float(s.min()), float(s.max()), n_grid)
    y_uniform = np.interp(s_uniform, s, y)

    y_scaled, center, scale = _robust_standardize(y_uniform)

    ds = float(s_uniform[1] - s_uniform[0])
    if not np.isfinite(ds) or ds <= 0:
        raise ValueError("Invalid uniform log-energy coordinate step.")

    return s_uniform, y_uniform, y_scaled, ds, center, scale


def prepare_rb2000164_embeddings(parquet_path, embedding_root):
    """
    Build the exact type of 2D...10D delay-embedding arrays consumed by the
    Lyapunov-spectrum analysis from the processed RB2000164 radiation curves.

    One analysis curve = one sample_id × one physical response observable.
    """
    source_path = Path(parquet_path)

    if source_path.name.endswith(".csv.gz") or source_path.suffix.lower() == ".csv":
        df_n003 = pd.read_csv(source_path)
    elif source_path.suffix.lower() == ".parquet":
        try:
            df_n003 = pd.read_parquet(source_path)
        except ImportError as exc:
            csv_fallback = source_path.with_suffix("").with_suffix(".csv.gz")
            if csv_fallback.exists():
                print(
                    "Parquet engine unavailable; using compressed CSV fallback:",
                    csv_fallback,
                    flush=True,
                )
                source_path = csv_fallback
                df_n003 = pd.read_csv(source_path)
            else:
                raise ImportError(
                    "Parquet support requires pyarrow or fastparquet, and no CSV fallback was found."
                ) from exc
    else:
        raise ValueError(f"Unsupported RB2000164 input format: {source_path}")

    print("Loaded processed RB2000164:", source_path, flush=True)
    print("Rows:", len(df_n003), flush=True)
    print("Columns:", list(df_n003.columns), flush=True)

    sample_col = _find_column(
        df_n003,
        exact=("sample_id", "sample", "sample_name"),
        contains=("sample",),
        exclude=("uncert",),
    )
    energy_col = _find_column(
        df_n003,
        exact=("energy_eV", "energy_ev", "neutron_energy_eV", "energy_in_eV"),
        contains=("energy",),
        exclude=("uncert", "lower", "upper"),
    )
    transmission_col = _find_column(
        df_n003,
        exact=("transmission",),
        contains=("transmission",),
        exclude=("uncert", "sigma", "error"),
    )
    removal_col = _find_column(
        df_n003,
        exact=(
            "macroscopic_removal_cross_section",
            "macroscopic_removal_cross_section_cm_inv",
            "sigma_r",
            "Sigma_R",
            "sigma_r_cm_inv",
            "Sigma_R_cm_inv",
        ),
        contains=("removal", "sigma_r"),
        exclude=("uncert", "error"),
    )

    if sample_col is None:
        raise KeyError("Could not identify sample_id column in RB2000164.")
    if energy_col is None:
        raise KeyError("Could not identify neutron-energy column in RB2000164.")
    if transmission_col is None:
        raise KeyError("Could not identify transmission column in RB2000164.")

    response_cols = [("transmission", transmission_col)]
    if removal_col is not None:
        response_cols.append(("macroscopic_removal_cross_section", removal_col))

    # Rebuild only the derived embedding inputs for this notebook.
    if os.path.isdir(embedding_root):
        shutil.rmtree(embedding_root)

    dirs = {
        2: os.path.join(embedding_root, "2dembedding_data"),
        3: os.path.join(embedding_root, "3dembedding_data"),
    }
    for m in range(4, 11):
        dirs[m] = os.path.join(embedding_root, "embeddings_4to10")

    for d in set(dirs.values()):
        os.makedirs(d, exist_ok=True)

    meta_rows = []

    for sample_id, part in df_n003.groupby(sample_col, sort=True, dropna=False):
        sample_label = str(sample_id)

        for response_label, value_col in response_cols:
            try:
                s_u, y_raw, y_scaled, ds, center, scale = _prepare_uniform_log_energy_curve(
                    part, energy_col, value_col
                )
            except Exception as exc:
                print(
                    f"SKIP sample={sample_label} response={response_label}: "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )
                continue

            safe_sample = _safe_filename(sample_label)
            safe_response = _safe_filename(response_label)
            channel_name = f"{safe_sample}__{safe_response}"

            for m in EMBEDDING_DIMS:
                X = _delay_embed_1d(
                    y_scaled,
                    emb_dim=m,
                    tau=EMBEDDING_TAU,
                )

                out_dir = dirs[m]
                fp = os.path.join(out_dir, f"{m}dembedded_{channel_name}.npy")
                np.save(fp, X)

                meta_rows.append({
                    "file": os.path.basename(fp),
                    "path": os.path.relpath(fp, embedding_root),
                    "dataset_id": "N003",
                    "dataset_name": "ISIS RB2000164 barite-enriched concrete",
                    "sample_id": sample_label,
                    "observable": response_label,
                    "source_column": str(value_col),
                    "energy_column": str(energy_col),
                    "ordered_coordinate": "log(E/E_min)",
                    "coordinate_step": ds,
                    "embedding_dimension": m,
                    "embedding_tau_samples": EMBEDDING_TAU,
                    "n_uniform_points": len(s_u),
                    "raw_center_for_scaling": center,
                    "raw_scale_for_scaling": scale,
                    "source_file": public_project_path(source_path),
                })

    meta = pd.DataFrame(meta_rows)

    if meta.empty:
        raise RuntimeError(
            "No radiation embedding trajectories were created from RB2000164."
        )

    meta.to_csv(EMBEDDING_METADATA_CSV, index=False)

    print(
        f"Prepared {len(meta)} real-data embedding trajectories "
        f"from {meta['sample_id'].nunique()} samples.",
        flush=True,
    )
    print("Embedding metadata:", EMBEDDING_METADATA_CSV, flush=True)

    return meta


embedding_meta_df = prepare_rb2000164_embeddings(
    N003_PATH,
    embedding_root,
)

embedding_meta_by_path = {
    os.path.abspath(os.path.join(embedding_root, row["path"])): row
    for _, row in embedding_meta_df.iterrows()
}

existing_dirs, file_paths = gather_embedding_files(embedding_root)

if len(file_paths) == 0:
    raise FileNotFoundError(
        "No radiation embedding .npy files were found after generation.\n"
        f"Embedding root searched:\n  {embedding_root}\n"
        "Expected subdirectories:\n"
        f"  {os.path.join(embedding_root, '2dembedding_data')}\n"
        f"  {os.path.join(embedding_root, '3dembedding_data')}\n"
        f"  {os.path.join(embedding_root, 'embeddings_4to10')}"
    )

if MAX_FILES is not None:
    file_paths = file_paths[:MAX_FILES]

print("\nUsing radiation embedding directories:", flush=True)
for d in existing_dirs:
    print(" ", d, flush=True)

print(f"\nFound {len(file_paths)} radiation embedding trajectories.", flush=True)
print(f"Processed source: {N003_PATH}", flush=True)
print(f"Output CSV: {OUT_CSV}", flush=True)
print(f"Analysis dir: {analysis_dir}", flush=True)
print(f"FAST_OPTION: {FAST_OPTION}", flush=True)
print(f"K_NEIGHBORS: {K_NEIGHBORS}", flush=True)
print(f"THEILER: {THEILER}", flush=True)
print(f"STRIDE: {STRIDE}", flush=True)
print(f"MAX_STEPS_PER_FILE: {MAX_STEPS_PER_FILE}", flush=True)
print(f"FAST_MAX_POINTS: {FAST_MAX_POINTS}", flush=True)
print(f"SHOW_PLOTS_IN_NOTEBOOK: {SHOW_PLOTS_IN_NOTEBOOK}", flush=True)
print(f"SAVE_PLOTS_TO_DISK: {SAVE_PLOTS_TO_DISK}", flush=True)
print(f"IPython display available: {_HAVE_IPYTHON_DISPLAY}", flush=True)
print(f"cKDTree available: {_HAVE_KDTREE}", flush=True)

records = []
run_start = time.time()

for idx, fp in enumerate(file_paths, start=1):
    file_start = time.time()

    file_name = os.path.basename(fp)
    stem = os.path.splitext(file_name)[0]
    abs_fp = os.path.abspath(fp)
    meta = embedding_meta_by_path.get(abs_fp, {})

    rec = {
        "file": file_name,
        "stem": stem,
        "path": os.path.relpath(fp, embedding_root),
        "source_dir": os.path.basename(os.path.dirname(fp)),
        "channel": infer_channel_name_from_stem(stem),
        "dataset_id": meta.get("dataset_id", "N003"),
        "sample_id": meta.get("sample_id"),
        "observable": meta.get("observable"),
        "ordered_coordinate": meta.get("ordered_coordinate", "log(E/E_min)"),
        "source_file": meta.get("source_file", public_project_path(N003_PATH)),
        "embedding_dimension": None,
        "status": "ok",
        "error": "",
        "n_samples_original": None,
        "n_samples_used": None,
        "state_dim": None,
        "subsample_step": 1,
        "dt_used": None,
        "steps_used": None,
        "elapsed_sec": None,
    }

    print(f"\n[{idx}/{len(file_paths)}] Processing: {file_name}", flush=True)

    try:
        data = np.load(fp, allow_pickle=False)

        if data.ndim == 2:
            X = np.asarray(data, dtype=float)
        else:
            x = np.asarray(data, dtype=float).reshape(-1)
            X = _delay_embed_1d(x, emb_dim=7, tau=1)

        rec["n_samples_original"] = int(X.shape[0])

        coordinate_step = float(meta.get("coordinate_step", 1.0))
        if not np.isfinite(coordinate_step) or coordinate_step <= 0:
            raise ValueError("Invalid ordered-coordinate step in embedding metadata.")

        X, local_dt, subsample_step = _maybe_fast_subsample(
            X,
            coordinate_step,
            max_points=FAST_MAX_POINTS,
        )

        rec["n_samples_used"] = int(X.shape[0])
        rec["state_dim"] = int(X.shape[1])
        rec["subsample_step"] = int(subsample_step)
        rec["dt_used"] = float(local_dt)
        rec["embedding_dimension"] = infer_embedding_dim_from_name(stem, X)

        print(
            f"    dataset={rec['dataset_id']} "
            f"| sample={rec['sample_id']} "
            f"| observable={rec['observable']} "
            f"| ordered_coordinate={rec['ordered_coordinate']} "
            f"| original_shape=({rec['n_samples_original']}, {rec['state_dim']}) "
            f"| used_shape={X.shape} "
            f"| dim={rec['embedding_dimension']} "
            f"| ds_used={local_dt:.8g} "
            f"| subsample_step={subsample_step}",
            flush=True,
        )

        exps, used = lyapunov_spectrum_from_trajectory(
            X,
            k_neighbors=K_NEIGHBORS,
            theiler=THEILER,
            stride=STRIDE,
            dt=local_dt,
            max_steps=MAX_STEPS_PER_FILE,
            query_extra=QUERY_EXTRA,
            chunk_size=CHUNK_SIZE,
        )

        rec["steps_used"] = int(used)

        for i, val in enumerate(exps, start=1):
            rec[f"Exp{i}"] = float(val)

        print(
            f"    done | steps_used={used} "
            f"| first exponents={np.array2string(exps[:min(5, len(exps))], precision=6)}",
            flush=True,
        )

    except Exception as e:
        rec["status"] = "error"
        rec["error"] = f"{type(e).__name__}: {e}"
        print(f"    ERROR: {rec['error']}", flush=True)

    rec["elapsed_sec"] = time.time() - file_start
    records.append(rec)

    print(f"    curve_time={rec['elapsed_sec']:.2f} sec", flush=True)

    if (idx % CHECKPOINT_EVERY == 0) or (idx == len(file_paths)):
        df_checkpoint = save_checkpoint(records, OUT_CSV)

        # Preserve radiation metadata columns if save_checkpoint used the
        # preferred radiation-column order.
        for col in [
            "dataset_id", "sample_id", "observable",
            "ordered_coordinate", "source_file"
        ]:
            if col not in df_checkpoint.columns:
                df_checkpoint[col] = [r.get(col) for r in records]
        df_checkpoint.to_csv(OUT_CSV, index=False)

        total_elapsed = time.time() - run_start
        n_done = len(records)
        avg_time = total_elapsed / max(n_done, 1)
        remaining = len(file_paths) - n_done
        est_remaining = remaining * avg_time

        print(
            f"    checkpoint saved | completed={n_done}/{len(file_paths)} "
            f"| total_elapsed={total_elapsed / 60:.2f} min "
            f"| est_remaining={est_remaining / 60:.2f} min",
            flush=True,
        )

df = save_checkpoint(records, OUT_CSV)

# Ensure radiation identity/coordinate columns are present in final table.
for col in [
    "dataset_id", "sample_id", "observable",
    "ordered_coordinate", "source_file"
]:
    if col not in df.columns:
        df[col] = [r.get(col) for r in records]

df.to_csv(OUT_CSV, index=False)

print("\nSaved raw radiation Lyapunov spectrum CSV:", OUT_CSV, flush=True)
print("\nRaw head:", flush=True)
print(df.head(), flush=True)

n_errors = int((df["status"] == "error").sum()) if "status" in df.columns else 0
print(f"\nErrors: {n_errors} of {len(df)}", flush=True)

print("\nComputing expert spectrum metrics...", flush=True)
df_expert = add_expert_metrics(df)

# Add physically neutral aliases without deleting the existing
# mathematical quantities.
if "chaos_timescale" in df_expert.columns:
    df_expert["divergence_coordinate_scale"] = df_expert["chaos_timescale"]

if "hyperchaos_flag" in df_expert.columns:
    df_expert["multiple_positive_directions_flag"] = df_expert["hyperchaos_flag"]

table_paths = save_expert_tables(df_expert, analysis_dir)

# Save a radiation-semantics companion table.
radiation_semantics_csv = os.path.join(
    analysis_dir,
    f"{CELL4_RESULT_STEM}_semantics.csv",
)
df_expert.to_csv(radiation_semantics_csv, index=False)
table_paths["radiation_semantics_csv"] = radiation_semantics_csv

print("\nSaved expert tables:", flush=True)
for name, path in table_paths.items():
    print(f"  {name}: {path}", flush=True)

print("\nGenerating the expert visualization family...", flush=True)
plot_paths = make_expert_plots(df_expert, analysis_dir)

print_expert_console_summary(df_expert)

print("\nRADIATION INTERPRETATION NOTE", flush=True)
print(
    "Positive fitted exponents indicate local separation along the ordered "
    "log-energy reconstruction. They are not temporal-chaos claims.",
    flush=True,
)
print(
    "The 'chaos_timescale' column is retained for mathematical "
    "traceability but is duplicated as 'divergence_coordinate_scale'.",
    flush=True,
)
print(
    "The 'hyperchaos_flag' is retained for mathematical "
    "traceability but is duplicated as 'multiple_positive_directions_flag'.",
    flush=True,
)

print("\nSaved plots:", flush=True)
for p in plot_paths:
    print(" ", p, flush=True)

print("\nDone.", flush=True)
print(f"Raw CSV: {OUT_CSV}", flush=True)
print(f"Expert analysis folder: {analysis_dir}", flush=True)

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">

<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Lyapunov-Spectrum Summary Visualization</h3>

<p>
This cell does not estimate a new dynamical invariant. It reads the Lyapunov-spectrum results produced in the preceding cell and summarizes them by exponent index, radiation curve, embedding dimension, runtime, and related finite-sample statistics.
</p>

<p>
When spectra are averaged, the exponent positions must be sorted consistently so that
<b>λ₁ ≥ λ₂ ≥ ⋯</b> across rows. The plotted mean is therefore an empirical mean of like-ranked exponents,
not a Fourier transform or a spectral-density estimate.
</p>

<p>
For <b>M</b> processed radiation curves, the mean exponent at rank <b>i</b> is:
</p>

<p style="text-align:center;font-size:16px;">
<b>λ̄<sub>i</sub> = (1/M) Σ<sub>m=1</sub><sup>M</sup> λ<sub>i</sub><sup>(m)</sup></b>
</p>

<p><b>Cell role.</b> The plotting stage presents the full result structure using the black/red radiation plotting style.</p>

</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">

<p><b>Core idea.</b>
A Lyapunov exponent describes how rapidly two initially nearby reconstructed states separate or approach one another as the ordered coordinate advances.
A positive exponent means small differences grow on average, a negative exponent means they shrink, and a value near zero corresponds to a direction with little average exponential change.
</p>

<p><b>Radiation interpretation.</b>
For this N003 analysis, the independent variable is the ordered log-energy coordinate rather than clock time.
The numerical values therefore quantify stretching and contraction along the reconstructed energy-response trajectory.
They should be compared only when the same preprocessing, coordinate definition, embedding, neighborhood, and estimator settings are used.
</p>

<p><b>Visualization caution.</b>
Color, ordering, ranking, and marker size make patterns easier to inspect but do not create additional statistical evidence beyond the calculated values and their uncertainty or robustness diagnostics.
</p>

</div>

In [ ]:
# -------------------------------------------------------
# PLOT + SAVE LYAPUNOV RESULTS
# Radiation dynamical-systems analysis — Cell 6
# Black background + red ticks/lines
# Uses the actual N003 results produced in Cell 4
# -------------------------------------------------------

import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------------------------------------------------------
# Radiation-research repository
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

# -------------------------------------------------------
# Flat Cell-6 results/plot output.
# -------------------------------------------------------
PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL4_RESULT_STEM = "cell_04_lyapunov_spectrum_results"
CELL6_RESULT_STEM = "cell_06_lyapunov_summary_results"

# Load Cell-4 results when df is not already in memory.
try:
    df
except NameError:
    preferred_csv = str(PHASE2_RESULTS_DIR / f"{CELL4_RESULT_STEM}.csv")

    if os.path.exists(preferred_csv):
        OUT_CSV = preferred_csv
    else:
        csv_files = sorted(
            glob.glob(str(PHASE2_RESULTS_DIR / "cell_04_lyapunov_spectrum_results*.csv"))
        )
        if len(csv_files) == 0:
            raise FileNotFoundError(
                "No Cell-4 Lyapunov result CSV found. Run Cell 4 first."
            )
        OUT_CSV = csv_files[0]

    print("Loading radiation Lyapunov CSV:", OUT_CSV)
    df = pd.read_csv(OUT_CSV)

plot_dir = str(PHASE2_RESULTS_DIR)

# Save the exact table used by Cell 6.
cell6_table_csv = PHASE2_RESULTS_DIR / f"{CELL6_RESULT_STEM}.csv"
df.to_csv(cell6_table_csv, index=False)

print(f"Cell 6 result: {cell6_table_csv}")
print(f"Cell 6 plots:  {CELL6_RESULT_STEM}_plots_*.png")
# -------------------------------------------------------
# Keep only successful rows
# -------------------------------------------------------
df_ok = df[df["status"] == "ok"].copy()

# Plotting-only reliability metadata comes from Cell 4's QC table. The raw
# source result CSV intentionally does not duplicate reliable_steps_flag.
qc_csv = PHASE2_RESULTS_DIR / f"{CELL4_RESULT_STEM}_quality_control.csv"
if qc_csv.exists():
    qc = pd.read_csv(qc_csv)
    keys = [c for c in ("file","channel","embedding_dimension") if c in df_ok.columns and c in qc.columns]
    if len(keys) >= 2 and "reliable_steps_flag" in qc.columns:
        df_ok = df_ok.merge(
            qc[keys + ["reliable_steps_flag"]].drop_duplicates(keys),
            on=keys,
            how="left",
            validate="many_to_one",
        )

if "reliable_steps_flag" not in df_ok.columns:
    # Presentation fallback only. Global feature eligibility later uses strict QC.
    df_ok["reliable_steps_flag"] = True
else:
    df_ok["reliable_steps_flag"] = df_ok["reliable_steps_flag"].fillna(False).astype(bool)

# Find exponent columns automatically: Exp1, Exp2, ...
exp_cols = sorted(
    [c for c in df_ok.columns if c.startswith("Exp")],
    key=lambda x: int(x.replace("Exp", ""))
    if x.replace("Exp", "").isdigit()
    else 10**9,
)

if len(df_ok) == 0 or len(exp_cols) == 0:
    print("No successful radiation Lyapunov exponent rows to plot.")

else:
    red = "#FF2B2B"
    red_soft = "#FF6B6B"
    red_dim = "#8B0000"
    white = "#F2F2F2"
    black = "#000000"

    def style_ax(ax):
        ax.set_facecolor(black)
        ax.tick_params(axis="both", colors=red)

        ax.xaxis.label.set_color(red)
        ax.yaxis.label.set_color(red)
        ax.title.set_color(red)

        for spine in ax.spines.values():
            spine.set_color(red)

        ax.grid(True, color=red_dim, alpha=0.25)

    spectra = df_ok[exp_cols].to_numpy(dtype=float)
    x = np.arange(1, len(exp_cols) + 1)

    # ---------------------------------------------------
    # 1. Mean Lyapunov spectrum across all radiation curves
    # ---------------------------------------------------
    mean_spectrum = np.nanmean(spectra, axis=0)
    std_spectrum = np.nanstd(spectra, axis=0)

    fig, ax = plt.subplots(figsize=(9, 6), facecolor=black)
    style_ax(ax)

    ax.plot(
        x,
        mean_spectrum,
        marker="o",
        linewidth=2.5,
        color=red,
        label="Mean ranked exponent",
    )

    ax.fill_between(
        x,
        mean_spectrum - std_spectrum,
        mean_spectrum + std_spectrum,
        color=red,
        alpha=0.15,
        label="±1 SD across radiation curves",
    )

    ax.axhline(
        0,
        color=white,
        linestyle="--",
        alpha=0.75,
        label="Zero exponent",
    )

    ax.set_title("Mean Lyapunov Spectrum — Processed Radiation Curves")
    ax.set_xlabel("Exponent rank")
    ax.set_ylabel("Lyapunov exponent per unit log-energy coordinate")

    leg = ax.legend(facecolor=black, edgecolor=red)
    for text in leg.get_texts():
        text.set_color(red)

    plt.tight_layout()
    out_path = os.path.join(plot_dir, f"{CELL6_RESULT_STEM}_plots_01_mean_lyapunov_spectrum.png")
    plt.savefig(out_path, dpi=300, facecolor=black)
    plt.show()

    print("Saved:", out_path)

    # ---------------------------------------------------
    # 2. All spectra overlaid
    #
    # Legend is intentionally conceptual rather than one entry
    # per radiation curve, which would be unreadable.
    # ---------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6), facecolor=black)
    style_ax(ax)

    for row in spectra:
        ax.plot(
            x,
            row,
            marker="o",
            linewidth=1.2,
            alpha=0.28,
            color=red,
        )

    ax.axhline(
        0,
        color=white,
        linestyle="--",
        alpha=0.75,
    )

    ax.set_title("All Lyapunov Spectra — Processed Radiation Curves")
    ax.set_xlabel("Exponent rank")
    ax.set_ylabel("Lyapunov exponent per unit log-energy coordinate")

    legend_handles = [
        Line2D(
            [0], [0],
            color=red,
            marker="o",
            linewidth=1.4,
            label="One processed radiation spectrum",
        ),
        Line2D(
            [0], [0],
            color=white,
            linestyle="--",
            linewidth=1.2,
            label="Zero exponent",
        ),
    ]

    leg = ax.legend(
        handles=legend_handles,
        facecolor=black,
        edgecolor=red,
        loc="best",
    )
    for text in leg.get_texts():
        text.set_color(red)

    plt.tight_layout()
    out_path = os.path.join(plot_dir, f"{CELL6_RESULT_STEM}_plots_02_all_lyapunov_spectra.png")
    plt.savefig(out_path, dpi=300, facecolor=black)
    plt.show()

    print("Saved:", out_path)

    # ---------------------------------------------------
    # 3. Largest Lyapunov exponent per radiation curve
    # ---------------------------------------------------
    largest_exp = np.nanmax(spectra, axis=1)

    fig, ax = plt.subplots(figsize=(12, 6), facecolor=black)
    style_ax(ax)

    ax.plot(
        np.arange(len(largest_exp)),
        largest_exp,
        marker="o",
        linewidth=1.8,
        color=red,
        label="Largest fitted exponent",
    )

    ax.axhline(
        0,
        color=white,
        linestyle="--",
        alpha=0.75,
        label="Zero exponent",
    )

    ax.set_title("Largest Lyapunov Exponent by Radiation Curve")
    ax.set_xlabel("Radiation curve index")
    ax.set_ylabel("Largest exponent per unit log-energy coordinate")

    leg = ax.legend(facecolor=black, edgecolor=red)
    for text in leg.get_texts():
        text.set_color(red)

    plt.tight_layout()
    out_path = os.path.join(
        plot_dir,
        f"{CELL6_RESULT_STEM}_plots_03_largest_lyapunov_exponent_by_radiation_curve.png",
    )
    plt.savefig(out_path, dpi=300, facecolor=black)
    plt.show()

    print("Saved:", out_path)

    # ---------------------------------------------------
    # 4. Runtime per radiation curve
    # ---------------------------------------------------
    if "elapsed_sec" in df_ok.columns:
        fig, ax = plt.subplots(figsize=(12, 6), facecolor=black)
        style_ax(ax)

        ax.plot(
            np.arange(len(df_ok)),
            df_ok["elapsed_sec"].to_numpy(dtype=float),
            marker="o",
            linewidth=1.8,
            color=red,
        )

        ax.set_title("Runtime per Radiation Curve")
        ax.set_xlabel("Radiation curve index")
        ax.set_ylabel("Elapsed seconds")

        plt.tight_layout()
        out_path = os.path.join(plot_dir, f"{CELL6_RESULT_STEM}_plots_04_runtime_per_radiation_curve.png")
        plt.savefig(out_path, dpi=300, facecolor=black)
        plt.show()

        print("Saved:", out_path)

    # ---------------------------------------------------
    # 5. Mean Lyapunov spectrum by embedding dimension
    # ---------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=black)
    style_ax(ax)

    line_styles = ["-", "--", "-.", ":"]
    markers = ["o", "s", "^", "D", "v", "P", "X", "*", "h"]

    for j, dim in enumerate(
        sorted(df_ok["embedding_dimension"].dropna().unique())
    ):
        sub = df_ok[df_ok["embedding_dimension"] == dim].copy()

        dim_exp_cols = sorted(
            [c for c in exp_cols if sub[c].notna().any()],
            key=lambda x: int(x.replace("Exp", "")),
        )

        if len(dim_exp_cols) == 0:
            continue

        sub_spectra = sub[dim_exp_cols].to_numpy(dtype=float)
        mean_spec = np.nanmean(sub_spectra, axis=0)
        x_dim = np.arange(1, len(dim_exp_cols) + 1)

        relfrac=float(sub["reliable_steps_flag"].mean()) if "reliable_steps_flag" in sub.columns else 1.0
        qc_low=relfrac < 0.8

        ax.plot(
            x_dim,
            mean_spec,
            marker=markers[j % len(markers)],
            linestyle=":" if qc_low else line_styles[j % len(line_styles)],
            linewidth=2,
            alpha=0.50 if qc_low else 0.90,
            color=red_dim if qc_low else red,
            label=f"{int(dim)}D" + (" [QC-low]" if qc_low else ""),
        )

    ax.axhline(
        0,
        color=white,
        linestyle="--",
        alpha=0.75,
    )

    ax.set_title("Mean Lyapunov Spectrum by Embedding Dimension")
    ax.set_xlabel("Exponent rank")
    ax.set_ylabel("Mean exponent per unit log-energy coordinate")

    leg = ax.legend(facecolor=black, edgecolor=red)
    for text in leg.get_texts():
        text.set_color(red)

    plt.tight_layout()
    out_path = os.path.join(
        plot_dir,
        f"{CELL6_RESULT_STEM}_plots_05_mean_lyapunov_spectrum_by_dimension.png",
    )
    plt.savefig(out_path, dpi=300, facecolor=black)
    plt.show()

    print("Saved:", out_path)

    # ---------------------------------------------------
    # 6. Largest Lyapunov exponent vs embedding dimension
    # ---------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 6), facecolor=black)
    style_ax(ax)

    dims = []
    largest_means = []
    largest_stds = []

    for dim in sorted(
        df_ok["embedding_dimension"].dropna().unique()
    ):
        sub = df_ok[df_ok["embedding_dimension"] == dim].copy()

        dim_exp_cols = sorted(
            [c for c in exp_cols if sub[c].notna().any()],
            key=lambda x: int(x.replace("Exp", "")),
        )

        if len(dim_exp_cols) == 0:
            continue

        sub_spectra = sub[dim_exp_cols].to_numpy(dtype=float)
        vals = np.nanmax(sub_spectra, axis=1)

        dims.append(dim)
        largest_means.append(np.nanmean(vals))
        largest_stds.append(np.nanstd(vals))

    dims = np.array(dims, dtype=float)
    largest_means = np.array(largest_means, dtype=float)
    largest_stds = np.array(largest_stds, dtype=float)

    ax.plot(
        dims,
        largest_means,
        marker="o",
        linewidth=2.5,
        color=red,
        label="Mean largest exponent",
    )

    ax.fill_between(
        dims,
        largest_means - largest_stds,
        largest_means + largest_stds,
        color=red,
        alpha=0.15,
        label="±1 SD across radiation curves",
    )

    rel_by_dim = (
        df_ok.groupby("embedding_dimension")["reliable_steps_flag"].mean()
        if "reliable_steps_flag" in df_ok.columns
        else pd.Series(dtype=float)
    )
    bad_dims = [d for d in dims if float(rel_by_dim.get(d, 1.0)) < 0.8]
    if bad_dims:
        mean_lookup = dict(zip(dims, largest_means))
        ax.scatter(
            bad_dims,
            [mean_lookup[d] for d in bad_dims],
            s=95,
            facecolors="none",
            edgecolors=white,
            linewidths=1.2,
            label="QC-low embedding dimension",
            zorder=6,
        )
        first_bad=min(bad_dims)
        ax.axvline(first_bad-0.5,color=white,linestyle=":",alpha=0.45)

    ax.axhline(
        0,
        color=white,
        linestyle="--",
        alpha=0.75,
        label="Zero exponent",
    )

    ax.set_title("Largest Lyapunov Exponent vs Embedding Dimension")
    ax.set_xlabel("Embedding dimension")
    ax.set_ylabel("Mean largest exponent per unit log-energy coordinate")

    leg = ax.legend(facecolor=black, edgecolor=red)
    for text in leg.get_texts():
        text.set_color(red)

    plt.tight_layout()
    out_path = os.path.join(
        plot_dir,
        f"{CELL6_RESULT_STEM}_plots_06_largest_exponent_vs_embedding_dimension.png",
    )
    plt.savefig(out_path, dpi=300, facecolor=black)
    plt.show()

    print("Saved:", out_path)

    # ---------------------------------------------------
    # 7. Print counts per embedding dimension
    # ---------------------------------------------------
    print("\nRadiation curves per embedding dimension:")
    print(df_ok.groupby("embedding_dimension").size())

    print("\nAll Cell-6 plots saved directly in:")
    print(PHASE2_RESULTS_DIR)


# Arnold Tongue

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">

<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Radiation-Derived Kuramoto Arnold Tongues</h3>

<p>
The raw N003 transmission curves are smooth, strongly trend-dominated functions of neutron energy. A direct mean-subtracted FFT therefore selects the same lowest non-zero Fourier bin for almost every sample, which makes the inferred natural frequencies nearly identical and produces a nearly trivial synchronization map.
</p>

<p>
This cell uses an externally driven Kuramoto model with a <b>radiation-to-natural-frequency mapping</b> designed for monotone shielding-response curves.
</p>

<h3 style="color:#FF4D4D;">Radiation characteristic-frequency extraction</h3>
<p>
For each measured N003 transmission curve <b>T(E)</b>, the code first forms optical depth <b>τ(E) = −ln T(E)</b>, smooths it gently on the common log-energy grid, differentiates it with respect to <b>s = ln(E/E<sub>min</sub>)</b>, removes the remaining linear trend, applies a Hann window, and computes the Fourier power spectrum of that shape-sensitive derivative signal.
</p>

<p>
The code records both a quadratic sub-bin estimate of the dominant spectral peak and the spectral centroid. The spectral centroid is used as the sample's characteristic spatial frequency because these radiation responses are broad-band smooth curves. This avoids one-bin quantization while keeping the Fourier analysis central to the method.
</p>

<h3 style="color:#FF4D4D;">Frequency units</h3>
<p>
The natural-frequency mapping is unit-consistent inside the reduced model. Each measured characteristic spatial frequency is divided by the median characteristic frequency to form a positive dimensionless cycles-per-model-unit frequency, and the Kuramoto natural angular frequency is then <b>ω<sub>i</sub> = 2π f<sub>i,norm</sub></b>. The external drive remains <b>2πb</b>, so natural and drive frequencies now occupy the same angular-frequency scale.
</p>

<h3 style="color:#FF4D4D;">Arnold-tongue scan</h3>
<p>
The drive-frequency range is centered on the data-derived natural-frequency population rather than being imposed independently. Drive amplitude and coupling strength are scaled to the measured natural-frequency dispersion, and the amplitude grid includes <b>a = 0</b> so every driven result has a genuine no-drive baseline.
</p>

<p>
Each grid point is averaged across several deterministic random initial phase configurations. This suppresses the isolated single-pixel artifacts produced by one initial phase realization and makes the tongue structure a property of the reduced model rather than of one arbitrary initial condition.
</p>

</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Interpretation.</b> The measured radiation data determine the reduced-model natural-frequency distribution. The Kuramoto integration itself remains a model experiment. Arnold tongues, synchronization gain, drive locking, and frequency-error reduction therefore describe the explicitly defined radiation-informed phase model; they are not claims that the shielding material literally oscillates in time.</p>
<p><b>Plotting.</b> Full-range 0–1 synchronization maps are retained. Contrast plots are now baseline-referenced and use robust symmetric limits, preventing one numerical outlier from making the rest of a heatmap appear black.</p>
</div>

In [ ]:
import os
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.integrate import solve_ivp
from scipy import signal

SPEED_PROFILES = {
    "ultra": {"grid_n": 12, "t_end": 10.0, "n_t_eval": 150, "rtol": 1e-3, "atol": 1e-5, "solver_method": "RK45", "dpi": 140, "contours": False, "n_initializations": 1, "description": "Very fast preview."},
    "fast": {"grid_n": 25, "t_end": 24.0, "n_t_eval": 300, "rtol": 3e-4, "atol": 1e-6, "solver_method": "RK45", "dpi": 170, "contours": False, "n_initializations": 2, "description": "Fast exploratory run."},
    "balanced": {"grid_n": 40, "t_end": 45.0, "n_t_eval": 500, "rtol": 1e-4, "atol": 1e-7, "solver_method": "RK45", "dpi": 200, "contours": True, "n_initializations": 3, "description": "Good default run."},
    "full": {"grid_n": 50, "t_end": 70.0, "n_t_eval": 700, "rtol": 1e-5, "atol": 1e-7, "solver_method": "RK45", "dpi": 220, "contours": True, "n_initializations": 5, "description": "Full-quality run."},
}

def parse_speed_mode():
    shared_speed = str(globals().get("FAST_OPTION", "")).strip().lower()
    if shared_speed:
        if shared_speed not in SPEED_PROFILES:
            raise ValueError("FAST_OPTION must be one of: full, balanced, fast, ultra.")
        return shared_speed

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--speed",
        choices=list(SPEED_PROFILES.keys()),
        default="full",
        help="Choose simulation speed: ultra, fast, balanced, or full.",
    )
    args, _ = parser.parse_known_args()

    env_speed = os.environ.get("KURAMOTO_SPEED", "").strip().lower()
    if env_speed:
        if env_speed not in SPEED_PROFILES:
            raise ValueError(
                f"Invalid KURAMOTO_SPEED={env_speed!r}. "
                f"Choose one of: {list(SPEED_PROFILES.keys())}"
            )
        return env_speed

    return args.speed

speed_mode = parse_speed_mode()
speed_cfg = SPEED_PROFILES[speed_mode]

PROJECT_ROOT = _resolve_project_root_portable()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Radiation-research repository not found at:\n  {PROJECT_ROOT}")

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]
radiation_path = next((p for p in N003_SOURCE_CANDIDATES if p.exists()), None)
if radiation_path is None:
    raise FileNotFoundError("Processed RB2000164 dataset not found.\nExpected one of:\n" + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES))

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL9_RESULT_STEM = f"cell_09_arnold_tongue_kuramoto_{speed_mode}_results"

out_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)
# Remove existing artifacts for plot families that are regenerated below.
for _family in (
    "kuramoto_drive_locking_arnold_tongue",
    "kuramoto_collective_drive_locking",
    "kuramoto_drive_frequency_error",
    "kuramoto_mean_sync_visible_scale",
    "kuramoto_mean_sync_full_range",
    "kuramoto_omega_by_radiation_sample",
    "mean_sync_vs_b", "mean_sync_vs_a",
    "frequency_error_vs_b", "frequency_error_vs_a",
    "drive_lock_vs_b", "drive_lock_vs_a",
):
    for _artifact in Path(plots_dir).glob(f"{CELL9_RESULT_STEM}_plots_{_family}*.png"):
        _artifact.unlink()



print(f"Cell 9 outputs will be saved directly in: {PHASE2_RESULTS_DIR}")
print(f"Cell 9 result prefix: {CELL9_RESULT_STEM}")

rng_seed = 42
grid_n = speed_cfg["grid_n"]
tau_start = 0.0
tau_end = speed_cfg["t_end"]
n_tau_eval = speed_cfg["n_t_eval"]
transient_fraction = 0.5
solver_method = speed_cfg["solver_method"]
rtol = speed_cfg["rtol"]
atol = speed_cfg["atol"]
plot_dpi = speed_cfg["dpi"]
plot_contours = speed_cfg["contours"]
n_initializations = speed_cfg["n_initializations"]

MAX_COORD_POINTS = 4096
MIN_COORD_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

RED_SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list("radiation_black_red", [BLACK, RED_DIM, RED, RED_SOFT], N=256)
RED_REVERSED_CMAP = LinearSegmentedColormap.from_list("radiation_red_black", [RED_SOFT, RED, RED_DIM, BLACK], N=256)
RED_DIVERGING_CMAP = LinearSegmentedColormap.from_list("radiation_red_black_red", [RED_DIM, BLACK, RED_SOFT], N=256)

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": ACCENT, "axes.labelcolor": ACCENT, "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT, "xtick.color": ACCENT, "ytick.color": ACCENT,
    "grid.color": RED_DIM, "grid.alpha": 0.25,
    "legend.facecolor": BG, "legend.edgecolor": ACCENT, "legend.labelcolor": ACCENT,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=ACCENT)
    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)

    for spine in ax.spines.values():
        spine.set_color(ACCENT)

    ax.grid(True, alpha=0.20, color=ACCENT)

def style_colorbar(cbar):
    cbar.outline.set_edgecolor(ACCENT)
    cbar.ax.tick_params(color=ACCENT, labelcolor=ACCENT)

    label = cbar.ax.get_ylabel()
    if label:
        cbar.set_label(label, color=ACCENT)

    for label in cbar.ax.get_yticklabels():
        label.set_color(ACCENT)

def save_show_close(fig, out_path, dpi=None):
    if dpi is None:
        dpi = plot_dpi

    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)

def robust_limits(
    arr,
    lower=2,
    upper=98,
    pad_fraction=0.08,
    min_span=1e-8,
    bounds=None,
):
    arr = np.asarray(arr, dtype=float)
    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        if bounds is not None:
            return bounds
        return 0.0, 1.0

    vmin, vmax = np.nanpercentile(finite, [lower, upper])

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    center = 0.5 * (vmin + vmax)
    span = vmax - vmin

    if not np.isfinite(span) or span < min_span:
        span = max(abs(center) * 1e-6, min_span)
        vmin = center - 0.5 * span
        vmax = center + 0.5 * span
    else:
        pad = pad_fraction * span
        vmin -= pad
        vmax += pad

    if bounds is not None:
        lo, hi = bounds
        vmin = max(lo, vmin)
        vmax = min(hi, vmax)

    if vmax <= vmin:
        delta = max(abs(center) * 1e-6, min_span)
        vmin = center - delta
        vmax = center + delta

        if bounds is not None:
            lo, hi = bounds
            vmin = max(lo, vmin)
            vmax = min(hi, vmax)

        if vmax <= vmin:
            if bounds is not None:
                return bounds
            return center - min_span, center + min_span

    return float(vmin), float(vmax)

def ensure_time_channel_layout(x, n_channels_expected):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D ordered-response array, got shape {x.shape}")

    if x.shape[1] == n_channels_expected:
        return x

    if x.shape[0] == n_channels_expected:
        return x.T

    raise ValueError(f"Unexpected ordered-response shape: {x.shape}")

def estimate_dominant_frequencies(signal_matrix, fs, band=(1.0, 45.0)):
    signal_matrix = np.asarray(signal_matrix, dtype=float)
    n_samples, n_channels = signal_matrix.shape

    X = signal_matrix - np.mean(signal_matrix, axis=0, keepdims=True)
    fft_vals = np.fft.rfft(X, axis=0)
    freqs = np.fft.rfftfreq(n_samples, d=1.0 / fs)

    band_mask = (freqs >= band[0]) & (freqs <= band[1])
    if not np.any(band_mask):
        raise ValueError(f"No FFT frequencies found inside band {band}")

    band_freqs = freqs[band_mask]
    band_power = np.abs(fft_vals[band_mask, :])

    peak_idx = np.argmax(band_power, axis=0)
    dom_freqs = band_freqs[peak_idx]

    return dom_freqs

def build_natural_frequencies_from_radiation(signal_matrix, fs, band=(1.0, 45.0)):
    dom_freqs = estimate_dominant_frequencies(signal_matrix, fs, band=band)
    omega = (dom_freqs - np.mean(dom_freqs)) / (np.std(dom_freqs) + eps)
    return omega, dom_freqs

def kuramoto_rhs(t, theta, omega, K, a, b):
    z = np.mean(np.exp(1j * theta))
    r = np.abs(z)
    psi = np.angle(z)

    coupling = K * r * np.sin(psi - theta)

    drive_phase = 2.0 * np.pi * b * t
    drive = a * np.sin(drive_phase - theta)

    return omega + coupling + drive

def simulate_kuramoto(omega, K, a, b, theta0, t_span, t_eval):
    sol = solve_ivp(
        kuramoto_rhs,
        t_span=t_span,
        y0=theta0,
        t_eval=t_eval,
        args=(omega, K, a, b),
        method=solver_method,
        rtol=rtol,
        atol=atol,
    )

    if not sol.success:
        raise RuntimeError(sol.message)

    theta = sol.y
    z = np.mean(np.exp(1j * theta), axis=0)
    r = np.abs(z)

    return sol.t, theta, r

def plot_heatmap(
    data,
    title,
    colorbar_label,
    out_name,
    cmap_name=RED_SEQUENTIAL_CMAP,
    vmin=None,
    vmax=None,
    robust=False,
    bounds=None,
    contour=False,
    contour_color=None,
):
    fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
    style_ax(ax)

    arr = np.asarray(data, dtype=float)

    if robust:
        vmin, vmax = robust_limits(
            arr,
            lower=2,
            upper=98,
            pad_fraction=0.08,
            min_span=1e-8,
            bounds=bounds,
        )

    if isinstance(cmap_name, str):
        cmap = plt.get_cmap(cmap_name).copy()
    else:
        cmap = cmap_name.copy()
    cmap.set_bad(BG)

    im = ax.imshow(
        np.ma.masked_invalid(arr),
        extent=[b_values.min(), b_values.max(), a_values.min(), a_values.max()],
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )

    if contour and plot_contours:
        try:
            finite = arr[np.isfinite(arr)]
            if finite.size > 0:
                cmin = np.nanmin(finite) if vmin is None else vmin
                cmax = np.nanmax(finite) if vmax is None else vmax
                if cmax > cmin:
                    B, A = np.meshgrid(b_values, a_values)
                    levels = np.linspace(cmin, cmax, 8)
                    ax.contour(
                        B,
                        A,
                        arr,
                        levels=levels,
                        colors=contour_color or ACCENT,
                        linewidths=0.45,
                        alpha=0.35,
                    )
        except Exception as e:
            print(f"Skipping contour overlay for {out_name}:", e)

    ax.set_title(title)
    ax.set_xlabel("Drive frequency b (cycles per model-coordinate unit)")
    ax.set_ylabel("Drive amplitude a (angular-frequency units)")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(colorbar_label)
    style_colorbar(cbar)

    out_path = os.path.join(plots_dir, out_name)
    save_show_close(fig, out_path)

    return out_path, vmin, vmax

def plot_line_with_band(x, y, ystd, title, xlabel, ylabel, out_name, ylim_bounds=None, physical_bounds=None):
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
    style_ax(ax)

    ax.plot(
        x,
        y,
        color=ACCENT,
        marker="o",
        linewidth=2.0,
    )

    if ystd is not None:
        band_low=np.asarray(y,float)-np.asarray(ystd,float)
        band_high=np.asarray(y,float)+np.asarray(ystd,float)
        bounds=physical_bounds if physical_bounds is not None else ylim_bounds
        if bounds is not None:
            lower_bound,upper_bound=bounds
            if lower_bound is not None:
                band_low=np.maximum(band_low,float(lower_bound))
            if upper_bound is not None:
                band_high=np.minimum(band_high,float(upper_bound))
        ax.fill_between(
            x,
            band_low,
            band_high,
            color=ACCENT,
            alpha=0.15,
        )

    if ylim_bounds is not None:
        ymin, ymax = robust_limits(
            y,
            lower=0,
            upper=100,
            pad_fraction=0.15,
            min_span=1e-8,
            bounds=ylim_bounds,
        )
        ax.set_ylim(ymin, ymax)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    out_path = os.path.join(plots_dir, out_name)
    save_show_close(fig, out_path)

    return out_path

def _find_radiation_column(df, exact=(), contains=(), exclude=()):
    lower_map = {str(c).lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    matches = []
    for c in df.columns:
        lc = str(c).lower()
        if any(token.lower() in lc for token in contains) and not any(token.lower() in lc for token in exclude):
            matches.append(c)
    if not matches:
        return None
    return sorted(matches, key=lambda c: (len(str(c)), str(c)))[0]


def load_rb2000164_transmission_matrix(source_path):
    source_path = Path(source_path)
    if source_path.name.endswith('.csv.gz') or source_path.suffix.lower() == '.csv':
        df = pd.read_csv(source_path)
    elif source_path.suffix.lower() == '.parquet':
        try:
            df = pd.read_parquet(source_path)
        except ImportError as exc:
            csv_fallback = source_path.with_suffix('').with_suffix('.csv.gz')
            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(source_path)
            else:
                raise ImportError('Parquet support unavailable and no CSV fallback found.') from exc
    else:
        raise ValueError(f'Unsupported RB2000164 input format: {source_path}')

    sample_col = _find_radiation_column(df, exact=('sample_id','sample','sample_name'), contains=('sample',), exclude=('uncert',))
    energy_col = _find_radiation_column(df, exact=('energy_eV','energy_ev','neutron_energy_eV','energy_in_eV'), contains=('energy',), exclude=('uncert','lower','upper'))
    transmission_col = _find_radiation_column(df, exact=('transmission',), contains=('transmission',), exclude=('uncert','sigma','error'))

    if sample_col is None or energy_col is None or transmission_col is None:
        raise KeyError(f'Could not resolve required N003 columns: sample={sample_col}, energy={energy_col}, transmission={transmission_col}')

    curves = {}
    for sample_id, part in df.groupby(sample_col, sort=True, dropna=False):
        q = part[[energy_col, transmission_col]].copy()
        q[energy_col] = pd.to_numeric(q[energy_col], errors='coerce')
        q[transmission_col] = pd.to_numeric(q[transmission_col], errors='coerce')
        q = q.replace([np.inf,-np.inf], np.nan).dropna()
        q = q[(q[energy_col] > 0) & (q[transmission_col] > 0)]
        q = q.groupby(energy_col, as_index=False, sort=True)[transmission_col].mean().sort_values(energy_col)
        if len(q) < MIN_COORD_POINTS:
            print(f'Skipping sample {sample_id}: only {len(q)} unique positive-energy points.', flush=True)
            continue
        curves[str(sample_id)] = (np.log(q[energy_col].to_numpy(float)), q[transmission_col].to_numpy(float))

    if len(curves) < 3:
        raise RuntimeError('Need at least three sufficiently sampled N003 transmission curves.')

    common_lo = max(v[0].min() for v in curves.values())
    common_hi = min(v[0].max() for v in curves.values())
    if not np.isfinite(common_lo) or not np.isfinite(common_hi) or common_hi <= common_lo:
        raise RuntimeError('No common log-energy support across retained N003 samples.')

    n_grid = min(MAX_COORD_POINTS, min(len(v[0]) for v in curves.values()))
    if n_grid < MIN_COORD_POINTS:
        raise RuntimeError(f'Common grid only has {n_grid} points; need at least {MIN_COORD_POINTS}.')

    logE_grid = np.linspace(common_lo, common_hi, n_grid)
    labels, columns = [], []
    for sample_id in sorted(curves):
        logE, T = curves[sample_id]
        columns.append(np.interp(logE_grid, logE, T))
        labels.append(sample_id)

    matrix = np.column_stack(columns)
    s_grid = logE_grid - logE_grid[0]
    ds = float(s_grid[1] - s_grid[0])
    return matrix, labels, s_grid, ds, df, sample_col, energy_col, transmission_col, source_path


def make_radiation_shape_signals(transmission_matrix, ds):
    """Convert smooth monotone T(E) curves into shape-sensitive optical-depth-gradient signals."""
    T = np.asarray(transmission_matrix, dtype=float)
    if T.ndim != 2:
        raise ValueError('transmission_matrix must be 2D: coordinate x sample')

    n = T.shape[0]
    # About 3% of the curve length, odd, and large enough for cubic smoothing.
    window = max(11, int(round(0.03 * n)))
    if window % 2 == 0:
        window += 1
    if window >= n:
        window = n - 1 if (n - 1) % 2 == 1 else n - 2
    window = max(window, 5)

    features = np.empty_like(T, dtype=float)
    for j in range(T.shape[1]):
        t = np.clip(T[:, j], 1e-12, None)
        optical_depth = -np.log(t)
        smooth = signal.savgol_filter(optical_depth, window_length=window, polyorder=3, mode='interp')
        slope = np.gradient(smooth, ds)
        shape = signal.detrend(slope, type='linear')
        shape -= np.mean(shape)
        sd = np.std(shape)
        if not np.isfinite(sd) or sd <= 0:
            raise RuntimeError(f'Degenerate radiation shape signal for sample column {j}.')
        features[:, j] = shape / sd
    return features, window


def estimate_radiation_characteristic_frequencies(shape_matrix, fs):
    """
    Fourier characteristic frequency for each radiation shape signal.

    Returns a continuous sub-bin peak estimate plus the spectral centroid.
    The centroid is used for the Kuramoto population because these shielding
    response curves are broad-band rather than narrow-band oscillations.
    """
    X = np.asarray(shape_matrix, dtype=float)
    n, n_channels = X.shape
    window = np.hanning(n)[:, None]
    fft_vals = np.fft.rfft(X * window, axis=0)
    freqs = np.fft.rfftfreq(n, d=1.0/fs)
    power = np.abs(fft_vals) ** 2

    dfreq = freqs[1] - freqs[0]
    nyq = freqs[-1]
    low = max(3.0 * dfreq, 0.01 * nyq)
    high = 0.90 * nyq
    mask = (freqs >= low) & (freqs <= high)
    idx_band = np.where(mask)[0]
    if len(idx_band) < 5:
        raise RuntimeError('Insufficient Fourier bins in radiation characteristic-frequency band.')

    subbin_peaks = np.zeros(n_channels)
    centroids = np.zeros(n_channels)
    spreads = np.zeros(n_channels)

    for j in range(n_channels):
        p = power[:, j]
        p_band = p[idx_band]
        peak_global_idx = int(idx_band[np.argmax(p_band)])
        peak_f = freqs[peak_global_idx]

        # Quadratic interpolation in log power for sub-bin peak location.
        if 0 < peak_global_idx < len(freqs)-1:
            y1 = np.log(p[peak_global_idx-1] + eps)
            y2 = np.log(p[peak_global_idx] + eps)
            y3 = np.log(p[peak_global_idx+1] + eps)
            denom = (y1 - 2.0*y2 + y3)
            if np.isfinite(denom) and abs(denom) > 1e-14:
                delta = 0.5 * (y1 - y3) / denom
                delta = float(np.clip(delta, -0.5, 0.5))
                peak_f = freqs[peak_global_idx] + delta * dfreq

        weights = p_band + eps
        centroid = float(np.sum(freqs[idx_band] * weights) / np.sum(weights))
        spread = float(np.sqrt(np.sum(((freqs[idx_band]-centroid)**2) * weights) / np.sum(weights)))

        subbin_peaks[j] = float(peak_f)
        centroids[j] = centroid
        spreads[j] = spread

    if np.any(~np.isfinite(centroids)) or np.any(centroids <= 0):
        raise RuntimeError('Non-finite/invalid radiation spectral centroids.')

    relative_spread = float(np.std(centroids) / max(np.mean(centroids), eps))
    if relative_spread < 1e-5:
        raise RuntimeError(
            'N003 radiation characteristic frequencies remain degenerate even after trend removal. '
            'This dataset should not be forced into the Kuramoto Arnold-tongue method.'
        )

    return subbin_peaks, centroids, spreads, (low, high)


def build_natural_frequencies_from_radiation(characteristic_freqs):
    """Map positive characteristic frequencies to unit-consistent Kuramoto angular frequencies."""
    f = np.asarray(characteristic_freqs, dtype=float)
    median_f = float(np.median(f))
    if not np.isfinite(median_f) or median_f <= 0:
        raise RuntimeError('Invalid median characteristic radiation frequency.')

    normalized_cycles = f / median_f
    omega = 2.0 * np.pi * normalized_cycles

    if np.std(omega) < 1e-6:
        raise RuntimeError('Natural-frequency population is numerically degenerate; Arnold tongue is not identifiable.')

    return omega, normalized_cycles


def robust_symmetric_limit(arr, percentile=98.0, floor=1e-10):
    v = np.asarray(arr, float)
    finite = np.abs(v[np.isfinite(v)])
    if finite.size == 0:
        return floor
    lim = float(np.nanpercentile(finite, percentile))
    return max(lim, floor)


print('\nKuramoto speed profile')
print('======================')
print(f'Selected speed: {speed_mode}')
print(speed_cfg['description'])
for key, value in speed_cfg.items():
    if key != 'description':
        print(f'  {key}: {value}')

(
    radiation_matrix,
    radiation_samples,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    actual_source_path,
) = load_rb2000164_transmission_matrix(radiation_path)

N = len(radiation_samples)
radiation_matrix = ensure_time_channel_layout(radiation_matrix, N)
shape_matrix, smoothing_window = make_radiation_shape_signals(radiation_matrix, coordinate_step)
coordinate_sampling_rate = 1.0 / coordinate_step

subbin_peak_freqs, characteristic_freqs, spectral_spreads, spectral_band = estimate_radiation_characteristic_frequencies(
    shape_matrix,
    coordinate_sampling_rate,
)
omega, normalized_cycles = build_natural_frequencies_from_radiation(characteristic_freqs)

omega_spread = float(np.std(omega))
K = max(0.05, 0.75 * omega_spread)

# Drive frequency b is in cycles/model-unit, matching omega = 2*pi*f.
center_b = float(np.median(normalized_cycles))
half_width_b = max(0.35, 2.5 * float(np.std(normalized_cycles)))
b_lo = max(0.02, center_b - half_width_b)
b_hi = center_b + half_width_b
b_values = np.linspace(b_lo, b_hi, grid_n)

# Include the genuine no-drive baseline a=0 and scale forcing to the observed dispersion.
a_max = max(0.50, 4.0 * omega_spread)
a_values = np.linspace(0.0, a_max, grid_n)

print(f'\nLoaded processed radiation matrix: {radiation_matrix.shape[0]} coordinate points x {N} samples')
print(f'Processed source: {public_project_path(actual_source_path)}')
print(f'Common coordinate: s = ln(E/E_min), ds={coordinate_step:.8g}')
print(f'Optical-depth-gradient smoothing window: {smoothing_window} points')
print(f'Characteristic-frequency Fourier band: {spectral_band}')
print(f'Natural omega spread: {omega_spread:.8g}')
print(f'Fixed Kuramoto coupling K: {K:.8g}')
print(f'Drive-frequency b range: [{b_lo:.6g}, {b_hi:.6g}] cycles/model-unit')
print(f'Drive-amplitude a range: [0, {a_max:.6g}]')
print(f'Initial-phase realizations per grid point: {n_initializations}')

frequency_df = pd.DataFrame({
    'sample_id': radiation_samples,
    'subbin_peak_frequency': subbin_peak_freqs,
    'spectral_centroid_frequency': characteristic_freqs,
    'spectral_spread': spectral_spreads,
    'normalized_cycles_per_model_unit': normalized_cycles,
    'omega_rad_per_model_unit': omega,
})
frequency_csv = os.path.join(out_dir, f"{CELL9_RESULT_STEM}_characteristic_frequencies.csv")
frequency_df.to_csv(frequency_csv, index=False)
display(frequency_df)

# Diagnostic guardrail: do not proceed with an almost-degenerate oscillator population.
if np.std(normalized_cycles) < 1e-5:
    raise RuntimeError('Normalized radiation frequency population is too narrow for an informative Arnold-tongue scan.')

rng_master = np.random.default_rng(rng_seed)
initial_conditions = [
    rng_master.uniform(0.0, 2.0*np.pi, N)
    for _ in range(n_initializations)
]

tau_eval = np.linspace(tau_start, tau_end, n_tau_eval)
tau_span = (tau_start, tau_end)
transient_start_idx = int(transient_fraction * len(tau_eval))

synchronization_array = np.zeros((len(a_values), len(b_values)), dtype=float)
sync_std_array = np.zeros_like(synchronization_array)
drive_lock_array = np.zeros_like(synchronization_array)
collective_drive_lock_array = np.zeros_like(synchronization_array)
freq_error_array = np.zeros_like(synchronization_array)

print('\nStarting Arnold tongue grid')
print('===========================')
print(f'Grid: {len(a_values)} x {len(b_values)} = {len(a_values)*len(b_values)} parameter pairs')
print(f'Each pair averaged over {n_initializations} initial phase realization(s)')

for i, a in enumerate(a_values):
    print(f'Processing amplitude row {i+1}/{len(a_values)} | a={a:.6g}')
    for j, b in enumerate(b_values):
        rep_sync, rep_sync_std = [], []
        rep_drive, rep_collective, rep_freq_error = [], [], []

        for theta0 in initial_conditions:
            tau, theta, r = simulate_kuramoto(
                omega=omega,
                K=K,
                a=float(a),
                b=float(b),
                theta0=theta0,
                t_span=tau_span,
                t_eval=tau_eval,
            )

            r_ss = r[transient_start_idx:]
            theta_ss = theta[:, transient_start_idx:]
            tau_ss = tau[transient_start_idx:]

            rep_sync.append(float(np.mean(r_ss)))
            rep_sync_std.append(float(np.std(r_ss)))

            drive_phase = 2.0*np.pi*b*tau_ss
            relative_phase = np.exp(1j*(theta_ss - drive_phase[None,:]))
            per_oscillator_drive_lock = np.abs(np.mean(relative_phase, axis=1))
            rep_drive.append(float(np.mean(per_oscillator_drive_lock)))

            z_ss = np.mean(np.exp(1j*theta_ss), axis=0)
            collective_phase = np.angle(z_ss)
            rep_collective.append(float(np.abs(np.mean(np.exp(1j*(collective_phase-drive_phase))))))

            unwrapped_theta = np.unwrap(theta_ss, axis=1)
            observed_omega_each = (unwrapped_theta[:,-1]-unwrapped_theta[:,0]) / (tau_ss[-1]-tau_ss[0])
            drive_omega = 2.0*np.pi*b
            rep_freq_error.append(float(np.mean(np.abs(observed_omega_each-drive_omega))))

        synchronization_array[i,j] = np.mean(rep_sync)
        sync_std_array[i,j] = np.mean(rep_sync_std)
        drive_lock_array[i,j] = np.mean(rep_drive)
        collective_drive_lock_array[i,j] = np.mean(rep_collective)
        freq_error_array[i,j] = np.mean(rep_freq_error)

# Baseline-referenced maps: row zero is a=0, the genuine no-drive condition.
sync_gain = synchronization_array - synchronization_array[0:1,:]
sync_variability_change = sync_std_array - sync_std_array[0:1,:]
drive_lock_gain = drive_lock_array - drive_lock_array[0:1,:]
freq_error_improvement = freq_error_array[0:1,:] - freq_error_array

npz_path = os.path.join(out_dir, f"{CELL9_RESULT_STEM}.npz")
np.savez_compressed(
    npz_path,
    synchronization_array=synchronization_array,
    sync_std_array=sync_std_array,
    drive_lock_array=drive_lock_array,
    collective_drive_lock_array=collective_drive_lock_array,
    freq_error_array=freq_error_array,
    sync_gain=sync_gain,
    sync_variability_change=sync_variability_change,
    drive_lock_gain=drive_lock_gain,
    freq_error_improvement=freq_error_improvement,
    a_values=a_values,
    b_values=b_values,
    omega=omega,
    normalized_cycles=normalized_cycles,
    characteristic_freqs=characteristic_freqs,
    subbin_peak_freqs=subbin_peak_freqs,
    spectral_spreads=spectral_spreads,
    radiation_samples=np.array(radiation_samples, dtype=object),
    K=K,
    source_file=public_project_path(actual_source_path),
    speed_mode=speed_mode,
    n_initializations=n_initializations,
)

# Long-form results table.
grid_rows = []
for i, a in enumerate(a_values):
    for j, b in enumerate(b_values):
        grid_rows.append({
            'speed_mode': speed_mode,
            'a': float(a),
            'b_cycles_per_model_unit': float(b),
            'drive_omega_rad_per_model_unit': float(2.0*np.pi*b),
            'mean_sync': float(synchronization_array[i,j]),
            'std_sync': float(sync_std_array[i,j]),
            'drive_lock': float(drive_lock_array[i,j]),
            'collective_drive_lock': float(collective_drive_lock_array[i,j]),
            'frequency_error': float(freq_error_array[i,j]),
            'sync_gain_vs_no_drive': float(sync_gain[i,j]),
            'sync_variability_change_vs_no_drive': float(sync_variability_change[i,j]),
            'drive_lock_gain_vs_no_drive': float(drive_lock_gain[i,j]),
            'frequency_error_improvement_vs_no_drive': float(freq_error_improvement[i,j]),
        })
grid_df = pd.DataFrame(grid_rows)
grid_csv = os.path.join(out_dir, f"{CELL9_RESULT_STEM}.csv")
grid_df.to_csv(grid_csv, index=False)

print('\nModel diagnostics')
print('---------------------------')
print(f'Characteristic frequency CV: {np.std(characteristic_freqs)/np.mean(characteristic_freqs):.6g}')
print(f'omega min/max: {omega.min():.6g} / {omega.max():.6g}')
print(f'mean synchronization min/max: {synchronization_array.min():.6g} / {synchronization_array.max():.6g}')
print(f'drive locking min/max: {drive_lock_array.min():.6g} / {drive_lock_array.max():.6g}')
print(f'frequency error min/max: {freq_error_array.min():.6g} / {freq_error_array.max():.6g}')

# ------------------------------------------------------------------
# Plotting.
# ------------------------------------------------------------------
plot_heatmap(
    drive_lock_array,
    f'Drive-Locking Arnold Tongue — Radiation Model\nspeed: {speed_mode}',
    'Mean oscillator-to-drive locking index',
    f"{CELL9_RESULT_STEM}_plots_kuramoto_drive_locking_arnold_tongue.png",
    cmap_name=RED_SEQUENTIAL_CMAP,
    vmin=0.0, vmax=1.0, robust=False, contour=True,
)

plot_heatmap(
    collective_drive_lock_array,
    f'Collective Phase Drive Locking — Radiation Model\nspeed: {speed_mode}',
    'Collective phase-to-drive locking index',
    f"{CELL9_RESULT_STEM}_plots_kuramoto_collective_drive_locking.png",
    cmap_name=RED_SEQUENTIAL_CMAP,
    vmin=0.0, vmax=1.0, robust=False, contour=True,
)

plot_heatmap(
    freq_error_array,
    f'Frequency Error Relative to External Drive — Radiation Model\nspeed: {speed_mode}',
    'Mean |observed omega - drive omega|',
    f"{CELL9_RESULT_STEM}_plots_kuramoto_drive_frequency_error.png",
    cmap_name=RED_REVERSED_CMAP,
    robust=True, bounds=(0.0, np.inf), contour=True,
)

plot_heatmap(
    synchronization_array,
    f'Kuramoto Mean Synchronization — Radiation Model\nspeed: {speed_mode}',
    'Mean synchronization r',
    f"{CELL9_RESULT_STEM}_plots_kuramoto_mean_sync_visible_scale.png",
    cmap_name=RED_SEQUENTIAL_CMAP,
    robust=True, bounds=(0.0,1.0), contour=True,
)

plot_heatmap(
    synchronization_array,
    f'Kuramoto Mean Synchronization, Full 0-to-1 Scale\nspeed: {speed_mode}',
    'Mean synchronization r',
    f"{CELL9_RESULT_STEM}_plots_kuramoto_mean_sync_full_range.png",
    cmap_name=RED_SEQUENTIAL_CMAP,
    vmin=0.0, vmax=1.0, robust=False, bounds=(0.0,1.0), contour=False,
)

# Robust symmetric baseline-referenced maps.
for arr, title, label, filename in [
    (sync_gain, 'Drive-Induced Synchronization Gain', 'Mean sync minus a=0 baseline', f"{CELL9_RESULT_STEM}_plots_kuramoto_sync_gain_vs_no_drive.png"),
    (sync_variability_change, 'Synchronization Variability Change', 'Sync std minus a=0 baseline', f"{CELL9_RESULT_STEM}_plots_kuramoto_sync_variability_change_vs_no_drive.png"),
    (drive_lock_gain, 'Drive-Locking Gain', 'Drive lock minus a=0 baseline', f"{CELL9_RESULT_STEM}_plots_kuramoto_drive_lock_gain_vs_no_drive.png"),
    (freq_error_improvement, 'Frequency-Error Improvement', 'a=0 error minus driven error', f"{CELL9_RESULT_STEM}_plots_kuramoto_frequency_error_improvement.png"),
]:
    lim = robust_symmetric_limit(arr, percentile=98.0)
    plot_heatmap(
        arr,
        f'{title}\nspeed: {speed_mode}',
        label,
        filename,
        cmap_name=RED_DIVERGING_CMAP,
        vmin=-lim, vmax=lim, robust=False, contour=False,
    )

# Characteristic radiation frequencies by sample.
fig, ax = plt.subplots(figsize=(10,8), facecolor=BG)
style_ax(ax)
order = np.argsort(characteristic_freqs)
ax.barh(
    np.array(radiation_samples)[order],
    characteristic_freqs[order],
    color=ACCENT,
    edgecolor=RED_SOFT,
)
ax.set_title('Characteristic Radiation Spatial Frequency by N003 Sample')
ax.set_xlabel('Spectral-centroid frequency (cycles per unit log-energy)')
ax.set_ylabel('Measured sample')
save_show_close(fig, os.path.join(plots_dir, f"{CELL9_RESULT_STEM}_plots_characteristic_radiation_spatial_frequency_by_sample.png"))

# Natural angular frequencies: now positive and non-degenerate.
fig, ax = plt.subplots(figsize=(10,8), facecolor=BG)
style_ax(ax)
order = np.argsort(omega)
ax.barh(np.array(radiation_samples)[order], omega[order], color=ACCENT, edgecolor=RED_SOFT)
ax.axvline(np.median(omega), color=WHITE, linestyle='--', linewidth=1.0, alpha=0.75, label='Median omega')
ax.set_title('Kuramoto Natural Angular Frequencies Derived from N003 Transmission')
ax.set_xlabel('Natural angular frequency omega (rad/model-unit)')
ax.set_ylabel('Measured sample')
leg = ax.legend(loc='best', frameon=True)
for txt in leg.get_texts():
    txt.set_color(RED)
save_show_close(fig, os.path.join(plots_dir, f"{CELL9_RESULT_STEM}_plots_kuramoto_omega_by_radiation_sample.png"))

# Peak versus centroid is a useful extraction diagnostic; unlike omega-vs-frequency,
# it is not a guaranteed linear transformation.
fig, ax = plt.subplots(figsize=(8,6), facecolor=BG)
style_ax(ax)
ax.scatter(subbin_peak_freqs, characteristic_freqs, color=ACCENT, edgecolor=WHITE, linewidth=0.6, s=65)
for rank, (sample, xp, yc) in enumerate(zip(radiation_samples, subbin_peak_freqs, characteristic_freqs)):
    ax.annotate(sample, (xp,yc), xytext=(5, 4 + 8*(rank%3)), textcoords='offset points', color=RED_SOFT, fontsize=8)
lo = min(np.min(subbin_peak_freqs), np.min(characteristic_freqs))
hi = max(np.max(subbin_peak_freqs), np.max(characteristic_freqs))
ax.plot([lo,hi],[lo,hi], color=WHITE, linestyle='--', alpha=0.55, linewidth=1.0)
ax.set_title('Radiation Fourier Peak vs Spectral-Centroid Frequency')
ax.set_xlabel('Quadratic sub-bin peak frequency')
ax.set_ylabel('Spectral-centroid characteristic frequency')
save_show_close(fig, os.path.join(plots_dir, f"{CELL9_RESULT_STEM}_plots_radiation_peak_vs_centroid_frequency.png"))

# Same marginal summaries as signal cell.
plot_line_with_band(b_values, np.nanmean(synchronization_array,axis=0), np.nanstd(synchronization_array,axis=0),
                    f'Mean Synchronization Averaged over Amplitude\nspeed: {speed_mode}',
                    'Drive frequency b (cycles/model-unit)', 'Mean synchronization r', f"{CELL9_RESULT_STEM}_plots_mean_sync_vs_b.png", (0.0,1.0))
plot_line_with_band(a_values, np.nanmean(synchronization_array,axis=1), np.nanstd(synchronization_array,axis=1),
                    f'Mean Synchronization Averaged over Drive Frequency\nspeed: {speed_mode}',
                    'Drive amplitude a', 'Mean synchronization r', f"{CELL9_RESULT_STEM}_plots_mean_sync_vs_a.png", (0.0,1.0))
plot_line_with_band(b_values, np.nanmean(drive_lock_array,axis=0), np.nanstd(drive_lock_array,axis=0),
                    f'Drive Locking Averaged over Amplitude\nspeed: {speed_mode}',
                    'Drive frequency b (cycles/model-unit)', 'Drive-locking index', f"{CELL9_RESULT_STEM}_plots_drive_lock_vs_b.png", (0.0,1.0))
plot_line_with_band(a_values, np.nanmean(drive_lock_array,axis=1), np.nanstd(drive_lock_array,axis=1),
                    f'Drive Locking Averaged over Drive Frequency\nspeed: {speed_mode}',
                    'Drive amplitude a', 'Drive-locking index', f"{CELL9_RESULT_STEM}_plots_drive_lock_vs_a.png", (0.0,1.0))
plot_line_with_band(b_values, np.nanmean(freq_error_array,axis=0), np.nanstd(freq_error_array,axis=0),
                    f'Frequency Error Averaged over Amplitude\nspeed: {speed_mode}',
                    'Drive frequency b (cycles/model-unit)', 'Mean frequency error', f"{CELL9_RESULT_STEM}_plots_frequency_error_vs_b.png", None, physical_bounds=(0.0,None))
plot_line_with_band(a_values, np.nanmean(freq_error_array,axis=1), np.nanstd(freq_error_array,axis=1),
                    f'Frequency Error Averaged over Drive Frequency\nspeed: {speed_mode}',
                    'Drive amplitude a', 'Mean frequency error', f"{CELL9_RESULT_STEM}_plots_frequency_error_vs_a.png", None, physical_bounds=(0.0,None))

summary_txt = os.path.join(out_dir, f"{CELL9_RESULT_STEM}.txt")
with open(summary_txt, 'w') as f:
    f.write('Radiation-Informed Kuramoto Arnold-Tongue Analysis\n')
    f.write('===========================================================\n\n')
    f.write(f'Source: {public_project_path(actual_source_path)}\n')
    f.write(f'Speed mode: {speed_mode}\n')
    f.write(f'N samples: {N}\n')
    f.write(f'Characteristic-frequency CV: {np.std(characteristic_freqs)/np.mean(characteristic_freqs):.10g}\n')
    f.write(f'K: {K:.10g}\n')
    f.write(f'a range: {a_values.min():.10g} to {a_values.max():.10g}\n')
    f.write(f'b range: {b_values.min():.10g} to {b_values.max():.10g}\n')
    f.write(f'Initializations per grid point: {n_initializations}\n')
    f.write('Natural-frequency map: omega_i = 2*pi*(f_i/median(f))\n')
    f.write('Radiation feature: detrended derivative of smoothed optical depth -ln(T) vs log-energy\n')
    f.write('Characteristic f_i: spectral centroid; quadratic sub-bin peak retained as diagnostic\n')
    f.write('Contrast maps: driven value minus a=0 baseline; robust symmetric 98th-percentile color limits\n')

print(f'\nSaved NPZ: {npz_path}')
print(f'Saved grid CSV: {grid_csv}')
print(f'Saved characteristic-frequency CSV: {frequency_csv}')
print(f'Saved summary: {summary_txt}')
print(f'Saved plots to: {plots_dir}')


# Mode Locked

<div style="font-size:14px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:20px;border-radius:8px;margin:10px;display:flex;flex-wrap:nowrap;justify-content:space-between;line-height:1.55;">

<div style="flex:1;margin-right:10px;">
<h2 style="color:#FF4D4D;">Introduction</h2>
<p>
This cell applies circle-map fixed-point-locking analysis to the processed
<b>N003 / ISIS RB2000164 neutron-transmission data</b>.
The circle map itself remains the same reduced nonlinear phase model; the radiation data provide the distribution of initial phases.
</p>

<h2 style="color:#FF4D4D;">Mathematical Foundations</h2>
<p>
The standard circle map is
<b>θ<sub>n+1</sub> = θ<sub>n</sub> + Ω − [K/(2π)] sin(2πθ<sub>n</sub>) mod 1</b>.
Here <b>Ω</b> is the model rotation parameter and <b>K</b> controls nonlinearity/coupling.
</p>

<h2 style="color:#FF4D4D;">Radiation Phase Construction</h2>
<p>
Raw transmission curves are strongly trend-dominated and do not possess a direct temporal phase.
The Hilbert-phase operation is applied along the uniformly sampled log-energy coordinate;
the processed transmission curves are first converted to optical depth
<b>τ(E) = −ln T(E)</b>, gently smoothed, differentiated with respect to
<b>s = ln(E/E<sub>min</sub>)</b>, linearly detrended, and standardized.
The Hilbert transform is then applied along this ordered log-energy coordinate.
</p>
</div>

<div style="flex:1;margin-left:10px;">
<h2 style="color:#FF4D4D;">Cross-Sample Circular Mean</h2>
<p>
Each measured N003 sample produces one analytic-signal phase curve over log-energy.
At every coordinate point, the circular mean is computed across the measured radiation ensemble.
That radiation-derived mean phase supplies the initial-condition ensemble for the circle map.
</p>

<h2 style="color:#FF4D4D;">Mode-Locking Analysis</h2>
<p>
For each pair of model parameters <b>Ω</b> and <b>K</b>, the map is iterated from the radiation-derived phase samples.
The reported quantity is the proportion of initial conditions whose final wrapped one-step displacement falls below the fixed-point tolerance.
</p>

<h2 style="color:#FF4D4D;">Interpretation</h2>
<p>
This is an <b>effective fixed-point-locking model conditioned on radiation-derived phase coordinates</b>.
It is not evidence that shielding material literally evolves as a circle map, and it is not a full rational
<b>p:q</b> Arnold-tongue analysis. Rotation-number plateaus are treated separately in the next mathematical block.
</p>
</div>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Radiation object.</b> The data mapping uses <b>N003.SEQ_SAMPLE / N003.TRAJ_E</b> with processed transmission <b>T(E)</b> over the ordered neutron-energy coordinate.</p>
<p><b>Phase caveat.</b> Hilbert phase is most naturally interpreted for oscillatory signals. Here it is used as a formal analytic-signal phase of a detrended attenuation-shape feature over log-energy. The code therefore reports the cross-sample phase resultant length as a quality diagnostic; low resultant length means the circular mean phase is weakly defined.</p>
<p><b>Plotting.</b> All figures retain the project convention of black backgrounds with red scientific accents. Results and plots are written directly into <b>results/phase2/</b> with the Cell-13 result stem.</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">
<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Circle-Map Fixed-Point Locking</h3>

<p>
The standard circle map evolves a phase on the unit circle:
<b>θ<sub>n+1</sub> = θ<sub>n</sub> + Ω − [K/(2π)] sin(2πθ<sub>n</sub>) mod 1</b>.
This cell uses radiation-derived analytic-signal phases as initial conditions and measures the proportion of trajectories whose final wrapped one-step displacement is below a tolerance.
That is an empirical <b>fixed-point-locking probability</b>, not a complete <b>p:q</b> Arnold-tongue calculation.
</p>

<p>
For a fixed point modulo an integer <b>m</b>, the map must satisfy
<b>Ω − m = [K/(2π)] sin(2πθ*)</b>.
A necessary existence condition is therefore
<b>|Ω − m| ≤ K/(2π)</b>.
The plotted theoretical boundaries are the fixed-point existence boundaries of the circle map.
</p>

<p><b>Cell role.</b>
The grid scan reports fixed-point locking only. Rational rotation-number plateaus and general mode-locking ratios are analyzed separately in the next circle-map cell set.
</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Radiation interpretation.</b> The phase variable here is a formal phase coordinate extracted from attenuation-shape structure along log-energy. The circle-map parameters <b>Ω</b> and <b>K</b> are model parameters; they are not measured neutron frequencies or material constants.</p>
<p><b>What the colors mean.</b> A heatmap value of 1 means every retained radiation-derived initial phase converged to the cell's fixed-point criterion at that parameter pair; 0 means none did. Intermediate values represent a basin fraction over the radiation-derived phase ensemble.</p>
</div>

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import hilbert, savgol_filter, detrend

# -------------------------------------------------------
# SPEED MODE — shared four-level convention
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError("FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'.")

_SPEED_COUNT_FACTOR = {
    "full": 1.0,
    "balanced": 0.60,
    "fast": 0.35,
    "ultra": 0.18,
}[FAST_OPTION]

def _speed_count(original, minimum=1):
    return int(original) if FAST_OPTION == "full" else max(int(minimum), int(round(float(original) * _SPEED_COUNT_FACTOR)))

# -------------------------------------------------------
# CONFIG — project repository and flat result directory
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next((p for p in N003_SOURCE_CANDIDATES if p.exists()), None)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL13_RESULT_STEM = "cell_13_circle_map_fixed_point_locking_results"

out_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

# Grid resolution and speed convention.
n_omega = _speed_count(150, 45)
n_K = _speed_count(150, 45)
omegas = np.linspace(0.0, 1.0, n_omega)
K_values = np.linspace(0.0, 4.0 * np.pi, n_K)

# Circle-map iteration controls.
iterations = _speed_count(60, 25)
tol = 1e-6

# Radiation-derived phase subsampling.
max_phase_samples = _speed_count(5000, 1200)

# Common log-energy resampling.
MAX_COORD_POINTS = 5000
MIN_COORD_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

RED_SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    "circle_map_black_red",
    [BLACK, RED_DIM, RED, RED_SOFT],
    N=256,
)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=ACCENT)

    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)

    for spine in ax.spines.values():
        spine.set_color(ACCENT)

    ax.grid(True, alpha=0.20, color=ACCENT)

def style_colorbar(cbar):
    cbar.outline.set_edgecolor(ACCENT)
    cbar.ax.tick_params(color=ACCENT, labelcolor=ACCENT)

    for label in cbar.ax.get_yticklabels():
        label.set_color(ACCENT)

    cbar.ax.yaxis.label.set_color(ACCENT)

def save_show_close(fig, out_path, dpi=220):
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)

def ensure_time_channel_layout(x, n_channels_expected):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D ordered-response array, got shape {x.shape}")

    if x.shape[1] == n_channels_expected:
        return x

    if x.shape[0] == n_channels_expected:
        return x.T

    raise ValueError(f"Unexpected ordered-response shape: {x.shape}")

def to_cycle_phase(rad_phase):
    """
    Convert radians to phase on [0, 1).
    """
    return np.mod(rad_phase / (2.0 * np.pi), 1.0)

def circular_mean_phase(phases_rad, axis=1):
    """
    Circular mean phase in radians.
    """
    z = np.mean(np.exp(1j * phases_rad), axis=axis)
    return np.angle(z)

def uniform_subsample(x, max_n):
    x = np.asarray(x)

    if len(x) <= max_n:
        return x

    idx = np.linspace(0, len(x) - 1, max_n).astype(int)
    return x[idx]

def circle_map(theta, omega, K):
    """
    Standard circle map on phase cycles [0, 1).

    theta_{n+1} = theta_n + Omega - (K / 2pi) sin(2pi theta_n) mod 1
    """
    theta_next = theta + omega - (K / (2.0 * np.pi)) * np.sin(2.0 * np.pi * theta)
    return np.mod(theta_next, 1.0)

def wrapped_cycle_distance(a, b):
    """
    Shortest distance on the unit circle in cycle coordinates.
    """
    return np.abs(((a - b + 0.5) % 1.0) - 0.5)

def fixed_point_locked_proportion_row(K, omegas, phases, iterations=60, tol=1e-6):
    """
    Vectorized over all omegas for one fixed K.
    Also vectorized over all initial phases.

    This returns the proportion of radiation-derived initial phase samples that
    end near a fixed point according to the final one-step wrapped distance.

    This is fixed-point locking, not full p:q rotation-number Arnold tongues.

    Returns
    -------
    proportions : shape (n_omega,)
    """
    omegas = np.asarray(omegas, dtype=float)
    phases = np.asarray(phases, dtype=float)

    # theta shape: (n_omega, n_phase)
    theta = np.broadcast_to(phases[None, :], (len(omegas), len(phases))).copy()
    omega_col = omegas[:, None]

    for _ in range(iterations):
        theta_next = circle_map(theta, omega_col, K)
        theta = theta_next

    # Check final one-step movement
    theta_next = circle_map(theta, omega_col, K)
    final_delta = wrapped_cycle_distance(theta_next, theta)

    locked = final_delta < tol
    return locked.mean(axis=1)

# -------------------------------------------------------
# RADIATION DATA HELPERS
# -------------------------------------------------------
def _find_radiation_column(df, exact=(), contains=(), exclude=()):
    lower_map = {str(c).lower(): c for c in df.columns}

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    matches = []
    for c in df.columns:
        lc = str(c).lower()
        if (
            any(token.lower() in lc for token in contains)
            and not any(token.lower() in lc for token in exclude)
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(matches, key=lambda c: (len(str(c)), str(c)))[0]


def load_rb2000164_transmission_matrix(source_path):
    """
    Build one common-log-energy transmission curve per measured N003 sample.
    """
    source_path = Path(source_path)

    if source_path.name.endswith(".csv.gz") or source_path.suffix.lower() == ".csv":
        df = pd.read_csv(source_path)
    elif source_path.suffix.lower() == ".parquet":
        try:
            df = pd.read_parquet(source_path)
        except ImportError as exc:
            csv_fallback = source_path.with_suffix("").with_suffix(".csv.gz")
            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(source_path)
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback was found."
                ) from exc
    else:
        raise ValueError(f"Unsupported RB2000164 input format: {source_path}")

    sample_col = _find_radiation_column(
        df,
        exact=("sample_id", "sample", "sample_name"),
        contains=("sample",),
        exclude=("uncert",),
    )
    energy_col = _find_radiation_column(
        df,
        exact=("energy_eV", "energy_ev", "neutron_energy_eV", "energy_in_eV"),
        contains=("energy",),
        exclude=("uncert", "lower", "upper"),
    )
    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=("uncert", "sigma", "error"),
    )

    if sample_col is None:
        raise KeyError("Could not identify N003 sample column.")
    if energy_col is None:
        raise KeyError("Could not identify N003 neutron-energy column.")
    if transmission_col is None:
        raise KeyError("Could not identify N003 transmission column.")

    curves = {}

    for sample_id, part in df.groupby(sample_col, sort=True, dropna=False):
        q = part[[energy_col, transmission_col]].copy()
        q[energy_col] = pd.to_numeric(q[energy_col], errors="coerce")
        q[transmission_col] = pd.to_numeric(q[transmission_col], errors="coerce")
        q = q.replace([np.inf, -np.inf], np.nan).dropna()
        q = q[
            (q[energy_col] > 0)
            & (q[transmission_col] > 0)
        ]

        q = (
            q.groupby(energy_col, as_index=False, sort=True)[transmission_col]
            .mean()
            .sort_values(energy_col)
        )

        if len(q) < MIN_COORD_POINTS:
            print(
                f"Skipping sample {sample_id}: only {len(q)} usable points.",
                flush=True,
            )
            continue

        curves[str(sample_id)] = (
            np.log(q[energy_col].to_numpy(dtype=float)),
            q[transmission_col].to_numpy(dtype=float),
        )

    if len(curves) < 3:
        raise RuntimeError(
            "Need at least three sufficiently sampled N003 transmission curves."
        )

    common_lo = max(v[0].min() for v in curves.values())
    common_hi = min(v[0].max() for v in curves.values())

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval exists across retained N003 samples."
        )

    n_grid = min(
        MAX_COORD_POINTS,
        min(len(v[0]) for v in curves.values()),
    )

    if n_grid < MIN_COORD_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(common_lo, common_hi, n_grid)

    labels = []
    columns = []

    for sample_id in sorted(curves):
        logE, transmission = curves[sample_id]
        interp_T = np.interp(logE_grid, logE, transmission)
        labels.append(sample_id)
        columns.append(interp_T)

    matrix = np.column_stack(columns)
    s_grid = logE_grid - logE_grid[0]
    ds = float(s_grid[1] - s_grid[0])

    return (
        matrix,
        labels,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        source_path,
    )


def radiation_shape_features(transmission_matrix, ds):
    """
    Shape-sensitive radiation feature used before Hilbert phase extraction.

    T(E)
      -> optical depth -ln(T)
      -> Savitzky-Golay smoothing
      -> derivative with respect to log-energy
      -> linear detrending
      -> standardization
    """
    T = np.asarray(transmission_matrix, dtype=float)

    if T.ndim != 2:
        raise ValueError(
            "transmission_matrix must be 2D: coordinate x measured sample."
        )

    n = T.shape[0]

    window = max(11, int(round(0.03 * n)))

    if window % 2 == 0:
        window += 1

    if window >= n:
        window = n - 1 if (n - 1) % 2 == 1 else n - 2

    window = max(window, 5)

    features = np.empty_like(T, dtype=float)

    for j in range(T.shape[1]):
        transmission = np.clip(T[:, j], 1e-12, None)
        optical_depth = -np.log(transmission)

        smoothed = savgol_filter(
            optical_depth,
            window_length=window,
            polyorder=3,
            mode="interp",
        )

        slope = np.gradient(smoothed, ds)
        shape = detrend(slope, type="linear")
        shape = shape - np.mean(shape)

        scale = np.std(shape)

        if not np.isfinite(scale) or scale <= 0:
            raise RuntimeError(
                f"Degenerate radiation phase feature for sample column {j}."
            )

        features[:, j] = shape / scale

    return features, window


# -------------------------------------------------------
# LOAD RADIATION CURVES AND EXTRACT ANALYTIC-SIGNAL PHASE
# -------------------------------------------------------
(
    transmission_matrix,
    radiation_samples,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    actual_source_path,
) = load_rb2000164_transmission_matrix(N003_PATH)

n_coordinate_samples, n_samples = transmission_matrix.shape

shape_matrix, smoothing_window = radiation_shape_features(
    transmission_matrix,
    coordinate_step,
)

shape_matrix = ensure_time_channel_layout(
    shape_matrix,
    n_samples,
)

analytic_signal = hilbert(shape_matrix, axis=0)
inst_phase_rad = np.angle(analytic_signal)

# Circular mean across measured radiation samples.
mean_phase_vector = np.mean(
    np.exp(1j * inst_phase_rad),
    axis=1,
)

phase_resultant_length = np.abs(mean_phase_vector)
avg_phase_rad = np.angle(mean_phase_vector)
avg_phase_cycles = to_cycle_phase(avg_phase_rad)

# Uniform phase subsampling for speed.
avg_phase_cycles_sub = uniform_subsample(
    avg_phase_cycles,
    max_phase_samples,
)
phase_resultant_sub = uniform_subsample(
    phase_resultant_length,
    max_phase_samples,
)

print(f"Using processed radiation source: {public_project_path(actual_source_path)}")
print(
    f"Loaded radiation matrix: "
    f"{n_coordinate_samples} log-energy points x {n_samples} measured samples"
)
print(f"Measured samples: {radiation_samples}")
print(f"Optical-depth-gradient smoothing window: {smoothing_window}")
print(f"Original radiation phase samples: {len(avg_phase_cycles)}")
print(f"Using radiation phase samples:    {len(avg_phase_cycles_sub)}")
print(
    "Mean cross-sample phase resultant length: "
    f"{np.mean(phase_resultant_length):.6f}"
)

# -------------------------------------------------------
# FIXED-POINT LOCKING GRID COMPUTATION
# -------------------------------------------------------
locked = np.zeros(
    (len(K_values), len(omegas)),
    dtype=float,
)

for i, K in enumerate(K_values):
    locked[i, :] = fixed_point_locked_proportion_row(
        K=K,
        omegas=omegas,
        phases=avg_phase_cycles_sub,
        iterations=iterations,
        tol=tol,
    )

    if (
        (i + 1) % 10 == 0
        or i == 0
        or i == len(K_values) - 1
    ):
        print(
            f"Processed K row {i + 1}/{len(K_values)}",
            flush=True,
        )

# -------------------------------------------------------
# SAVE RESULTS — flat results/phase2 convention
# -------------------------------------------------------
npz_path = os.path.join(
    out_dir,
    f"{CELL13_RESULT_STEM}.npz",
)

np.savez_compressed(
    npz_path,
    locked=locked,
    omegas=omegas,
    K_values=K_values,
    avg_phase_cycles_sub=avg_phase_cycles_sub,
    avg_phase_cycles_full=avg_phase_cycles,
    phase_resultant_length_full=phase_resultant_length,
    phase_resultant_length_sub=phase_resultant_sub,
    radiation_samples=np.array(radiation_samples, dtype=object),
    s_grid=s_grid,
    coordinate_step=coordinate_step,
    smoothing_window=smoothing_window,
    iterations=iterations,
    tol=tol,
    max_phase_samples=max_phase_samples,
    source_file=public_project_path(actual_source_path),
    speed_mode=FAST_OPTION,
)

grid_rows = []

for i, K in enumerate(K_values):
    for j, omega in enumerate(omegas):
        grid_rows.append({
            "Omega": float(omega),
            "K": float(K),
            "proportion_fixed_point_locked": float(locked[i, j]),
        })

grid_df = pd.DataFrame(grid_rows)

grid_csv = os.path.join(
    out_dir,
    f"{CELL13_RESULT_STEM}.csv",
)
grid_df.to_csv(grid_csv, index=False)

phase_csv = os.path.join(
    out_dir,
    f"{CELL13_RESULT_STEM}_radiation_phase_samples.csv",
)

pd.DataFrame({
    "subsample_index": np.arange(len(avg_phase_cycles_sub)),
    "phase_cycles": avg_phase_cycles_sub,
    "cross_sample_resultant_length": phase_resultant_sub,
}).to_csv(
    phase_csv,
    index=False,
)

summary_txt = os.path.join(
    out_dir,
    f"{CELL13_RESULT_STEM}.txt",
)

# -------------------------------------------------------
# DIAGNOSTICS
# -------------------------------------------------------
locked_min = float(np.nanmin(locked))
locked_max = float(np.nanmax(locked))
locked_mean = float(np.nanmean(locked))

phase_min = float(np.nanmin(avg_phase_cycles_sub))
phase_max = float(np.nanmax(avg_phase_cycles_sub))
phase_mean = float(np.nanmean(avg_phase_cycles_sub))
phase_std = float(np.nanstd(avg_phase_cycles_sub))

resultant_min = float(np.nanmin(phase_resultant_sub))
resultant_max = float(np.nanmax(phase_resultant_sub))
resultant_mean = float(np.nanmean(phase_resultant_sub))
resultant_median = float(np.nanmedian(phase_resultant_sub))

with open(summary_txt, "w") as f:
    f.write(
        "Circle Map Fixed-Point Locking Using Radiation-Derived "
        "Analytic-Signal Phase\n"
    )
    f.write(
        "==============================================================\n\n"
    )
    f.write(
        "Interpretation:\n"
        "  Fixed-point locking probability for the circle map.\n"
        "  This is not a full p:q Arnold-tongue calculation.\n"
        "  Radiation phase is an analytic-signal phase of a detrended "
        "optical-depth-gradient feature over log-energy.\n\n"
    )
    f.write(f"Source: {public_project_path(actual_source_path)}\n")
    f.write("Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n")
    f.write("Observable: transmission T(E)\n")
    f.write(f"Measured samples: {n_samples}\n")
    f.write(f"Coordinate points: {n_coordinate_samples}\n")
    f.write(f"Coordinate step: {coordinate_step}\n")
    f.write(f"Smoothing window: {smoothing_window}\n")
    f.write(
        f"Omega range: [{omegas.min()}, {omegas.max()}] "
        f"with {len(omegas)} points\n"
    )
    f.write(
        f"K range: [{K_values.min()}, {K_values.max()}] "
        f"with {len(K_values)} points\n"
    )
    f.write(f"Iterations: {iterations}\n")
    f.write(f"Tolerance: {tol}\n")
    f.write(
        f"Original radiation phase samples: {len(avg_phase_cycles)}\n"
    )
    f.write(
        f"Used radiation phase samples: {len(avg_phase_cycles_sub)}\n"
    )
    f.write(
        f"Mean cross-sample phase resultant length: "
        f"{resultant_mean:.10f}\n"
    )
    f.write(
        f"Median cross-sample phase resultant length: "
        f"{resultant_median:.10f}\n"
    )

print("\nFixed-point locking diagnostics:")
print(f"  locked min  = {locked_min:.6f}")
print(f"  locked max  = {locked_max:.6f}")
print(f"  locked mean = {locked_mean:.6f}")

print("\nRadiation-derived phase diagnostics:")
print(f"  phase min   = {phase_min:.6f} cycles")
print(f"  phase max   = {phase_max:.6f} cycles")
print(f"  phase mean  = {phase_mean:.6f} cycles")
print(f"  phase std   = {phase_std:.6f} cycles")
print(f"  resultant mean   = {resultant_mean:.6f}")
print(f"  resultant median = {resultant_median:.6f}")

print(
    "\nNOTE: The map reports fixed-point locking of the reduced circle-map "
    "model initialized by radiation-derived phase coordinates. "
    "It is not a physical temporal-locking claim."
)

# -------------------------------------------------------
# PLOT 1: FIXED-POINT LOCKING HEATMAP
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)
style_ax(ax)

im = ax.imshow(
    locked,
    extent=[
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ],
    origin="lower",
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
    cmap=RED_SEQUENTIAL_CMAP,
)

ax.set_title(
    "Circle Map Fixed-Point Locking Probability — Radiation Phase"
)
ax.set_xlabel("Circle-map frequency parameter Ω")
ax.set_ylabel("Nonlinearity / coupling parameter K")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Proportion fixed-point locked")
style_colorbar(cbar)

plot1_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_01_fixed_point_locking_heatmap.png",
)
save_show_close(fig, plot1_path)

# -------------------------------------------------------
# PLOT 2: THEORETICAL FIXED-POINT EXISTENCE OVERLAY
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)
style_ax(ax)

im = ax.imshow(
    locked,
    extent=[
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ],
    origin="lower",
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
    cmap=RED_SEQUENTIAL_CMAP,
)

omega_line = np.linspace(
    omegas.min(),
    omegas.max(),
    1000,
)

# Necessary fixed-point existence condition:
# |Omega - m| <= K/(2*pi), with principal branches m=0 and m=1.
K_boundary_m0 = 2.0 * np.pi * np.abs(
    omega_line - 0.0
)
K_boundary_m1 = 2.0 * np.pi * np.abs(
    omega_line - 1.0
)

ax.plot(
    omega_line,
    K_boundary_m0,
    color=WHITE,
    linestyle="--",
    linewidth=1.5,
    alpha=0.90,
    label="Existence boundary, m=0",
)

ax.plot(
    omega_line,
    K_boundary_m1,
    color=RED_SOFT,
    linestyle=":",
    linewidth=1.8,
    alpha=0.90,
    label="Existence boundary, m=1",
)

ax.set_ylim(
    K_values.min(),
    K_values.max(),
)

ax.set_title(
    "Fixed-Point Locking with Theoretical Existence Boundaries"
)
ax.set_xlabel("Circle-map frequency parameter Ω")
ax.set_ylabel("Nonlinearity / coupling parameter K")

leg = ax.legend(
    facecolor=BG,
    edgecolor=ACCENT,
)

for text in leg.get_texts():
    text.set_color(ACCENT)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Proportion fixed-point locked")
style_colorbar(cbar)

plot2_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_02_fixed_point_locking_with_boundaries.png",
)
save_show_close(fig, plot2_path)

# -------------------------------------------------------
# PLOT 3: MEAN FIXED-POINT LOCKING ACROSS K
# -------------------------------------------------------
mean_locked_over_K = np.nanmean(
    locked,
    axis=0,
)
std_locked_over_K = np.nanstd(
    locked,
    axis=0,
)

fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)
style_ax(ax)

ax.plot(
    omegas,
    mean_locked_over_K,
    color=ACCENT,
    linewidth=2.0,
)

ax.fill_between(
    omegas,
    np.clip(mean_locked_over_K - std_locked_over_K, 0.0, 1.0),
    np.clip(mean_locked_over_K + std_locked_over_K, 0.0, 1.0),
    color=RED_SOFT,
    alpha=0.18,
)

ax.set_ylim(0.0, 1.0)
ax.set_title("Mean Fixed-Point Locking Across K")
ax.set_xlabel("Circle-map frequency parameter Ω")
ax.set_ylabel("Mean proportion fixed-point locked")

plot3_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_03_mean_fixed_point_locking_across_K.png",
)
save_show_close(fig, plot3_path)

# -------------------------------------------------------
# PLOT 4: MEAN FIXED-POINT LOCKING ACROSS OMEGA
# -------------------------------------------------------
mean_locked_over_omega = np.nanmean(
    locked,
    axis=1,
)
std_locked_over_omega = np.nanstd(
    locked,
    axis=1,
)

fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)
style_ax(ax)

ax.plot(
    K_values,
    mean_locked_over_omega,
    color=ACCENT,
    linewidth=2.0,
)

ax.fill_between(
    K_values,
    np.clip(
        mean_locked_over_omega - std_locked_over_omega,
        0.0,
        1.0,
    ),
    np.clip(
        mean_locked_over_omega + std_locked_over_omega,
        0.0,
        1.0,
    ),
    color=RED_SOFT,
    alpha=0.18,
)

ax.set_ylim(0.0, 1.0)
ax.set_title("Mean Fixed-Point Locking Across Ω")
ax.set_xlabel("Nonlinearity / coupling parameter K")
ax.set_ylabel("Mean proportion fixed-point locked")

plot4_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_04_mean_fixed_point_locking_across_omega.png",
)
save_show_close(fig, plot4_path)

# -------------------------------------------------------
# PLOT 5: RADIATION-DERIVED MEAN PHASE HISTOGRAM
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)
style_ax(ax)

ax.hist(
    avg_phase_cycles_sub,
    bins=60,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_title(
    "Histogram of Radiation-Derived Circular Mean Phase Samples"
)
ax.set_xlabel("Circular mean phase (cycles)")
ax.set_ylabel("Count")

plot5_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_05_radiation_mean_phase_histogram.png",
)
save_show_close(fig, plot5_path)

# -------------------------------------------------------
# PLOT 6: RADIATION-DERIVED PHASE SAMPLES OVER LOG-ENERGY SUBSAMPLE
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 5),
    facecolor=BG,
)
style_ax(ax)

ax.plot(
    np.arange(len(avg_phase_cycles_sub)),
    avg_phase_cycles_sub,
    color=ACCENT,
    linewidth=1.2,
)

ax.set_title(
    "Radiation-Derived Circular Mean Phase Samples"
)
ax.set_xlabel("Uniform log-energy subsample index")
ax.set_ylabel("Phase (cycles)")

plot6_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_06_radiation_mean_phase_samples.png",
)
save_show_close(fig, plot6_path)

# -------------------------------------------------------
# PLOT 7: THRESHOLDED LOCKING MAP
# -------------------------------------------------------
lock_threshold = 0.5
locked_binary = locked >= lock_threshold

fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)
style_ax(ax)

im = ax.imshow(
    locked_binary.astype(float),
    extent=[
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ],
    origin="lower",
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
    cmap=RED_SEQUENTIAL_CMAP,
)

ax.plot(
    omega_line,
    K_boundary_m0,
    color=WHITE,
    linestyle="--",
    linewidth=1.5,
    alpha=0.90,
)

ax.plot(
    omega_line,
    K_boundary_m1,
    color=RED_SOFT,
    linestyle=":",
    linewidth=1.8,
    alpha=0.90,
)

ax.set_ylim(
    K_values.min(),
    K_values.max(),
)

ax.set_title(
    f"Thresholded Fixed-Point Locking Map ≥ {lock_threshold}"
)
ax.set_xlabel("Circle-map frequency parameter Ω")
ax.set_ylabel("Nonlinearity / coupling parameter K")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Locked region indicator")
style_colorbar(cbar)

plot7_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_07_thresholded_fixed_point_locking_map.png",
)
save_show_close(fig, plot7_path)

# -------------------------------------------------------
# PLOT 8: CROSS-SAMPLE PHASE RESULTANT LENGTH
# Added radiation-specific quality diagnostic.
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 5),
    facecolor=BG,
)
style_ax(ax)

ax.plot(
    np.arange(len(phase_resultant_sub)),
    phase_resultant_sub,
    color=ACCENT,
    linewidth=1.2,
)

ax.axhline(
    resultant_mean,
    color=WHITE,
    linestyle="--",
    linewidth=1.0,
    alpha=0.75,
    label=f"Mean resultant = {resultant_mean:.3f}",
)

ax.set_ylim(0.0, 1.0)
ax.set_title(
    "Cross-Sample Phase Resultant Length Along Log-Energy"
)
ax.set_xlabel("Uniform log-energy subsample index")
ax.set_ylabel("Circular resultant length")

leg = ax.legend(
    loc="best",
    frameon=True,
)
for text in leg.get_texts():
    text.set_color(ACCENT)

plot8_path = os.path.join(
    plots_dir,
    f"{CELL13_RESULT_STEM}_plots_08_phase_resultant_length.png",
)
save_show_close(fig, plot8_path)

# -------------------------------------------------------
# APPEND PLOTTING DIAGNOSTICS TO SUMMARY
# -------------------------------------------------------
with open(summary_txt, "a") as f:
    f.write("\n\nPlotting diagnostics\n")
    f.write("====================\n")
    f.write(
        "Interpretation: fixed-point locking probability from the "
        "radiation-informed circle map.\n"
    )
    f.write(
        "This is not a full rotation-number Arnold tongue calculation.\n\n"
    )
    f.write(f"locked min  = {locked_min:.10f}\n")
    f.write(f"locked max  = {locked_max:.10f}\n")
    f.write(f"locked mean = {locked_mean:.10f}\n")
    f.write(f"phase min   = {phase_min:.10f} cycles\n")
    f.write(f"phase max   = {phase_max:.10f} cycles\n")
    f.write(f"phase mean  = {phase_mean:.10f} cycles\n")
    f.write(f"phase std   = {phase_std:.10f} cycles\n")
    f.write(
        f"phase resultant min    = {resultant_min:.10f}\n"
    )
    f.write(
        f"phase resultant max    = {resultant_max:.10f}\n"
    )
    f.write(
        f"phase resultant mean   = {resultant_mean:.10f}\n"
    )
    f.write(
        f"phase resultant median = {resultant_median:.10f}\n"
    )
    f.write(
        f"lock threshold for binary map = {lock_threshold:.4f}\n"
    )

print(f"\nSaved NPZ: {npz_path}")
print(f"Saved grid CSV: {grid_csv}")
print(f"Saved radiation phase CSV: {phase_csv}")
print(f"Saved summary TXT: {summary_txt}")
print(f"Saved plots directly to: {PHASE2_RESULTS_DIR}")


# Arnold Tongues Rotations

<div style="font-size:14px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:20px;border-radius:8px;margin:10px;display:flex;flex-wrap:nowrap;justify-content:space-between;line-height:1.55;">

<div style="flex:1;margin-right:10px;">
<h2 style="color:#FF4D4D;">Introduction</h2>
<p>
This block extends the preceding fixed-point calculation to the more general
<b>rotation number</b> of the lifted circle map.
The actual processed <b>N003 / ISIS RB2000164 transmission curves</b> provide
sample-specific phase initial conditions, while the circle map supplies the reduced nonlinear dynamics.
</p>

<h2 style="color:#FF4D4D;">Circle Map</h2>
<p>
The lifted map is
<b>Θ<sub>n+1</sub> = Θ<sub>n</sub> + Ω − [K/(2π)] sin(2πΘ<sub>n</sub>)</b>.
Unlike the wrapped fixed-point calculation, the phase is not reduced modulo one while the map is iterated.
This makes it possible to measure the accumulated average rotation.
</p>

<h2 style="color:#FF4D4D;">Radiation Initial Conditions</h2>
<p>
Each measured transmission response is converted to a shape-sensitive attenuation feature:
<b>T(E) → −ln T(E) → smoothing → d/dln(E) → detrending → standardization</b>.
The Hilbert transform then gives an analytic-signal phase along log-energy.
For each measured sample, the circular mean of that phase provides the circle-map initial condition,
with one initial phase supplied by each radiation curve.
</p>
</div>

<div style="flex:1;margin-left:10px;">
<h2 style="color:#FF4D4D;">Average Rotation Number</h2>
<p>
After a transient, the lifted phase is advanced for a finite measurement window.
The rotation number is estimated from the mean phase advance per iterate.
Plateaus near rational values indicate the familiar mode-locking structure of the circle map.
</p>

<h2 style="color:#FF4D4D;">Radiation Comparison</h2>
<p>
The full <b>Ω × K</b> rotation-number surface is calculated separately for each measured radiation sample.
The code also calculates the mean surface across samples, cross-sample variability,
and proximity to low-order rational winding ratios.
</p>

<h2 style="color:#FF4D4D;">Interpretation</h2>
<p>
The rotation number belongs to the <b>reduced circle-map model</b>, not directly to neutron transport.
Differences among N003 samples enter through their radiation-derived initial phase.
The resulting mode-locking structure should therefore be interpreted as a model response conditioned on the measured attenuation-shape phases.
</p>
</div>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Important distinction.</b> Cell 13 measured fixed-point locking. This cell measures the lifted-map rotation number and therefore captures general rational mode-locking plateaus rather than only fixed points.</p>
<p><b>Quality diagnostic.</b> The circular resultant length of the analytic phase is saved for every radiation sample. A very small resultant means that the sample's circular-mean initial phase is weakly defined.</p>
<p><b>Output convention.</b> All result tables and plots are written directly into <b>results/phase2/</b>. Plot filenames use the same Cell-17 result stem followed by <b>_plots_</b>.</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">
<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Circle-Map Rotation Number and Rational Mode Locking</h3>

<p>
The lifted circle map is iterated without reducing the state modulo one:
<b>Θ<sub>n+1</sub> = Θ<sub>n</sub> + Ω − [K/(2π)] sin(2πΘ<sub>n</sub>)</b>.
The finite-window rotation-number estimate is the accumulated phase advance divided by the number of measured iterates.
</p>

<p>
A plateau of the rotation number near a rational value <b>p/q</b> is the standard signature of rational mode locking.
The model parameter region supporting such a plateau is an Arnold tongue.
This cell therefore generalizes the fixed-point criterion from the previous block.
</p>

<p>
The numerical rotation estimate is also reported modulo one for the auxiliary modulo-one views.
That convention is retained for mathematical traceability.
The unreduced mean increment is also saved as an additional diagnostic so that wrap-boundary behavior can be inspected rather than hidden.
</p>

<p><b>Radiation role.</b>
The N003 measurements determine only the sample-specific initial phases.
The circle-map parameters <b>Ω</b> and <b>K</b> remain effective model parameters and are not measured neutron frequencies or material constants.
</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Rational-locking diagnostic.</b> In addition to the original rotation surfaces, the code compares the mean rotation surface with all reduced fractions <b>p/q</b> having denominator <b>q ≤ 8</b>. This does not change the circle-map mathematics; it simply makes low-order plateaus easier to identify quantitatively.</p>
<p><b>Interpretation guardrail.</b> A rational plateau is a property of the reduced map. It does not establish periodic neutron transport or temporal synchronization inside shielding material.</p>
</div>

In [ ]:
import os
from pathlib import Path
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.signal import hilbert, savgol_filter, detrend

# -------------------------------------------------------
# SPEED MODE
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError(
        "FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'."
    )

_SPEED_COUNT_FACTOR = {
    "full": 1.0,
    "balanced": 0.60,
    "fast": 0.35,
    "ultra": 0.18,
}[FAST_OPTION]

def _speed_count(original, minimum=1):
    return int(original) if FAST_OPTION == "full" else max(int(minimum), int(round(float(original) * _SPEED_COUNT_FACTOR)))

# -------------------------------------------------------
# CONFIG — project repository and flat result directory
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next(
    (p for p in N003_SOURCE_CANDIDATES if p.exists()),
    None,
)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL17_RESULT_STEM = "cell_17_circle_map_rotation_numbers_results"

out_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

# Circle-map grid/range controlled by the speed convention.
n_omega = _speed_count(300, 70)
n_K = _speed_count(300, 70)
omegas = np.linspace(0.0, 1.0, n_omega)
K_values = np.linspace(0.0, 2.0 * np.pi, n_K)

iterations = _speed_count(1000, 200)
transient = _speed_count(100, 40)

MAX_COORD_POINTS = 5000
MIN_COORD_POINTS = 512
MAX_RATIONAL_DENOMINATOR = 8
RATIONAL_TOL = 0.01

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

RED_ROTATION_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "rotation_black_red",
    [BLACK, RED_DIM, RED, RED_SOFT],
    N=256,
)

RED_DIVERGING_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "rotation_red_black_red",
    [RED_DIM, BLACK, RED_SOFT],
    N=256,
)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

def ensure_time_channel_layout(x, n_channels_expected):
    x = np.asarray(x, dtype=float)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D ordered-response array, got shape {x.shape}")

    if x.shape[1] == n_channels_expected:
        return x
    if x.shape[0] == n_channels_expected:
        return x.T

    raise ValueError(f"Unexpected ordered-response shape: {x.shape}")

def circular_mean_phase(phases_rad, axis=0):
    z = np.mean(np.exp(1j * phases_rad), axis=axis)
    return np.angle(z)

def to_cycle_phase(rad_phase):
    return np.mod(rad_phase / (2.0 * np.pi), 1.0)

def circle_map_lift(theta, omega, K):
    """
    Lifted circle map.
    Do NOT reduce modulo 1 here; rotation number needs the lifted increment.
    """
    return theta + omega - (K / (2.0 * np.pi)) * np.sin(2.0 * np.pi * theta)

def average_rotation_number_grid(omegas, K, theta0, iterations=1000, transient=100):
    """
    Vectorized over all omega values for one fixed K and one channel.
    Uses the lifted map and computes:
        rho = mean increment per iterate mod 1
            = ((theta_T - theta_0) / T) mod 1
    """
    omegas = np.asarray(omegas, dtype=float)
    theta = np.full_like(omegas, fill_value=float(theta0), dtype=float)

    # Transient
    for _ in range(transient):
        theta = circle_map_lift(theta, omegas, K)

    theta_start = theta.copy()

    # Measurement window
    for _ in range(iterations):
        theta = circle_map_lift(theta, omegas, K)

    rho = ((theta - theta_start) / max(iterations, 1)) % 1.0
    return rho

# -------------------------------------------------------
# ADDITIONAL DIAGNOSTIC: unreduced mean rotation increment
# -------------------------------------------------------
def average_rotation_number_grid_unwrapped(
    omegas,
    K,
    theta0,
    iterations=1000,
    transient=100,
):
    """
    Lifted-map calculation retaining the
    unreduced mean increment instead of applying modulo one.
    """
    omegas = np.asarray(omegas, dtype=float)
    theta = np.full_like(
        omegas,
        fill_value=float(theta0),
        dtype=float,
    )

    for _ in range(transient):
        theta = circle_map_lift(
            theta,
            omegas,
            K,
        )

    theta_start = theta.copy()

    for _ in range(iterations):
        theta = circle_map_lift(
            theta,
            omegas,
            K,
        )

    return (
        theta - theta_start
    ) / max(iterations, 1)


# -------------------------------------------------------
# RADIATION DATA HELPERS
# -------------------------------------------------------
def _find_radiation_column(
    df,
    exact=(),
    contains=(),
    exclude=(),
):
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    matches = []

    for c in df.columns:
        lc = str(c).lower()

        if (
            any(
                token.lower() in lc
                for token in contains
            )
            and not any(
                token.lower() in lc
                for token in exclude
            )
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda c: (
            len(str(c)),
            str(c),
        ),
    )[0]


def load_rb2000164_transmission_matrix(source_path):
    source_path = Path(source_path)

    if (
        source_path.name.endswith(".csv.gz")
        or source_path.suffix.lower() == ".csv"
    ):
        df = pd.read_csv(
            source_path,
        )

    elif source_path.suffix.lower() == ".parquet":
        try:
            df = pd.read_parquet(
                source_path,
            )
        except ImportError as exc:
            csv_fallback = (
                source_path
                .with_suffix("")
                .with_suffix(".csv.gz")
            )

            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(
                    source_path,
                )
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback found."
                ) from exc

    else:
        raise ValueError(
            f"Unsupported RB2000164 input format: {source_path}"
        )

    sample_col = _find_radiation_column(
        df,
        exact=(
            "sample_id",
            "sample",
            "sample_name",
        ),
        contains=("sample",),
        exclude=("uncert",),
    )

    energy_col = _find_radiation_column(
        df,
        exact=(
            "energy_eV",
            "energy_ev",
            "neutron_energy_eV",
            "energy_in_eV",
        ),
        contains=("energy",),
        exclude=(
            "uncert",
            "lower",
            "upper",
        ),
    )

    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=(
            "uncert",
            "sigma",
            "error",
        ),
    )

    if sample_col is None:
        raise KeyError(
            "Could not identify N003 sample column."
        )

    if energy_col is None:
        raise KeyError(
            "Could not identify N003 neutron-energy column."
        )

    if transmission_col is None:
        raise KeyError(
            "Could not identify N003 transmission column."
        )

    curves = {}

    for sample_id, part in df.groupby(
        sample_col,
        sort=True,
        dropna=False,
    ):
        q = part[
            [
                energy_col,
                transmission_col,
            ]
        ].copy()

        q[energy_col] = pd.to_numeric(
            q[energy_col],
            errors="coerce",
        )

        q[transmission_col] = pd.to_numeric(
            q[transmission_col],
            errors="coerce",
        )

        q = (
            q.replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        q = q[
            (q[energy_col] > 0)
            & (q[transmission_col] > 0)
        ]

        q = (
            q.groupby(
                energy_col,
                as_index=False,
                sort=True,
            )[transmission_col]
            .mean()
            .sort_values(
                energy_col,
            )
        )

        if len(q) < MIN_COORD_POINTS:
            print(
                f"Skipping sample {sample_id}: "
                f"only {len(q)} usable points.",
                flush=True,
            )
            continue

        curves[str(sample_id)] = (
            np.log(
                q[energy_col]
                .to_numpy(
                    dtype=float,
                )
            ),
            q[transmission_col]
            .to_numpy(
                dtype=float,
            ),
        )

    if len(curves) < 3:
        raise RuntimeError(
            "Need at least three sufficiently sampled N003 transmission curves."
        )

    common_lo = max(
        v[0].min()
        for v in curves.values()
    )

    common_hi = min(
        v[0].max()
        for v in curves.values()
    )

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval exists across retained N003 samples."
        )

    n_grid = min(
        MAX_COORD_POINTS,
        min(
            len(v[0])
            for v in curves.values()
        ),
    )

    if n_grid < MIN_COORD_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(
        common_lo,
        common_hi,
        n_grid,
    )

    labels = []
    columns = []

    for sample_id in sorted(curves):
        logE, transmission = curves[
            sample_id
        ]

        interp_T = np.interp(
            logE_grid,
            logE,
            transmission,
        )

        labels.append(
            sample_id,
        )
        columns.append(
            interp_T,
        )

    matrix = np.column_stack(
        columns,
    )

    s_grid = (
        logE_grid
        - logE_grid[0]
    )

    ds = float(
        s_grid[1]
        - s_grid[0]
    )

    return (
        matrix,
        labels,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        source_path,
    )


def radiation_shape_features(
    transmission_matrix,
    ds,
):
    """
    Create the same phase-bearing radiation feature used in Cell 13.
    """
    T = np.asarray(
        transmission_matrix,
        dtype=float,
    )

    if T.ndim != 2:
        raise ValueError(
            "transmission_matrix must be 2D."
        )

    n = T.shape[0]

    window = max(
        11,
        int(
            round(
                0.03 * n
            )
        ),
    )

    if window % 2 == 0:
        window += 1

    if window >= n:
        window = (
            n - 1
            if (n - 1) % 2 == 1
            else n - 2
        )

    window = max(
        window,
        5,
    )

    features = np.empty_like(
        T,
        dtype=float,
    )

    for j in range(
        T.shape[1]
    ):
        transmission = np.clip(
            T[:, j],
            1e-12,
            None,
        )

        optical_depth = -np.log(
            transmission,
        )

        smoothed = savgol_filter(
            optical_depth,
            window_length=window,
            polyorder=3,
            mode="interp",
        )

        slope = np.gradient(
            smoothed,
            ds,
        )

        shape = detrend(
            slope,
            type="linear",
        )

        shape = (
            shape
            - np.mean(shape)
        )

        scale = np.std(
            shape,
        )

        if (
            not np.isfinite(scale)
            or scale <= 0
        ):
            raise RuntimeError(
                f"Degenerate radiation phase feature "
                f"for sample column {j}."
            )

        features[:, j] = (
            shape / scale
        )

    return (
        features,
        window,
    )


def circular_phase_resultant(
    phases_rad,
    axis=0,
):
    return np.abs(
        np.mean(
            np.exp(
                1j * phases_rad
            ),
            axis=axis,
        )
    )


def low_order_rational_catalog(
    max_denominator=8,
    rho_min=0.0,
    rho_max=1.0,
):
    values = {}

    lo = float(min(rho_min, rho_max))
    hi = float(max(rho_min, rho_max))

    for q in range(
        1,
        max_denominator + 1,
    ):
        p_lo = int(np.floor(lo * q)) - 1
        p_hi = int(np.ceil(hi * q)) + 1

        for p in range(
            p_lo,
            p_hi + 1,
        ):
            frac = Fraction(
                p,
                q,
            )

            value = float(
                frac,
            )

            if (
                lo - 1.0 / q <= value <= hi + 1.0 / q
            ):
                values[
                    (
                        frac.numerator,
                        frac.denominator,
                    )
                ] = value

    rows = [
        (
            p,
            q,
            value,
        )
        for (
            p,
            q
        ), value
        in values.items()
    ]

    return sorted(
        rows,
        key=lambda item: (
            item[2],
            item[1],
            item[0],
        ),
    )


def nearest_low_order_rational(
    rho_array,
    max_denominator=8,
):
    rho = np.asarray(
        rho_array,
        dtype=float,
    )

    finite = rho[np.isfinite(rho)]
    if finite.size == 0:
        raise ValueError("No finite rotation numbers supplied")

    catalog = low_order_rational_catalog(
        max_denominator=max_denominator,
        rho_min=float(np.min(finite)),
        rho_max=float(np.max(finite)),
    )

    distances = np.stack(
        [
            np.abs(
                rho - value
            )
            for _, _, value
            in catalog
        ],
        axis=0,
    )

    idx = np.argmin(
        distances,
        axis=0,
    )

    min_distance = np.take_along_axis(
        distances,
        idx[None, ...],
        axis=0,
    )[0]

    rational_value = np.empty_like(
        rho,
        dtype=float,
    )

    numerator = np.empty_like(
        rho,
        dtype=int,
    )

    denominator = np.empty_like(
        rho,
        dtype=int,
    )

    for cat_idx, (
        p,
        q,
        value,
    ) in enumerate(catalog):
        mask = (
            idx == cat_idx
        )

        rational_value[mask] = value
        numerator[mask] = p
        denominator[mask] = q

    return (
        rational_value,
        numerator,
        denominator,
        min_distance,
        catalog,
    )


# -------------------------------------------------------
# LOAD RADIATION DATA AND CONSTRUCT SAMPLE-SPECIFIC PHASES
# -------------------------------------------------------
(
    transmission_matrix,
    radiation_samples,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    actual_source_path,
) = load_rb2000164_transmission_matrix(
    N003_PATH,
)

n_coordinate_samples, n_samples = (
    transmission_matrix.shape
)

shape_matrix, smoothing_window = (
    radiation_shape_features(
        transmission_matrix,
        coordinate_step,
    )
)

shape_matrix = (
    ensure_time_channel_layout(
        shape_matrix,
        n_samples,
    )
)

analytic_signal = hilbert(
    shape_matrix,
    axis=0,
)

inst_phase_rad = np.angle(
    analytic_signal,
)

# One initial phase per measured radiation sample:
# one initial phase per radiation curve.
theta0_rad = circular_mean_phase(
    inst_phase_rad,
    axis=0,
)

theta0_cycles = to_cycle_phase(
    theta0_rad,
)

phase_resultant = circular_phase_resultant(
    inst_phase_rad,
    axis=0,
)

theta0_df = pd.DataFrame({
    "sample_id": radiation_samples,
    "theta0_rad": theta0_rad,
    "theta0_cycles": theta0_cycles,
    "phase_resultant_length": phase_resultant,
})

print(
    f"Loaded radiation phase matrix: "
    f"{n_coordinate_samples} log-energy points "
    f"x {n_samples} measured samples"
)

print(
    f"Processed source: {public_project_path(actual_source_path)}"
)

print(
    f"Optical-depth-gradient smoothing window: "
    f"{smoothing_window}"
)

print(
    "\nInitial circle-map phases per measured radiation sample:"
)

print(
    theta0_df.to_string(
        index=False,
    )
)

# -------------------------------------------------------
# COMPUTE ROTATION NUMBERS
# -------------------------------------------------------
rotation_numbers_all = np.zeros(
    (
        n_samples,
        len(K_values),
        len(omegas),
    ),
    dtype=float,
)

rotation_numbers_unwrapped_all = np.zeros_like(
    rotation_numbers_all,
)

summary_rows = []

for sample_idx, sample_id in enumerate(
    radiation_samples
):
    theta0 = theta0_cycles[
        sample_idx
    ]

    print(
        f"\nProcessing sample "
        f"{sample_idx + 1}/{n_samples}: "
        f"{sample_id} | "
        f"theta0={theta0:.6f}"
    )

    sample_rho = np.zeros(
        (
            len(K_values),
            len(omegas),
        ),
        dtype=float,
    )

    sample_rho_unwrapped = np.zeros_like(
        sample_rho,
    )

    for i, K in enumerate(
        K_values
    ):
        sample_rho[i, :] = (
            average_rotation_number_grid(
                omegas=omegas,
                K=K,
                theta0=theta0,
                iterations=iterations,
                transient=transient,
            )
        )

        sample_rho_unwrapped[i, :] = (
            average_rotation_number_grid_unwrapped(
                omegas=omegas,
                K=K,
                theta0=theta0,
                iterations=iterations,
                transient=transient,
            )
        )

        if (
            (i + 1) % 25 == 0
            or i == 0
            or i == len(K_values) - 1
        ):
            print(
                f"  K row "
                f"{i + 1}/{len(K_values)}",
                flush=True,
            )

    rotation_numbers_all[
        sample_idx
    ] = sample_rho

    rotation_numbers_unwrapped_all[
        sample_idx
    ] = sample_rho_unwrapped

    summary_rows.append({
        "sample_id": sample_id,
        "theta0_cycles": float(
            theta0,
        ),
        "phase_resultant_length": float(
            phase_resultant[
                sample_idx
            ]
        ),
        "rotation_mean_mod1": float(
            np.mean(
                sample_rho
            )
        ),
        "rotation_std_mod1": float(
            np.std(
                sample_rho
            )
        ),
        "rotation_min_mod1": float(
            np.min(
                sample_rho
            )
        ),
        "rotation_max_mod1": float(
            np.max(
                sample_rho
            )
        ),
        "rotation_mean_unwrapped": float(
            np.mean(
                sample_rho_unwrapped
            )
        ),
        "rotation_std_unwrapped": float(
            np.std(
                sample_rho_unwrapped
            )
        ),
    })

summary_df = pd.DataFrame(
    summary_rows,
)

# -------------------------------------------------------
# CROSS-SAMPLE SURFACES
# -------------------------------------------------------
rotation_mean_surface = np.mean(
    rotation_numbers_all,
    axis=0,
)

rotation_std_surface = np.std(
    rotation_numbers_all,
    axis=0,
)

rotation_unwrapped_mean_surface = np.mean(
    rotation_numbers_unwrapped_all,
    axis=0,
)

(
    nearest_rational_value,
    nearest_rational_p,
    nearest_rational_q,
    rational_distance,
    rational_catalog,
) = nearest_low_order_rational(
    rotation_unwrapped_mean_surface,
    max_denominator=MAX_RATIONAL_DENOMINATOR,
)

low_order_lock_mask = (
    rational_distance
    <= RATIONAL_TOL
)

# -------------------------------------------------------
# SAVE RESULTS
# -------------------------------------------------------
npz_path = os.path.join(
    out_dir,
    f"{CELL17_RESULT_STEM}.npz",
)

np.savez_compressed(
    npz_path,
    rotation_numbers_all=rotation_numbers_all,
    rotation_numbers_unwrapped_all=rotation_numbers_unwrapped_all,
    rotation_mean_surface=rotation_mean_surface,
    rotation_std_surface=rotation_std_surface,
    rotation_unwrapped_mean_surface=rotation_unwrapped_mean_surface,
    nearest_rational_value=nearest_rational_value,
    nearest_rational_p=nearest_rational_p,
    nearest_rational_q=nearest_rational_q,
    rational_distance=rational_distance,
    low_order_lock_mask=low_order_lock_mask,
    omegas=omegas,
    K_values=K_values,
    radiation_samples=np.array(
        radiation_samples,
        dtype=object,
    ),
    theta0_rad=theta0_rad,
    theta0_cycles=theta0_cycles,
    phase_resultant_length=phase_resultant,
    s_grid=s_grid,
    coordinate_step=coordinate_step,
    smoothing_window=smoothing_window,
    iterations=iterations,
    transient=transient,
    source_file=str(
        actual_source_path,
    ),
    speed_mode=FAST_OPTION,
)

summary_csv = os.path.join(
    out_dir,
    f"{CELL17_RESULT_STEM}.csv",
)

theta0_csv = os.path.join(
    out_dir,
    f"{CELL17_RESULT_STEM}_initial_conditions.csv",
)

summary_df.to_csv(
    summary_csv,
    index=False,
)

theta0_df.to_csv(
    theta0_csv,
    index=False,
)

# Full mean-grid table.
grid_rows = []

for i, K in enumerate(
    K_values
):
    for j, omega in enumerate(
        omegas
    ):
        grid_rows.append({
            "Omega": float(
                omega
            ),
            "K": float(
                K
            ),
            "rotation_mean_mod1": float(
                rotation_mean_surface[
                    i,
                    j,
                ]
            ),
            "rotation_std_across_samples": float(
                rotation_std_surface[
                    i,
                    j,
                ]
            ),
            "rotation_mean_unwrapped": float(
                rotation_unwrapped_mean_surface[
                    i,
                    j,
                ]
            ),
            "nearest_low_order_rational": float(
                nearest_rational_value[
                    i,
                    j,
                ]
            ),
            "nearest_rational_p": int(
                nearest_rational_p[
                    i,
                    j,
                ]
            ),
            "nearest_rational_q": int(
                nearest_rational_q[
                    i,
                    j,
                ]
            ),
            "rational_distance": float(
                rational_distance[
                    i,
                    j,
                ]
            ),
            "low_order_lock_within_tolerance": bool(
                low_order_lock_mask[
                    i,
                    j,
                ]
            ),
        })

grid_csv = os.path.join(
    out_dir,
    f"{CELL17_RESULT_STEM}_grid.csv",
)

pd.DataFrame(
    grid_rows,
).to_csv(
    grid_csv,
    index=False,
)

summary_txt = os.path.join(
    out_dir,
    f"{CELL17_RESULT_STEM}.txt",
)

with open(
    summary_txt,
    "w",
) as f:
    f.write(
        "Circle-Map Rotation Numbers from Radiation-Derived Phase\n"
    )
    f.write(
        "=======================================================\n\n"
    )

    f.write(
        f"Source: {public_project_path(actual_source_path)}\n"
    )

    f.write(
        "Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n"
    )

    f.write(
        "Observable: transmission T(E)\n"
    )

    f.write(
        f"Measured samples: {n_samples}\n"
    )

    f.write(
        f"Coordinate points: {n_coordinate_samples}\n"
    )

    f.write(
        f"Coordinate step: {coordinate_step}\n"
    )

    f.write(
        f"Smoothing window: {smoothing_window}\n"
    )

    f.write(
        f"Omega range: "
        f"[{omegas.min()}, {omegas.max()}] "
        f"with {len(omegas)} points\n"
    )

    f.write(
        f"K range: "
        f"[{K_values.min()}, {K_values.max()}] "
        f"with {len(K_values)} points\n"
    )

    f.write(
        f"Iterations: {iterations}\n"
    )

    f.write(
        f"Transient: {transient}\n"
    )

    f.write(
        f"Rational denominator limit: "
        f"{MAX_RATIONAL_DENOMINATOR}\n"
    )

    f.write(
        f"Rational-lock tolerance: "
        f"{RATIONAL_TOL}\n\n"
    )

    f.write(
        "Per-sample summary:\n"
    )

    for _, row in summary_df.iterrows():
        f.write(
            f"  {row['sample_id']}: "
            f"theta0_cycles="
            f"{row['theta0_cycles']:.6f}, "
            f"phase_resultant="
            f"{row['phase_resultant_length']:.6f}, "
            f"mean_mod1="
            f"{row['rotation_mean_mod1']:.6f}, "
            f"std_mod1="
            f"{row['rotation_std_mod1']:.6f}, "
            f"mean_unwrapped="
            f"{row['rotation_mean_unwrapped']:.6f}\n"
        )

    f.write(
        "\nInterpretation:\n"
        "  Rotation-number plateaus are properties of the reduced circle map.\n"
        "  Radiation data enter through sample-specific analytic-signal initial phases.\n"
        "  They are not direct neutron-transport frequencies.\n"
    )

# -------------------------------------------------------
# PLOT 1: CONTACT SHEET OF ALL MEASURED SAMPLES
# -------------------------------------------------------
rows = 3
cols = 3

fig, axs = plt.subplots(
    rows,
    cols,
    figsize=(15, 13),
    constrained_layout=True,
    facecolor=BG,
)

for ax in axs.flat:
    style_ax(
        ax,
    )

fig.suptitle(
    "Modulo-1 Rotation Surfaces — Diagnostic View",
    color=ACCENT,
    fontsize=14,
)

for ax, sample_id, rot in zip(
    axs.flat,
    radiation_samples,
    rotation_numbers_all,
):
    im = ax.imshow(
        rot,
        extent=(
            omegas.min(),
            omegas.max(),
            K_values.min(),
            K_values.max(),
        ),
        aspect="auto",
        origin="lower",
        cmap=RED_ROTATION_CMAP,
        vmin=0.0,
        vmax=1.0,
        interpolation="nearest",
    )

    ax.set_title(
        str(sample_id),
        fontsize=10,
        color=ACCENT,
    )

    ax.set_xlabel(
        "Ω",
    )

    ax.set_ylabel(
        "K",
    )

cbar = fig.colorbar(
    im,
    ax=axs.ravel().tolist(),
    shrink=0.82,
)

cbar.set_label(
    "Rotation number mod 1",
    color=ACCENT,
)

cbar.outline.set_edgecolor(
    ACCENT,
)

cbar.ax.tick_params(
    colors=ACCENT,
)

plot1_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_01_all_samples_rotation_contact_sheet.png",
)

plt.savefig(
    plot1_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 2: MEAN ROTATION NUMBER PER SAMPLE
# -------------------------------------------------------
plot_df = summary_df.sort_values(
    "rotation_mean_mod1",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df["rotation_mean_mod1"].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_title(
    "Mean Circle-Map Rotation Number by N003 Sample — Modulo-1 Diagnostic"
)

ax.set_xlabel(
    "Mean rotation number mod 1"
)

ax.set_ylabel(
    "Measured sample"
)

plot2_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_02_mean_rotation_number_by_sample.png",
)

plt.tight_layout()
plt.savefig(
    plot2_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 3: ROTATION NUMBER VARIABILITY PER SAMPLE
# -------------------------------------------------------
plot_df = summary_df.sort_values(
    "rotation_std_mod1",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df["rotation_std_mod1"].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_title(
    "Rotation-Number Variability by N003 Sample — Modulo-1 Diagnostic"
)

ax.set_xlabel(
    "Rotation-number standard deviation (mod 1)"
)

ax.set_ylabel(
    "Measured sample"
)

plot3_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_03_rotation_number_std_by_sample.png",
)

plt.tight_layout()
plt.savefig(
    plot3_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 4: INITIAL CONDITIONS BY SAMPLE
# -------------------------------------------------------
plot_df = theta0_df.sort_values(
    "theta0_cycles",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df["theta0_cycles"].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_title(
    "Circle-Map Initial Phase by N003 Sample"
)

ax.set_xlabel(
    "Initial phase (cycles)"
)

ax.set_ylabel(
    "Measured sample"
)

plot4_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_04_initial_phase_by_sample.png",
)

plt.tight_layout()
plt.savefig(
    plot4_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 5: MEAN ROTATION SURFACE ACROSS SAMPLES
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

im = ax.imshow(
    rotation_mean_surface,
    extent=(
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ),
    origin="lower",
    aspect="auto",
    cmap=RED_ROTATION_CMAP,
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
)

ax.set_title(
    "Mean Circle-Map Rotation Number Across N003 Samples — Modulo-1 Diagnostic"
)

ax.set_xlabel(
    "Circle-map frequency parameter Ω"
)

ax.set_ylabel(
    "Nonlinearity / coupling parameter K"
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Mean rotation number mod 1",
    color=ACCENT,
)

cbar.outline.set_edgecolor(
    ACCENT,
)

cbar.ax.tick_params(
    colors=ACCENT,
)

plot5_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_05_mean_rotation_surface.png",
)

plt.tight_layout()
plt.savefig(
    plot5_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 6: CROSS-SAMPLE ROTATION VARIABILITY
# -------------------------------------------------------
vmax_std = float(
    np.nanpercentile(
        rotation_std_surface,
        99.0,
    )
)

vmax_std = max(
    vmax_std,
    1e-12,
)

fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

im = ax.imshow(
    rotation_std_surface,
    extent=(
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ),
    origin="lower",
    aspect="auto",
    cmap=RED_ROTATION_CMAP,
    vmin=0.0,
    vmax=vmax_std,
    interpolation="nearest",
)

ax.set_title(
    "Cross-Sample Rotation-Number Variability"
)

ax.set_xlabel(
    "Circle-map frequency parameter Ω"
)

ax.set_ylabel(
    "Nonlinearity / coupling parameter K"
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Std across measured samples",
    color=ACCENT,
)

cbar.outline.set_edgecolor(
    ACCENT,
)

cbar.ax.tick_params(
    colors=ACCENT,
)

plot6_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_06_cross_sample_rotation_variability.png",
)

plt.tight_layout()
plt.savefig(
    plot6_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 7: DISTANCE TO NEAREST LOW-ORDER RATIONAL
# -------------------------------------------------------
rational_display_max = max(
    RATIONAL_TOL * 5.0,
    float(
        np.nanmax(
            rational_distance,
        )
    ),
)

fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

im = ax.imshow(
    rational_distance,
    extent=(
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ),
    origin="lower",
    aspect="auto",
    cmap=RED_ROTATION_CMAP.reversed(),
    vmin=0.0,
    vmax=rational_display_max,
    interpolation="nearest",
)

ax.contour(
    omegas,
    K_values,
    low_order_lock_mask.astype(float),
    levels=[0.5],
    colors=[WHITE],
    linewidths=0.9,
)

ax.set_title(
    "Distance to Nearest Low-Order Rational Rotation"
)

ax.set_xlabel(
    "Circle-map frequency parameter Ω"
)

ax.set_ylabel(
    "Nonlinearity / coupling parameter K"
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Absolute distance in rotation number",
    color=ACCENT,
)

cbar.outline.set_edgecolor(
    ACCENT,
)

cbar.ax.tick_params(
    colors=ACCENT,
)

plot7_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_07_distance_to_low_order_rational.png",
)

plt.tight_layout()
plt.savefig(
    plot7_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 8: LOW-ORDER RATIONAL LOCK MASK
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

im = ax.imshow(
    low_order_lock_mask.astype(float),
    extent=(
        omegas.min(),
        omegas.max(),
        K_values.min(),
        K_values.max(),
    ),
    origin="lower",
    aspect="auto",
    cmap=RED_ROTATION_CMAP,
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
)

ax.set_title(
    f"Low-Order Rational Rotation Regions "
    f"(q ≤ {MAX_RATIONAL_DENOMINATOR}, "
    f"|Δρ| ≤ {RATIONAL_TOL})"
)

ax.set_xlabel(
    "Circle-map frequency parameter Ω"
)

ax.set_ylabel(
    "Nonlinearity / coupling parameter K"
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Rational-lock indicator",
    color=ACCENT,
)

cbar.outline.set_edgecolor(
    ACCENT,
)

cbar.ax.tick_params(
    colors=ACCENT,
)

plot8_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_08_low_order_rational_lock_mask.png",
)

plt.tight_layout()
plt.savefig(
    plot8_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

# -------------------------------------------------------
# PLOT 9: PHASE RESULTANT LENGTH BY SAMPLE
# -------------------------------------------------------
plot_df = theta0_df.sort_values(
    "phase_resultant_length",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax,
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df["phase_resultant_length"].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_xlim(
    0.0,
    1.0,
)

ax.set_title(
    "Analytic-Phase Resultant Length by N003 Sample"
)

ax.set_xlabel(
    "Circular resultant length"
)

ax.set_ylabel(
    "Measured sample"
)

plot9_path = os.path.join(
    plots_dir,
    f"{CELL17_RESULT_STEM}_plots_09_phase_resultant_length_by_sample.png",
)

plt.tight_layout()
plt.savefig(
    plot9_path,
    dpi=190,
    bbox_inches="tight",
    facecolor=BG,
)

plt.show()
plt.close(
    fig,
)

print(
    f"\nSaved NPZ: {npz_path}"
)

print(
    f"Saved summary CSV: {summary_csv}"
)

print(
    f"Saved initial-condition CSV: {theta0_csv}"
)

print(
    f"Saved grid CSV: {grid_csv}"
)

print(
    f"Saved summary TXT: {summary_txt}"
)

print(
    f"Saved plots directly to: {PHASE2_RESULTS_DIR}"
)


<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">
<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Circle-Map Pushforward of Radiation-Derived Phase Distributions</h3>

<p>
Instead of reducing each measured radiation response to a single phase, this cell retains an
<b>ensemble of analytic-signal phases along the ordered log-energy coordinate</b>.
For every measured N003 transmission curve, that empirical phase distribution is repeatedly pushed through the circle map.
Histograms of the final phases approximate the model's finite-time pushforward measure.
</p>

<p>
The map is
<b>F(θ) = θ + Ω − [K/(2π)] sin(2πθ) mod 1</b>,
and the distribution after <b>n</b> iterations is the pushforward
<b>p<sub>n</sub> = (F<sup>n</sup>)<sub>#</sub> p<sub>0</sub></b>.
</p>

<p>
Circular concentration is summarized with the resultant length
<b>R = |Σ p<sub>k</sub> exp(i2πθ<sub>k</sub>)|</b>.
A perfectly uniform circular distribution has <b>R ≈ 0</b>; concentration around one phase drives <b>R</b> toward one.
The cell also records normalized histogram concentration based on <b>Σp²</b>.
</p>

<p><b>Radiation phase construction.</b>
Each processed transmission curve is mapped as
<b>T(E) → −ln T(E) → smoothing → d/dln(E) → detrending → standardization → Hilbert phase</b>.
This is the same radiation phase representation used in the preceding circle-map cells and avoids applying Hilbert phase directly to the strongly monotonic transmission trend.
</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Finite-time, not automatically invariant.</b>
The saved distribution is evaluated after a finite number of map iterations.
The primary result is described as a <b>finite-time pushforward distribution</b>.
An additional total-variation stabilization diagnostic compares the histogram after the nominal iteration count with the histogram after extra iterations. Small values support numerical stabilization; they do not by themselves prove existence or uniqueness of an invariant density.</p>

<p><b>Interpretation guardrail.</b>
The distributional locking belongs to the reduced circle-map experiment.
It does not imply that neutron transport literally evolves through circle-map time steps.
Radiation measurements enter through the empirical initial phase ensemble only.</p>

<p><b>Output convention.</b>
All result tables and plots are saved directly into <b>results/phase2/</b>.
Every plot uses the same Cell-19 result stem followed by <b>_plots_</b>.</p>
</div>

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import (
    butter,
    sosfiltfilt,
    hilbert,
    savgol_filter,
    detrend,
)

# -------------------------------------------------------
# SPEED MODE — shared four-level convention
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError(
        "FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'."
    )

_SPEED_COUNT_FACTOR = {
    "full": 1.0,
    "balanced": 0.60,
    "fast": 0.35,
    "ultra": 0.18,
}[FAST_OPTION]

def _speed_count(original, minimum=1):
    return int(original) if FAST_OPTION == "full" else max(int(minimum), int(round(float(original) * _SPEED_COUNT_FACTOR)))

# -------------------------------------------------------
# CONFIG — project repository and flat results directory
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next(
    (p for p in N003_SOURCE_CANDIDATES if p.exists()),
    None,
)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL19_RESULT_STEM = "cell_19_circle_map_phase_pushforward_results"

out_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

# -------------------------------------------------------
# CIRCLE-MAP PARAMETERS — preserve cell settings
# -------------------------------------------------------
Omega = 1.0 / 3.0
K_values = np.linspace(
    0.0,
    4.0 * np.pi,
    _speed_count(300, 80),
)

iterations = _speed_count(50, 25)

# Extra iterations used only for finite-time stabilization diagnostics.
stabilization_extra_iterations = _speed_count(20, 10)

# -------------------------------------------------------
# HISTOGRAM SETTINGS
# -------------------------------------------------------
n_bins = _speed_count(50, 28)
bin_edges = np.linspace(
    0.0,
    1.0,
    n_bins + 1,
)
bin_centers = 0.5 * (
    bin_edges[:-1]
    + bin_edges[1:]
)

max_samples_per_sample = _speed_count(
    5000,
    1200,
)

rng_seed = 42

# Common radiation-coordinate grid.
MAX_COORD_POINTS = 5000
MIN_COORD_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

RED_SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    "phase_pushforward_black_red",
    [BLACK, RED_DIM, RED, RED_SOFT],
    N=256,
)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=ACCENT)

    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)

    for spine in ax.spines.values():
        spine.set_color(ACCENT)

    ax.grid(True, alpha=0.20, color=ACCENT)

def style_colorbar(cbar):
    cbar.outline.set_edgecolor(ACCENT)
    cbar.ax.tick_params(color=ACCENT, labelcolor=ACCENT)

    for label in cbar.ax.get_yticklabels():
        label.set_color(ACCENT)

    cbar.ax.yaxis.label.set_color(ACCENT)

def save_show_close(fig, out_path, dpi=220):
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)

def ensure_time_channel_layout(x, n_channels_expected):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D ordered-response array, got shape {x.shape}")

    if x.shape[1] == n_channels_expected:
        return x

    if x.shape[0] == n_channels_expected:
        return x.T

    raise ValueError(f"Unexpected ordered-response shape: {x.shape}")

def bandpass_filter_signal(x, fs, band, order=4):
    x = np.asarray(x, dtype=float)

    low, high = band
    nyq = 0.5 * fs

    if low <= 0 and high >= nyq:
        return x

    if low <= 0:
        sos = butter(order, high / nyq, btype="lowpass", output="sos")
    elif high >= nyq:
        sos = butter(order, low / nyq, btype="highpass", output="sos")
    else:
        sos = butter(order, [low / nyq, high / nyq], btype="bandpass", output="sos")

    return sosfiltfilt(sos, x, axis=0)

def radians_to_cycles(rad_phase):
    return np.mod(rad_phase / (2.0 * np.pi), 1.0)

def uniform_subsample(x, max_n, rng=None):
    x = np.asarray(x)

    if len(x) <= max_n:
        return x

    if rng is None:
        idx = np.linspace(0, len(x) - 1, max_n).astype(int)
    else:
        idx = np.sort(rng.choice(len(x), size=max_n, replace=False))

    return x[idx]

def circle_map(theta, Omega, K):
    """
    Standard circle map on phase cycles [0, 1):

        theta_{n+1} = theta_n + Omega - (K / 2pi) sin(2pi theta_n) mod 1
    """
    theta_next = theta + Omega - (K / (2.0 * np.pi)) * np.sin(2.0 * np.pi * theta)
    return np.mod(theta_next, 1.0)

def circular_mean_cycles(values, weights=None):
    """
    Circular mean of values in cycle units [0, 1).
    Returns mean phase in cycles and resultant length R.
    """
    values = np.asarray(values, dtype=float)
    angles = 2.0 * np.pi * values

    if weights is None:
        weights = np.ones_like(values, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    weights = weights / (np.sum(weights) + eps)

    z = np.sum(weights * np.exp(1j * angles))
    mean_cycles = np.mod(np.angle(z) / (2.0 * np.pi), 1.0)
    resultant_length = np.abs(z)

    return mean_cycles, resultant_length

def normalized_concentration_from_prob(prob):
    """
    Concentration from a probability vector.

    sum(p^2) ranges from:
      1/n_bins for uniform
      1.0 for all mass in one bin

    This returns normalized concentration in [0, 1].
    """
    prob = np.asarray(prob, dtype=float)

    raw = np.sum(prob ** 2, axis=-1)
    uniform_val = 1.0 / prob.shape[-1]

    norm = (raw - uniform_val) / (1.0 - uniform_val + eps)
    return np.clip(norm, 0.0, 1.0)

# -------------------------------------------------------
# RADIATION DATA HELPERS
# -------------------------------------------------------
def _find_radiation_column(
    df,
    exact=(),
    contains=(),
    exclude=(),
):
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[
                name.lower()
            ]

    matches = []

    for c in df.columns:
        lc = str(c).lower()

        if (
            any(
                token.lower() in lc
                for token in contains
            )
            and not any(
                token.lower() in lc
                for token in exclude
            )
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda c: (
            len(str(c)),
            str(c),
        ),
    )[0]


def load_rb2000164_transmission_matrix(
    source_path,
):
    """
    Place every measured N003 transmission curve on one common
    uniformly sampled log-energy coordinate.
    """
    source_path = Path(
        source_path
    )

    if (
        source_path.name.endswith(
            ".csv.gz"
        )
        or source_path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            source_path
        )

    elif (
        source_path.suffix.lower()
        == ".parquet"
    ):
        try:
            df = pd.read_parquet(
                source_path
            )
        except ImportError as exc:
            csv_fallback = (
                source_path
                .with_suffix("")
                .with_suffix(".csv.gz")
            )

            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(
                    source_path
                )
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback found."
                ) from exc

    else:
        raise ValueError(
            f"Unsupported RB2000164 input format: {source_path}"
        )

    sample_col = _find_radiation_column(
        df,
        exact=(
            "sample_id",
            "sample",
            "sample_name",
        ),
        contains=("sample",),
        exclude=("uncert",),
    )

    energy_col = _find_radiation_column(
        df,
        exact=(
            "energy_eV",
            "energy_ev",
            "neutron_energy_eV",
            "energy_in_eV",
        ),
        contains=("energy",),
        exclude=(
            "uncert",
            "lower",
            "upper",
        ),
    )

    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=(
            "uncert",
            "sigma",
            "error",
        ),
    )

    if sample_col is None:
        raise KeyError(
            "Could not identify N003 sample column."
        )

    if energy_col is None:
        raise KeyError(
            "Could not identify N003 neutron-energy column."
        )

    if transmission_col is None:
        raise KeyError(
            "Could not identify N003 transmission column."
        )

    curves = {}

    for sample_id, part in df.groupby(
        sample_col,
        sort=True,
        dropna=False,
    ):
        q = part[
            [
                energy_col,
                transmission_col,
            ]
        ].copy()

        q[energy_col] = pd.to_numeric(
            q[energy_col],
            errors="coerce",
        )

        q[transmission_col] = (
            pd.to_numeric(
                q[transmission_col],
                errors="coerce",
            )
        )

        q = (
            q.replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        q = q[
            (q[energy_col] > 0)
            & (q[transmission_col] > 0)
        ]

        q = (
            q.groupby(
                energy_col,
                as_index=False,
                sort=True,
            )[transmission_col]
            .mean()
            .sort_values(
                energy_col
            )
        )

        if len(q) < MIN_COORD_POINTS:
            print(
                f"Skipping sample {sample_id}: "
                f"only {len(q)} usable points.",
                flush=True,
            )
            continue

        curves[
            str(sample_id)
        ] = (
            np.log(
                q[energy_col]
                .to_numpy(
                    dtype=float
                )
            ),
            q[transmission_col]
            .to_numpy(
                dtype=float
            ),
        )

    if len(curves) < 3:
        raise RuntimeError(
            "Need at least three sufficiently sampled N003 transmission curves."
        )

    common_lo = max(
        v[0].min()
        for v in curves.values()
    )

    common_hi = min(
        v[0].max()
        for v in curves.values()
    )

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval exists across retained N003 samples."
        )

    n_grid = min(
        MAX_COORD_POINTS,
        min(
            len(v[0])
            for v in curves.values()
        ),
    )

    if n_grid < MIN_COORD_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(
        common_lo,
        common_hi,
        n_grid,
    )

    labels = []
    columns = []

    for sample_id in sorted(curves):
        logE, transmission = (
            curves[sample_id]
        )

        interp_T = np.interp(
            logE_grid,
            logE,
            transmission,
        )

        labels.append(
            sample_id
        )

        columns.append(
            interp_T
        )

    matrix = np.column_stack(
        columns
    )

    s_grid = (
        logE_grid
        - logE_grid[0]
    )

    ds = float(
        s_grid[1]
        - s_grid[0]
    )

    return (
        matrix,
        labels,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        source_path,
    )


def radiation_shape_features(
    transmission_matrix,
    ds,
):
    """
    Radiation phase-bearing representation:
      T(E)
        -> optical depth -ln(T)
        -> Savitzky-Golay smoothing
        -> derivative with respect to log-energy
        -> linear detrending
        -> standardization
    """
    T = np.asarray(
        transmission_matrix,
        dtype=float,
    )

    if T.ndim != 2:
        raise ValueError(
            "transmission_matrix must be 2D."
        )

    n = T.shape[0]

    window = max(
        11,
        int(
            round(
                0.03 * n
            )
        ),
    )

    if window % 2 == 0:
        window += 1

    if window >= n:
        window = (
            n - 1
            if (n - 1) % 2 == 1
            else n - 2
        )

    window = max(
        window,
        5,
    )

    features = np.empty_like(
        T,
        dtype=float,
    )

    for j in range(
        T.shape[1]
    ):
        transmission = np.clip(
            T[:, j],
            1e-12,
            None,
        )

        optical_depth = -np.log(
            transmission
        )

        smoothed = savgol_filter(
            optical_depth,
            window_length=window,
            polyorder=3,
            mode="interp",
        )

        slope = np.gradient(
            smoothed,
            ds,
        )

        shape = detrend(
            slope,
            type="linear",
        )

        shape = (
            shape
            - np.mean(shape)
        )

        scale = np.std(
            shape
        )

        if (
            not np.isfinite(scale)
            or scale <= 0
        ):
            raise RuntimeError(
                "Degenerate radiation phase feature "
                f"for sample column {j}."
            )

        features[:, j] = (
            shape / scale
        )

    return (
        features,
        window,
    )


def histogram_total_variation(
    p,
    q,
):
    """
    Total-variation distance between two discrete probability vectors.
    Range: [0, 1].
    """
    p = np.asarray(
        p,
        dtype=float,
    )

    q = np.asarray(
        q,
        dtype=float,
    )

    return 0.5 * np.sum(
        np.abs(
            p - q
        ),
        axis=-1,
    )


# -------------------------------------------------------
# LOAD RADIATION DATA AND EXTRACT PHASE ENSEMBLES
# -------------------------------------------------------
(
    transmission_matrix,
    radiation_samples,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    actual_source_path,
) = load_rb2000164_transmission_matrix(
    N003_PATH
)

n_coordinate_samples, n_samples = (
    transmission_matrix.shape
)

shape_matrix, smoothing_window = (
    radiation_shape_features(
        transmission_matrix,
        coordinate_step,
    )
)

shape_matrix = (
    ensure_time_channel_layout(
        shape_matrix,
        n_samples,
    )
)

analytic_signal = hilbert(
    shape_matrix,
    axis=0,
)

inst_phase_rad = np.angle(
    analytic_signal
)

inst_phase_cycles = radians_to_cycles(
    inst_phase_rad
)

rng = np.random.default_rng(
    rng_seed
)

print(
    f"Loaded radiation phase matrix: "
    f"{n_coordinate_samples} log-energy points "
    f"x {n_samples} measured samples"
)

print(
    f"Processed source: {public_project_path(actual_source_path)}"
)

print(
    f"Optical-depth-gradient smoothing window: "
    f"{smoothing_window}"
)

print(
    f"Fixed circle-map Omega: {Omega}"
)

print(
    f"K grid: {len(K_values)} points over "
    f"[{K_values.min():.6g}, {K_values.max():.6g}]"
)

print(
    f"Nominal map iterations: {iterations}"
)

print(
    f"Extra stabilization iterations: "
    f"{stabilization_extra_iterations}"
)

# -------------------------------------------------------
# INITIAL PHASE DISTRIBUTION PER RADIATION SAMPLE
# -------------------------------------------------------
initial_phase_hist_prob = np.zeros(
    (
        n_samples,
        n_bins,
    ),
    dtype=float,
)

initial_phase_resultant = np.zeros(
    n_samples,
    dtype=float,
)

for sample_idx in range(
    n_samples
):
    theta_init_full = (
        inst_phase_cycles[
            :,
            sample_idx,
        ]
    )

    counts, _ = np.histogram(
        theta_init_full,
        bins=bin_edges,
        density=False,
    )

    initial_phase_hist_prob[
        sample_idx
    ] = (
        counts.astype(float)
        / (
            np.sum(counts)
            + eps
        )
    )

    _, initial_R = (
        circular_mean_cycles(
            theta_init_full
        )
    )

    initial_phase_resultant[
        sample_idx
    ] = initial_R

# -------------------------------------------------------
# FINITE-TIME PUSHFORWARD DISTRIBUTIONS PER SAMPLE
# -------------------------------------------------------
all_hist_prob = np.zeros(
    (
        n_samples,
        len(K_values),
        n_bins,
    ),
    dtype=float,
)

all_hist_prob_stabilization = np.zeros_like(
    all_hist_prob
)

all_circ_mean = np.zeros(
    (
        n_samples,
        len(K_values),
    ),
    dtype=float,
)

all_resultant = np.zeros_like(
    all_circ_mean
)

all_concentration = np.zeros_like(
    all_circ_mean
)

all_stabilization_tv = np.zeros_like(
    all_circ_mean
)

sample_summary_rows = []

for sample_idx, sample_id in enumerate(
    radiation_samples
):
    theta_init = (
        inst_phase_cycles[
            :,
            sample_idx,
        ]
    )

    theta_init = uniform_subsample(
        theta_init,
        max_samples_per_sample,
        rng=rng,
    )

    print(
        f"\nProcessing radiation sample "
        f"{sample_idx + 1}/{n_samples}: "
        f"{sample_id} | "
        f"initial phase samples={len(theta_init)}"
    )

    for i, K in enumerate(
        K_values
    ):
        theta = theta_init.copy()

        for _ in range(
            iterations
        ):
            theta = circle_map(
                theta,
                Omega,
                K,
            )

        counts, _ = np.histogram(
            theta,
            bins=bin_edges,
            density=False,
        )

        prob = (
            counts.astype(float)
            / (
                np.sum(counts)
                + eps
            )
        )

        all_hist_prob[
            sample_idx,
            i,
            :,
        ] = prob

        cm, R = circular_mean_cycles(
            theta
        )

        all_circ_mean[
            sample_idx,
            i,
        ] = cm

        all_resultant[
            sample_idx,
            i,
        ] = R

        all_concentration[
            sample_idx,
            i,
        ] = (
            normalized_concentration_from_prob(
                prob
            )
        )

        theta_stabilization = (
            theta.copy()
        )

        for _ in range(
            stabilization_extra_iterations
        ):
            theta_stabilization = (
                circle_map(
                    theta_stabilization,
                    Omega,
                    K,
                )
            )

        counts_stab, _ = np.histogram(
            theta_stabilization,
            bins=bin_edges,
            density=False,
        )

        prob_stab = (
            counts_stab.astype(float)
            / (
                np.sum(counts_stab)
                + eps
            )
        )

        all_hist_prob_stabilization[
            sample_idx,
            i,
            :,
        ] = prob_stab

        all_stabilization_tv[
            sample_idx,
            i,
        ] = histogram_total_variation(
            prob,
            prob_stab,
        )

    sample_summary_rows.append({
        "sample_id": sample_id,
        "initial_phase_resultant_length": float(
            initial_phase_resultant[
                sample_idx
            ]
        ),
        "mean_normalized_concentration": float(
            np.mean(
                all_concentration[
                    sample_idx
                ]
            )
        ),
        "max_normalized_concentration": float(
            np.max(
                all_concentration[
                    sample_idx
                ]
            )
        ),
        "mean_resultant_length": float(
            np.mean(
                all_resultant[
                    sample_idx
                ]
            )
        ),
        "max_resultant_length": float(
            np.max(
                all_resultant[
                    sample_idx
                ]
            )
        ),
        "mean_circular_final_state": float(
            circular_mean_cycles(
                all_circ_mean[
                    sample_idx
                ]
            )[0]
        ),
        "mean_stabilization_TV": float(
            np.mean(
                all_stabilization_tv[
                    sample_idx
                ]
            )
        ),
        "max_stabilization_TV": float(
            np.max(
                all_stabilization_tv[
                    sample_idx
                ]
            )
        ),
    })

# -------------------------------------------------------
# AGGREGATE ACROSS MEASURED RADIATION SAMPLES
# -------------------------------------------------------
avg_hist_prob = np.mean(
    all_hist_prob,
    axis=0,
)

avg_hist_prob_stabilization = np.mean(
    all_hist_prob_stabilization,
    axis=0,
)

agg_concentration = (
    normalized_concentration_from_prob(
        avg_hist_prob
    )
)

agg_circ_mean = np.zeros(
    len(K_values),
    dtype=float,
)

agg_resultant = np.zeros(
    len(K_values),
    dtype=float,
)

for i in range(
    len(K_values)
):
    cm, R = circular_mean_cycles(
        bin_centers,
        weights=avg_hist_prob[
            i
        ],
    )

    agg_circ_mean[i] = cm
    agg_resultant[i] = R

agg_stabilization_tv = (
    histogram_total_variation(
        avg_hist_prob,
        avg_hist_prob_stabilization,
    )
)

summary_df = pd.DataFrame(
    sample_summary_rows
)

summary_df = (
    summary_df
    .sort_values(
        "mean_normalized_concentration"
    )
    .reset_index(
        drop=True
    )
)

# -------------------------------------------------------
# LONG-FORM TABLES
# -------------------------------------------------------
density_rows = []

for i, K in enumerate(
    K_values
):
    for b, center in enumerate(
        bin_centers
    ):
        density_rows.append({
            "K": float(K),
            "final_state_bin_center": float(
                center
            ),
            "avg_probability": float(
                avg_hist_prob[
                    i,
                    b,
                ]
            ),
            "avg_probability_after_extra_iterations": float(
                avg_hist_prob_stabilization[
                    i,
                    b,
                ]
            ),
        })

density_df = pd.DataFrame(
    density_rows
)

aggregate_diag_df = pd.DataFrame({
    "K": K_values,
    "aggregate_circular_mean": agg_circ_mean,
    "aggregate_resultant_length": agg_resultant,
    "aggregate_normalized_concentration": agg_concentration,
    "aggregate_stabilization_TV": agg_stabilization_tv,
})

initial_density_rows = []

for sample_idx, sample_id in enumerate(
    radiation_samples
):
    for b, center in enumerate(
        bin_centers
    ):
        initial_density_rows.append({
            "sample_id": sample_id,
            "initial_phase_bin_center": float(
                center
            ),
            "initial_probability": float(
                initial_phase_hist_prob[
                    sample_idx,
                    b,
                ]
            ),
        })

initial_density_df = pd.DataFrame(
    initial_density_rows
)

# -------------------------------------------------------
# SAVE RESULTS
# -------------------------------------------------------
npz_path = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}.npz",
)

np.savez_compressed(
    npz_path,
    all_hist_prob=all_hist_prob,
    all_hist_prob_stabilization=all_hist_prob_stabilization,
    avg_hist_prob=avg_hist_prob,
    avg_hist_prob_stabilization=avg_hist_prob_stabilization,
    initial_phase_hist_prob=initial_phase_hist_prob,
    all_circ_mean=all_circ_mean,
    all_resultant=all_resultant,
    all_concentration=all_concentration,
    all_stabilization_tv=all_stabilization_tv,
    K_values=K_values,
    bin_edges=bin_edges,
    bin_centers=bin_centers,
    Omega=Omega,
    iterations=iterations,
    stabilization_extra_iterations=stabilization_extra_iterations,
    max_samples_per_sample=max_samples_per_sample,
    radiation_samples=np.array(
        radiation_samples,
        dtype=object,
    ),
    s_grid=s_grid,
    coordinate_step=coordinate_step,
    smoothing_window=smoothing_window,
    initial_phase_resultant=initial_phase_resultant,
    agg_concentration=agg_concentration,
    agg_circ_mean=agg_circ_mean,
    agg_resultant=agg_resultant,
    agg_stabilization_tv=agg_stabilization_tv,
    source_file=public_project_path(actual_source_path),
    speed_mode=FAST_OPTION,
)

density_csv = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}.csv",
)

summary_csv = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}_sample_summary.csv",
)

aggregate_diag_csv = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}_aggregate_diagnostics.csv",
)

initial_density_csv = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}_initial_phase_distributions.csv",
)

density_df.to_csv(
    density_csv,
    index=False,
)

summary_df.to_csv(
    summary_csv,
    index=False,
)

aggregate_diag_df.to_csv(
    aggregate_diag_csv,
    index=False,
)

initial_density_df.to_csv(
    initial_density_csv,
    index=False,
)

summary_txt = os.path.join(
    out_dir,
    f"{CELL19_RESULT_STEM}.txt",
)

with open(
    summary_txt,
    "w",
) as f:
    f.write(
        "Circle-Map Finite-Time Phase-Distribution Pushforward "
        "from Radiation-Derived Initial Phases\n"
    )

    f.write(
        "==============================================================="
        "=========================\n\n"
    )

    f.write(
        "Interpretation:\n"
        "  Each measured N003 sample supplies an empirical analytic-signal "
        "phase distribution along log-energy.\n"
        "  The circle map pushes that distribution forward for a finite "
        "number of model iterations.\n"
        "  Histogram concentration and circular resultant length summarize "
        "distributional concentration.\n"
        "  Stabilization TV compares the nominal histogram with the histogram "
        "after additional map iterations.\n"
        "  Small stabilization TV supports finite-time stabilization but "
        "does not prove a unique invariant density.\n\n"
    )

    f.write(
        f"Source: {public_project_path(actual_source_path)}\n"
    )

    f.write(
        "Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n"
    )

    f.write(
        "Observable: transmission T(E)\n"
    )

    f.write(
        f"Measured samples: {n_samples}\n"
    )

    f.write(
        f"Coordinate points: {n_coordinate_samples}\n"
    )

    f.write(
        f"Coordinate step: {coordinate_step}\n"
    )

    f.write(
        f"Smoothing window: {smoothing_window}\n"
    )

    f.write(
        f"Omega: {Omega}\n"
    )

    f.write(
        f"K range: [{K_values.min()}, {K_values.max()}] "
        f"with {len(K_values)} points\n"
    )

    f.write(
        f"Nominal iterations: {iterations}\n"
    )

    f.write(
        f"Extra stabilization iterations: "
        f"{stabilization_extra_iterations}\n"
    )

    f.write(
        f"Histogram bins: {n_bins}\n"
    )

    f.write(
        f"Max phase samples per radiation sample: "
        f"{max_samples_per_sample}\n\n"
    )

    f.write(
        "Aggregate diagnostics:\n"
    )

    f.write(
        f"  mean normalized concentration: "
        f"{np.mean(agg_concentration):.8f}\n"
    )

    f.write(
        f"  max normalized concentration:  "
        f"{np.max(agg_concentration):.8f}\n"
    )

    f.write(
        f"  mean resultant length:         "
        f"{np.mean(agg_resultant):.8f}\n"
    )

    f.write(
        f"  max resultant length:          "
        f"{np.max(agg_resultant):.8f}\n"
    )

    f.write(
        f"  mean stabilization TV:         "
        f"{np.mean(agg_stabilization_tv):.8f}\n"
    )

    f.write(
        f"  max stabilization TV:          "
        f"{np.max(agg_stabilization_tv):.8f}\n\n"
    )

    f.write(
        "Per-sample summary:\n"
    )

    for _, row in summary_df.iterrows():
        f.write(
            f"  {row['sample_id']}: "
            f"initial_R="
            f"{row['initial_phase_resultant_length']:.6f}, "
            f"mean_norm_concentration="
            f"{row['mean_normalized_concentration']:.6f}, "
            f"max_norm_concentration="
            f"{row['max_normalized_concentration']:.6f}, "
            f"mean_resultant_length="
            f"{row['mean_resultant_length']:.6f}, "
            f"max_resultant_length="
            f"{row['max_resultant_length']:.6f}, "
            f"mean_stabilization_TV="
            f"{row['mean_stabilization_TV']:.6f}, "
            f"max_stabilization_TV="
            f"{row['max_stabilization_TV']:.6f}\n"
        )

# -------------------------------------------------------
# PLOT 1: AGGREGATE FINAL-STATE PROBABILITY HEATMAP
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(ax)

im = ax.imshow(
    avg_hist_prob.T,
    extent=(
        K_values.min(),
        K_values.max(),
        0.0,
        1.0,
    ),
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap=RED_SEQUENTIAL_CMAP,
    vmin=0.0,
    vmax=max(
        float(
            np.nanpercentile(
                avg_hist_prob,
                99.5,
            )
        ),
        1e-12,
    ),
)

ax.set_title(
    "Circle-Map Final-State Probability "
    "from Radiation-Derived Phase\n"
    f"Ω = {Omega:.4f}"
)

ax.set_xlabel(
    "Circle-map nonlinearity K"
)

ax.set_ylabel(
    "Final state θ after iteration (cycles)"
)

cbar = plt.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Mean probability per bin"
)

style_colorbar(
    cbar
)

plot1_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_01_aggregate_final_state_probability_heatmap.png",
)

save_show_close(
    fig,
    plot1_path,
)

# -------------------------------------------------------
# PLOT 2: CIRCULAR MEAN FINAL STATE VS K
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    K_values,
    agg_circ_mean,
    color=ACCENT,
    linewidth=1.8,
)

ax.set_title(
    "Circular Mean Final State vs K"
)

ax.set_xlabel(
    "Circle-map nonlinearity K"
)

ax.set_ylabel(
    "Circular mean final state θ (cycles)"
)

plot2_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_02_circular_mean_final_state_vs_K.png",
)

save_show_close(
    fig,
    plot2_path,
)

# -------------------------------------------------------
# PLOT 3: NORMALIZED CONCENTRATION VS K
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    K_values,
    agg_concentration,
    color=ACCENT,
    linewidth=1.8,
)

ax.set_ylim(
    0.0,
    1.0,
)

ax.set_title(
    "Final-State Distribution Concentration vs K"
)

ax.set_xlabel(
    "Circle-map nonlinearity K"
)

ax.set_ylabel(
    "Normalized histogram concentration"
)

plot3_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_03_normalized_concentration_vs_K.png",
)

save_show_close(
    fig,
    plot3_path,
)

# -------------------------------------------------------
# PLOT 4: CIRCULAR RESULTANT LENGTH VS K
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    K_values,
    agg_resultant,
    color=ACCENT,
    linewidth=1.8,
)

ax.set_ylim(
    0.0,
    1.0,
)

ax.set_title(
    "Circular Resultant Length of Final-State Distribution vs K"
)

ax.set_xlabel(
    "Circle-map nonlinearity K"
)

ax.set_ylabel(
    "Resultant length R"
)

plot4_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_04_circular_resultant_length_vs_K.png",
)

save_show_close(
    fig,
    plot4_path,
)

# -------------------------------------------------------
# PLOT 5: SAMPLE-WISE CONCENTRATION SUMMARY
# -------------------------------------------------------
plot_df = summary_df.sort_values(
    "mean_normalized_concentration",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df[
        "mean_normalized_concentration"
    ].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_xlim(
    0.0,
    1.0,
)

ax.set_title(
    "Mean Final-State Concentration by N003 Sample"
)

ax.set_xlabel(
    "Mean normalized concentration"
)

ax.set_ylabel(
    "Measured sample"
)

plot5_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_05_mean_final_state_concentration_by_sample.png",
)

save_show_close(
    fig,
    plot5_path,
)

# -------------------------------------------------------
# PLOT 6: SAMPLE-WISE RESULTANT LENGTH SUMMARY
# -------------------------------------------------------
plot_df = summary_df.sort_values(
    "mean_resultant_length",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df[
        "mean_resultant_length"
    ].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_xlim(
    0.0,
    1.0,
)

ax.set_title(
    "Mean Circular Resultant Length by N003 Sample"
)

ax.set_xlabel(
    "Mean resultant length R"
)

ax.set_ylabel(
    "Measured sample"
)

plot6_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_06_mean_resultant_length_by_sample.png",
)

save_show_close(
    fig,
    plot6_path,
)

# -------------------------------------------------------
# PLOT 7: INITIAL RADIATION PHASE DISTRIBUTION BY SAMPLE
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax
)

im = ax.imshow(
    initial_phase_hist_prob,
    extent=(
        0.0,
        1.0,
        -0.5,
        n_samples - 0.5,
    ),
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap=RED_SEQUENTIAL_CMAP,
    vmin=0.0,
    vmax=max(
        float(
            np.nanpercentile(
                initial_phase_hist_prob,
                99.5,
            )
        ),
        1e-12,
    ),
)

ax.set_title(
    "Initial Radiation Analytic-Phase Distribution by N003 Sample"
)

ax.set_xlabel(
    "Initial phase θ (cycles)"
)

ax.set_ylabel(
    "Measured sample"
)

ax.set_yticks(
    np.arange(
        n_samples
    )
)

ax.set_yticklabels(
    radiation_samples
)

cbar = plt.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Probability per bin"
)

style_colorbar(
    cbar
)

plot7_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_07_initial_radiation_phase_distribution_by_sample.png",
)

save_show_close(
    fig,
    plot7_path,
)

# -------------------------------------------------------
# PLOT 8: SELECTED FINAL-STATE DISTRIBUTIONS
# -------------------------------------------------------
selected_K_values = [
    K_values[0],
    K_values[
        len(K_values) // 4
    ],
    K_values[
        len(K_values) // 2
    ],
    K_values[
        3 * len(K_values) // 4
    ],
    K_values[-1],
]

fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

line_styles = [
    "-",
    "--",
    "-.",
    ":",
    "-",
]

alphas = [
    1.0,
    0.85,
    0.70,
    0.55,
    0.40,
]

for idx, K_sel in enumerate(
    selected_K_values
):
    k_idx = int(
        np.argmin(
            np.abs(
                K_values
                - K_sel
            )
        )
    )

    ax.plot(
        bin_centers,
        avg_hist_prob[
            k_idx
        ],
        color=ACCENT,
        linestyle=line_styles[
            idx
            % len(
                line_styles
            )
        ],
        alpha=alphas[
            idx
        ],
        linewidth=1.8,
        label=(
            f"K = "
            f"{K_values[k_idx]:.2f}"
        ),
    )

ax.set_title(
    "Selected Final-State Probability Distributions"
)

ax.set_xlabel(
    "Final state θ (cycles)"
)

ax.set_ylabel(
    "Probability per bin"
)

leg = ax.legend(
    facecolor=BG,
    edgecolor=ACCENT,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot8_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_08_selected_final_state_distributions.png",
)

save_show_close(
    fig,
    plot8_path,
)

# -------------------------------------------------------
# PLOT 9: FINITE-TIME STABILIZATION DIAGNOSTIC
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    K_values,
    agg_stabilization_tv,
    color=ACCENT,
    linewidth=1.8,
)

ax.axhline(
    0.05,
    color=WHITE,
    linestyle="--",
    linewidth=1.0,
    alpha=0.75,
    label="TV = 0.05 reference",
)

ax.set_ylim(
    0.0,
    min(
        1.0,
        max(
            0.10,
            1.08
            * float(
                np.nanmax(
                    agg_stabilization_tv
                )
            ),
        ),
    ),
)

ax.set_title(
    "Finite-Time Distribution Stabilization vs K"
)

ax.set_xlabel(
    "Circle-map nonlinearity K"
)

ax.set_ylabel(
    "Total-variation distance after extra iterations"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot9_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_09_distribution_stabilization_TV_vs_K.png",
)

save_show_close(
    fig,
    plot9_path,
)

# -------------------------------------------------------
# PLOT 10: STABILIZATION SUMMARY BY RADIATION SAMPLE
# -------------------------------------------------------
plot_df = summary_df.sort_values(
    "mean_stabilization_TV",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df["sample_id"].values,
    plot_df[
        "mean_stabilization_TV"
    ].values,
    color=ACCENT,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_xlim(
    0.0,
    min(
        1.0,
        max(
            0.10,
            1.08
            * float(
                plot_df[
                    "mean_stabilization_TV"
                ].max()
            ),
        ),
    ),
)

ax.set_title(
    "Mean Finite-Time Distribution Stabilization by N003 Sample"
)

ax.set_xlabel(
    "Mean total-variation distance"
)

ax.set_ylabel(
    "Measured sample"
)

plot10_path = os.path.join(
    plots_dir,
    f"{CELL19_RESULT_STEM}_plots_10_mean_distribution_stabilization_by_sample.png",
)

save_show_close(
    fig,
    plot10_path,
)

# -------------------------------------------------------
# FINAL PRINTS
# -------------------------------------------------------
print(
    f"\nSaved NPZ: {npz_path}"
)

print(
    f"Saved density CSV: {density_csv}"
)

print(
    f"Saved sample summary CSV: {summary_csv}"
)

print(
    f"Saved aggregate diagnostics CSV: {aggregate_diag_csv}"
)

print(
    f"Saved initial phase distribution CSV: {initial_density_csv}"
)

print(
    f"Saved summary TXT: {summary_txt}"
)

print(
    f"Saved plots directly to: {PHASE2_RESULTS_DIR}"
)


<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">

<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Nearest-Neighbor Distance-Distribution Entropy in Radiation Embedding Space</h3>

<p>
For each reconstructed radiation state, the code finds the Euclidean distance to its nearest neighbor.
Those distances are binned into an empirical probability distribution <b>p<sub>b</sub></b> and summarized by Shannon entropy:
<b>H<sub>NN</sub> = −Σ p<sub>b</sub> log<sub>2</sub> p<sub>b</sub></b>.
The statistic describes how heterogeneous the local spacing is in the sampled embedding geometry.
</p>

<p>
The quantity labeled <b>KS-like metric</b> is <b>not a Kolmogorov–Sinai entropy estimator</b>.
KS entropy is a dynamical entropy rate defined through refining measurable partitions.
Entropy of nearest-neighbor distances is a geometric finite-sample descriptor.
The radiation notebook therefore uses the physically neutral name <b>nearest-neighbor distance entropy</b>.
</p>

<p>
The processed <b>N003 / ISIS RB2000164</b> responses are sampled on a common uniform
<b>s = ln(E/E<sub>min</sub>)</b> coordinate and delay-embedded at dimensions 2 through 10.
Both measured transmission <b>T(E)</b> and the processed macroscopic removal cross section
<b>Σ<sub>R</sub>(E)</b> are used when available, so the analysis can distinguish embedding effects from observable effects.
</p>

<p>
Each response curve is robustly standardized before embedding. Because the histogram bins are generated from each curve's own nearest-neighbor distances, a simple positive rescaling does not create an artificial entropy ranking; the standardization mainly improves numerical conditioning.
</p>

</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">

<p><b>Ordered-coordinate caveat.</b>
Adjacent log-energy states can naturally be nearest neighbors because the radiation responses are smooth ordered curves.
The main metric uses the raw nearest-neighbor definition without excluding neighboring indices.
To make that limitation visible, the analysis additionally records the nearest-neighbor index separation and the fraction of nearest neighbors that are immediately adjacent along the ordered coordinate.
A high adjacent-neighbor fraction means the entropy is strongly describing local sampling geometry rather than a global reconstructed-state structure.
</p>

<p><b>Embedding-dimension caveat.</b>
Nearest-neighbor geometry changes systematically with embedding dimension.
The notebook therefore includes both raw dimension trends and within-dimension z-score plots.
Cross-dimension rankings should not be interpreted as a dimension-independent physical invariant.</p>

<p><b>Output convention.</b>
All tables and plots are saved directly into <b>results/phase2/</b>.
Every plot uses the Cell-21 result filename followed by <b>_plots_</b>.</p>

</div>

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree
from scipy.stats import entropy

# -------------------------------------------------------
# SPEED MODE — shared four-level convention
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError(
        "FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'."
    )

_SPEED_COUNT_FACTOR = {
    "full": 1.0,
    "balanced": 0.60,
    "fast": 0.35,
    "ultra": 0.18,
}[FAST_OPTION]

def _speed_count(original, minimum=1):
    return int(original) if FAST_OPTION == "full" else max(int(minimum), int(round(float(original) * _SPEED_COUNT_FACTOR)))

# -------------------------------------------------------
# CONFIG — project repository and flat results directory
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next(
    (p for p in N003_SOURCE_CANDIDATES if p.exists()),
    None,
)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL21_RESULT_STEM = "cell_21_nearest_neighbor_distance_entropy_results"

out_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

EMBEDDING_DIMS = tuple(range(2, 11))
TIME_DELAY = 1
N_BINS = 80

MAX_POINTS = _speed_count(
    20000,
    minimum=500,
)

MAX_UNIFORM_POINTS = 5000
MIN_UNIFORM_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

RED_SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    "nn_entropy_black_red",
    [BLACK, RED_DIM, RED, RED_SOFT],
    N=256,
)

RED_DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "nn_entropy_red_black_red",
    [RED_DIM, BLACK, RED_SOFT],
    N=256,
)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})

def load_npy_safely(file_path):
    """
    Loads .npy files that may contain normal arrays or object arrays.
    Only use allow_pickle=True for files you trust.
    """
    arr = np.load(file_path, allow_pickle=True)

    # If it is a scalar object array, unwrap it.
    if arr.dtype == object and arr.shape == ():
        arr = arr.item()

    # If it is a dict-like saved object, try to extract the first ndarray.
    if isinstance(arr, dict):
        arrays = [v for v in arr.values() if isinstance(v, np.ndarray)]
        if len(arrays) == 0:
            raise ValueError("Loaded object is a dict but contains no ndarray values.")
        arr = arrays[0]

    # If it is an object array containing arrays/lists, try to stack/convert.
    if isinstance(arr, np.ndarray) and arr.dtype == object:
        try:
            arr = np.asarray(arr.tolist(), dtype=float)
        except Exception:
            arr = np.asarray(arr, dtype=float)

    arr = np.asarray(arr, dtype=float)
    arr = np.squeeze(arr)

    if arr.size == 0:
        raise ValueError("Loaded array is empty.")

    return arr

def delay_embedding_1d(series, dimension, delay):
    """
    Delay-embed a 1D time series into shape (n_vectors, dimension).
    """
    series = np.asarray(series, dtype=float).reshape(-1)
    n = len(series)

    n_vectors = n - (dimension - 1) * delay

    if n_vectors <= 1:
        raise ValueError(
            f"Time series too short for dimension={dimension}, delay={delay}. "
            f"Length was {n}."
        )

    embedded = np.column_stack([
        series[i * delay : i * delay + n_vectors]
        for i in range(dimension)
    ])

    return embedded

def prepare_data_for_ks(data, embedding_dimension, time_delay):
    """
    If data is 1D, delay-embed it.
    If data is already 2D, use it as an existing embedding.
    """
    data = np.asarray(data, dtype=float)
    data = np.squeeze(data)

    # Remove NaN/inf
    if data.ndim == 1:
        data = data[np.isfinite(data)]
        X = delay_embedding_1d(data, embedding_dimension, time_delay)

    elif data.ndim == 2:
        # Assume already embedded: rows=samples, cols=features.
        # Remove rows with NaN/inf.
        mask = np.all(np.isfinite(data), axis=1)
        X = data[mask]

        if X.shape[0] <= 1:
            raise ValueError("Not enough valid rows after removing NaN/inf.")

    else:
        # Flatten higher-dimensional data as a fallback.
        flat = data.reshape(-1)
        flat = flat[np.isfinite(flat)]
        X = delay_embedding_1d(flat, embedding_dimension, time_delay)

    return X

def nearest_neighbor_distances_fast(X):
    """
    Fast nearest-neighbor distances using cKDTree.
    Query k=2 because the nearest neighbor is the point itself.
    """
    X = np.asarray(X, dtype=float)

    tree = cKDTree(X)

    try:
        dists, idx = tree.query(X, k=2, workers=-1)
    except TypeError:
        # Older scipy versions may not support workers
        dists, idx = tree.query(X, k=2)

    nearest = dists[:, 1]
    nearest = nearest[np.isfinite(nearest)]
    nearest = nearest[nearest > 0]

    return nearest

def entropy_of_nearest_neighbor_distances(nearest_distances, n_bins=80):
    """
    Entropy of the nearest-neighbor distance distribution.

    This is a KS-like / entropy-like metric, not a rigorous
    Kolmogorov-Sinai entropy estimator.
    """
    nearest_distances = np.asarray(nearest_distances, dtype=float)
    nearest_distances = nearest_distances[np.isfinite(nearest_distances)]
    nearest_distances = nearest_distances[nearest_distances > 0]

    if nearest_distances.size == 0:
        raise ValueError("No valid positive nearest-neighbor distances found.")

    counts, edges = np.histogram(nearest_distances, bins=n_bins, density=False)
    prob = counts.astype(float) / max(np.sum(counts), 1)

    prob = prob[prob > 0]

    return float(entropy(prob, base=2))

def calculate_ks_metric(data, embedding_dimension=3, time_delay=1, max_points=20000):
    """
    Approximate entropy-like metric from nearest-neighbor distances.

    Note:
    This is not a rigorous Kolmogorov-Sinai entropy estimator.
    It is an entropy of the nearest-neighbor distance distribution.
    """
    X = prepare_data_for_ks(data, embedding_dimension, time_delay)

    # Optional subsampling for speed
    if max_points is not None and X.shape[0] > max_points:
        idx = np.linspace(0, X.shape[0] - 1, max_points).astype(int)
        X = X[idx]

    if X.shape[0] < 3:
        raise ValueError("Need at least 3 embedded points.")

    nearest = nearest_neighbor_distances_fast(X)

    if nearest.size == 0:
        raise ValueError("No nearest-neighbor distances found.")

    ks_metric = entropy_of_nearest_neighbor_distances(nearest)

    # Return 4 values because the batch loop expects 4
    return float(ks_metric), int(X.shape[0]), int(X.shape[1]), nearest

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(axis="both", colors=ACCENT)

    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)

    for spine in ax.spines.values():
        spine.set_color(ACCENT)

    ax.grid(True, color=ACCENT, alpha=0.18)

def style_colorbar(cbar):
    cbar.outline.set_edgecolor(ACCENT)
    cbar.ax.tick_params(color=ACCENT, labelcolor=ACCENT)
    cbar.ax.yaxis.label.set_color(ACCENT)

    for label in cbar.ax.get_yticklabels():
        label.set_color(ACCENT)

def save_show_close(fig, out_path, dpi=240):
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)

def infer_embedding_family(row):
    file_name = str(row["file"]).lower()
    source_dir = str(row.get("source_dir", "")).lower()

    if "tsne" in file_name or "tsne" in source_dir:
        return "t-SNE"

    if "umap" in file_name or "umap" in source_dir:
        return "UMAP"

    if re.match(r"^\d+dembedded_", file_name):
        return "Delay embedding"

    return str(row.get("source_dir", "Unknown"))

def infer_embedding_dimension(row):
    file_name = str(row["file"])

    m = re.match(r"^(\d+)dembedded_", file_name)
    if m:
        return int(m.group(1))

    # t-SNE and UMAP outputs are usually 2D here.
    return int(row["state_dim"])

def infer_channel(row):
    file_name = str(row["file"])
    stem = os.path.splitext(file_name)[0]

    if "dembedded_" in stem:
        return stem.split("dembedded_", 1)[1]

    prefixes = [
        "tsne_embedding_",
        "umap_embedding_",
        "umap_m10_",
        "umap_m9_",
        "umap_m8_",
        "umap_m7_",
        "umap_m6_",
        "umap_m5_",
        "umap_m4_",
        "umap_m3_",
        "umap_m2_",
        "umap_",
        "tsne_",
    ]

    for prefix in prefixes:
        if stem.startswith(prefix):
            return stem.replace(prefix, "", 1)

    return stem

def robust_z(x):
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return 0.6745 * (x - med) / mad

# -------------------------------------------------------
# RADIATION DATA HELPERS
# -------------------------------------------------------
def _find_radiation_column(
    df,
    exact=(),
    contains=(),
    exclude=(),
):
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[
                name.lower()
            ]

    matches = []

    for c in df.columns:
        lc = str(c).lower()

        if (
            any(
                token.lower() in lc
                for token in contains
            )
            and not any(
                token.lower() in lc
                for token in exclude
            )
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda c: (
            len(str(c)),
            str(c),
        ),
    )[0]


def _robust_standardize(
    x,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    med = np.nanmedian(
        x
    )

    mad = np.nanmedian(
        np.abs(
            x - med
        )
    )

    scale = 1.4826 * mad

    if (
        not np.isfinite(scale)
        or scale <= 0
    ):
        scale = np.nanstd(
            x
        )

    if (
        not np.isfinite(scale)
        or scale <= 0
    ):
        raise ValueError(
            "Degenerate response curve."
        )

    return (
        x - med
    ) / scale


def load_rb2000164_common_curves(
    source_path,
):
    """
    Return actual processed radiation curves on one common uniform
    log-energy coordinate.

    Curves are keyed by:
      sample_id x observable
    """
    source_path = Path(
        source_path
    )

    if (
        source_path.name.endswith(
            ".csv.gz"
        )
        or source_path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            source_path
        )

    elif (
        source_path.suffix.lower()
        == ".parquet"
    ):
        try:
            df = pd.read_parquet(
                source_path
            )
        except ImportError as exc:
            csv_fallback = (
                source_path
                .with_suffix("")
                .with_suffix(".csv.gz")
            )

            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(
                    source_path
                )
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback found."
                ) from exc

    else:
        raise ValueError(
            f"Unsupported RB2000164 input format: {source_path}"
        )

    sample_col = _find_radiation_column(
        df,
        exact=(
            "sample_id",
            "sample",
            "sample_name",
        ),
        contains=("sample",),
        exclude=("uncert",),
    )

    energy_col = _find_radiation_column(
        df,
        exact=(
            "energy_eV",
            "energy_ev",
            "neutron_energy_eV",
            "energy_in_eV",
        ),
        contains=("energy",),
        exclude=(
            "uncert",
            "lower",
            "upper",
        ),
    )

    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=(
            "uncert",
            "sigma",
            "error",
        ),
    )

    sigma_col = _find_radiation_column(
        df,
        exact=(
            "Sigma_R_cm_inv",
            "sigma_r_cm_inv",
            "Sigma_R",
            "sigma_r",
            "macroscopic_removal_cross_section_cm_inv",
            "macroscopic_removal_cross_section",
        ),
        contains=(
            "sigma_r",
            "removal",
        ),
        exclude=(
            "uncert",
            "error",
        ),
    )

    if sample_col is None:
        raise KeyError(
            "Could not identify N003 sample column."
        )

    if energy_col is None:
        raise KeyError(
            "Could not identify N003 neutron-energy column."
        )

    if transmission_col is None:
        raise KeyError(
            "Could not identify N003 transmission column."
        )

    response_columns = [
        (
            "transmission",
            transmission_col,
        )
    ]

    if sigma_col is not None:
        response_columns.append(
            (
                "Sigma_R_cm_inv",
                sigma_col,
            )
        )

    # First build cleaned per-sample/per-observable arrays.
    raw_curves = {}

    for sample_id, part in df.groupby(
        sample_col,
        sort=True,
        dropna=False,
    ):
        sample_id = str(
            sample_id
        )

        for observable, value_col in response_columns:
            q = part[
                [
                    energy_col,
                    value_col,
                ]
            ].copy()

            q[energy_col] = pd.to_numeric(
                q[energy_col],
                errors="coerce",
            )

            q[value_col] = pd.to_numeric(
                q[value_col],
                errors="coerce",
            )

            q = (
                q.replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .dropna()
            )

            q = q[
                q[energy_col] > 0
            ]

            if observable == "transmission":
                q = q[
                    q[value_col] > 0
                ]

            else:
                q = q[
                    q[value_col] >= 0
                ]

            q = (
                q.groupby(
                    energy_col,
                    as_index=False,
                    sort=True,
                )[value_col]
                .mean()
                .sort_values(
                    energy_col
                )
            )

            if len(q) < MIN_UNIFORM_POINTS:
                print(
                    f"Skipping {sample_id} / {observable}: "
                    f"only {len(q)} usable points.",
                    flush=True,
                )
                continue

            raw_curves[
                (
                    sample_id,
                    observable,
                )
            ] = (
                np.log(
                    q[energy_col]
                    .to_numpy(
                        dtype=float
                    )
                ),
                q[value_col]
                .to_numpy(
                    dtype=float
                ),
            )

    if not raw_curves:
        raise RuntimeError(
            "No usable processed radiation curves found."
        )

    common_lo = max(
        item[0].min()
        for item in raw_curves.values()
    )

    common_hi = min(
        item[0].max()
        for item in raw_curves.values()
    )

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval across radiation curves."
        )

    n_grid = min(
        MAX_UNIFORM_POINTS,
        min(
            len(item[0])
            for item in raw_curves.values()
        ),
    )

    if n_grid < MIN_UNIFORM_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(
        common_lo,
        common_hi,
        n_grid,
    )

    s_grid = (
        logE_grid
        - logE_grid[0]
    )

    ds = float(
        s_grid[1]
        - s_grid[0]
    )

    curves = []

    for (
        sample_id,
        observable,
    ), (
        logE,
        values,
    ) in sorted(
        raw_curves.items()
    ):
        interp_values = np.interp(
            logE_grid,
            logE,
            values,
        )

        scaled = _robust_standardize(
            interp_values
        )

        curves.append({
            "sample_id": sample_id,
            "observable": observable,
            "values_raw": interp_values,
            "values_scaled": scaled,
        })

    return (
        curves,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        sigma_col,
        source_path,
    )


def nearest_neighbor_index_diagnostics(
    X,
):
    """
    Additional radiation diagnostic.

    Returns nearest-neighbor distances plus the absolute index separation
    between each point and its nearest geometric neighbor.
    """
    X = np.asarray(
        X,
        dtype=float,
    )

    tree = cKDTree(
        X
    )

    try:
        dists, idx = tree.query(
            X,
            k=2,
            workers=-1,
        )
    except TypeError:
        dists, idx = tree.query(
            X,
            k=2,
        )

    nearest = dists[
        :,
        1,
    ]

    nearest_idx = idx[
        :,
        1,
    ]

    source_idx = np.arange(
        len(X)
    )

    valid = (
        np.isfinite(nearest)
        & (nearest > 0)
        & np.isfinite(
            nearest_idx
        )
    )

    nearest = nearest[
        valid
    ]

    index_gap = np.abs(
        nearest_idx[
            valid
        ]
        - source_idx[
            valid
        ]
    )

    return (
        nearest,
        index_gap,
    )


# -------------------------------------------------------
# LOAD ACTUAL PROCESSED RADIATION CURVES
# -------------------------------------------------------
(
    radiation_curves,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    sigma_col,
    actual_source_path,
) = load_rb2000164_common_curves(
    N003_PATH
)

print(
    f"Using processed radiation source: {public_project_path(actual_source_path)}"
)

print(
    f"Common log-energy points: {len(s_grid)}"
)

print(
    f"Coordinate step: {coordinate_step:.8g}"
)

print(
    f"Usable sample/observable curves: {len(radiation_curves)}"
)

print(
    "Observables: "
    + ", ".join(
        sorted(
            set(
                item["observable"]
                for item in radiation_curves
            )
        )
    )
)

# -------------------------------------------------------
# COMPUTE ORIGINAL NN-DISTANCE ENTROPY METRIC
# -------------------------------------------------------
records = []
nearest_distance_examples = {}

for curve_idx, item in enumerate(
    radiation_curves,
    start=1,
):
    sample_id = item[
        "sample_id"
    ]

    observable = item[
        "observable"
    ]

    series = item[
        "values_scaled"
    ]

    for dimension in EMBEDDING_DIMS:
        rec = {
            "sample_id": sample_id,
            "observable": observable,
            "embedding_family": "Delay embedding",
            "embedding_dimension": int(
                dimension
            ),
            "time_delay": int(
                TIME_DELAY
            ),
            "status": "ok",
            "error": "",
            "nn_distance_entropy_bits": np.nan,
            "ks_metric": np.nan,
            "n_points_used": np.nan,
            "state_dim": np.nan,
            "nearest_distance_mean": np.nan,
            "nearest_distance_std": np.nan,
            "nearest_distance_min": np.nan,
            "nearest_distance_max": np.nan,
            "nn_cv": np.nan,
            "log10_nn_mean": np.nan,
            "nearest_index_gap_mean": np.nan,
            "nearest_index_gap_median": np.nan,
            "nearest_index_gap_min": np.nan,
            "adjacent_neighbor_fraction": np.nan,
        }

        try:
            X = delay_embedding_1d(
                series,
                dimension,
                TIME_DELAY,
            )

            if (
                MAX_POINTS is not None
                and X.shape[0] > MAX_POINTS
            ):
                selected = np.linspace(
                    0,
                    X.shape[0] - 1,
                    MAX_POINTS,
                ).astype(int)

                X = X[
                    selected
                ]

            if X.shape[0] < 3:
                raise ValueError(
                    "Need at least three embedded points."
                )

            nearest = (
                nearest_neighbor_distances_fast(
                    X
                )
            )

            nn_entropy = (
                entropy_of_nearest_neighbor_distances(
                    nearest,
                    n_bins=N_BINS,
                )
            )

            (
                nearest_diag,
                index_gap,
            ) = nearest_neighbor_index_diagnostics(
                X
            )

            # The two distance arrays should describe the same k=2 query.
            # Use the raw nearest-neighbor array for the main metric.
            rec[
                "nn_distance_entropy_bits"
            ] = float(
                nn_entropy
            )

            # Keep source column name for mathematical traceability.
            rec[
                "ks_metric"
            ] = float(
                nn_entropy
            )

            rec[
                "n_points_used"
            ] = int(
                X.shape[0]
            )

            rec[
                "state_dim"
            ] = int(
                X.shape[1]
            )

            rec[
                "nearest_distance_mean"
            ] = float(
                np.mean(
                    nearest
                )
            )

            rec[
                "nearest_distance_std"
            ] = float(
                np.std(
                    nearest
                )
            )

            rec[
                "nearest_distance_min"
            ] = float(
                np.min(
                    nearest
                )
            )

            rec[
                "nearest_distance_max"
            ] = float(
                np.max(
                    nearest
                )
            )

            rec[
                "nn_cv"
            ] = float(
                np.std(nearest)
                / (
                    np.mean(nearest)
                    + eps
                )
            )

            rec[
                "log10_nn_mean"
            ] = float(
                np.log10(
                    np.mean(nearest)
                    + eps
                )
            )

            if len(index_gap):
                rec[
                    "nearest_index_gap_mean"
                ] = float(
                    np.mean(
                        index_gap
                    )
                )

                rec[
                    "nearest_index_gap_median"
                ] = float(
                    np.median(
                        index_gap
                    )
                )

                rec[
                    "nearest_index_gap_min"
                ] = float(
                    np.min(
                        index_gap
                    )
                )

                rec[
                    "adjacent_neighbor_fraction"
                ] = float(
                    np.mean(
                        index_gap == 1
                    )
                )

            example_key = (
                f"{sample_id}__"
                f"{observable}__"
                f"d{dimension}"
            )

            if len(
                nearest_distance_examples
            ) < 8:
                nearest_distance_examples[
                    example_key
                ] = nearest

        except Exception as exc:
            rec[
                "status"
            ] = "error"

            rec[
                "error"
            ] = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            print(
                f"ERROR sample={sample_id} "
                f"observable={observable} "
                f"dimension={dimension}: "
                f"{rec['error']}",
                flush=True,
            )

        records.append(
            rec
        )

    print(
        f"Completed radiation curve "
        f"{curve_idx}/{len(radiation_curves)}: "
        f"{sample_id} / {observable}",
        flush=True,
    )

df = pd.DataFrame(
    records
)

raw_csv = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}.csv",
)

df.to_csv(
    raw_csv,
    index=False,
)

df_ok = df[
    df["status"] == "ok"
].copy()

if df_ok.empty:
    raise RuntimeError(
        "No successful nearest-neighbor entropy rows."
    )

# -------------------------------------------------------
# ENRICHED METADATA / WITHIN-DIMENSION STANDARDIZATION
# -------------------------------------------------------
df_ok[
    "channel"
] = (
    df_ok["sample_id"].astype(str)
    + "__"
    + df_ok["observable"].astype(str)
)

group_cols = [
    "observable",
    "embedding_dimension",
]

df_ok[
    "entropy_z_within_observable_dim"
] = (
    df_ok.groupby(
        group_cols
    )[
        "nn_distance_entropy_bits"
    ]
    .transform(
        lambda x: (
            x - x.mean()
        ) / (
            x.std(
                ddof=0
            )
            + eps
        )
    )
)

df_ok[
    "entropy_robust_z_within_observable_dim"
] = (
    df_ok.groupby(
        group_cols
    )[
        "nn_distance_entropy_bits"
    ]
    .transform(
        robust_z
    )
)

enriched_csv = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}_enriched.csv",
)

df_ok.to_csv(
    enriched_csv,
    index=False,
)

# -------------------------------------------------------
# SUMMARIES
# -------------------------------------------------------
dimension_summary = (
    df_ok.groupby(
        [
            "observable",
            "embedding_dimension",
        ]
    )
    .agg(
        n=(
            "nn_distance_entropy_bits",
            "count",
        ),
        entropy_mean=(
            "nn_distance_entropy_bits",
            "mean",
        ),
        entropy_std=(
            "nn_distance_entropy_bits",
            "std",
        ),
        entropy_median=(
            "nn_distance_entropy_bits",
            "median",
        ),
        nn_mean=(
            "nearest_distance_mean",
            "mean",
        ),
        nn_cv_mean=(
            "nn_cv",
            "mean",
        ),
        adjacent_neighbor_fraction_mean=(
            "adjacent_neighbor_fraction",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "observable",
            "embedding_dimension",
        ]
    )
)

dimension_summary_csv = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}_dimension_summary.csv",
)

dimension_summary.to_csv(
    dimension_summary_csv,
    index=False,
)

sample_summary = (
    df_ok.groupby(
        [
            "sample_id",
            "observable",
        ]
    )
    .agg(
        entropy_mean=(
            "nn_distance_entropy_bits",
            "mean",
        ),
        entropy_std=(
            "nn_distance_entropy_bits",
            "std",
        ),
        entropy_min=(
            "nn_distance_entropy_bits",
            "min",
        ),
        entropy_max=(
            "nn_distance_entropy_bits",
            "max",
        ),
        adjacent_neighbor_fraction_mean=(
            "adjacent_neighbor_fraction",
            "mean",
        ),
        nearest_index_gap_median=(
            "nearest_index_gap_median",
            "median",
        ),
    )
    .reset_index()
)

sample_summary_csv = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}_sample_summary.csv",
)

sample_summary.to_csv(
    sample_summary_csv,
    index=False,
)

# High / low within-comparable-group outliers.
outlier_df = df_ok.copy()

outlier_df[
    "label"
] = (
    outlier_df["sample_id"].astype(str)
    + " | "
    + outlier_df["observable"].astype(str)
    + " | d="
    + outlier_df["embedding_dimension"].astype(str)
)

top_low = (
    outlier_df
    .sort_values(
        "entropy_robust_z_within_observable_dim",
        ascending=True,
    )
    .head(10)
)

top_high = (
    outlier_df
    .sort_values(
        "entropy_robust_z_within_observable_dim",
        ascending=False,
    )
    .head(10)
)

outlier_plot_df = pd.concat(
    [
        top_low,
        top_high,
    ],
    axis=0,
).sort_values(
    "entropy_robust_z_within_observable_dim"
)

outlier_csv = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}_high_low_outliers.csv",
)

outlier_plot_df.to_csv(
    outlier_csv,
    index=False,
)

# -------------------------------------------------------
# PLOT 1: ENTROPY BY EMBEDDING DIMENSION
# -------------------------------------------------------
observables = sorted(
    df_ok["observable"]
    .dropna()
    .unique()
)

fig, ax = plt.subplots(
    figsize=(11, 6),
    facecolor=BG,
)

style_ax(
    ax
)

rng = np.random.default_rng(
    42
)

offsets = np.linspace(
    -0.12,
    0.12,
    max(
        len(observables),
        1,
    ),
)

for obs_idx, observable in enumerate(
    observables
):
    sub = df_ok[
        df_ok["observable"]
        == observable
    ]

    grouped = (
        sub.groupby(
            "embedding_dimension"
        )[
            "nn_distance_entropy_bits"
        ]
        .agg(
            [
                "mean",
                "std",
            ]
        )
        .reset_index()
    )

    x = (
        grouped[
            "embedding_dimension"
        ].to_numpy(
            dtype=float
        )
        + offsets[
            obs_idx
        ]
    )

    ax.errorbar(
        x,
        grouped["mean"],
        yerr=grouped[
            "std"
        ].fillna(0),
        color=(
            RED
            if obs_idx == 0
            else RED_SOFT
        ),
        marker=(
            "o"
            if obs_idx == 0
            else "s"
        ),
        linewidth=2.0,
        capsize=3,
        label=observable,
    )

    for dimension in EMBEDDING_DIMS:
        vals = sub[
            sub["embedding_dimension"]
            == dimension
        ][
            "nn_distance_entropy_bits"
        ].to_numpy(
            dtype=float
        )

        jitter = rng.normal(
            0.0,
            0.035,
            size=len(vals),
        )

        ax.scatter(
            np.full(
                len(vals),
                dimension
                + offsets[
                    obs_idx
                ],
            )
            + jitter,
            vals,
            color=(
                RED
                if obs_idx == 0
                else RED_SOFT
            ),
            edgecolors=WHITE,
            linewidths=0.35,
            alpha=0.40,
            s=25,
        )

ax.set_title(
    "Nearest-Neighbor Distance Entropy vs Embedding Dimension"
)

ax.set_xlabel(
    "Delay embedding dimension"
)

ax.set_ylabel(
    "Nearest-neighbor distance entropy (bits)"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot1_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_01_entropy_by_embedding_dimension.png",
)

save_show_close(
    fig,
    plot1_path,
)

# -------------------------------------------------------
# PLOT 2: OBSERVABLE COMPARISON
# -------------------------------------------------------
observable_summary = (
    df_ok.groupby(
        "observable"
    )[
        "nn_distance_entropy_bits"
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "count",
        ]
    )
    .reset_index()
    .sort_values(
        "mean"
    )
)

fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    observable_summary[
        "observable"
    ],
    observable_summary[
        "mean"
    ],
    xerr=observable_summary[
        "std"
    ].fillna(0),
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

for y, row in enumerate(
    observable_summary.itertuples()
):
    ax.text(
        row.mean,
        y,
        f"  n={row.count}",
        color=RED_SOFT,
        va="center",
        fontsize=9,
    )

ax.set_title(
    "Radiation Observable Comparison: Mean NN-Distance Entropy"
)

ax.set_xlabel(
    "Mean nearest-neighbor distance entropy (bits)"
)

ax.set_ylabel(
    "Radiation observable"
)

plot2_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_02_mean_entropy_by_observable.png",
)

save_show_close(
    fig,
    plot2_path,
)

# -------------------------------------------------------
# PLOT 3: SAMPLE x DIMENSION RAW HEATMAP — TRANSMISSION
# -------------------------------------------------------
for plot_number, observable in enumerate(
    observables,
    start=3,
):
    sub = df_ok[
        df_ok["observable"]
        == observable
    ].copy()

    pivot = sub.pivot_table(
        index="sample_id",
        columns="embedding_dimension",
        values="nn_distance_entropy_bits",
        aggfunc="mean",
    )

    pivot = pivot.loc[
        pivot.mean(
            axis=1
        )
        .sort_values()
        .index
    ]

    fig, ax = plt.subplots(
        figsize=(11, 7),
        facecolor=BG,
    )

    style_ax(
        ax
    )

    im = ax.imshow(
        pivot.to_numpy(
            dtype=float
        ),
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap=RED_SEQUENTIAL_CMAP,
    )

    ax.set_title(
        f"{observable}: Sample × Embedding-Dimension NN-Distance Entropy"
    )

    ax.set_xlabel(
        "Embedding dimension"
    )

    ax.set_ylabel(
        "Measured sample"
    )

    ax.set_xticks(
        np.arange(
            len(
                pivot.columns
            )
        )
    )

    ax.set_xticklabels(
        [
            str(
                int(c)
            )
            for c in pivot.columns
        ]
    )

    ax.set_yticks(
        np.arange(
            len(
                pivot.index
            )
        )
    )

    ax.set_yticklabels(
        pivot.index
    )

    cbar = plt.colorbar(
        im,
        ax=ax,
    )

    cbar.set_label(
        "NN-distance entropy (bits)"
    )

    style_colorbar(
        cbar
    )

    plot_path = os.path.join(
        plots_dir,
        f"{CELL21_RESULT_STEM}_plots_{plot_number:02d}_{observable}_sample_by_dimension_heatmap.png",
    )

    save_show_close(
        fig,
        plot_path,
    )

# -------------------------------------------------------
# PLOT 5: WITHIN-DIMENSION Z-SCORE HEATMAP
# -------------------------------------------------------
z_pivot = df_ok.pivot_table(
    index=[
        "sample_id",
        "observable",
    ],
    columns="embedding_dimension",
    values="entropy_z_within_observable_dim",
    aggfunc="mean",
)

z_pivot = z_pivot.loc[
    z_pivot.mean(
        axis=1
    )
    .sort_values()
    .index
]

z_values = z_pivot.to_numpy(
    dtype=float
)

vmax = float(
    np.nanmax(
        np.abs(
            z_values
        )
    )
)

if (
    not np.isfinite(vmax)
    or vmax <= 0
):
    vmax = 1.0

fig, ax = plt.subplots(
    figsize=(11, 10),
    facecolor=BG,
)

style_ax(
    ax
)

im = ax.imshow(
    z_values,
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    cmap=RED_DIVERGING_CMAP,
    vmin=-vmax,
    vmax=vmax,
)

ax.set_title(
    "Radiation Curves: NN-Entropy Z-Score Within Observable and Dimension"
)

ax.set_xlabel(
    "Embedding dimension"
)

ax.set_ylabel(
    "Sample / observable"
)

ax.set_xticks(
    np.arange(
        len(
            z_pivot.columns
        )
    )
)

ax.set_xticklabels(
    [
        str(
            int(c)
        )
        for c in z_pivot.columns
    ]
)

ax.set_yticks(
    np.arange(
        len(
            z_pivot.index
        )
    )
)

ax.set_yticklabels(
    [
        f"{sample} | {observable}"
        for sample, observable
        in z_pivot.index
    ],
    fontsize=8,
)

cbar = plt.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Z-score within observable / dimension"
)

style_colorbar(
    cbar
)

plot5_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_05_within_dimension_zscore_heatmap.png",
)

save_show_close(
    fig,
    plot5_path,
)

# -------------------------------------------------------
# PLOT 6: TRAJECTORIES ACROSS EMBEDDING DIMENSION
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(11, 6),
    facecolor=BG,
)

style_ax(
    ax
)

for (
    sample_id,
    observable,
), sub in df_ok.groupby(
    [
        "sample_id",
        "observable",
    ]
):
    sub = sub.sort_values(
        "embedding_dimension"
    )

    ax.plot(
        sub[
            "embedding_dimension"
        ],
        sub[
            "nn_distance_entropy_bits"
        ],
        color=(
            RED
            if observable
            == "transmission"
            else RED_SOFT
        ),
        alpha=0.24,
        linewidth=1.0,
    )

for obs_idx, observable in enumerate(
    observables
):
    grouped = (
        df_ok[
            df_ok["observable"]
            == observable
        ]
        .groupby(
            "embedding_dimension"
        )[
            "nn_distance_entropy_bits"
        ]
        .agg(
            [
                "mean",
                "std",
            ]
        )
        .reset_index()
    )

    ax.plot(
        grouped[
            "embedding_dimension"
        ],
        grouped[
            "mean"
        ],
        color=(
            RED
            if obs_idx == 0
            else RED_SOFT
        ),
        linewidth=3.0,
        marker=(
            "o"
            if obs_idx == 0
            else "s"
        ),
        label=(
            f"{observable} mean"
        ),
    )

    ax.fill_between(
        grouped[
            "embedding_dimension"
        ],
        grouped[
            "mean"
        ]
        - grouped[
            "std"
        ].fillna(0),
        grouped[
            "mean"
        ]
        + grouped[
            "std"
        ].fillna(0),
        color=(
            RED
            if obs_idx == 0
            else RED_SOFT
        ),
        alpha=0.10,
    )

ax.set_title(
    "NN-Distance Entropy Trajectories Across Embedding Dimension"
)

ax.set_xlabel(
    "Embedding dimension"
)

ax.set_ylabel(
    "NN-distance entropy (bits)"
)

leg = ax.legend(
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot6_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_06_entropy_trajectories_across_dimension.png",
)

save_show_close(
    fig,
    plot6_path,
)

# -------------------------------------------------------
# PLOT 7: SAMPLE RANKING WITHIN EACH OBSERVABLE
# -------------------------------------------------------
rank_df = (
    df_ok.groupby(
        [
            "sample_id",
            "observable",
        ]
    )[
        "nn_distance_entropy_bits"
    ]
    .agg(
        [
            "mean",
            "std",
            "count",
        ]
    )
    .reset_index()
)

rank_df[
    "label"
] = (
    rank_df[
        "sample_id"
    ].astype(str)
    + " | "
    + rank_df[
        "observable"
    ].astype(str)
)

rank_df = rank_df.sort_values(
    "mean"
)

fig, ax = plt.subplots(
    figsize=(11, 9),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    rank_df[
        "label"
    ],
    rank_df[
        "mean"
    ],
    xerr=rank_df[
        "std"
    ].fillna(0),
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.82,
)

ax.set_title(
    "Mean NN-Distance Entropy by Radiation Sample and Observable"
)

ax.set_xlabel(
    "Mean entropy across embedding dimensions (bits)"
)

ax.set_ylabel(
    "Sample / observable"
)

plot7_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_07_sample_observable_ranking.png",
)

save_show_close(
    fig,
    plot7_path,
)

# -------------------------------------------------------
# PLOT 8: ENTROPY VS MEAN NN SCALE
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 6),
    facecolor=BG,
)

style_ax(
    ax
)

marker_map = {
    "transmission": "o",
    "Sigma_R_cm_inv": "s",
}

for observable in observables:
    sub = df_ok[
        df_ok["observable"]
        == observable
    ]

    ax.scatter(
        sub[
            "log10_nn_mean"
        ],
        sub[
            "nn_distance_entropy_bits"
        ],
        color=(
            RED
            if observable
            == "transmission"
            else RED_SOFT
        ),
        alpha=0.48,
        s=42,
        marker=marker_map.get(
            observable,
            "x",
        ),
        edgecolors=WHITE,
        linewidths=0.35,
        label=observable,
    )

ax.set_title(
    "NN-Distance Entropy vs Mean Nearest-Neighbor Scale"
)

ax.set_xlabel(
    "log10 mean nearest-neighbor distance"
)

ax.set_ylabel(
    "NN-distance entropy (bits)"
)

leg = ax.legend(
    frameon=True,
    fontsize=8,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot8_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_08_entropy_vs_log_nn_scale.png",
)

save_show_close(
    fig,
    plot8_path,
)

# -------------------------------------------------------
# PLOT 9: ENTROPY VS NN COEFFICIENT OF VARIATION
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 6),
    facecolor=BG,
)

style_ax(
    ax
)

for observable in observables:
    sub = df_ok[
        df_ok["observable"]
        == observable
    ]

    ax.scatter(
        sub[
            "nn_cv"
        ],
        sub[
            "nn_distance_entropy_bits"
        ],
        color=(
            RED
            if observable
            == "transmission"
            else RED_SOFT
        ),
        alpha=0.48,
        s=42,
        marker=marker_map.get(
            observable,
            "x",
        ),
        edgecolors=WHITE,
        linewidths=0.35,
        label=observable,
    )

ax.set_title(
    "NN-Distance Entropy vs Nearest-Neighbor Distance Variability"
)

ax.set_xlabel(
    "Nearest-neighbor coefficient of variation"
)

ax.set_ylabel(
    "NN-distance entropy (bits)"
)

leg = ax.legend(
    frameon=True,
    fontsize=8,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot9_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_09_entropy_vs_nn_cv.png",
)

save_show_close(
    fig,
    plot9_path,
)

# -------------------------------------------------------
# PLOT 10: HIGH / LOW WITHIN-GROUP OUTLIERS
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(12, 9),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    outlier_plot_df[
        "label"
    ],
    outlier_plot_df[
        "entropy_robust_z_within_observable_dim"
    ],
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.axvline(
    0.0,
    color=WHITE,
    linestyle="--",
    alpha=0.65,
)

ax.set_title(
    "Strongest Low and High NN-Entropy Outliers Within Comparable Groups"
)

ax.set_xlabel(
    "Robust z-score within observable / embedding dimension"
)

ax.set_ylabel(
    "Radiation curve"
)

plot10_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_10_high_low_outliers_robust_zscore.png",
)

save_show_close(
    fig,
    plot10_path,
)

# -------------------------------------------------------
# PLOT 11: ORDERED-COORDINATE ADJACENCY DIAGNOSTIC
# -------------------------------------------------------
adj_summary = (
    df_ok.groupby(
        [
            "observable",
            "embedding_dimension",
        ]
    )[
        "adjacent_neighbor_fraction"
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(10, 6),
    facecolor=BG,
)

style_ax(
    ax
)

for obs_idx, observable in enumerate(
    observables
):
    sub = adj_summary[
        adj_summary["observable"]
        == observable
    ]

    ax.errorbar(
        sub[
            "embedding_dimension"
        ],
        sub[
            "mean"
        ],
        yerr=sub[
            "std"
        ].fillna(0),
        color=(
            RED
            if obs_idx == 0
            else RED_SOFT
        ),
        marker=(
            "o"
            if obs_idx == 0
            else "s"
        ),
        linewidth=2.0,
        capsize=3,
        label=observable,
    )

ax.set_ylim(
    0.0,
    1.0,
)

ax.set_title(
    "Nearest Neighbors That Are Adjacent Along Log-Energy"
)

ax.set_xlabel(
    "Embedding dimension"
)

ax.set_ylabel(
    "Fraction with |Δ index| = 1"
)

leg = ax.legend(
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot11_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_11_adjacent_neighbor_fraction.png",
)

save_show_close(
    fig,
    plot11_path,
)

# -------------------------------------------------------
# PLOT 12: POINT-COUNT / DIMENSION SANITY CHECK
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 6),
    facecolor=BG,
)

style_ax(
    ax
)

for observable in observables:
    sub = df_ok[
        df_ok["observable"]
        == observable
    ]

    ax.scatter(
        sub[
            "n_points_used"
        ],
        sub[
            "state_dim"
        ],
        s=42,
        color=(
            RED
            if observable
            == "transmission"
            else RED_SOFT
        ),
        alpha=0.48,
        marker=marker_map.get(
            observable,
            "x",
        ),
        edgecolors=WHITE,
        linewidths=0.35,
        label=observable,
    )

ax.set_title(
    "Diagnostic: Points Used vs State Dimension"
)

ax.set_xlabel(
    "Number of embedded points used"
)

ax.set_ylabel(
    "State dimension"
)

leg = ax.legend(
    frameon=True,
    fontsize=8,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot12_path = os.path.join(
    plots_dir,
    f"{CELL21_RESULT_STEM}_plots_12_points_used_vs_state_dimension.png",
)

save_show_close(
    fig,
    plot12_path,
)

# -------------------------------------------------------
# SUMMARY TEXT
# -------------------------------------------------------
summary_txt = os.path.join(
    out_dir,
    f"{CELL21_RESULT_STEM}.txt",
)

with open(
    summary_txt,
    "w",
) as f:
    f.write(
        "Nearest-Neighbor Distance-Distribution Entropy "
        "in Radiation Delay-Embedding Space\n"
    )

    f.write(
        "=============================================================\n\n"
    )

    f.write(
        "Interpretation:\n"
        "  This is Shannon entropy of the nearest-neighbor distance "
        "distribution, not Kolmogorov-Sinai entropy.\n"
        "  It is a finite-sample geometric descriptor of spacing "
        "heterogeneity in delay-embedding space.\n"
        "  Radiation curves are ordered by log-energy rather than time.\n"
        "  Adjacent-coordinate nearest neighbors are quantified separately "
        "because they can dominate smooth ordered trajectories.\n\n"
    )

    f.write(
        f"Source: {public_project_path(actual_source_path)}\n"
    )

    f.write(
        "Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n"
    )

    f.write(
        f"Observables: {observables}\n"
    )

    f.write(
        f"Embedding dimensions: {EMBEDDING_DIMS}\n"
    )

    f.write(
        f"Delay: {TIME_DELAY}\n"
    )

    f.write(
        f"Histogram bins: {N_BINS}\n"
    )

    f.write(
        f"MAX_POINTS: {MAX_POINTS}\n"
    )

    f.write(
        f"Common log-energy points: {len(s_grid)}\n"
    )

    f.write(
        f"Coordinate step: {coordinate_step}\n"
    )

    f.write(
        f"Successful rows: {len(df_ok)} / {len(df)}\n\n"
    )

    f.write(
        "Overall diagnostics:\n"
    )

    f.write(
        f"  entropy min: "
        f"{df_ok['nn_distance_entropy_bits'].min():.8f} bits\n"
    )

    f.write(
        f"  entropy max: "
        f"{df_ok['nn_distance_entropy_bits'].max():.8f} bits\n"
    )

    f.write(
        f"  entropy mean: "
        f"{df_ok['nn_distance_entropy_bits'].mean():.8f} bits\n"
    )

    f.write(
        f"  mean adjacent-neighbor fraction: "
        f"{df_ok['adjacent_neighbor_fraction'].mean():.8f}\n"
    )

    f.write(
        f"  median adjacent-neighbor fraction: "
        f"{df_ok['adjacent_neighbor_fraction'].median():.8f}\n\n"
    )

    f.write(
        "Embedding-dimension summary:\n"
    )

    for _, row in dimension_summary.iterrows():
        f.write(
            f"  {row['observable']} d={int(row['embedding_dimension'])}: "
            f"entropy_mean={row['entropy_mean']:.6f}, "
            f"entropy_std={row['entropy_std']:.6f}, "
            f"adjacent_fraction_mean="
            f"{row['adjacent_neighbor_fraction_mean']:.6f}\n"
        )

print(
    f"\nSaved raw CSV: {raw_csv}"
)

print(
    f"Saved enriched CSV: {enriched_csv}"
)

print(
    f"Saved dimension summary: {dimension_summary_csv}"
)

print(
    f"Saved sample summary: {sample_summary_csv}"
)

print(
    f"Saved outlier table: {outlier_csv}"
)

print(
    f"Saved summary TXT: {summary_txt}"
)

print(
    f"Saved plots directly to: {PHASE2_RESULTS_DIR}"
)


### Pyragas Method

<div style="font-size:14px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:20px;border-radius:8px;margin:10px;display:flex;flex-wrap:nowrap;justify-content:space-between;line-height:1.55;">

<div style="flex:1;margin-right:10px;">
<h2 style="color:#FF4D4D;">Introduction</h2>
<p>
This block applies mutual-information delay selection and a Pyragas-style delayed-difference transformation to the processed
<b>N003 / ISIS RB2000164 radiation-response curves</b>.
The independent coordinate is the ordered dimensionless log-energy
<b>s = ln(E/E<sub>min</sub>)</b>, not physical time.
</p>

<h2 style="color:#FF4D4D;">Radiation Data Preparation</h2>
<p>
All measured sample/observable curves are placed on one common uniformly sampled log-energy grid.
Transmission <b>T(E)</b> and processed macroscopic removal cross section <b>Σ<sub>R</sub>(E)</b> are retained when available.
Each curve is robustly standardized for delay estimation, while the delayed-difference transformation itself is also applied to the physical processed response values.
</p>

<h2 style="color:#FF4D4D;">Coordinate-Lag Selection</h2>
<p>
Mutual information measures statistical dependence between a response curve and a coordinate-shifted copy:
<b>I(Δs) = I[X(s); X(s−Δs)]</b>.
The first local minimum of the lagged mutual-information curve is used as a decorrelation scale when available.
</p>
</div>

<div style="flex:1;margin-left:10px;">
<h2 style="color:#FF4D4D;">Robust Radiation Delay</h2>
<p>
A delay curve is estimated separately for every measured transmission sample.
Because choosing a single arbitrary sample would make the radiation analysis unnecessarily sample-dependent,
the common feedback delay is the <b>median selected log-energy lag across the measured transmission samples</b>.
This method includes a radiation-specific robustness check.
</p>

<h2 style="color:#FF4D4D;">Pyragas-Style Transformation</h2>
<p>
The stabilizing delayed-difference form is
<b>x<sub>ctrl</sub>(s) = x(s) + g[x(s−Δs) − x(s)]</b>.
For <b>0 &lt; g &lt; 1</b>, this is a convex delayed mixture of the observed response and its shifted copy.
</p>

<h2 style="color:#FF4D4D;">Interpretation</h2>
<p>
This is a <b>post-processing transformation along an ordered radiation coordinate</b>.
It is not evidence that the shielding material was placed in a closed temporal feedback loop and it does not establish chaos control.
Its meaningful outputs are the selected coordinate scale, the size of the delayed-difference correction, and how that correction changes local response roughness.
</p>
</div>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Mutual-information caveat.</b> The MI estimate depends on finite sampling and the regression-based estimator. The notebook therefore stores each transmission sample's selected lag and the complete per-sample MI curves, rather than reporting only one number.</p>
<p><b>Delayed-feedback caveat.</b> A smaller derivative RMS after transformation indicates smoothing of the ordered response; it should not be described as stabilization of an underlying physical orbit.</p>
<p><b>Output convention.</b> All results and plots are written directly into <b>results/phase2/</b> using the Cell-25 result stem followed by <b>_plots_</b> for figures.</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">
<h3 style="color:#FF4D4D;margin-top:0;">Theory and Method — Mutual-Information Coordinate Delay and Pyragas-Style Delayed Difference</h3>

<p>
The lagged mutual-information curve is evaluated between the current radiation response and a shifted copy along log-energy:
<b>I(Δs) = I[X(s); X(s−Δs)]</b>.
A first local minimum is used when one exists; otherwise the minimum over the admissible search range is retained.
</p>

<p>
The delayed-difference transformation is
<b>u(s) = g[x(s−Δs) − x(s)]</b> and
<b>x<sub>ctrl</sub>(s) = x(s) + u(s)</b>.
This is the stabilizing sign convention used by the method.
</p>

<p>
Integer decimation, when needed, preserves the physical lag coordinate by updating the effective log-energy sampling density.
The final selected lag is always converted back to the original uniform radiation grid and stored both as a sample count and as
<b>Δs</b> in log-energy units.
</p>
</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">
<p><b>Method components.</b> Mutual-information lag selection, integer decimation, first-local-minimum selection, the delayed-copy construction, and the stabilizing delayed-difference sign are all retained.</p>
<p><b>What changes physically.</b> Time delay becomes an <b>ordered-coordinate lag</b>. The x-axis of every comparison plot is log-energy coordinate rather than time. “Control strength” is reported as a transformation magnitude, not as evidence of successful feedback control of radiation transport.</p>
</div>

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression

# -------------------------------------------------------
# SPEED MODE — shared four-level convention
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "fast")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError(
        "FAST_OPTION must be one of: 'full', 'balanced', 'fast', 'ultra'."
    )

_SPEED_COUNT_FACTOR = {
    "full": 1.0,
    "balanced": 0.60,
    "fast": 0.35,
    "ultra": 0.18,
}[FAST_OPTION]

def _speed_count(original, minimum=1):
    return int(original) if FAST_OPTION == "full" else max(int(minimum), int(round(float(original) * _SPEED_COUNT_FACTOR)))

# -------------------------------------------------------
# CONFIG — project repository and flat results directory
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next(
    (p for p in N003_SOURCE_CANDIDATES if p.exists()),
    None,
)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL25_RESULT_STEM = "cell_25_pyragas_delayed_feedback_results"

results_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

# Pyragas-style delayed difference gain.
gain = 0.1

# Search a broad but finite fraction of the ordered radiation coordinate.
MIN_LAG_FRACTION = 0.005
MAX_LAG_FRACTION = 0.20

# Number of coarse MI lag candidates. A local integer refinement is then run
# around the selected coarse minimum.
N_MI_LAG_CANDIDATES = _speed_count(240, 55)
MI_REFINEMENT_HALF_WIDTH = _speed_count(12, 4)

# Integer decimation still preserves the coordinate lag.
max_mi_samples = _speed_count(25000, 3500)

HEARTBEAT_EVERY_SEC = 10

MAX_UNIFORM_POINTS = 5000
MIN_UNIFORM_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps = 1e-12

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(axis="both", colors=ACCENT)

    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)

    for spine in ax.spines.values():
        spine.set_color(ACCENT)

    ax.grid(True, color=ACCENT, alpha=0.18)

def save_show_close(fig, out_path, dpi=220):
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.show()
    plt.close(fig)
    print("Saved:", out_path, flush=True)

def heartbeat(message, every_sec=10, force=False):
    global last_heartbeat
    now = time.time()

    if force or (now - last_heartbeat >= every_sec):
        print(message, flush=True)
        last_heartbeat = now

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - np.nanmean(x)) / (np.nanstd(x) + 1e-12)

def demean(x):
    x = np.asarray(x, dtype=float)
    return x - np.nanmean(x)

def safe_decimate_for_mi(signal, fs, max_samples=25000):
    """
    Decimate by an integer factor so lag timing remains interpretable.

    Returns
    -------
    signal_work : ndarray
    fs_work : float
    decimation_factor : int
    """
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if max_samples is None or len(signal) <= max_samples:
        return signal, float(fs), 1

    decimation_factor = int(np.ceil(len(signal) / max_samples))
    signal_work = signal[::decimation_factor]
    fs_work = fs / decimation_factor

    return signal_work, fs_work, decimation_factor

def estimate_delay_mutual_information(
    signal,
    sampling_rate,
    min_lag_ms=20,
    max_lag_ms=1000,
    max_samples=25000,
):
    """
    Estimate delay using mutual information.

    Estimator safeguards:
    - ignores tiny lags below min_lag_ms
    - uses integer decimation so lag timing remains valid
    - returns lag in original sample units
    """
    heartbeat("Preparing signal for mutual information delay estimate...", force=True)

    signal = np.asarray(signal, dtype=float).reshape(-1)
    signal = signal[np.isfinite(signal)]

    # Demean/z-score for MI stability
    signal = zscore(signal)

    signal_work, fs_work, decimation_factor = safe_decimate_for_mi(
        signal,
        sampling_rate,
        max_samples=max_samples
    )

    min_lag_work = max(1, int(round((min_lag_ms / 1000.0) * fs_work)))
    max_lag_work = max(min_lag_work + 1, int(round((max_lag_ms / 1000.0) * fs_work)))
    max_lag_work = min(max_lag_work, len(signal_work) // 3)

    if max_lag_work <= min_lag_work:
        raise ValueError(
            f"Signal too short after decimation for lag range. "
            f"min_lag_work={min_lag_work}, max_lag_work={max_lag_work}"
        )

    lags_work = np.arange(min_lag_work, max_lag_work + 1)
    mi_values = np.zeros(len(lags_work), dtype=float)

    print(
        f"MI working samples: {len(signal_work)} | "
        f"decimation factor: {decimation_factor} | "
        f"effective fs: {fs_work:.3f} Hz",
        flush=True
    )

    print(
        f"Searching lags from {min_lag_ms} ms to {max_lag_ms} ms "
        f"({len(lags_work)} lag values)",
        flush=True
    )

    heartbeat("Starting mutual information loop...", force=True)
    start = time.time()

    for idx, lag in enumerate(lags_work):
        heartbeat(
            f"Still running MI... {idx + 1}/{len(lags_work)} "
            f"| lag={lag} work samples "
            f"| elapsed={time.time() - start:.1f}s",
            every_sec=HEARTBEAT_EVERY_SEC
        )

        x = signal_work[:-lag].reshape(-1, 1)
        y = signal_work[lag:]

        mi_values[idx] = mutual_info_regression(
            x,
            y,
            discrete_features=False,
            random_state=42
        )[0]

    heartbeat("Finished mutual information loop.", force=True)

    # First local minimum after min_lag
    local_minima = np.where(
        (mi_values[1:-1] < mi_values[:-2]) &
        (mi_values[1:-1] < mi_values[2:])
    )[0] + 1

    if len(local_minima) > 0:
        best_idx = int(local_minima[0])
        selection_method = "first local minimum after minimum lag"
    else:
        best_idx = int(np.argmin(mi_values))
        selection_method = "global minimum in allowed lag range"

    delay_work_samples = int(lags_work[best_idx])
    delay_seconds = delay_work_samples / fs_work
    delay_samples_original = int(round(delay_seconds * sampling_rate))

    lags_seconds = lags_work / fs_work
    lags_original_samples = np.round(lags_seconds * sampling_rate).astype(int)

    print("Delay selection method:", selection_method, flush=True)

    return {
        "delay_seconds": float(delay_seconds),
        "delay_samples": int(delay_samples_original),
        "lags_seconds": lags_seconds,
        "lags_original_samples": lags_original_samples,
        "mi_values": mi_values,
        "selection_method": selection_method,
        "decimation_factor": int(decimation_factor),
        "fs_work": float(fs_work),
    }

def apply_pyragas_control(signal, delay_samples, gain=0.1, sign="stabilizing"):
    """
    Apply simple Pyragas-style delayed feedback.

    Stabilizing version:
        controlled = signal + gain * (delayed - signal)

    This is usually more sensible than adding signal - delayed, which can
    amplify high-frequency differences.
    """
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if delay_samples <= 0:
        raise ValueError("delay_samples must be positive.")

    if delay_samples >= len(signal):
        raise ValueError("delay_samples must be smaller than signal length.")

    delayed_signal = np.empty_like(signal)
    delayed_signal[:delay_samples] = signal[0]
    delayed_signal[delay_samples:] = signal[:-delay_samples]

    if sign == "stabilizing":
        feedback = gain * (delayed_signal - signal)
    elif sign == "destabilizing":
        feedback = gain * (signal - delayed_signal)
    else:
        raise ValueError("sign must be 'stabilizing' or 'destabilizing'.")

    controlled_signal = signal + feedback

    return controlled_signal, feedback, delayed_signal

# -------------------------------------------------------
# RADIATION DATA HELPERS
# -------------------------------------------------------
def _find_radiation_column(
    df,
    exact=(),
    contains=(),
    exclude=(),
):
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[
                name.lower()
            ]

    matches = []

    for c in df.columns:
        lc = str(c).lower()

        if (
            any(
                token.lower() in lc
                for token in contains
            )
            and not any(
                token.lower() in lc
                for token in exclude
            )
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda c: (
            len(str(c)),
            str(c),
        ),
    )[0]


def _robust_standardize(
    x,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    med = np.nanmedian(
        x
    )

    mad = np.nanmedian(
        np.abs(
            x - med
        )
    )

    scale = 1.4826 * mad

    if (
        not np.isfinite(scale)
        or scale <= 0
    ):
        scale = np.nanstd(
            x
        )

    if (
        not np.isfinite(scale)
        or scale <= 0
    ):
        raise ValueError(
            "Degenerate radiation response curve."
        )

    return (
        x - med
    ) / scale


def load_rb2000164_common_curves(
    source_path,
):
    """
    Load actual processed N003 responses and place all usable sample/observable
    curves on one common uniform log-energy coordinate.
    """
    source_path = Path(
        source_path
    )

    if (
        source_path.name.endswith(
            ".csv.gz"
        )
        or source_path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            source_path
        )

    elif (
        source_path.suffix.lower()
        == ".parquet"
    ):
        try:
            df = pd.read_parquet(
                source_path
            )
        except ImportError as exc:
            csv_fallback = (
                source_path
                .with_suffix("")
                .with_suffix(".csv.gz")
            )

            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(
                    source_path
                )
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback found."
                ) from exc

    else:
        raise ValueError(
            f"Unsupported RB2000164 input format: {source_path}"
        )

    sample_col = _find_radiation_column(
        df,
        exact=(
            "sample_id",
            "sample",
            "sample_name",
        ),
        contains=("sample",),
        exclude=("uncert",),
    )

    energy_col = _find_radiation_column(
        df,
        exact=(
            "energy_eV",
            "energy_ev",
            "neutron_energy_eV",
            "energy_in_eV",
        ),
        contains=("energy",),
        exclude=(
            "uncert",
            "lower",
            "upper",
        ),
    )

    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=(
            "uncert",
            "sigma",
            "error",
        ),
    )

    sigma_col = _find_radiation_column(
        df,
        exact=(
            "Sigma_R_cm_inv",
            "sigma_r_cm_inv",
            "Sigma_R",
            "sigma_r",
            "macroscopic_removal_cross_section_cm_inv",
            "macroscopic_removal_cross_section",
        ),
        contains=(
            "sigma_r",
            "removal",
        ),
        exclude=(
            "uncert",
            "error",
        ),
    )

    if sample_col is None:
        raise KeyError(
            "Could not identify N003 sample column."
        )

    if energy_col is None:
        raise KeyError(
            "Could not identify N003 neutron-energy column."
        )

    if transmission_col is None:
        raise KeyError(
            "Could not identify N003 transmission column."
        )

    response_cols = [
        (
            "transmission",
            transmission_col,
        )
    ]

    if sigma_col is not None:
        response_cols.append(
            (
                "Sigma_R_cm_inv",
                sigma_col,
            )
        )

    raw_curves = {}

    for sample_id, part in df.groupby(
        sample_col,
        sort=True,
        dropna=False,
    ):
        sample_id = str(
            sample_id
        )

        for observable, value_col in response_cols:
            q = part[
                [
                    energy_col,
                    value_col,
                ]
            ].copy()

            q[energy_col] = pd.to_numeric(
                q[energy_col],
                errors="coerce",
            )

            q[value_col] = pd.to_numeric(
                q[value_col],
                errors="coerce",
            )

            q = (
                q.replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .dropna()
            )

            q = q[
                q[energy_col] > 0
            ]

            if observable == "transmission":
                q = q[
                    q[value_col] > 0
                ]
            else:
                q = q[
                    q[value_col] >= 0
                ]

            q = (
                q.groupby(
                    energy_col,
                    as_index=False,
                    sort=True,
                )[value_col]
                .mean()
                .sort_values(
                    energy_col
                )
            )

            if len(q) < MIN_UNIFORM_POINTS:
                print(
                    f"Skipping {sample_id} / {observable}: "
                    f"only {len(q)} usable points.",
                    flush=True,
                )
                continue

            raw_curves[
                (
                    sample_id,
                    observable,
                )
            ] = (
                np.log(
                    q[energy_col]
                    .to_numpy(
                        dtype=float
                    )
                ),
                q[value_col]
                .to_numpy(
                    dtype=float
                ),
            )

    if not raw_curves:
        raise RuntimeError(
            "No usable N003 radiation curves."
        )

    common_lo = max(
        x[0].min()
        for x in raw_curves.values()
    )

    common_hi = min(
        x[0].max()
        for x in raw_curves.values()
    )

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval across usable N003 curves."
        )

    n_grid = min(
        MAX_UNIFORM_POINTS,
        min(
            len(x[0])
            for x in raw_curves.values()
        ),
    )

    if n_grid < MIN_UNIFORM_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(
        common_lo,
        common_hi,
        n_grid,
    )

    s_grid = (
        logE_grid
        - logE_grid[0]
    )

    ds = float(
        s_grid[1]
        - s_grid[0]
    )

    curves = []

    for (
        sample_id,
        observable,
    ), (
        logE,
        values,
    ) in sorted(
        raw_curves.items()
    ):
        raw_interp = np.interp(
            logE_grid,
            logE,
            values,
        )

        curves.append({
            "sample_id": sample_id,
            "observable": observable,
            "values_raw": raw_interp,
            "values_scaled": _robust_standardize(
                raw_interp
            ),
        })

    return (
        curves,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        sigma_col,
        source_path,
    )


def mutual_information_for_lags(
    signal,
    lags,
):
    """
    Regression-based mutual-information calculation,
    evaluated on an explicit integer-lag grid.
    """
    signal = np.asarray(
        signal,
        dtype=float,
    ).reshape(-1)

    signal = signal[
        np.isfinite(signal)
    ]

    signal = zscore(
        signal
    )

    lags = np.asarray(
        lags,
        dtype=int,
    )

    mi_values = np.zeros(
        len(lags),
        dtype=float,
    )

    for idx, lag in enumerate(
        lags
    ):
        if (
            lag <= 0
            or lag >= len(signal) - 2
        ):
            mi_values[
                idx
            ] = np.nan
            continue

        x = signal[
            :-lag
        ].reshape(
            -1,
            1,
        )

        y = signal[
            lag:
        ]

        mi_values[
            idx
        ] = mutual_info_regression(
            x,
            y,
            discrete_features=False,
            random_state=42,
        )[0]

    return mi_values


def select_first_local_minimum(
    lags,
    mi_values,
):
    lags = np.asarray(
        lags,
        dtype=int,
    )

    mi_values = np.asarray(
        mi_values,
        dtype=float,
    )

    valid = np.isfinite(
        mi_values
    )

    if np.sum(valid) < 3:
        raise RuntimeError(
            "Not enough finite MI values for delay selection."
        )

    lags_v = lags[
        valid
    ]

    mi_v = mi_values[
        valid
    ]

    local_minima = np.where(
        (
            mi_v[1:-1]
            < mi_v[:-2]
        )
        & (
            mi_v[1:-1]
            < mi_v[2:]
        )
    )[0] + 1

    if len(
        local_minima
    ) > 0:
        best = int(
            local_minima[0]
        )
        method = (
            "first local minimum"
        )
    else:
        best = int(
            np.argmin(
                mi_v
            )
        )
        method = (
            "global minimum in allowed coordinate-lag range"
        )

    return (
        int(
            lags_v[
                best
            ]
        ),
        method,
    )


def estimate_delay_mutual_information_coordinate(
    signal,
    coordinate_step,
    min_lag_fraction=0.005,
    max_lag_fraction=0.20,
    max_samples=None,
    n_candidates=120,
    refinement_half_width=8,
):
    """
    Radiation-coordinate analogue of the source MI delay estimator.

    Returns lag in ORIGINAL uniform-grid samples and in log-energy units.
    Integer decimation preserves the lag coordinate.
    """
    signal = np.asarray(
        signal,
        dtype=float,
    ).reshape(-1)

    if len(signal) < 20:
        raise ValueError(
            "Radiation curve too short for MI delay estimation."
        )

    # Use the source integer-decimation helper. Here fs is interpreted as
    # samples per unit log-energy rather than samples per second.
    sampling_density = (
        1.0
        / coordinate_step
    )

    signal_work, density_work, decimation_factor = (
        safe_decimate_for_mi(
            signal,
            sampling_density,
            max_samples=max_samples,
        )
    )

    n_work = len(
        signal_work
    )

    min_lag_work = max(
        1,
        int(
            round(
                min_lag_fraction
                * n_work
            )
        ),
    )

    max_lag_work = min(
        n_work // 3,
        max(
            min_lag_work + 2,
            int(
                round(
                    max_lag_fraction
                    * n_work
                )
            ),
        ),
    )

    coarse_lags = np.unique(
        np.round(
            np.linspace(
                min_lag_work,
                max_lag_work,
                min(
                    n_candidates,
                    max_lag_work
                    - min_lag_work
                    + 1,
                ),
            )
        ).astype(
            int
        )
    )

    coarse_mi = mutual_information_for_lags(
        signal_work,
        coarse_lags,
    )

    coarse_best, coarse_method = (
        select_first_local_minimum(
            coarse_lags,
            coarse_mi,
        )
    )

    if len(coarse_lags) > 1:
        coarse_spacing = max(
            1,
            int(
                np.median(
                    np.diff(
                        coarse_lags
                    )
                )
            ),
        )
    else:
        coarse_spacing = 1

    radius = max(
        refinement_half_width,
        coarse_spacing,
    )

    refine_lo = max(
        min_lag_work,
        coarse_best - radius,
    )

    refine_hi = min(
        max_lag_work,
        coarse_best + radius,
    )

    refine_lags = np.arange(
        refine_lo,
        refine_hi + 1,
        dtype=int,
    )

    refine_mi = mutual_information_for_lags(
        signal_work,
        refine_lags,
    )

    best_work, refine_method = (
        select_first_local_minimum(
            refine_lags,
            refine_mi,
        )
    )

    lag_original_samples = int(
        best_work
        * decimation_factor
    )

    lag_coordinate = float(
        lag_original_samples
        * coordinate_step
    )

    # Return a sorted unique lag grid for plotting.
    combined_lags_work = np.concatenate(
        [
            coarse_lags,
            refine_lags,
        ]
    )

    combined_mi = np.concatenate(
        [
            coarse_mi,
            refine_mi,
        ]
    )

    order = np.argsort(
        combined_lags_work
    )

    combined_lags_work = (
        combined_lags_work[
            order
        ]
    )

    combined_mi = (
        combined_mi[
            order
        ]
    )

    unique_lags_work, first_idx = np.unique(
        combined_lags_work,
        return_index=True,
    )

    unique_mi = combined_mi[
        first_idx
    ]

    lags_original_samples = (
        unique_lags_work
        * decimation_factor
    ).astype(
        int
    )

    lags_coordinate = (
        lags_original_samples
        * coordinate_step
    )

    return {
        "delay_samples": lag_original_samples,
        "delay_coordinate": lag_coordinate,
        "lags_original_samples": lags_original_samples,
        "lags_coordinate": lags_coordinate,
        "mi_values": unique_mi,
        "selection_method": (
            f"coarse: {coarse_method}; "
            f"refined: {refine_method}"
        ),
        "decimation_factor": int(
            decimation_factor
        ),
        "coordinate_sampling_density_work": float(
            density_work
        ),
    }


def rms(
    x,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    return float(
        np.sqrt(
            np.mean(
                x ** 2
            )
        )
    )


def derivative_rms(
    x,
    ds,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    return rms(
        np.gradient(
            x,
            ds,
        )
    )


# -------------------------------------------------------
# LOAD ACTUAL PROCESSED RADIATION CURVES
# -------------------------------------------------------
heartbeat(
    "Loading processed radiation curves...",
    force=True,
)

(
    radiation_curves,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    sigma_col,
    actual_source_path,
) = load_rb2000164_common_curves(
    N003_PATH
)

coordinate_span = float(
    s_grid[-1]
    - s_grid[0]
)

print(
    f"Loaded processed radiation source: {public_project_path(actual_source_path)}",
    flush=True,
)

print(
    f"Common uniform log-energy points: {len(s_grid)}",
    flush=True,
)

print(
    f"Coordinate step ds: {coordinate_step:.8g}",
    flush=True,
)

print(
    f"Coordinate span: {coordinate_span:.8g}",
    flush=True,
)

print(
    f"Usable sample/observable curves: {len(radiation_curves)}",
    flush=True,
)

# -------------------------------------------------------
# MI DELAY ESTIMATION — EACH MEASURED TRANSMISSION SAMPLE
# -------------------------------------------------------
transmission_curves = [
    item
    for item in radiation_curves
    if item["observable"]
    == "transmission"
]

if len(
    transmission_curves
) < 3:
    raise RuntimeError(
        "Need at least three transmission curves for robust coordinate-lag selection."
    )

mi_delay_rows = []
mi_curve_rows = []
mi_results_by_sample = {}

for idx, item in enumerate(
    transmission_curves,
    start=1,
):
    heartbeat(
        f"Estimating radiation coordinate delay "
        f"{idx}/{len(transmission_curves)}: "
        f"{item['sample_id']}",
        force=True,
    )

    result = (
        estimate_delay_mutual_information_coordinate(
            item[
                "values_scaled"
            ],
            coordinate_step=coordinate_step,
            min_lag_fraction=MIN_LAG_FRACTION,
            max_lag_fraction=MAX_LAG_FRACTION,
            max_samples=max_mi_samples,
            n_candidates=N_MI_LAG_CANDIDATES,
            refinement_half_width=MI_REFINEMENT_HALF_WIDTH,
        )
    )

    sample_id = item[
        "sample_id"
    ]

    mi_results_by_sample[
        sample_id
    ] = result

    mi_delay_rows.append({
        "sample_id": sample_id,
        "delay_samples": int(
            result[
                "delay_samples"
            ]
        ),
        "delay_log_energy": float(
            result[
                "delay_coordinate"
            ]
        ),
        "delay_fraction_of_coordinate_span": float(
            result[
                "delay_coordinate"
            ]
            / (
                coordinate_span
                + eps
            )
        ),
        "selection_method": result[
            "selection_method"
        ],
        "decimation_factor": int(
            result[
                "decimation_factor"
            ]
        ),
    })

    for lag_samples, lag_s, mi in zip(
        result[
            "lags_original_samples"
        ],
        result[
            "lags_coordinate"
        ],
        result[
            "mi_values"
        ],
    ):
        mi_curve_rows.append({
            "sample_id": sample_id,
            "lag_samples": int(
                lag_samples
            ),
            "lag_log_energy": float(
                lag_s
            ),
            "mutual_information": float(
                mi
            ),
        })

mi_delay_df = pd.DataFrame(
    mi_delay_rows
)

mi_curve_df = pd.DataFrame(
    mi_curve_rows
)

# Robust common lag across measured transmission samples.
common_delay_samples = int(
    round(
        np.median(
            mi_delay_df[
                "delay_samples"
            ].to_numpy(
                dtype=float
            )
        )
    )
)

common_delay_samples = max(
    1,
    min(
        common_delay_samples,
        len(s_grid) - 2,
    ),
)

common_delay_coordinate = float(
    common_delay_samples
    * coordinate_step
)

print(
    "\nPer-sample MI-selected delays:",
    flush=True,
)

print(
    mi_delay_df.to_string(
        index=False
    ),
    flush=True,
)

print(
    f"\nCommon robust delay: "
    f"{common_delay_samples} grid samples "
    f"= {common_delay_coordinate:.8g} log-energy units",
    flush=True,
)

# -------------------------------------------------------
# APPLY PYRAGAS-STYLE TRANSFORMATION TO ALL CURVES
# -------------------------------------------------------
heartbeat(
    "Applying delayed-difference transformation to radiation curves...",
    force=True,
)

effect_rows = []
controlled_curves = {}

for idx, item in enumerate(
    radiation_curves,
    start=1,
):
    sample_id = item[
        "sample_id"
    ]

    observable = item[
        "observable"
    ]

    raw = np.asarray(
        item[
            "values_raw"
        ],
        dtype=float,
    )

    controlled, feedback, delayed = (
        apply_pyragas_control(
            raw,
            delay_samples=common_delay_samples,
            gain=gain,
            sign="stabilizing",
        )
    )

    key = (
        sample_id,
        observable,
    )

    controlled_curves[
        key
    ] = {
        "original": raw,
        "controlled": controlled,
        "feedback": feedback,
        "delayed": delayed,
    }

    original_scale = rms(
        raw
    )

    feedback_scale = rms(
        feedback
    )

    rough_orig = derivative_rms(
        raw,
        coordinate_step,
    )

    rough_ctrl = derivative_rms(
        controlled,
        coordinate_step,
    )

    effect_rows.append({
        "sample_id": sample_id,
        "observable": observable,
        "delay_samples": common_delay_samples,
        "delay_log_energy": common_delay_coordinate,
        "gain": gain,
        "rms_original": original_scale,
        "rms_controlled": rms(
            controlled
        ),
        "rms_feedback": feedback_scale,
        "relative_feedback_rms": float(
            feedback_scale
            / (
                original_scale
                + eps
            )
        ),
        "std_original": float(
            np.std(
                raw
            )
        ),
        "std_controlled": float(
            np.std(
                controlled
            )
        ),
        "variance_ratio_controlled_to_original": float(
            np.var(
                controlled
            )
            / (
                np.var(
                    raw
                )
                + eps
            )
        ),
        "roughness_original_derivative_rms": rough_orig,
        "roughness_controlled_derivative_rms": rough_ctrl,
        "roughness_ratio_controlled_to_original": float(
            rough_ctrl
            / (
                rough_orig
                + eps
            )
        ),
        "correlation_original_controlled": float(
            np.corrcoef(
                raw,
                controlled,
            )[0, 1]
        ),
    })

effect_df = pd.DataFrame(
    effect_rows
)

# -------------------------------------------------------
# SAVE NUMERICAL OUTPUTS
# -------------------------------------------------------
mi_delay_csv = os.path.join(
    results_dir,
    f"{CELL25_RESULT_STEM}_mi_delays.csv",
)

mi_curve_csv = os.path.join(
    results_dir,
    f"{CELL25_RESULT_STEM}_mi_curves.csv",
)

effect_csv = os.path.join(
    results_dir,
    f"{CELL25_RESULT_STEM}.csv",
)

mi_delay_df.to_csv(
    mi_delay_csv,
    index=False,
)

mi_curve_df.to_csv(
    mi_curve_csv,
    index=False,
)

effect_df.to_csv(
    effect_csv,
    index=False,
)

# Store all curves in a rectangular array ordered exactly as radiation_curves.
curve_labels = np.array(
    [
        (
            f"{item['sample_id']}"
            f"__{item['observable']}"
        )
        for item in radiation_curves
    ],
    dtype=object,
)

original_matrix = np.column_stack(
    [
        controlled_curves[
            (
                item[
                    "sample_id"
                ],
                item[
                    "observable"
                ],
            )
        ][
            "original"
        ]
        for item in radiation_curves
    ]
)

controlled_matrix = np.column_stack(
    [
        controlled_curves[
            (
                item[
                    "sample_id"
                ],
                item[
                    "observable"
                ],
            )
        ][
            "controlled"
        ]
        for item in radiation_curves
    ]
)

feedback_matrix = np.column_stack(
    [
        controlled_curves[
            (
                item[
                    "sample_id"
                ],
                item[
                    "observable"
                ],
            )
        ][
            "feedback"
        ]
        for item in radiation_curves
    ]
)

out_npz = os.path.join(
    results_dir,
    f"{CELL25_RESULT_STEM}.npz",
)

np.savez_compressed(
    out_npz,
    s_grid=s_grid,
    coordinate_step=coordinate_step,
    curve_labels=curve_labels,
    original_matrix=original_matrix,
    controlled_matrix=controlled_matrix,
    feedback_matrix=feedback_matrix,
    common_delay_samples=common_delay_samples,
    common_delay_coordinate=common_delay_coordinate,
    gain=gain,
    source_file=public_project_path(actual_source_path),
    speed_mode=FAST_OPTION,
)

# -------------------------------------------------------
# SUMMARY TEXT
# -------------------------------------------------------
summary_txt = os.path.join(
    results_dir,
    f"{CELL25_RESULT_STEM}.txt",
)

with open(
    summary_txt,
    "w",
) as f:
    f.write(
        "Pyragas-Style Delayed-Difference Transformation "
        "of Processed Radiation Responses\n"
    )

    f.write(
        "==============================================================\n\n"
    )

    f.write(
        "Interpretation:\n"
        "  Mutual information selects an ordered-coordinate lag along "
        "uniform log-energy, not a temporal delay.\n"
        "  The transformed response is x_ctrl(s)=x(s)+g[x(s-delta)-x(s)].\n"
        "  This is post-processing of radiation response curves, not an "
        "implemented closed-loop physical control system.\n\n"
    )

    f.write(
        f"Source: {public_project_path(actual_source_path)}\n"
    )

    f.write(
        "Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n"
    )

    f.write(
        f"Coordinate points: {len(s_grid)}\n"
    )

    f.write(
        f"Coordinate step: {coordinate_step}\n"
    )

    f.write(
        f"Coordinate span: {coordinate_span}\n"
    )

    f.write(
        f"Gain: {gain}\n"
    )

    f.write(
        f"Common delay: {common_delay_samples} grid samples\n"
    )

    f.write(
        f"Common delay: {common_delay_coordinate} log-energy units\n"
    )

    f.write(
        f"Transmission samples used for MI delay selection: "
        f"{len(transmission_curves)}\n\n"
    )

    f.write(
        "Per-transmission-sample MI delays:\n"
    )

    for _, row in mi_delay_df.iterrows():
        f.write(
            f"  {row['sample_id']}: "
            f"{int(row['delay_samples'])} samples, "
            f"delta_s={row['delay_log_energy']:.8g}, "
            f"method={row['selection_method']}\n"
        )

    f.write(
        "\nTransformation diagnostics:\n"
    )

    for _, row in effect_df.iterrows():
        f.write(
            f"  {row['sample_id']} / {row['observable']}: "
            f"relative_feedback_rms="
            f"{row['relative_feedback_rms']:.6f}, "
            f"roughness_ratio="
            f"{row['roughness_ratio_controlled_to_original']:.6f}, "
            f"variance_ratio="
            f"{row['variance_ratio_controlled_to_original']:.6f}, "
            f"corr(original,controlled)="
            f"{row['correlation_original_controlled']:.6f}\n"
        )

# -------------------------------------------------------
# PLOT 1: MI CURVES AND SELECTED DELAYS
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(11, 6),
    facecolor=BG,
)

style_ax(
    ax
)

for sample_id, sub in mi_curve_df.groupby(
    "sample_id"
):
    sub = sub.sort_values(
        "lag_log_energy"
    )

    ax.plot(
        sub[
            "lag_log_energy"
        ],
        sub[
            "mutual_information"
        ],
        color=RED,
        linewidth=1.0,
        alpha=0.25,
    )

# Median MI curve on a common lag grid using interpolation.
lag_min = max(
    sub[
        "lag_log_energy"
    ].min()
    for _, sub
    in mi_curve_df.groupby(
        "sample_id"
    )
)

lag_max = min(
    sub[
        "lag_log_energy"
    ].max()
    for _, sub
    in mi_curve_df.groupby(
        "sample_id"
    )
)

common_lag_plot = np.linspace(
    lag_min,
    lag_max,
    250,
)

mi_interp = []

for _, sub in mi_curve_df.groupby(
    "sample_id"
):
    sub = sub.sort_values(
        "lag_log_energy"
    )

    mi_interp.append(
        np.interp(
            common_lag_plot,
            sub[
                "lag_log_energy"
            ],
            sub[
                "mutual_information"
            ],
        )
    )

median_mi_plot = np.median(
    np.vstack(
        mi_interp
    ),
    axis=0,
)

ax.plot(
    common_lag_plot,
    median_mi_plot,
    color=RED_SOFT,
    linewidth=2.5,
    label="Median MI across transmission samples",
)

ax.axvline(
    common_delay_coordinate,
    color=WHITE,
    linestyle="--",
    linewidth=1.3,
    alpha=0.9,
    label=(
        f"Median selected delay = "
        f"{common_delay_coordinate:.4g}"
    ),
)

ax.set_title(
    "Mutual-Information Coordinate-Lag Selection"
)

ax.set_xlabel(
    "Lag Δs along log-energy"
)

ax.set_ylabel(
    "Mutual information"
)

leg = ax.legend(
    facecolor=BG,
    edgecolor=ACCENT,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot1_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_01_mutual_information_coordinate_delay.png",
)

save_show_close(
    fig,
    plot1_path,
)

# -------------------------------------------------------
# REPRESENTATIVE CURVE FOR SOURCE-ROLE COMPARISON PLOTS
# -------------------------------------------------------
representative = sorted(
    transmission_curves,
    key=lambda item: item[
        "sample_id"
    ],
)[0]

rep_key = (
    representative[
        "sample_id"
    ],
    representative[
        "observable"
    ],
)

rep_data = controlled_curves[
    rep_key
]

orig = zscore(
    rep_data[
        "original"
    ]
)

ctrl = zscore(
    rep_data[
        "controlled"
    ]
)

feedback = rep_data[
    "feedback"
]

# -------------------------------------------------------
# PLOT 2: ORIGINAL VS TRANSFORMED — FULL COORDINATE
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(12, 6),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    s_grid,
    orig,
    alpha=0.60,
    color=RED,
    linewidth=1.0,
    label="Original response, z-scored",
)

ax.plot(
    s_grid,
    ctrl,
    alpha=0.88,
    color=RED_SOFT,
    linewidth=1.0,
    linestyle="--",
    label="Delayed-difference transformed, z-scored",
)

ax.set_title(
    f"N003 {representative['sample_id']} Transmission: "
    f"Original vs Delayed-Difference Transform"
)

ax.set_xlabel(
    "Ordered coordinate s = ln(E/E_min)"
)

ax.set_ylabel(
    "Z-scored response"
)

leg = ax.legend(
    facecolor=BG,
    edgecolor=ACCENT,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot2_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_02_original_vs_transformed_full.png",
)

save_show_close(
    fig,
    plot2_path,
)

# -------------------------------------------------------
# PLOT 3: ZOOMED ORIGINAL VS TRANSFORMED
# -------------------------------------------------------
zoom_fraction = 0.15
zoom_n = max(
    10,
    int(
        round(
            zoom_fraction
            * len(
                s_grid
            )
        )
    ),
)

fig, ax = plt.subplots(
    figsize=(12, 6),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    s_grid[
        :zoom_n
    ],
    orig[
        :zoom_n
    ],
    alpha=0.68,
    color=RED,
    linewidth=1.2,
    label="Original response, z-scored",
)

ax.plot(
    s_grid[
        :zoom_n
    ],
    ctrl[
        :zoom_n
    ],
    alpha=0.92,
    color=RED_SOFT,
    linewidth=1.2,
    linestyle="--",
    label="Delayed-difference transformed, z-scored",
)

ax.set_title(
    f"Zoomed Ordered-Coordinate Comparison | "
    f"first {100*zoom_fraction:.0f}% of common log-energy span"
)

ax.set_xlabel(
    "Ordered coordinate s = ln(E/E_min)"
)

ax.set_ylabel(
    "Z-scored response"
)

leg = ax.legend(
    facecolor=BG,
    edgecolor=ACCENT,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot3_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_03_original_vs_transformed_zoom.png",
)

save_show_close(
    fig,
    plot3_path,
)

# -------------------------------------------------------
# PLOT 4: FEEDBACK TERM
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(12, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.plot(
    s_grid,
    zscore(
        feedback
    ),
    color=RED,
    linewidth=1.0,
    alpha=0.85,
)

ax.set_title(
    f"Delayed-Difference Feedback Term | "
    f"{representative['sample_id']} transmission"
)

ax.set_xlabel(
    "Ordered coordinate s = ln(E/E_min)"
)

ax.set_ylabel(
    "Feedback term, z-scored"
)

plot4_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_04_feedback_term.png",
)

save_show_close(
    fig,
    plot4_path,
)

# -------------------------------------------------------
# PLOT 5: RELATIVE FEEDBACK STRENGTH BY CURVE
# -------------------------------------------------------
plot_df = effect_df.copy()

plot_df[
    "label"
] = (
    plot_df[
        "sample_id"
    ].astype(str)
    + " | "
    + plot_df[
        "observable"
    ].astype(str)
)

plot_df = plot_df.sort_values(
    "relative_feedback_rms"
)

fig, ax = plt.subplots(
    figsize=(11, 9),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df[
        "label"
    ],
    plot_df[
        "relative_feedback_rms"
    ],
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.84,
)

ax.set_title(
    "Relative Delayed-Difference Magnitude by Radiation Curve"
)

ax.set_xlabel(
    "RMS feedback / RMS original response"
)

ax.set_ylabel(
    "Sample / observable"
)

plot5_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_05_relative_feedback_strength.png",
)

save_show_close(
    fig,
    plot5_path,
)

# -------------------------------------------------------
# PLOT 6: SELECTED MI DELAY BY TRANSMISSION SAMPLE
# -------------------------------------------------------
plot_df = mi_delay_df.sort_values(
    "delay_log_energy"
)

fig, ax = plt.subplots(
    figsize=(10, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df[
        "sample_id"
    ],
    plot_df[
        "delay_log_energy"
    ],
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.84,
)

ax.axvline(
    common_delay_coordinate,
    color=WHITE,
    linestyle="--",
    linewidth=1.2,
    alpha=0.85,
    label="Median common delay",
)

ax.set_title(
    "Mutual-Information Selected Coordinate Lag by N003 Sample"
)

ax.set_xlabel(
    "Selected lag Δs along log-energy"
)

ax.set_ylabel(
    "Measured transmission sample"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot6_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_06_selected_delay_by_sample.png",
)

save_show_close(
    fig,
    plot6_path,
)

# -------------------------------------------------------
# PLOT 7: ORDERED-RESPONSE ROUGHNESS RATIO
# -------------------------------------------------------
plot_df = effect_df.copy()

plot_df[
    "label"
] = (
    plot_df[
        "sample_id"
    ].astype(str)
    + " | "
    + plot_df[
        "observable"
    ].astype(str)
)

plot_df = plot_df.sort_values(
    "roughness_ratio_controlled_to_original"
)

fig, ax = plt.subplots(
    figsize=(11, 9),
    facecolor=BG,
)

style_ax(
    ax
)

ax.barh(
    plot_df[
        "label"
    ],
    plot_df[
        "roughness_ratio_controlled_to_original"
    ],
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.84,
)

ax.axvline(
    1.0,
    color=WHITE,
    linestyle="--",
    linewidth=1.2,
    alpha=0.85,
    label="No roughness change",
)

ax.set_title(
    "Effect of Delayed-Difference Transform on Response Roughness"
)

ax.set_xlabel(
    "Derivative RMS ratio: transformed / original"
)

ax.set_ylabel(
    "Sample / observable"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot7_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_07_roughness_ratio.png",
)

save_show_close(
    fig,
    plot7_path,
)

# -------------------------------------------------------
# PLOT 8: VARIANCE RATIO VS FEEDBACK MAGNITUDE
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 6),
    facecolor=BG,
)

style_ax(
    ax
)

observables = sorted(
    effect_df[
        "observable"
    ].unique()
)

for obs_idx, observable in enumerate(
    observables
):
    sub = effect_df[
        effect_df[
            "observable"
        ]
        == observable
    ]

    ax.scatter(
        sub[
            "relative_feedback_rms"
        ],
        sub[
            "variance_ratio_controlled_to_original"
        ],
        color=(
            RED
            if obs_idx == 0
            else RED_SOFT
        ),
        edgecolor=WHITE,
        linewidth=0.45,
        s=55,
        alpha=0.72,
        label=observable,
    )

ax.axhline(
    1.0,
    color=WHITE,
    linestyle="--",
    linewidth=1.0,
    alpha=0.65,
)

ax.set_title(
    "Transformation Magnitude vs Response Variance Change"
)

ax.set_xlabel(
    "RMS feedback / RMS original response"
)

ax.set_ylabel(
    "Variance ratio: transformed / original"
)

leg = ax.legend(
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot8_path = os.path.join(
    plots_dir,
    f"{CELL25_RESULT_STEM}_plots_08_feedback_vs_variance_ratio.png",
)

save_show_close(
    fig,
    plot8_path,
)

heartbeat(
    "Cell 25 complete.",
    force=True,
)

print(
    f"\nSaved primary results CSV: {effect_csv}"
)

print(
    f"Saved MI delay CSV: {mi_delay_csv}"
)

print(
    f"Saved MI curve CSV: {mi_curve_csv}"
)

print(
    f"Saved NPZ: {out_npz}"
)

print(
    f"Saved summary TXT: {summary_txt}"
)

print(
    f"Saved plots directly to: {PHASE2_RESULTS_DIR}"
)


### Kakutani's theorem

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:18px;border-radius:8px;margin:10px;line-height:1.55;">

<h3 style="color:#FF4D4D;margin-top:0;">Kakutani Fixed-Point Theorem — Radiation Finite-Data Construction</h3>

<p>
Kakutani's theorem is a <b>set-valued</b> fixed-point theorem.
For a nonempty compact convex set <b>K</b> and an upper-hemicontinuous correspondence
<b>F: K ⇉ K</b> with nonempty compact convex values, at least one point
<b>z*</b> satisfies <b>z* ∈ F(z*)</b>.
</p>

<p>
The single-valued statement that a continuous map <b>g: K → K</b> has a fixed point is Brouwer's theorem.
A single-valued map is also a special case of Kakutani through the singleton correspondence
<b>F(z) = {g(z)}</b>.
The construction explicitly forms a set-valued enlargement
<b>F(z) = K ∩ B̄(g(z), ε)</b>.
</p>

<h3 style="color:#FF4D4D;">Radiation State Construction</h3>

<p>
The processed <b>N003 / ISIS RB2000164 transmission curves</b> are first placed on one common uniform
<b>s = ln(E/E<sub>min</sub>)</b> grid.
At every log-energy coordinate, the vector of transmission responses across measured samples defines one multivariate radiation state.
Each sample-response channel is centered and standardized, and singular-value decomposition produces a two-dimensional PCA state plane.
</p>

<p>
The ordered state sequence is therefore
<b>z(s) = [PC1(s), PC2(s)]</b>.
A one-step affine model is fitted:
<b>z(s+Δs) ≈ A z(s) + b</b>.
The compact convex domain is the observed PCA bounding box
<b>K = [ℓ<sub>1</sub>,u<sub>1</sub>] × [ℓ<sub>2</sub>,u<sub>2</sub>]</b>,
and the clipped map is
<b>g(z) = Π<sub>K</sub>(Az+b)</b>.
</p>

<p>
The correspondence radius <b>ε</b> is the median one-step residual norm:
<b>ε = median ||z<sub>i+1</sub> − g(z<sub>i</sub>)||</b>.
The numerical search minimizes <b>||z − g(z)||</b> over <b>K</b>.
A candidate satisfies the constructed Kakutani inclusion whenever
<b>||z* − g(z*)|| ≤ ε</b>.
</p>

</div>

<div style="font-size:13px;font-family:'Times New Roman',Times,serif;background-color:#202020;color:#D8D8D8;padding:14px 16px;border-radius:8px;margin:12px 10px;line-height:1.6;border-left:3px solid #8B0000;">

<p><b>Ordered-coordinate interpretation.</b>
The one-step model advances along log-energy, not physical time.
A fixed point of the fitted map is therefore a fixed state of the <b>radiation-derived ordered-coordinate model</b>.
It is not evidence of a stationary temporal state of the shielding material.</p>

<p><b>Theorem versus data fit.</b>
The clipped affine map is continuous and maps the compact convex box into itself, so Brouwer already guarantees a fixed point of <b>g</b>.
The Kakutani correspondence is a valid set-valued enlargement that incorporates the empirical one-step residual scale.
Neither theorem guarantees that the affine model is a high-fidelity description of neutron transport; model-fit diagnostics must be examined separately.</p>

<p><b>Numerical diagnostics.</b>
The cell therefore saves PCA explained variance, affine one-step RMSE/R², the residual distribution, distance from the numerical fixed point to the observed PCA trajectory, the correspondence radius <b>ε</b>, the fixed-point residual, and the inclusion margin <b>ε − ||z*−g(z*)||</b>.</p>

<p><b>Output convention.</b>
All result tables and plots are written directly into <b>results/phase2/</b>.
Every plot uses the Cell-28 result stem followed by <b>_plots_</b>.</p>

</div>

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy import signal
from scipy.optimize import least_squares

try:
    from IPython.display import display
except Exception:
    display = print

# -------------------------------------------------------
# SPEED MODE — preserve cell convention
# -------------------------------------------------------
FAST_OPTION = globals().get("FAST_OPTION", "ultra")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError(
        "FAST_OPTION must be one of: full, balanced, fast, ultra"
    )

_KAK_SPEED = {
    "full": 1,
    "balanced": 2,
    "fast": 4,
    "ultra": 8,
}[FAST_OPTION]

# -------------------------------------------------------
# PROJECT REPOSITORY / FLAT RESULTS DIRECTORY
# -------------------------------------------------------
PROJECT_ROOT = _resolve_project_root_portable()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Radiation-research repository not found at:\n"
        f"  {PROJECT_ROOT}"
    )

N003_SOURCE_CANDIDATES = [
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.csv.gz",
    PROJECT_ROOT
    / "data/processed/RB2000164/rb2000164_physics_ready_strict.parquet",
]

N003_PATH = next(
    (p for p in N003_SOURCE_CANDIDATES if p.exists()),
    None,
)

if N003_PATH is None:
    raise FileNotFoundError(
        "Processed RB2000164 dataset not found.\nExpected one of:\n"
        + "\n".join(f"  {p}" for p in N003_SOURCE_CANDIDATES)
    )

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results/phase2/N003"
os.makedirs(PHASE2_RESULTS_DIR, exist_ok=True)

CELL28_RESULT_STEM = "cell_28_kakutani_fixed_point_results"

results_dir = str(PHASE2_RESULTS_DIR)
plots_dir = str(PHASE2_RESULTS_DIR)

MAX_UNIFORM_POINTS = 5000
MIN_UNIFORM_POINTS = 512

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
BLACK = "#000000"
ACCENT = RED
BG = BLACK
eps_num = 1e-12

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": RED_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
})


def style_ax(ax):
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)
    ax.tick_params(colors=ACCENT)
    ax.xaxis.label.set_color(ACCENT)
    ax.yaxis.label.set_color(ACCENT)
    ax.title.set_color(ACCENT)
    ax.grid(True, color=RED_DIM, alpha=0.22)


def save_show_close(fig, path):
    fig.savefig(
        path,
        dpi=220,
        bbox_inches="tight",
        facecolor=BG,
    )
    plt.show()
    plt.close(fig)


def _find_radiation_column(
    df,
    exact=(),
    contains=(),
    exclude=(),
):
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in exact:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    matches = []

    for c in df.columns:
        lc = str(c).lower()

        if (
            any(token.lower() in lc for token in contains)
            and not any(token.lower() in lc for token in exclude)
        ):
            matches.append(c)

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda c: (len(str(c)), str(c)),
    )[0]


def load_rb2000164_transmission_matrix(source_path):
    """
    Build one common-log-energy transmission curve for each measured N003 sample.

    Matrix shape:
      coordinate points x measured samples
    """
    source_path = Path(source_path)

    if source_path.name.endswith(".csv.gz") or source_path.suffix.lower() == ".csv":
        df = pd.read_csv(source_path)

    elif source_path.suffix.lower() == ".parquet":
        try:
            df = pd.read_parquet(source_path)
        except ImportError as exc:
            csv_fallback = (
                source_path
                .with_suffix("")
                .with_suffix(".csv.gz")
            )

            if csv_fallback.exists():
                source_path = csv_fallback
                df = pd.read_csv(source_path)
            else:
                raise ImportError(
                    "Parquet support unavailable and no CSV fallback found."
                ) from exc

    else:
        raise ValueError(
            f"Unsupported RB2000164 input format: {source_path}"
        )

    sample_col = _find_radiation_column(
        df,
        exact=("sample_id", "sample", "sample_name"),
        contains=("sample",),
        exclude=("uncert",),
    )

    energy_col = _find_radiation_column(
        df,
        exact=(
            "energy_eV",
            "energy_ev",
            "neutron_energy_eV",
            "energy_in_eV",
        ),
        contains=("energy",),
        exclude=("uncert", "lower", "upper"),
    )

    transmission_col = _find_radiation_column(
        df,
        exact=("transmission",),
        contains=("transmission",),
        exclude=("uncert", "sigma", "error"),
    )

    if sample_col is None:
        raise KeyError("Could not identify N003 sample column.")

    if energy_col is None:
        raise KeyError("Could not identify N003 neutron-energy column.")

    if transmission_col is None:
        raise KeyError("Could not identify N003 transmission column.")

    curves = {}

    for sample_id, part in df.groupby(
        sample_col,
        sort=True,
        dropna=False,
    ):
        q = part[
            [energy_col, transmission_col]
        ].copy()

        q[energy_col] = pd.to_numeric(
            q[energy_col],
            errors="coerce",
        )

        q[transmission_col] = pd.to_numeric(
            q[transmission_col],
            errors="coerce",
        )

        q = (
            q.replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        q = q[
            (q[energy_col] > 0)
            & (q[transmission_col] > 0)
        ]

        q = (
            q.groupby(
                energy_col,
                as_index=False,
                sort=True,
            )[transmission_col]
            .mean()
            .sort_values(energy_col)
        )

        if len(q) < MIN_UNIFORM_POINTS:
            print(
                f"Skipping sample {sample_id}: "
                f"only {len(q)} usable transmission points.",
                flush=True,
            )
            continue

        curves[str(sample_id)] = (
            np.log(
                q[energy_col].to_numpy(dtype=float)
            ),
            q[transmission_col].to_numpy(dtype=float),
        )

    if len(curves) < 3:
        raise RuntimeError(
            "Need at least three sufficiently sampled N003 transmission curves."
        )

    common_lo = max(
        logE.min()
        for logE, _ in curves.values()
    )

    common_hi = min(
        logE.max()
        for logE, _ in curves.values()
    )

    if (
        not np.isfinite(common_lo)
        or not np.isfinite(common_hi)
        or common_hi <= common_lo
    ):
        raise RuntimeError(
            "No common log-energy interval across retained N003 samples."
        )

    n_grid = min(
        MAX_UNIFORM_POINTS,
        min(
            len(logE)
            for logE, _ in curves.values()
        ),
    )

    if n_grid < MIN_UNIFORM_POINTS:
        raise RuntimeError(
            f"Common radiation grid has only {n_grid} points."
        )

    logE_grid = np.linspace(
        common_lo,
        common_hi,
        n_grid,
    )

    labels = []
    columns = []

    for sample_id in sorted(curves):
        logE, transmission = curves[sample_id]

        interp_T = np.interp(
            logE_grid,
            logE,
            transmission,
        )

        labels.append(sample_id)
        columns.append(interp_T)

    matrix = np.column_stack(columns)

    s_grid = (
        logE_grid
        - logE_grid[0]
    )

    ds = float(
        s_grid[1]
        - s_grid[0]
    )

    return (
        matrix,
        labels,
        s_grid,
        ds,
        df,
        sample_col,
        energy_col,
        transmission_col,
        source_path,
    )


# -------------------------------------------------------
# LOAD REAL RADIATION STATE MATRIX
# -------------------------------------------------------
(
    _KX_raw,
    radiation_samples,
    s_grid,
    coordinate_step,
    radiation_df,
    sample_col,
    energy_col,
    transmission_col,
    actual_source_path,
) = load_rb2000164_transmission_matrix(
    N003_PATH
)

# Subsample ordered states for computational efficiency.
_KX = np.asarray(
    _KX_raw[::_KAK_SPEED],
    dtype=float,
)

s_used = s_grid[::_KAK_SPEED]

if len(s_used) != len(_KX):
    raise RuntimeError(
        "Radiation coordinate and state matrix became misaligned."
    )

coordinate_step_used = float(
    coordinate_step
    * _KAK_SPEED
)

# Preserve the cell preprocessing:
# constant detrending and per-channel z-standardization.
_KX = signal.detrend(
    _KX,
    axis=0,
    type="constant",
)

_channel_std = _KX.std(
    axis=0,
    keepdims=True,
)

if np.any(
    ~np.isfinite(_channel_std)
) or np.any(
    _channel_std <= 0
):
    raise RuntimeError(
        "At least one radiation transmission channel is degenerate."
    )

_KX = (
    _KX
    - _KX.mean(
        axis=0,
        keepdims=True,
    )
) / (
    _channel_std
    + 1e-12
)

print(
    f"Using processed radiation source: {public_project_path(actual_source_path)}"
)

print(
    f"Radiation state matrix after speed subsampling: "
    f"{_KX.shape[0]} coordinate states x {_KX.shape[1]} measured samples"
)

print(
    f"Measured transmission samples: {radiation_samples}"
)

print(
    f"Ordered-coordinate step used: {coordinate_step_used:.8g}"
)

# -------------------------------------------------------
# PCA STATE PLANE — SVD construction
# -------------------------------------------------------
_pca_input = _KX[
    :,
    :min(16, _KX.shape[1])
]

_U, _s, _Vt = np.linalg.svd(
    _pca_input,
    full_matrices=False,
)

if len(_s) < 2:
    raise RuntimeError(
        "Radiation state matrix does not support a two-dimensional PCA plane."
    )

_Z = (
    _U[:, :2]
    * _s[:2]
)

_pca_variance = (
    _s ** 2
)

_pca_explained = (
    _pca_variance
    / (
        np.sum(_pca_variance)
        + eps_num
    )
)

_Z0 = _Z[:-1]
_Z1 = _Z[1:]

# -------------------------------------------------------
# AFFINE ONE-STEP MODEL
# -------------------------------------------------------
_B = np.linalg.lstsq(
    np.c_[
        _Z0,
        np.ones(
            len(_Z0)
        ),
    ],
    _Z1,
    rcond=None,
)[0]

_A = _B[:2].T
_bvec = _B[2]

_lo = _Z.min(axis=0)
_hi = _Z.max(axis=0)

# Cell-28 map using the radiation-derived
# PCA state plane.
def _kak_g(z):
    return np.clip(
        _A @ np.asarray(z)
        + _bvec,
        _lo,
        _hi,
    )


_Z1_pred = np.array(
    [
        _kak_g(z)
        for z in _Z0
    ]
)

_resid_vector = (
    _Z1
    - _Z1_pred
)

_resid = np.linalg.norm(
    _resid_vector,
    axis=1,
)

# Preserve cell correspondence radius: median one-step residual.
_eps = float(
    np.quantile(
        _resid,
        0.5,
    )
)

# One-step fit diagnostics.
_rmse_pc1 = float(
    np.sqrt(
        np.mean(
            (
                _Z1[:, 0]
                - _Z1_pred[:, 0]
            ) ** 2
        )
    )
)

_rmse_pc2 = float(
    np.sqrt(
        np.mean(
            (
                _Z1[:, 1]
                - _Z1_pred[:, 1]
            ) ** 2
        )
    )
)

_rmse_2d = float(
    np.sqrt(
        np.mean(
            np.sum(
                _resid_vector ** 2,
                axis=1,
            )
        )
    )
)


def _r2_score(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)

    ss_res = float(
        np.sum(
            (
                y - yhat
            ) ** 2
        )
    )

    ss_tot = float(
        np.sum(
            (
                y - np.mean(y)
            ) ** 2
        )
    )

    if ss_tot <= 0:
        return np.nan

    return float(
        1.0
        - ss_res / ss_tot
    )


_r2_pc1 = _r2_score(
    _Z1[:, 0],
    _Z1_pred[:, 0],
)

_r2_pc2 = _r2_score(
    _Z1[:, 1],
    _Z1_pred[:, 1],
)

_r2_joint = _r2_score(
    _Z1.reshape(-1),
    _Z1_pred.reshape(-1),
)

# -------------------------------------------------------
# NUMERICAL FIXED-POINT SEARCH
# -------------------------------------------------------
_seed_count = {
    "full": 12,
    "balanced": 8,
    "fast": 5,
    "ultra": 3,
}[FAST_OPTION]

_seed_indices = np.linspace(
    0,
    len(_Z) - 1,
    _seed_count,
).astype(int)

_seeds = np.vstack([
    _Z.mean(axis=0),
    _Z[_seed_indices],
])

_best = None

for _seed in _seeds:
    _fit = least_squares(
        lambda z: z - _kak_g(z),
        _seed,
        bounds=(_lo, _hi),
        max_nfev={
            "full": 1000,
            "balanced": 600,
            "fast": 350,
            "ultra": 180,
        }[FAST_OPTION],
    )

    _r = float(
        np.linalg.norm(
            _fit.x
            - _kak_g(_fit.x)
        )
    )

    if (
        _best is None
        or _r < _best[0]
    ):
        _best = (
            _r,
            _fit.x,
            _fit,
        )

_r, _zstar, _best_fit = _best

_g_zstar = _kak_g(
    _zstar
)

_inside_F = bool(
    _r
    <= _eps
    + 1e-10
)

_inclusion_margin = float(
    _eps - _r
)

_inside_K = bool(
    np.all(
        _zstar >= _lo - 1e-10
    )
    and np.all(
        _zstar <= _hi + 1e-10
    )
)

_nearest_observed_distance = float(
    np.min(
        np.linalg.norm(
            _Z - _zstar,
            axis=1,
        )
    )
)

# Whether clipping is active at the candidate.
_unclipped_g_zstar = (
    _A @ _zstar
    + _bvec
)

_clipping_active = bool(
    np.any(
        np.abs(
            _unclipped_g_zstar
            - _g_zstar
        )
        > 1e-10
    )
)

print(
    "Kakutani correspondence fixed-inclusion residual:",
    _r,
    "epsilon:",
    _eps,
    "inside F:",
    _inside_F,
)

print(
    "Inside compact convex box K:",
    _inside_K,
)

print(
    "Inclusion margin epsilon - residual:",
    _inclusion_margin,
)

print(
    "Nearest observed PCA-state distance to z_star:",
    _nearest_observed_distance,
)

print(
    "Affine one-step R2:",
    _r2_joint,
)

# -------------------------------------------------------
# RESULT TABLES
# -------------------------------------------------------
coordinate_df = pd.DataFrame({
    "coordinate": ["PC1", "PC2"],
    "z_star": _zstar,
    "g_z_star": _g_zstar,
    "unclipped_Az_plus_b": _unclipped_g_zstar,
    "lower_K": _lo,
    "upper_K": _hi,
})

display(
    coordinate_df
)

summary_df = pd.DataFrame([
    {
        "source_file": public_project_path(actual_source_path),
        "speed_mode": FAST_OPTION,
        "n_coordinate_states": int(_KX.shape[0]),
        "n_measured_samples": int(_KX.shape[1]),
        "coordinate_step_used": coordinate_step_used,
        "pca_pc1_explained_fraction": float(_pca_explained[0]),
        "pca_pc2_explained_fraction": float(_pca_explained[1]),
        "pca_first_two_explained_fraction": float(
            _pca_explained[0]
            + _pca_explained[1]
        ),
        "affine_rmse_pc1": _rmse_pc1,
        "affine_rmse_pc2": _rmse_pc2,
        "affine_rmse_2d": _rmse_2d,
        "affine_r2_pc1": _r2_pc1,
        "affine_r2_pc2": _r2_pc2,
        "affine_r2_joint": _r2_joint,
        "epsilon_median_one_step_residual": _eps,
        "fixed_inclusion_residual": _r,
        "inside_F": _inside_F,
        "inside_K": _inside_K,
        "inclusion_margin": _inclusion_margin,
        "nearest_observed_state_distance": _nearest_observed_distance,
        "clipping_active_at_z_star": _clipping_active,
        "optimizer_success": bool(_best_fit.success),
        "optimizer_nfev": int(_best_fit.nfev),
    }
])

model_df = pd.DataFrame(
    [
        {
            "matrix_element": "A11",
            "value": float(_A[0, 0]),
        },
        {
            "matrix_element": "A12",
            "value": float(_A[0, 1]),
        },
        {
            "matrix_element": "A21",
            "value": float(_A[1, 0]),
        },
        {
            "matrix_element": "A22",
            "value": float(_A[1, 1]),
        },
        {
            "matrix_element": "b1",
            "value": float(_bvec[0]),
        },
        {
            "matrix_element": "b2",
            "value": float(_bvec[1]),
        },
    ]
)

# -------------------------------------------------------
# SAVE NUMERICAL RESULTS
# -------------------------------------------------------
result_csv = os.path.join(
    results_dir,
    f"{CELL28_RESULT_STEM}.csv",
)

coordinate_csv = os.path.join(
    results_dir,
    f"{CELL28_RESULT_STEM}_coordinates.csv",
)

model_csv = os.path.join(
    results_dir,
    f"{CELL28_RESULT_STEM}_affine_model.csv",
)

summary_df.to_csv(
    result_csv,
    index=False,
)

coordinate_df.to_csv(
    coordinate_csv,
    index=False,
)

model_df.to_csv(
    model_csv,
    index=False,
)

npz_path = os.path.join(
    results_dir,
    f"{CELL28_RESULT_STEM}.npz",
)

np.savez_compressed(
    npz_path,
    radiation_samples=np.array(
        radiation_samples,
        dtype=object,
    ),
    s_grid=s_grid,
    s_used=s_used,
    KX_standardized=_KX,
    Z=_Z,
    Z0=_Z0,
    Z1=_Z1,
    Z1_pred=_Z1_pred,
    residual_norms=_resid,
    A=_A,
    b=_bvec,
    lower_K=_lo,
    upper_K=_hi,
    epsilon=_eps,
    z_star=_zstar,
    g_z_star=_g_zstar,
    fixed_inclusion_residual=_r,
    source_file=public_project_path(actual_source_path),
    speed_mode=FAST_OPTION,
)

summary_txt = os.path.join(
    results_dir,
    f"{CELL28_RESULT_STEM}.txt",
)

with open(
    summary_txt,
    "w",
) as f:
    f.write(
        "Kakutani / Brouwer Fixed-Point Construction "
        "from Processed Radiation Data\n"
    )

    f.write(
        "============================================================\n\n"
    )

    f.write(
        "Interpretation:\n"
        "  Processed N003 transmission responses define a multivariate "
        "state vector along uniform log-energy.\n"
        "  A two-dimensional PCA state plane is fitted with a one-step "
        "affine ordered-coordinate model.\n"
        "  The clipped map g(z)=Pi_K(Az+b) is continuous and maps the "
        "compact convex box K into itself.\n"
        "  Brouwer therefore guarantees a fixed point of g.\n"
        "  The set-valued correspondence F(z)=K intersect closed_ball(g(z),epsilon) "
        "is the Kakutani construction used here.\n"
        "  The numerical candidate satisfies the fixed-inclusion condition "
        "when ||z*-g(z*)|| <= epsilon.\n"
        "  This is an ordered-coordinate reduced-model result, not a claim "
        "of a temporal fixed state in the shielding material.\n\n"
    )

    f.write(
        f"Source: {public_project_path(actual_source_path)}\n"
    )

    f.write(
        "Radiation object: N003.SEQ_SAMPLE / N003.TRAJ_E\n"
    )

    f.write(
        "Observable: transmission T(E)\n"
    )

    f.write(
        f"Measured samples: {len(radiation_samples)}\n"
    )

    f.write(
        f"Coordinate states used: {_KX.shape[0]}\n"
    )

    f.write(
        f"Coordinate step used: {coordinate_step_used}\n"
    )

    f.write(
        f"PC1 explained fraction: {_pca_explained[0]:.10f}\n"
    )

    f.write(
        f"PC2 explained fraction: {_pca_explained[1]:.10f}\n"
    )

    f.write(
        f"PC1+PC2 explained fraction: "
        f"{(_pca_explained[0]+_pca_explained[1]):.10f}\n"
    )

    f.write(
        f"Affine one-step joint R2: {_r2_joint:.10f}\n"
    )

    f.write(
        f"Affine one-step 2D RMSE: {_rmse_2d:.10f}\n"
    )

    f.write(
        f"Epsilon, median one-step residual: {_eps:.10f}\n"
    )

    f.write(
        f"Fixed-inclusion residual: {_r:.10e}\n"
    )

    f.write(
        f"Inside F: {_inside_F}\n"
    )

    f.write(
        f"Inside K: {_inside_K}\n"
    )

    f.write(
        f"Inclusion margin: {_inclusion_margin:.10f}\n"
    )

    f.write(
        f"Nearest observed state distance: "
        f"{_nearest_observed_distance:.10f}\n"
    )

    f.write(
        f"Clipping active at z_star: {_clipping_active}\n"
    )

# -------------------------------------------------------
# PLOT 1: PCA STATE CLOUD + FIXED-INCLUSION BALL
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8, 7),
    facecolor=BG,
)

style_ax(
    ax
)

plot_step = max(
    1,
    len(_Z) // 1800,
)

ax.scatter(
    _Z[::plot_step, 0],
    _Z[::plot_step, 1],
    s=8,
    color=RED,
    edgecolors=WHITE,
    linewidths=0.15,
    alpha=0.28,
    label="Radiation PCA states",
)

ax.scatter(
    [_zstar[0]],
    [_zstar[1]],
    s=130,
    marker="x",
    color=WHITE,
    linewidths=2.2,
    label="Numerical z*",
    zorder=5,
)

ax.scatter(
    [_g_zstar[0]],
    [_g_zstar[1]],
    s=55,
    marker="o",
    facecolors="none",
    edgecolors=RED_SOFT,
    linewidths=1.5,
    label="g(z*)",
    zorder=5,
)

# Visualize the correspondence ball in the 2D PCA plane.
ball = Circle(
    _g_zstar,
    radius=_eps,
    fill=False,
    edgecolor=RED_SOFT,
    linestyle="--",
    linewidth=1.2,
    alpha=0.75,
)

ax.add_patch(
    ball
)

ax.set_xlim(
    _lo[0],
    _hi[0],
)

ax.set_ylim(
    _lo[1],
    _hi[1],
)

ax.set_title(
    "Radiation-Derived Kakutani Fixed-Point Inclusion — Reference/Training Construction"
)

ax.set_xlabel(
    "PC1"
)

ax.set_ylabel(
    "PC2"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot1_path = os.path.join(
    plots_dir,
    f"{CELL28_RESULT_STEM}_plots_01_pca_fixed_point_inclusion.png",
)

save_show_close(
    fig,
    plot1_path,
)

# -------------------------------------------------------
# PLOT 2: ONE-STEP RESIDUAL DISTRIBUTION
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.hist(
    _resid,
    bins=50,
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.82,
)

ax.axvline(
    _eps,
    color=WHITE,
    linestyle="--",
    linewidth=1.3,
    label=f"Median residual epsilon = {_eps:.4g}",
)

ax.set_title(
    "Affine One-Step Residual Distribution in PCA Space — Training/Reference Fit"
)

ax.set_xlabel(
    "||z_next - g(z)||"
)

ax.set_ylabel(
    "Count"
)

leg = ax.legend(
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot2_path = os.path.join(
    plots_dir,
    f"{CELL28_RESULT_STEM}_plots_02_one_step_residual_distribution.png",
)

save_show_close(
    fig,
    plot2_path,
)

# -------------------------------------------------------
# PLOT 3: OBSERVED VS PREDICTED PCA COORDINATES
# -------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.scatter(
    _Z1[:, 0],
    _Z1_pred[:, 0],
    s=14,
    color=RED,
    alpha=0.35,
    edgecolors=WHITE,
    linewidths=0.15,
    label=f"PC1, R²={_r2_pc1:.3f}",
)

ax.scatter(
    _Z1[:, 1],
    _Z1_pred[:, 1],
    s=14,
    color=RED_SOFT,
    alpha=0.35,
    edgecolors=WHITE,
    linewidths=0.15,
    label=f"PC2, R²={_r2_pc2:.3f}",
)

all_obs = np.concatenate([
    _Z1[:, 0],
    _Z1[:, 1],
    _Z1_pred[:, 0],
    _Z1_pred[:, 1],
])

diag_lo = float(
    np.nanmin(
        all_obs
    )
)

diag_hi = float(
    np.nanmax(
        all_obs
    )
)

ax.plot(
    [diag_lo, diag_hi],
    [diag_lo, diag_hi],
    color=WHITE,
    linestyle="--",
    linewidth=1.0,
    alpha=0.75,
)

ax.set_title(
    "Observed vs Predicted One-Step PCA Coordinates — Training/Reference Fit"
)

ax.set_xlabel(
    "Observed next coordinate"
)

ax.set_ylabel(
    "Predicted next coordinate"
)

leg = ax.legend(
    loc="best",
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot3_path = os.path.join(
    plots_dir,
    f"{CELL28_RESULT_STEM}_plots_03_observed_vs_predicted_pca.png",
)

save_show_close(
    fig,
    plot3_path,
)

# -------------------------------------------------------
# PLOT 4: CLIPPED AFFINE VECTOR FIELD IN K
# -------------------------------------------------------
grid_n = {
    "full": 28,
    "balanced": 24,
    "fast": 20,
    "ultra": 16,
}[FAST_OPTION]

gx = np.linspace(
    _lo[0],
    _hi[0],
    grid_n,
)

gy = np.linspace(
    _lo[1],
    _hi[1],
    grid_n,
)

GX, GY = np.meshgrid(
    gx,
    gy,
)

points = np.column_stack([
    GX.ravel(),
    GY.ravel(),
])

mapped = np.array([
    _kak_g(z)
    for z in points
])

delta = (
    mapped
    - points
)

fig, ax = plt.subplots(
    figsize=(8, 7),
    facecolor=BG,
)

style_ax(
    ax
)

ax.quiver(
    points[:, 0],
    points[:, 1],
    delta[:, 0],
    delta[:, 1],
    color=RED,
    alpha=0.55,
    angles="xy",
    scale_units="xy",
    scale=1,
    width=0.003,
)

ax.scatter(
    [_zstar[0]],
    [_zstar[1]],
    s=130,
    marker="x",
    color=WHITE,
    linewidths=2.2,
    label="Numerical z*",
    zorder=5,
)

ax.set_xlim(
    _lo[0],
    _hi[0],
)

ax.set_ylim(
    _lo[1],
    _hi[1],
)

ax.set_title(
    "Clipped Affine Map Displacement Field: g(z) - z — Reference Construction"
)

ax.set_xlabel(
    "PC1"
)

ax.set_ylabel(
    "PC2"
)

leg = ax.legend(
    frameon=True,
)

for text in leg.get_texts():
    text.set_color(
        ACCENT
    )

plot4_path = os.path.join(
    plots_dir,
    f"{CELL28_RESULT_STEM}_plots_04_clipped_affine_displacement_field.png",
)

save_show_close(
    fig,
    plot4_path,
)

# -------------------------------------------------------
# PLOT 5: PCA EXPLAINED VARIANCE
# -------------------------------------------------------
n_components_plot = min(
    9,
    len(_pca_explained),
)

fig, ax = plt.subplots(
    figsize=(9, 5),
    facecolor=BG,
)

style_ax(
    ax
)

ax.bar(
    np.arange(
        1,
        n_components_plot + 1,
    ),
    _pca_explained[
        :n_components_plot
    ],
    color=RED,
    edgecolor=RED_SOFT,
    alpha=0.85,
)

ax.set_title(
    "Radiation-State PCA Explained Variance"
)

ax.set_xlabel(
    "Principal component"
)

ax.set_ylabel(
    "Explained variance fraction"
)

plot5_path = os.path.join(
    plots_dir,
    f"{CELL28_RESULT_STEM}_plots_05_pca_explained_variance.png",
)

save_show_close(
    fig,
    plot5_path,
)

print(
    f"\nSaved primary results CSV: {result_csv}"
)

print(
    f"Saved coordinate CSV: {coordinate_csv}"
)

print(
    f"Saved affine model CSV: {model_csv}"
)

print(
    f"Saved NPZ: {npz_path}"
)

print(
    f"Saved summary TXT: {summary_txt}"
)

print(
    f"Saved plots directly to: {PHASE2_RESULTS_DIR}"
)


# Cross-Dataset Dynamical-Systems Atlas

<div style="font-size:14px;font-family:'Times New Roman',Times,serif;background-color:#181818;color:#D0D0D0;padding:20px;border-radius:8px;margin:10px;line-height:1.6;">
<h2 style="color:#FF4D4D;">Scientific execution rule</h2>
<p>Each mathematical cell is attempted only on Object-Bank-approved datasets and then passes a second structural gate. The gate is specific to the calculation: trajectory length, effective embedding support, number of comparable phase-bearing curves, common coordinate support, non-degeneracy, multichannel state dimension, and reader availability are checked before result files are created.</p>

<h2 style="color:#FF4D4D;">Coordinate semantics</h2>
<p>Positive energy coordinates are transformed with one dataset-level reference, <b>s = ln(E/E<sub>ref</sub>)</b>. Results therefore store both the native coordinate and the analysis coordinate, transformation, reference value, and unit. Depth, MFP, beam-voltage, and position coordinates remain in their native ordered coordinate unless a method explicitly states otherwise.</p>

<h2 style="color:#FF4D4D;">Quality protection</h2>
<p>Every numerical feature has a quality status and an <code>eligible_for_model</code> flag. Method-specific descriptors are retained even when they are not suitable predictive features. Raw circle-map locking, fixed-point inclusion residuals, unstable finite-time pushforwards, and nearest-neighbor entropies dominated by adjacent coordinate points are automatically marked diagnostic-only.</p>

<h2 style="color:#FF4D4D;">Output layout</h2>
<pre style="color:#FF6B6B;background:#000;padding:12px;border-radius:6px;">results/phase2/
    phase2_applicability_results.csv
    phase2_feature_registry.csv
    phase2_cross_dataset_summary.csv
    N003/              # validated reference Cells 0–28
    P001/
    N001/
    ...
</pre>
<p>There are no per-cell subdirectories. Failed analyses do not create empty or malformed CSV files; their status and reason are stored in the global applicability ledger.</p>
</div>

In [ ]:
import os
import re
import sys
import math
import json
import shutil
import warnings
import tempfile
import subprocess
import importlib
import importlib.util
from pathlib import Path
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree
from scipy.signal import hilbert, savgol_filter, detrend
from scipy.optimize import least_squares
from sklearn.feature_selection import mutual_info_regression

# ============================================================
# CROSS-DATASET PHASE-II INFRASTRUCTURE
# ============================================================
PROJECT_ROOT = _resolve_project_root_portable()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project repository not found: {PROJECT_ROOT}")

PHASE2_RESULTS_ROOT = PROJECT_ROOT / "results/phase2"
PHASE2_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_DATASETS = ['P001', 'N001', 'N002', 'P002', 'P003', 'C001', 'PN001', 'P004', 'N003', 'N004', 'E001', 'PN002', 'P005_SIM', 'PN003', 'PVAL_6MV', 'PVAL_10MV', 'PVAL_15MV', 'PVAL_16MV', 'PVAL_18MV', 'NVAL_MATHEW', 'NVAL_ZAMORANO']
REFERENCE_DATASET = "N003"
ATLAS_DATASETS = [d for d in PRIMARY_DATASETS if d != REFERENCE_DATASET]
ATLAS_CELL_TO_CODE = {4: 'CODE_0001', 6: 'CODE_0002', 9: 'CODE_0003', 13: 'CODE_0004', 17: 'CODE_0005', 19: 'CODE_0006', 21: 'CODE_0007', 25: 'CODE_0008', 28: 'CODE_0009'}
ATLAS_APPLICABILITY = {4: {'P001': {'status': 'ADAPTED', 'object_id': 'P001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N001': {'status': 'ADAPTED', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N002': {'status': 'ADAPTED', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P002': {'status': 'CONDITIONAL', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SPARSE_SEQUENCE', 'note': 'May be underpowered; predeclare estimator minimum length and retain null/undefined result if unmet.'}, 'P003': {'status': 'ADAPTED', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'C001': {'status': 'ADAPTED', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN001': {'status': 'ADAPTED', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P004': {'status': 'CONDITIONAL', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'LIMITED_PER_CURVE_LENGTH', 'note': 'Use the full 35-condition field/point cloud where valid; per-material five-point sequences are too short for high-order estimators.'}, 'N003': {'status': 'ADAPTED', 'object_id': 'N003.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N004': {'status': 'ADAPTED', 'object_id': 'N004.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Do not derive Sigma(E) or any thickness-based quantity from ungrounded sample path lengths. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'E001': {'status': 'ADAPTED', 'object_id': 'E001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN002': {'status': 'ADAPTED', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Keep reaction-cross-section semantics separate from photon attenuation. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P005_SIM': {'status': 'ADAPTED', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Result retains SIMULATED evidence status. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN003': {'status': 'ADAPTED', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_6MV': {'status': 'ADAPTED', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_10MV': {'status': 'ADAPTED', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_15MV': {'status': 'ADAPTED', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_16MV': {'status': 'ADAPTED', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_18MV': {'status': 'ADAPTED', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_MATHEW': {'status': 'ADAPTED', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_ZAMORANO': {'status': None, 'object_id': None, 'reason_code': 'INSUFFICIENT_SAMPLE_COUNT', 'note': 'Five positions per machine are insufficient for this estimator without inventing pseudo-samples.'}}, 6: {'P001': {'status': 'ADAPTED', 'object_id': 'P001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N001': {'status': 'ADAPTED', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N002': {'status': 'ADAPTED', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P002': {'status': 'CONDITIONAL', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SPARSE_SEQUENCE', 'note': 'May be underpowered; predeclare estimator minimum length and retain null/undefined result if unmet.'}, 'P003': {'status': 'ADAPTED', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'C001': {'status': 'ADAPTED', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN001': {'status': 'ADAPTED', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P004': {'status': 'CONDITIONAL', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'LIMITED_PER_CURVE_LENGTH', 'note': 'Use the full 35-condition field/point cloud where valid; per-material five-point sequences are too short for high-order estimators.'}, 'N003': {'status': 'ADAPTED', 'object_id': 'N003.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N004': {'status': 'ADAPTED', 'object_id': 'N004.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Do not derive Sigma(E) or any thickness-based quantity from ungrounded sample path lengths. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'E001': {'status': 'ADAPTED', 'object_id': 'E001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN002': {'status': 'ADAPTED', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Keep reaction-cross-section semantics separate from photon attenuation. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P005_SIM': {'status': 'ADAPTED', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Result retains SIMULATED evidence status. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN003': {'status': 'ADAPTED', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_6MV': {'status': 'ADAPTED', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_10MV': {'status': 'ADAPTED', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_15MV': {'status': 'ADAPTED', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_16MV': {'status': 'ADAPTED', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_18MV': {'status': 'ADAPTED', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_MATHEW': {'status': 'ADAPTED', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_ZAMORANO': {'status': None, 'object_id': None, 'reason_code': 'INSUFFICIENT_SAMPLE_COUNT', 'note': 'Five positions per machine are insufficient for this estimator without inventing pseudo-samples.'}}, 9: {'P001': {'status': 'ADAPTED', 'object_id': 'P001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N001': {'status': 'ADAPTED', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N002': {'status': 'ADAPTED', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P002': {'status': 'ADAPTED', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P003': {'status': 'ADAPTED', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'C001': {'status': 'ADAPTED', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN001': {'status': 'ADAPTED', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P004': {'status': 'ADAPTED', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N003': {'status': 'ADAPTED', 'object_id': 'N003.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N004': {'status': 'ADAPTED', 'object_id': 'N004.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Do not derive Sigma(E) or any thickness-based quantity from ungrounded sample path lengths. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'E001': {'status': 'ADAPTED', 'object_id': 'E001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN002': {'status': 'ADAPTED', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Keep reaction-cross-section semantics separate from photon attenuation. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P005_SIM': {'status': 'ADAPTED', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Result retains SIMULATED evidence status. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN003': {'status': 'ADAPTED', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_6MV': {'status': 'ADAPTED', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_10MV': {'status': 'ADAPTED', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_15MV': {'status': 'ADAPTED', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_16MV': {'status': 'ADAPTED', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_18MV': {'status': 'ADAPTED', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_MATHEW': {'status': 'ADAPTED', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_ZAMORANO': {'status': 'CONDITIONAL', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SPARSE_DEGENERATE_EXPECTED', 'note': 'Execute only if estimator minimum-sample rule is met; otherwise record a degenerate/N/A result.'}}, 13: {'P001': {'status': 'ADAPTED', 'object_id': 'P001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N001': {'status': 'ADAPTED', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N002': {'status': 'ADAPTED', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P002': {'status': 'ADAPTED', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P003': {'status': 'ADAPTED', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'C001': {'status': 'ADAPTED', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN001': {'status': 'ADAPTED', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P004': {'status': 'ADAPTED', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N003': {'status': 'ADAPTED', 'object_id': 'N003.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N004': {'status': 'ADAPTED', 'object_id': 'N004.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Do not derive Sigma(E) or any thickness-based quantity from ungrounded sample path lengths. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'E001': {'status': 'ADAPTED', 'object_id': 'E001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN002': {'status': 'ADAPTED', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Keep reaction-cross-section semantics separate from photon attenuation. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P005_SIM': {'status': 'ADAPTED', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Result retains SIMULATED evidence status. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN003': {'status': 'ADAPTED', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_6MV': {'status': 'ADAPTED', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_10MV': {'status': 'ADAPTED', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_15MV': {'status': 'ADAPTED', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_16MV': {'status': 'ADAPTED', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_18MV': {'status': 'ADAPTED', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_MATHEW': {'status': 'ADAPTED', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_ZAMORANO': {'status': 'CONDITIONAL', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SPARSE_DEGENERATE_EXPECTED', 'note': 'Execute only if estimator minimum-sample rule is met; otherwise record a degenerate/N/A result.'}}, 17: {'P001': {'status': 'ADAPTED', 'object_id': 'P001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N001': {'status': 'ADAPTED', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N002': {'status': 'ADAPTED', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P002': {'status': 'ADAPTED', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P003': {'status': 'ADAPTED', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'C001': {'status': 'ADAPTED', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN001': {'status': 'ADAPTED', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P004': {'status': 'ADAPTED', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N003': {'status': 'ADAPTED', 'object_id': 'N003.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'N004': {'status': 'ADAPTED', 'object_id': 'N004.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Do not derive Sigma(E) or any thickness-based quantity from ungrounded sample path lengths. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'E001': {'status': 'ADAPTED', 'object_id': 'E001.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN002': {'status': 'ADAPTED', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Keep reaction-cross-section semantics separate from photon attenuation. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'P005_SIM': {'status': 'ADAPTED', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Result retains SIMULATED evidence status. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PN003': {'status': 'ADAPTED', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_6MV': {'status': 'ADAPTED', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_10MV': {'status': 'ADAPTED', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_15MV': {'status': 'ADAPTED', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_16MV': {'status': 'ADAPTED', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'PVAL_18MV': {'status': 'ADAPTED', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Interpret only as PDD/depth-response mathematics; never as direct validation of the aggregate 1-D source-energy spectrum. Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_MATHEW': {'status': 'ADAPTED', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'TIME_TO_TRANSPORT_COORDINATE_REINTERPRETATION', 'note': 'Dynamics are with respect to the physical transport coordinate, not clock time.'}, 'NVAL_ZAMORANO': {'status': 'CONDITIONAL', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SPARSE_DEGENERATE_EXPECTED', 'note': 'Execute only if estimator minimum-sample rule is met; otherwise record a degenerate/N/A result.'}}, 19: {'P001': {'status': 'SUPPORT', 'object_id': 'P001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N001': {'status': 'SUPPORT', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N002': {'status': 'SUPPORT', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P002': {'status': 'SUPPORT', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P003': {'status': 'SUPPORT', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'C001': {'status': 'SUPPORT', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN001': {'status': 'SUPPORT', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P004': {'status': 'SUPPORT', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N003': {'status': 'SUPPORT', 'object_id': 'N003.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N004': {'status': 'SUPPORT', 'object_id': 'N004.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'E001': {'status': 'SUPPORT', 'object_id': 'E001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN002': {'status': 'SUPPORT', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P005_SIM': {'status': 'SUPPORT', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN003': {'status': 'SUPPORT', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_6MV': {'status': 'SUPPORT', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_10MV': {'status': 'SUPPORT', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_15MV': {'status': 'SUPPORT', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_16MV': {'status': 'SUPPORT', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_18MV': {'status': 'SUPPORT', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_MATHEW': {'status': 'SUPPORT', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_ZAMORANO': {'status': 'SUPPORT', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SUPPORTING_STEP', 'note': None}}, 21: {'P001': {'status': 'SUPPORT', 'object_id': 'P001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N001': {'status': 'SUPPORT', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N002': {'status': 'SUPPORT', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P002': {'status': 'SUPPORT', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P003': {'status': 'SUPPORT', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'C001': {'status': 'SUPPORT', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN001': {'status': 'SUPPORT', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P004': {'status': 'SUPPORT', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N003': {'status': 'SUPPORT', 'object_id': 'N003.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N004': {'status': 'SUPPORT', 'object_id': 'N004.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'E001': {'status': 'SUPPORT', 'object_id': 'E001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN002': {'status': 'SUPPORT', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P005_SIM': {'status': 'SUPPORT', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN003': {'status': 'SUPPORT', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_6MV': {'status': 'SUPPORT', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_10MV': {'status': 'SUPPORT', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_15MV': {'status': 'SUPPORT', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_16MV': {'status': 'SUPPORT', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_18MV': {'status': 'SUPPORT', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_MATHEW': {'status': 'SUPPORT', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_ZAMORANO': {'status': 'SUPPORT', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SUPPORTING_STEP', 'note': None}}, 25: {'P001': {'status': 'SUPPORT', 'object_id': 'P001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N001': {'status': 'SUPPORT', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N002': {'status': 'SUPPORT', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P002': {'status': 'SUPPORT', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P003': {'status': 'SUPPORT', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'C001': {'status': 'SUPPORT', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN001': {'status': 'SUPPORT', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P004': {'status': 'SUPPORT', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N003': {'status': 'SUPPORT', 'object_id': 'N003.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N004': {'status': 'SUPPORT', 'object_id': 'N004.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'E001': {'status': 'SUPPORT', 'object_id': 'E001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN002': {'status': 'SUPPORT', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P005_SIM': {'status': 'SUPPORT', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN003': {'status': 'SUPPORT', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_6MV': {'status': 'SUPPORT', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_10MV': {'status': 'SUPPORT', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_15MV': {'status': 'SUPPORT', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_16MV': {'status': 'SUPPORT', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_18MV': {'status': 'SUPPORT', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_MATHEW': {'status': 'SUPPORT', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_ZAMORANO': {'status': 'SUPPORT', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SUPPORTING_STEP', 'note': None}}, 28: {'P001': {'status': 'SUPPORT', 'object_id': 'P001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N001': {'status': 'SUPPORT', 'object_id': 'N001.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N002': {'status': 'SUPPORT', 'object_id': 'N002.TRAJ_DEPTH', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P002': {'status': 'SUPPORT', 'object_id': 'P002.SEQ_TVL', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P003': {'status': 'SUPPORT', 'object_id': 'P003.TRAJ_BUILD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'C001': {'status': 'SUPPORT', 'object_id': 'C001.SEQ_LINES', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN001': {'status': 'SUPPORT', 'object_id': 'PN001.TRAJ_YIELD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P004': {'status': 'SUPPORT', 'object_id': 'P004.SEQ_MAT', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N003': {'status': 'SUPPORT', 'object_id': 'N003.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'N004': {'status': 'SUPPORT', 'object_id': 'N004.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'E001': {'status': 'SUPPORT', 'object_id': 'E001.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN002': {'status': 'SUPPORT', 'object_id': 'PN002.TRAJ_RXN', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'P005_SIM': {'status': 'SUPPORT', 'object_id': 'P005_SIM.TRAJ_SPEC', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PN003': {'status': 'SUPPORT', 'object_id': 'PN003.TRAJ_EXFOR', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_6MV': {'status': 'SUPPORT', 'object_id': 'PVAL_6MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_10MV': {'status': 'SUPPORT', 'object_id': 'PVAL_10MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_15MV': {'status': 'SUPPORT', 'object_id': 'PVAL_15MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_16MV': {'status': 'SUPPORT', 'object_id': 'PVAL_16MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'PVAL_18MV': {'status': 'SUPPORT', 'object_id': 'PVAL_18MV.TRAJ_PDD', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_MATHEW': {'status': 'SUPPORT', 'object_id': 'NVAL_MATHEW.TRAJ_E', 'reason_code': 'SUPPORTING_STEP', 'note': None}, 'NVAL_ZAMORANO': {'status': 'SUPPORT', 'object_id': 'NVAL_ZAMORANO.SEQ_POS', 'reason_code': 'SUPPORTING_STEP', 'note': None}}}
ATLAS_DATASET_REGISTRY = {'P001': {'name': 'NIST XCOM ordinary concrete', 'particle': 'photon', 'evidence': 'EVALUATED/REFERENCE', 'role': 'Mass attenuation and mass energy-absorption coefficients for ordinary concrete.', 'path': 'radiation_benchmark_data/photon/nist_ordinary_concrete_attenuation.csv (+ composition companion)', 'richness': 'RICH_1D', 'guardrail': 'Keep attenuation physics separate from photonuclear MF=3 reaction data.'}, 'N001': {'name': 'JAERI/TIARA concrete ~43 MeV source', 'particle': 'neutron', 'evidence': 'EXPERIMENTAL', 'role': 'Measured source spectrum, transmitted spectra, in-shield depth response and dose-equivalent benchmark.', 'path': 'radiation_benchmark_data/neutron/jaeri_tiara_* selected at source_proton_MeV=43', 'richness': 'RICH_STATE_TRAJECTORY', 'guardrail': 'Depth/energy are transport coordinates, not time.'}, 'N002': {'name': 'JAERI/TIARA concrete ~68 MeV source', 'particle': 'neutron', 'evidence': 'EXPERIMENTAL', 'role': 'Higher-energy companion with measured transmission, depth and dose evidence.', 'path': 'radiation_benchmark_data/neutron/jaeri_tiara_* selected at source_proton_MeV=68', 'richness': 'RICH_STATE_TRAJECTORY', 'guardrail': 'Depth/energy are transport coordinates, not time.'}, 'P002': {'name': 'IAEA SRS-47 photon TVLs', 'particle': 'photon', 'evidence': 'AUTHORITATIVE REFERENCE', 'role': 'Broad-beam concrete TVL engineering reference.', 'path': 'results/phase1/canonical_data/public_benchmarks/P002_IAEA_SRS47_concrete_TVL.csv', 'richness': 'SPARSE_1D', 'guardrail': 'Secondary broad-beam check; not direct source-spectrum proof.'}, 'P003': {'name': 'Published FLUKA concrete buildup benchmark', 'particle': 'photon', 'evidence': 'INDEPENDENT SIMULATION/PUBLISHED', 'role': 'Independent buildup-factor/reference comparison.', 'path': 'results/phase1/canonical_data/public_benchmarks/P003_concrete_exposure_buildup_FLUKA.csv', 'richness': 'RICH_FIELD', 'guardrail': 'Independent simulation, not experimental truth.'}, 'C001': {'name': 'IAEA H-1 capture-gamma reference', 'particle': 'capture gamma', 'evidence': 'EVALUATED/REFERENCE', 'role': 'Evaluated capture-gamma production/reference layer.', 'path': 'results/phase1/canonical_data/public_benchmarks/C001_H1_capture_gamma.csv', 'richness': 'DISTRIBUTION', 'guardrail': 'Treat gamma-line yields/energies as capture-gamma evidence, not generic photon attenuation.'}, 'PN001': {'name': 'Published natural-W photoneutron benchmark', 'particle': 'photonuclear/neutron', 'evidence': 'PUBLISHED SIMULATION/REFERENCE', 'role': 'Independent natural-tungsten photoneutron benchmark.', 'path': 'results/phase1/canonical_data/public_benchmarks/PN001_tungsten_photoneutron_yield.csv', 'richness': 'RICH_1D', 'guardrail': 'Published simulation/reference evidence; do not relabel as experiment.'}, 'P004': {'name': 'Ogundare broad-beam photon shielding experiment', 'particle': 'photon', 'evidence': 'EXPERIMENTAL', 'role': '35 measurements across 7 materials × 5 energies with uncertainties.', 'path': 'processed_datasets/BROAD_BEAM_PHOTON/broad_beam_photon_physics_ready.parquet', 'richness': 'SMALL_FIELD', 'guardrail': 'Material is categorical unless ordered by an explicit physical descriptor such as density or Z_eff.'}, 'N003': {'name': 'ISIS RB2000164 barite-enriched concrete', 'particle': 'neutron', 'evidence': 'EXPERIMENTAL', 'role': 'Experimental T(E) and grounded macroscopic removal cross section for 9 samples.', 'path': 'processed_datasets/RB2000164/rb2000164_physics_ready_strict.parquet', 'richness': 'VERY_RICH_FIELD', 'guardrail': 'Grounded thickness permits Sigma_R; preserve sample traceability.'}, 'N004': {'name': 'ISIS RB2000209 standard-monitor concrete transmission', 'particle': 'neutron', 'evidence': 'EXPERIMENTAL_DERIVED_MONITOR_PROXY', 'role': '54,445-row standard-monitor transmission proxy over 1 meV–1 keV.', 'path': 'processed_datasets/RB2000209/rb2000209_standard_monitor_proxy_physics_ready_1meV_1keV.parquet', 'richness': 'VERY_RICH_FIELD', 'guardrail': 'No grounded sample path length: do not calculate or infer Sigma(E); do not call proxy GEM-derived.'}, 'E001': {'name': 'NIST ESTAR Portland concrete', 'particle': 'electron', 'evidence': 'EVALUATED/REFERENCE', 'role': '81-energy stopping-power, CSDA-range and radiation-yield reference.', 'path': 'processed_datasets/NIST_ESTAR/estar_portland_concrete.parquet', 'richness': 'RICH_MULTI_1D', 'guardrail': 'Material 144 is Portland concrete; direct Geant4 comparison requires composition audit.'}, 'PN002': {'name': 'IAEA Photonuclear Data Library PD-2019', 'particle': 'photonuclear', 'evidence': 'EVALUATED', 'role': '219 evaluations / 178,635 MF=3 reaction points.', 'path': 'processed_datasets/IAEA_PD2019/pd2019_mf3_cross_sections.parquet', 'richness': 'VERY_RICH_TENSOR', 'guardrail': 'Reaction cross sections remain semantically separate from XCOM attenuation coefficients.'}, 'P005_SIM': {'name': 'Photon Shielding Spectra Dataset (PSSD)', 'particle': 'photon', 'evidence': 'SIMULATED', 'role': '971,520-row authoritative canonical long-form simulated spectral reference (92 elements × 22 incident energies × 16 depths × 30 outgoing-energy bins).', 'path': 'data/processed/PSSD/pssd_canonical_spectra.parquet', 'richness': 'VERY_RICH_TENSOR', 'guardrail': 'Simulation only; do not composition-weight pure-element spectra to manufacture concrete spectra.'}, 'PN003': {'name': 'IAEA EXFOR experimental photonuclear data', 'particle': 'photonuclear/nuclear reaction', 'evidence': 'EXPERIMENTAL', 'role': '66,683 strict photon-induced MF=3 rows with uncertainty and dependence semantics.', 'path': 'registry-resolved processed IAEA_EXFOR canonical Parquet', 'richness': 'VERY_RICH_TENSOR', 'guardrail': 'Retain target, reaction, observable, frame, status, uncertainty, accession/subentry and units.'}, 'PVAL_6MV': {'name': 'TrueBeam 6 MV 40×40 LANDauer PDD', 'particle': 'photon beam validation', 'evidence': 'EXPERIMENTAL', 'role': 'Measured water PDD diagnostic for 6 MV.', 'path': 'processed_datasets/production_source_validation/PVAL_6MV_TRUEBEAM_40x40_LANDAUER/canonical_pdd_40x40.csv', 'richness': 'RICH_1D', 'guardrail': 'PDD is nonqualifying for direct aggregate 1-D source-spectrum promotion.'}, 'PVAL_10MV': {'name': 'TrueBeam 10 MV 40×40 LANDauer PDD', 'particle': 'photon beam validation', 'evidence': 'EXPERIMENTAL', 'role': 'Measured water PDD diagnostic for 10 MV.', 'path': 'processed_datasets/production_source_validation/PVAL_10MV_TRUEBEAM_40x40_LANDAUER/canonical_pdd_40x40.csv', 'richness': 'RICH_1D', 'guardrail': 'PDD is nonqualifying for direct aggregate 1-D source-spectrum promotion.'}, 'PVAL_15MV': {'name': 'TrueBeam 15 MV 40×40 LANDauer PDD', 'particle': 'photon beam validation', 'evidence': 'EXPERIMENTAL', 'role': 'Measured water PDD diagnostic for 15 MV.', 'path': 'processed_datasets/production_source_validation/PVAL_15MV_TRUEBEAM_40x40_LANDAUER/canonical_pdd_40x40.csv', 'richness': 'RICH_1D', 'guardrail': 'PDD is nonqualifying for direct aggregate 1-D source-spectrum promotion.'}, 'PVAL_16MV': {'name': 'Clinac 21IX 16 MV 40×40 Pacyniak PDD', 'particle': 'photon beam validation', 'evidence': 'EXPERIMENTAL', 'role': 'Measured 40×40 PDD; shape-focused comparison.', 'path': 'processed_datasets/production_source_validation/PVAL_16MV_CLINAC21IX_40x40_PACYNIAK/canonical_pdd_40x40.csv', 'richness': 'RICH_1D', 'guardrail': 'SSD is not explicitly tied to digitized figure; preserve shape-focused interpretation.'}, 'PVAL_18MV': {'name': 'Siemens Oncor 18 MV 40×40 Sawkey/Faddegon PDD', 'particle': 'photon beam validation', 'evidence': 'EXPERIMENTAL', 'role': 'Measured clinical flattening-filter-in 40×40 water PDD diagnostic.', 'path': 'processed_datasets/production_source_validation/PVAL_18MV_ONCOR_40x40_SAWKEY/canonical_pdd_40x40_clinical.csv', 'richness': 'RICH_1D', 'guardrail': 'PDD is nonqualifying for direct aggregate 1-D source-spectrum promotion.'}, 'NVAL_MATHEW': {'name': 'Mathew et al. TrueBeam STx 15 MV measured neutron spectra', 'particle': 'linac photoneutron', 'evidence': 'EXPERIMENTAL', 'role': 'Measured active/passive neutron spectra plus total fluence/dose quantities.', 'path': 'processed_datasets/production_source_validation/NVAL_15MV_TRUEBEAM_STX_MATHEW/canonical_measured_neutron_spectra.csv', 'richness': 'RICH_MULTI_1D', 'guardrail': 'Independent linac-neutron benchmark; not direct PROJECT_15MV_40x40 source-spectrum proof.'}, 'NVAL_ZAMORANO': {'name': 'Zamorano et al. 15 MV measured thermal-neutron fluence', 'particle': 'linac photoneutron', 'evidence': 'EXPERIMENTAL', 'role': 'Measured thermal-neutron fluence at five positions for TrueBeam and Elekta.', 'path': 'processed_datasets/production_source_validation/NVAL_15MV_ZAMORANO_THERMAL/canonical_measured_thermal_neutron_fluence.csv', 'richness': 'VERY_SPARSE_SPATIAL', 'guardrail': 'Only five positions per machine; high-order sequence/dynamics results may be degenerate or N/A.'}}

ATLAS_SPEED = globals().get("ATLAS_SPEED", globals().get("FAST_OPTION", "fast"))
if ATLAS_SPEED not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError("ATLAS_SPEED must be full, balanced, fast, or ultra")

ATLAS_CURVE_CAP = {"full": None, "balanced": 160, "fast": 64, "ultra": 32}[ATLAS_SPEED]
ATLAS_POINT_CAP = {"full": 10000, "balanced": 6000, "fast": 3500, "ultra": 1800}[ATLAS_SPEED]
ATLAS_RANDOM_SEED = 42
ATLAS_AUTO_INSTALL_PYARROW = globals().get("ATLAS_AUTO_INSTALL_PYARROW", True)
ATLAS_CLEAN_RESULTS = globals().get("ATLAS_CLEAN_RESULTS", True)
ATLAS_TV_STABLE_THRESHOLD = 0.10
ATLAS_TV_P90_THRESHOLD = 0.20
ATLAS_TV_BAD_FRACTION_MAX = 0.25
ATLAS_ADJACENCY_DOMINANCE_THRESHOLD = 0.50
ATLAS_MIN_MODEL_CURVES = 3
ATLAS_MIN_MODEL_POINTS = 200
ATLAS_MAX_THEILER_ENTROPY_SENSITIVITY_BITS = 0.25
ATLAS_MAX_GENERALIZATION_GAP = 0.50

RED = "#FF2B2B"
RED_SOFT = "#FF6B6B"
RED_DIM = "#8B0000"
WHITE = "#F2F2F2"
ORANGE = "#FF8C42"
YELLOW = "#FFD84D"
BLACK = "#000000"
ATLAS_CMAP = LinearSegmentedColormap.from_list("atlas_black_red", [BLACK, RED_DIM, RED, RED_SOFT], N=256)
ATLAS_DIV_CMAP = LinearSegmentedColormap.from_list("atlas_red_black_red", [RED_DIM, BLACK, RED_SOFT], N=256)
plt.rcParams.update({
    "figure.facecolor": BLACK, "axes.facecolor": BLACK, "savefig.facecolor": BLACK,
    "axes.edgecolor": RED, "axes.labelcolor": RED, "axes.titlecolor": RED,
    "xtick.color": RED, "ytick.color": RED, "text.color": RED,
    "grid.color": RED_DIM, "grid.alpha": 0.25,
    "legend.facecolor": BLACK, "legend.edgecolor": RED, "legend.labelcolor": RED,
})

ATLAS_FEATURE_ROWS = []
ATLAS_LEDGER_ROWS = []
# Cache finalized dataset/object bundles so large sources are parsed once per kernel run.
ATLAS_BUNDLE_CACHE = {}


class AtlasStructuralFail(RuntimeError):
    def __init__(self, code, message):
        super().__init__(message)
        self.code = str(code)


class AtlasDependencyFail(RuntimeError):
    def __init__(self, code, message):
        super().__init__(message)
        self.code = str(code)


def atlas_dataset_dir(dataset_id):
    p = PHASE2_RESULTS_ROOT / str(dataset_id)
    p.mkdir(parents=True, exist_ok=True)
    return p


def atlas_style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(BLACK)
    for sp in ax.spines.values():
        sp.set_color(RED)
    ax.tick_params(colors=RED)
    ax.grid(True, color=RED_DIM, alpha=0.25)
    if title is not None: ax.set_title(title, color=RED)
    if xlabel is not None: ax.set_xlabel(xlabel, color=RED)
    if ylabel is not None: ax.set_ylabel(ylabel, color=RED)


def atlas_save_fig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=210, facecolor=BLACK, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def atlas_safe_name(x, max_len=120):
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")
    return s[:max_len] or "item"


def atlas_quality_eligible(status):
    return status in {"valid", "validated_reference", "stable", "theiler_excluded"}


def atlas_register_feature(dataset_id, cell, method, object_id, observable, bundle,
                           feature, value, unit="", quality_status="valid", notes="",
                           ensemble_group="", n_curves=np.nan, n_points=np.nan):
    try:
        v = float(value)
        if not np.isfinite(v):
            return
    except Exception:
        return
    ATLAS_FEATURE_ROWS.append({
        "dataset_id": dataset_id,
        "cell": int(cell),
        "method": method,
        "object_id": object_id,
        "observable": observable,
        "ensemble_group": ensemble_group,
        "coordinate_native": bundle.get("coordinate_native_name", ""),
        "coordinate_analysis": bundle.get("coordinate_analysis_name", ""),
        "coordinate_transform": bundle.get("coordinate_transform", "identity"),
        "coordinate_reference": bundle.get("coordinate_reference", np.nan),
        "coordinate_unit": bundle.get("coordinate_unit", ""),
        "feature": feature,
        "value": v,
        "unit": unit,
        "quality_status": quality_status,
        "eligible_for_model": bool(atlas_quality_eligible(quality_status)),
        "n_curves": n_curves,
        "n_points": n_points,
        "notes": notes,
    })


def atlas_register_ledger(dataset_id, cell, object_id, bank_status, execution_status,
                          reason_code="", reason="", n_curves=np.nan, n_points=np.nan,
                          bundle=None, successful_groups=0):
    bundle = bundle or {}
    ATLAS_LEDGER_ROWS.append({
        "dataset_id": dataset_id,
        "cell": int(cell),
        "object_id": object_id,
        "object_bank_status": bank_status,
        "execution_status": execution_status,
        "reason_code": reason_code,
        "reason": reason,
        "n_curves": n_curves,
        "n_points": n_points,
        "successful_groups": successful_groups,
        "coordinate_native": bundle.get("coordinate_native_name", ""),
        "coordinate_analysis": bundle.get("coordinate_analysis_name", ""),
        "coordinate_transform": bundle.get("coordinate_transform", ""),
        "coordinate_unit": bundle.get("coordinate_unit", ""),
    })


def atlas_flush_ledgers():
    if ATLAS_FEATURE_ROWS:
        pd.DataFrame(ATLAS_FEATURE_ROWS).drop_duplicates(
            subset=["dataset_id","cell","method","object_id","observable","ensemble_group","feature"],
            keep="last"
        ).to_csv(PHASE2_RESULTS_ROOT / "phase2_feature_registry.csv", index=False)
    if ATLAS_LEDGER_ROWS:
        pd.DataFrame(ATLAS_LEDGER_ROWS).drop_duplicates(
            subset=["dataset_id","cell"], keep="last"
        ).to_csv(PHASE2_RESULTS_ROOT / "phase2_applicability_results.csv", index=False)


def atlas_app_entry(cell, dataset_id):
    return ATLAS_APPLICABILITY[int(cell)].get(dataset_id, {"status":None,"object_id":None,"reason_code":"NO_ENTRY","note":None})


def atlas_bank_allows(cell, dataset_id):
    status = atlas_app_entry(cell, dataset_id).get("status")
    return status in {"DIRECT", "ADAPTED", "CONDITIONAL", "SUPPORT", "DIRECT_OR_ADAPTED", "SUPPORT_PROCESS"}


def atlas_find_column(df_or_cols, exact=(), contains=(), exclude=()):
    cols = list(df_or_cols.columns) if hasattr(df_or_cols, "columns") else list(df_or_cols)
    lower = {str(c).lower(): c for c in cols}
    for x in exact:
        if str(x).lower() in lower:
            return lower[str(x).lower()]
    candidates=[]
    for c in cols:
        lc=str(c).lower()
        if contains and not any(str(x).lower() in lc for x in contains):
            continue
        if any(str(x).lower() in lc for x in exclude):
            continue
        candidates.append(c)
    return sorted(candidates, key=lambda c:(len(str(c)),str(c)))[0] if candidates else None


def atlas_resolve_by_basename(basename, preferred_contains=()):
    candidates = list(PROJECT_ROOT.rglob(basename))
    if preferred_contains:
        preferred=[p for p in candidates if all(t.lower() in str(p).lower() for t in preferred_contains)]
        if preferred: candidates=preferred
    return sorted(candidates,key=lambda p:(len(str(p)),str(p)))[0] if candidates else None


def atlas_try_install_pyarrow():
    """Ensure a fresh Python interpreter can import pyarrow.

    Installing pyarrow after pandas has already been imported in the active
    notebook can leave the current process with Arrow extension-registration
    conflicts.  The actual fallback reader therefore runs in a clean
    subprocess and never imports the newly installed pyarrow into this kernel.
    """
    probe = [sys.executable, "-c", "import pyarrow, pandas; print(pyarrow.__version__)"]
    try:
        subprocess.run(probe, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        if not ATLAS_AUTO_INSTALL_PYARROW:
            return False
    try:
        print("Installing pyarrow for a clean subprocess Parquet reader...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pyarrow"])
        subprocess.run(probe, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception as exc:
        warnings.warn(f"Automatic pyarrow installation/subprocess probe failed: {exc}")
        return False


def atlas_read_parquet_subprocess(path, columns=None):
    """Read Parquet through a fresh interpreter to avoid in-kernel Arrow conflicts."""
    if not atlas_try_install_pyarrow():
        raise AtlasDependencyFail("PARQUET_READER_UNAVAILABLE",
                                  "pyarrow could not be made available to a clean subprocess")
    path = str(Path(path).resolve())
    fd, tmp = tempfile.mkstemp(prefix="atlas_parquet_", suffix=".csv")
    os.close(fd)
    try:
        cols_json = json.dumps(list(columns) if columns is not None else None)
        code = r"""
import json, sys
import pyarrow.parquet as pq
src, dst, cols_json = sys.argv[1], sys.argv[2], sys.argv[3]
cols = json.loads(cols_json)
table = pq.read_table(src, columns=cols)
table.to_pandas().to_csv(dst, index=False)
"""
        subprocess.check_call([sys.executable, "-c", code, path, tmp, cols_json])
        return pd.read_csv(tmp)
    finally:
        try:
            os.remove(tmp)
        except OSError:
            pass


def atlas_table_columns(path):
    """Return table column names without materializing a large Parquet table."""
    path=Path(path)
    if path.suffix.lower()==".csv" or path.name.endswith(".csv.gz"):
        return list(pd.read_csv(path,nrows=0).columns)
    for alt in [path.with_suffix(".csv"), Path(str(path).replace(".parquet", ".csv.gz"))]:
        if alt.exists():
            return list(pd.read_csv(alt,nrows=0).columns)
    if path.suffix.lower()==".parquet":
        if atlas_try_install_pyarrow():
            code = r"""
import json, sys
import pyarrow.parquet as pq
print(json.dumps(pq.ParquetFile(sys.argv[1]).schema_arrow.names))
"""
            out=subprocess.check_output([sys.executable,"-c",code,str(path.resolve())],text=True)
            return json.loads(out.strip())
        if importlib.util.find_spec("duckdb") is not None:
            try:
                import duckdb
                con=duckdb.connect(database=":memory:")
                try:
                    return [r[0] for r in con.execute("DESCRIBE SELECT * FROM read_parquet(?)",[str(path)]).fetchall()]
                finally:
                    con.close()
            except Exception:
                pass
    raise AtlasDependencyFail("TABLE_SCHEMA_UNAVAILABLE",f"Could not read schema for {path}")


def atlas_read_table(path, columns=None):
    path=Path(path)
    if path.suffix.lower()==".csv" or path.name.endswith(".csv.gz"):
        return pd.read_csv(path, usecols=columns)
    if path.suffix.lower()==".parquet":
        for alt in [path.with_suffix(".csv"), Path(str(path).replace(".parquet", ".csv.gz"))]:
            if alt.exists():
                return pd.read_csv(alt, usecols=columns)
        first_exc=None
        try:
            return pd.read_parquet(path, columns=columns)
        except Exception as exc:
            first_exc=exc
        if importlib.util.find_spec("duckdb") is not None:
            try:
                import duckdb
                sel="*" if not columns else ",".join('"'+str(c).replace('"','""')+'"' for c in columns)
                con=duckdb.connect(database=":memory:")
                try:
                    return con.execute(f"SELECT {sel} FROM read_parquet(?)", [str(path)]).df()
                finally:
                    con.close()
            except Exception:
                pass
        if importlib.util.find_spec("polars") is not None:
            try:
                import polars as pl
                return pl.read_parquet(path, columns=columns).to_pandas()
            except Exception:
                pass
        try:
            return atlas_read_parquet_subprocess(path, columns=columns)
        except Exception as sub_exc:
            raise AtlasDependencyFail(
                "PARQUET_READER_UNAVAILABLE",
                f"Could not read Parquet source {path}. In-kernel readers and clean-subprocess pyarrow failed. "
                f"Original pandas error: {first_exc}; subprocess error: {sub_exc}"
            )
    raise ValueError(f"Unsupported table format: {path}")

def atlas_clean_curve(coord, values, min_points=3):
    x=np.asarray(pd.to_numeric(pd.Series(coord),errors="coerce"),float).reshape(-1)
    y=np.asarray(pd.to_numeric(pd.Series(values),errors="coerce"),float).reshape(-1)
    good=np.isfinite(x)&np.isfinite(y)
    x,y=x[good],y[good]
    if len(x)==0: raise AtlasStructuralFail("EMPTY_CURVE","empty curve")
    order=np.argsort(x); x,y=x[order],y[order]
    d=pd.DataFrame({"x":x,"y":y}).groupby("x",as_index=False,sort=True).mean()
    x,y=d.x.to_numpy(),d.y.to_numpy()
    if len(x)<min_points: raise AtlasStructuralFail("INSUFFICIENT_UNIQUE_POINTS",f"only {len(x)} unique coordinate points")
    return x,y


def atlas_robust_standardize(x):
    x=np.asarray(x,float)
    med=np.nanmedian(x); mad=np.nanmedian(np.abs(x-med)); scale=1.4826*mad
    if not np.isfinite(scale) or scale<=1e-15: scale=np.nanstd(x)
    if not np.isfinite(scale) or scale<=1e-15: raise AtlasStructuralFail("DEGENERATE_CURVE","curve has negligible variation")
    return (x-med)/scale


def atlas_curve_non_degenerate(y):
    y=np.asarray(y,float)
    if len(y)<3 or not np.all(np.isfinite(y)): return False
    spread=np.nanpercentile(y,95)-np.nanpercentile(y,5)
    scale=max(np.nanmax(np.abs(y)),1.0)
    return bool(np.isfinite(spread) and spread>1e-10*scale and len(np.unique(np.round(y,12)))>=5)


def atlas_trim_threshold_plateau(x,y,observable):
    obs=str(observable).lower(); y=np.asarray(y,float); x=np.asarray(x,float)
    if "cross_section" not in obs and "yield" not in obs:
        return x,y,False
    if np.nanmin(y)<0 or np.nanmax(y)<=0:
        return x,y,False
    threshold=max(np.nanmax(y)*1e-10, np.finfo(float).tiny)
    active=np.where(y>threshold)[0]
    if len(active)<12:
        return x,y,False
    lo=max(0,active[0]-2); hi=min(len(y),active[-1]+3)
    if hi-lo < len(y)*0.35:
        return x,y,False
    return x[lo:hi],y[lo:hi],bool(lo>0 or hi<len(y))


def atlas_cap_curves(curves):
    cap=ATLAS_CURVE_CAP
    curves=sorted(curves,key=lambda c:(str(c.get("ensemble_group","")),str(c.get("label",""))))
    if cap is None or len(curves)<=cap: return curves,False
    idx=np.linspace(0,len(curves)-1,cap).round().astype(int)
    return [curves[i] for i in idx],True


def atlas_coordinate_unit(native_name):
    n=str(native_name).lower()
    if "_cm" in n or n.endswith("cm") or "depth_cm" in n: return "cm"
    if "mfp" in n: return "MFP"
    if "mev" in n: return "MeV"
    if "kev" in n: return "keV"
    if "ev" in n: return "eV"
    if "mv" in n: return "MV"
    if "index" in n or "position" in n: return "index"
    return "native_coordinate"


def atlas_finalize_bundle(bundle):
    curves=bundle["curves"]
    if not curves: raise AtlasStructuralFail("NO_USABLE_CURVES","No usable scalar curves could be constructed")
    curves,cap=atlas_cap_curves(curves); bundle["curves"]=curves; bundle["curve_cap_applied"]=cap
    native=bundle["coordinate_native_name"]
    is_energy="energy" in native.lower()
    positive=[]
    if is_energy:
        for c in curves:
            xx=np.asarray(c["coordinate"],float); positive.extend(xx[np.isfinite(xx)&(xx>0)].tolist())
    if is_energy and positive:
        ref=float(np.min(positive)); bundle["coordinate_reference"]=ref
        bundle["coordinate_analysis_name"]=f"log({native}/E_ref)"
        bundle["coordinate_transform"]="ln(x / dataset_min_positive_energy)"
        bundle["coordinate_unit"]="dimensionless_log_energy"
    else:
        bundle["coordinate_reference"]=np.nan
        bundle["coordinate_analysis_name"]=native
        bundle["coordinate_transform"]="identity"
        bundle["coordinate_unit"]=atlas_coordinate_unit(native)
    return bundle


def atlas_transform_coordinate(bundle,x):
    x=np.asarray(x,float)
    if bundle.get("coordinate_transform","").startswith("ln("):
        ref=float(bundle["coordinate_reference"])
        if not np.all(x>0): raise AtlasStructuralFail("NONPOSITIVE_ENERGY","log-energy transform requires positive energy")
        return np.log(x/ref)
    return x.copy()


def atlas_prepare_curve(bundle,curve,min_points=3,max_points=None,uniform=True):
    x,y=atlas_clean_curve(curve["coordinate"],curve["values"],min_points=min_points)
    x,y,trimmed=atlas_trim_threshold_plateau(x,y,curve.get("observable",""))
    if not atlas_curve_non_degenerate(y): raise AtlasStructuralFail("DEGENERATE_CURVE",f"{curve.get('label')} is degenerate")
    x=atlas_transform_coordinate(bundle,x)
    n=min(len(x),max_points or ATLAS_POINT_CAP)
    if n<min_points: raise AtlasStructuralFail("INSUFFICIENT_POINTS",f"only {n} points after preparation")
    if uniform:
        xu=np.linspace(float(x.min()),float(x.max()),n); yu=np.interp(xu,x,y)
    else:
        if n<len(x):
            ii=np.linspace(0,len(x)-1,n).round().astype(int); xu=x[ii]; yu=y[ii]
        else: xu,yu=x,y
    dx=float(np.median(np.diff(xu))) if len(xu)>1 else np.nan
    return xu,yu,dx,{"threshold_trimmed":trimmed}


def atlas_common_matrix_from_curves(bundle,curves,min_points=20,max_channels=64):
    prepared=[]
    for c in curves:
        try:
            x,y,dx,meta=atlas_prepare_curve(bundle,c,min_points=min_points,max_points=ATLAS_POINT_CAP)
            prepared.append((c,x,y))
        except AtlasStructuralFail:
            pass
    if len(prepared)<2: return None,None,[]
    prepared=sorted(prepared,key=lambda z:z[0]["label"])
    if len(prepared)>max_channels:
        ii=np.linspace(0,len(prepared)-1,max_channels).round().astype(int); prepared=[prepared[i] for i in ii]
    lo=max(p[1].min() for p in prepared); hi=min(p[1].max() for p in prepared)
    n=min(len(p[1]) for p in prepared)
    if hi<=lo or n<min_points: return None,None,[]
    grid=np.linspace(lo,hi,n); M=[]; labels=[]
    for c,x,y in prepared:
        M.append(np.interp(grid,x,y)); labels.append(c["label"])
    return grid,np.column_stack(M),labels


def atlas_group_curves(bundle, minimum=1):
    groups={}
    for c in bundle["curves"]:
        groups.setdefault(c.get("ensemble_group") or c.get("observable") or "default",[]).append(c)
    return {g:v for g,v in groups.items() if len(v)>=minimum}


def atlas_build_default_state_groups(bundle):
    # Explicit state groups are used only when the loader already has the
    # physically correct multivariate state object (for example P005_SIM.TRAJ_SPEC).
    # All other datasets continue through the unchanged curve-to-state builder below.
    state_groups={}
    for g,state in bundle.get("explicit_state_groups",{}).items():
        coord=np.asarray(state.get("coordinate",[]),float)
        M=np.asarray(state.get("matrix",[]),float)
        labels=list(state.get("labels",[]))
        if M.ndim==2 and len(coord)==M.shape[0] and M.shape[0]>=3 and M.shape[1]>=2:
            state_groups[g]={"coordinate":coord,"matrix":M,"labels":labels}
    for g,curves in atlas_group_curves(bundle,minimum=2).items():
        if g in state_groups:
            continue
        coord,M,labels=atlas_common_matrix_from_curves(bundle,curves,min_points=12,max_channels=64)
        if M is not None: state_groups[g]={"coordinate":coord,"matrix":M,"labels":labels}
    bundle["state_groups"]=state_groups
    return bundle


def atlas_load_bundle(dataset_id, object_id):
    cache_key=(str(dataset_id),str(object_id))
    if cache_key in ATLAS_BUNDLE_CACHE:
        return ATLAS_BUNDLE_CACHE[cache_key]

    curves=[]; source_path=None; native=""; explicit_state_groups={}

    if dataset_id in {"N001","N002"}:
        mev="43MEV" if dataset_id=="N001" else "68MEV"
        basename=f"{dataset_id}_JAERI_TIARA_{mev}_R_E_t_x_reference.npz"
        source_path=PROJECT_ROOT/"results/phase1/ordered_response_fields"/basename
        if not source_path.exists(): source_path=atlas_resolve_by_basename(basename)
        if source_path is None: raise FileNotFoundError(basename)
        d=np.load(source_path,allow_pickle=False); coord=np.asarray(d["thickness_cm"],float); response=np.asarray(d["response"],float)
        E=0.5*(np.asarray(d["energy_lower_MeV"],float)+np.asarray(d["energy_upper_MeV"],float)); off=np.asarray(d["off_axis_cm"],float)
        native="shield_depth_cm"
        for ie,e in enumerate(E):
            for io,o in enumerate(off):
                try:x,y=atlas_clean_curve(coord,response[ie,:,io])
                except AtlasStructuralFail: continue
                curves.append({"label":f"E={e:.6g}MeV__offaxis={o:.6g}cm","observable":"neutron_response","ensemble_group":"neutron_response_channels","coordinate":x,"values":y})

    elif dataset_id=="P001":
        source_path=atlas_resolve_by_basename("nist_ordinary_concrete_attenuation.csv")
        if source_path is None: raise FileNotFoundError("nist_ordinary_concrete_attenuation.csv")
        df=atlas_read_table(source_path); native="photon_energy_MeV"
        ec=atlas_find_column(df,exact=("energy_MeV","photon_energy_MeV","Energy_MeV"),contains=("energy",),exclude=("row",))
        if ec is None: raise KeyError("P001 energy column not found")
        canonical=[]
        for label,cands in [
            ("mu_over_rho_cm2_g",("mu_over_rho_cm2_g","mass_attenuation_cm2_g")),
            ("mu_en_over_rho_cm2_g",("mu_en_over_rho_cm2_g","mass_energy_absorption_cm2_g")),
        ]:
            c=atlas_find_column(df,exact=cands)
            if c is not None:
                x,y=atlas_clean_curve(df[ec],df[c]); curves.append({"label":label,"observable":label,"ensemble_group":"attenuation_state","coordinate":x,"values":y}); canonical.append((label,y,x))
        if not curves: raise AtlasStructuralFail("NO_CANONICAL_OBSERVABLES","P001 canonical attenuation coefficients not found")

    elif dataset_id=="P002":
        source_path=atlas_resolve_by_basename("P002_IAEA_SRS47_concrete_TVL.csv")
        if source_path is None: raise FileNotFoundError("P002_IAEA_SRS47_concrete_TVL.csv")
        df=atlas_read_table(source_path); native="nominal_beam_MV"; coord=pd.to_numeric(df["nominal_beam_MV"],errors="coerce")
        for c in ["primary_beam_TVL_mm","leakage_90deg_TVL_mm"]:
            if c in df:
                try:x,y=atlas_clean_curve(coord,df[c])
                except AtlasStructuralFail: continue
                curves.append({"label":c,"observable":c,"ensemble_group":"TVL_curves","coordinate":x,"values":y})

    elif dataset_id=="P003":
        source_path=atlas_resolve_by_basename("P003_concrete_exposure_buildup_FLUKA.csv")
        if source_path is None: raise FileNotFoundError("P003_concrete_exposure_buildup_FLUKA.csv")
        df=atlas_read_table(source_path); native="penetration_MFP"
        for e,g in df.groupby("photon_energy_MeV",sort=True):
            x,y=atlas_clean_curve(g["penetration_mfp"],g["exposure_buildup_factor"])
            curves.append({"label":f"E={e}MeV","observable":"exposure_buildup_factor","ensemble_group":"buildup_energy_channels","coordinate":x,"values":y})

    elif dataset_id=="P004":
        source_path=atlas_resolve_by_basename("broad_beam_photon_physics_ready.csv") or atlas_resolve_by_basename("broad_beam_photon_physics_ready.parquet")
        if source_path is None: raise FileNotFoundError("broad_beam_photon_physics_ready")
        df=atlas_read_table(source_path); native="photon_energy_MeV"
        ec=atlas_find_column(df,exact=("energy_MeV","photon_energy_MeV","energy_mev"),contains=("energy",)); vc=atlas_find_column(df,exact=("mass_attenuation_cm2_g","mu_over_rho","mass_attenuation_coefficient_cm2_g"),contains=("attenuation",),exclude=("uncert",)); gc=atlas_find_column(df,exact=("material","material_name"),contains=("material",),exclude=("density",))
        if ec is None or vc is None: raise KeyError("P004 energy/attenuation columns not found")
        for label,g in (df.groupby(gc,sort=True) if gc else [("all",df)]):
            try:x,y=atlas_clean_curve(g[ec],g[vc])
            except AtlasStructuralFail: continue
            curves.append({"label":str(label),"observable":"mass_attenuation","ensemble_group":"materials","coordinate":x,"values":y})

    elif dataset_id=="N003":
        source_path=atlas_resolve_by_basename("rb2000164_physics_ready_strict.csv.gz") or atlas_resolve_by_basename("rb2000164_physics_ready_strict.parquet")
        if source_path is None: raise FileNotFoundError("RB2000164 strict processed file")
        df=atlas_read_table(source_path); native="neutron_energy_eV"; sc=atlas_find_column(df,exact=("sample_id",),contains=("sample",)); ec=atlas_find_column(df,exact=("energy_eV","energy_ev"),contains=("energy",))
        obs=[("transmission",atlas_find_column(df,exact=("transmission",),contains=("transmission",),exclude=("uncert",))), ("Sigma_R_cm_inv",atlas_find_column(df,exact=("Sigma_R_cm_inv","sigma_r_cm_inv"),contains=("sigma_r","removal"),exclude=("uncert",)))]
        for sid,g in df.groupby(sc,sort=True):
            for oname,vc in obs:
                if vc is None: continue
                try:x,y=atlas_clean_curve(g[ec],g[vc])
                except AtlasStructuralFail: continue
                group="transmission_samples" if oname=="transmission" else "removal_samples"
                curves.append({"label":f"{sid}__{oname}","sample_id":str(sid),"observable":oname,"ensemble_group":group,"coordinate":x,"values":y})

    elif dataset_id=="N004":
        source_path=atlas_resolve_by_basename("rb2000209_standard_monitor_proxy_physics_ready_1meV_1keV.csv.gz") or atlas_resolve_by_basename("rb2000209_standard_monitor_proxy_physics_ready_1meV_1keV.parquet")
        if source_path is None: raise FileNotFoundError("RB2000209 processed file")
        df=atlas_read_table(source_path); native="neutron_energy_eV"; sc=atlas_find_column(df,exact=("sample_id",),contains=("sample",)); ec=atlas_find_column(df,exact=("energy_eV","energy_ev"),contains=("energy",)); vc=atlas_find_column(df,exact=("transmission",),contains=("transmission",),exclude=("uncert",))
        for sid,g in (df.groupby(sc,sort=True) if sc else [("proxy",df)]):
            try:x,y=atlas_clean_curve(g[ec],g[vc])
            except AtlasStructuralFail: continue
            curves.append({"label":str(sid),"sample_id":str(sid),"observable":"transmission_proxy","ensemble_group":"transmission_proxy_samples","coordinate":x,"values":y})

    elif dataset_id=="E001":
        source_path=atlas_resolve_by_basename("estar_portland_concrete.csv") or atlas_resolve_by_basename("estar_portland_concrete.parquet")
        if source_path is None: raise FileNotFoundError("ESTAR Portland concrete")
        df=atlas_read_table(source_path); native="electron_energy_MeV"; ec=atlas_find_column(df,exact=("energy_MeV","kinetic_energy_MeV","electron_energy_MeV"),contains=("energy",),exclude=("mean_excitation",))
        if ec is None: raise KeyError("E001 energy column not found")
        # Keep only non-redundant canonical transport observables.  Total stopping
        # power is omitted because it is the algebraic sum of collision+radiative
        # stopping power and would otherwise create a dependent state channel.
        allowed_groups={
            "collision_stopping_power_MeV_cm2_g":"stopping_power_components",
            "radiative_stopping_power_MeV_cm2_g":"stopping_power_components",
            "csda_range_g_cm2":"csda_range",
            "radiation_yield":"radiation_yield",
        }
        for c,grp in allowed_groups.items():
            if c not in df: continue
            try:x,y=atlas_clean_curve(df[ec],df[c],min_points=20)
            except AtlasStructuralFail: continue
            curves.append({"label":c,"observable":c,"ensemble_group":grp,"coordinate":x,"values":y})
        if not curves: raise AtlasStructuralFail("NO_CANONICAL_OBSERVABLES","No canonical ESTAR transport observables found")

    elif dataset_id=="PN002":
        source_path=atlas_resolve_by_basename("pd2019_mf3_cross_sections.csv.gz") or atlas_resolve_by_basename("pd2019_mf3_cross_sections.parquet")
        if source_path is None: raise FileNotFoundError("PD2019 MF3 cross sections")
        df=atlas_read_table(source_path); native="incident_photon_energy_MeV"; ec=atlas_find_column(df,exact=("energy_MeV","incident_energy_MeV"),contains=("energy",)); vc=atlas_find_column(df,exact=("cross_section_barn","cross_section_b"),contains=("cross_section",),exclude=("uncert",)); groups=[c for c in [atlas_find_column(df,exact=("target_Z",)),atlas_find_column(df,exact=("target_A",)),atlas_find_column(df,exact=("MT",))] if c is not None]
        if ec is None or vc is None or not groups: raise KeyError("PN002 columns not found")
        for key,g in df.groupby(groups,sort=True):
            if len(g)<12: continue
            try:x,y=atlas_clean_curve(g[ec],g[vc])
            except AtlasStructuralFail: continue
            curves.append({"label":"rxn_"+atlas_safe_name(key),"observable":"cross_section_barn","ensemble_group":"reaction_cross_sections","coordinate":x,"values":y})

    elif dataset_id=="PN003":
        # Prefer the full strict experimental MF=3 table. Support the available table formats without requiring an optional Parquet engine.
        source_path=(atlas_resolve_by_basename("exfor_photonuclear_mf3_strict.parquet") or
                     atlas_resolve_by_basename("PN003_EXFOR_experimental_reaction_curve_index.parquet") or
                     atlas_resolve_by_basename("exfor_photonuclear_mf3_all.parquet"))
        if source_path is None: raise FileNotFoundError("PN003 EXFOR strict photonuclear table")
        df=atlas_read_table(source_path); native="incident_photon_energy_MeV"
        ec=atlas_find_column(df,exact=("incident_energy_MeV","energy_MeV","energy_ev","energy_eV"),contains=("energy",),exclude=("uncert","lower","upper")); vc=atlas_find_column(df,exact=("experimental_cross_section_barn","cross_section_barn","cross_section_b","value"),contains=("cross_section","value"),exclude=("uncert","error")); groups=[c for c in [atlas_find_column(df,exact=("target_Z",)),atlas_find_column(df,exact=("target_A",)),atlas_find_column(df,exact=("MT",)),atlas_find_column(df,exact=("entry",)),atlas_find_column(df,exact=("subentry",))] if c is not None]
        if ec is None or vc is None or not groups: raise KeyError(f"PN003 expected curve columns not found. Columns: {list(df.columns)}")
        # If energy is stored in eV but named generically, normalize to MeV only when magnitude proves it.
        energy=pd.to_numeric(df[ec],errors="coerce")
        if np.nanmedian(energy)>1e4:
            energy=energy/1e6
        for key,gidx in df.assign(_atlas_energy_MeV=energy).groupby(groups,sort=True):
            if len(gidx)<12: continue
            try:x,y=atlas_clean_curve(gidx["_atlas_energy_MeV"],gidx[vc])
            except AtlasStructuralFail: continue
            curves.append({"label":"exfor_"+atlas_safe_name(key),"observable":"experimental_cross_section_barn","ensemble_group":"experimental_reaction_cross_sections","coordinate":x,"values":y})

    elif dataset_id=="P005_SIM":
        # P005_SIM.TRAJ_SPEC contract:
        #   ordered coordinate = shielding depth / optical depth (depth_MFP)
        #   state z(s)          = complete outgoing-energy spectrum at that depth
        # Fixed (element, incident_energy_MeV) defines one vector-valued trajectory.
        #
        # Phase II consumes the authoritative canonical PSSD product directly.
        # Upstream filename parsing, Flux_* decoding, wide-table inference, semantic
        # recovery, and source-layout guessing do not belong in this analysis layer.
        source_path=PROJECT_ROOT/"data/processed/PSSD/pssd_canonical_spectra.parquet"
        if not source_path.is_file():
            raise FileNotFoundError(
                "Authoritative canonical PSSD dataset not found: "
                + str(source_path)
            )

        required_columns=[
            "particle",
            "atomic_number",
            "element",
            "incident_energy_MeV",
            "depth_MFP",
            "outgoing_energy_MeV",
            "relative_flux",
        ]
        header=list(atlas_table_columns(source_path))
        missing=[c for c in required_columns if c not in header]
        if missing:
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_SCHEMA_MISMATCH",
                "Canonical PSSD is missing required column(s): "
                + ", ".join(missing)
            )

        native="depth_MFP"
        expected_rows=971520
        expected_element_count=92
        expected_incident_count=22
        expected_depth_count=16
        expected_energy_count=30
        expected_trajectory_count=expected_element_count*expected_incident_count

        target_trajectory_count={"full":8,"balanced":6,"fast":4,"ultra":3}[ATLAS_SPEED]
        candidate_multiplier=3
        candidate_trajectory_count=max(
            target_trajectory_count,
            target_trajectory_count*candidate_multiplier,
        )

        df=atlas_read_parquet_subprocess(
            source_path,
            columns=required_columns,
        )
        raw_rows_seen=int(len(df))
        if raw_rows_seen!=expected_rows:
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_ROW_COUNT",
                f"Canonical PSSD has {raw_rows_seen} rows; expected {expected_rows}"
            )

        # Canonical fields must already be numerical/semantic. Phase II validates
        # the contract but does not repair strings, paths, filenames, or wide tables.
        particle=df["particle"].astype(str).str.strip().str.lower()
        atomic_number=pd.to_numeric(df["atomic_number"],errors="coerce")
        element=df["element"].astype(str).str.strip()
        incident=pd.to_numeric(df["incident_energy_MeV"],errors="coerce")
        depth_values=pd.to_numeric(df["depth_MFP"],errors="coerce")
        outgoing=pd.to_numeric(df["outgoing_energy_MeV"],errors="coerce")
        relative_flux=pd.to_numeric(df["relative_flux"],errors="coerce")

        valid=(
            particle.eq("photon")
            & np.isfinite(atomic_number)
            & element.ne("")
            & np.isfinite(incident)
            & np.isfinite(depth_values)
            & np.isfinite(outgoing)
            & np.isfinite(relative_flux)
            & (relative_flux>=0)
        )
        if not bool(valid.all()):
            bad_count=int((~valid).sum())
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_INVALID_ROWS",
                f"Canonical PSSD contains {bad_count} row(s) violating the "
                "seven-column numerical/photon/nonnegative-flux contract"
            )

        spec=pd.DataFrame({
            "_atlas_Z":np.asarray(atomic_number,dtype=float),
            "_atlas_element":element.to_numpy(),
            "_atlas_incident":np.asarray(incident,dtype=float),
            "_atlas_depth":np.asarray(depth_values,dtype=float),
            "_atlas_outgoing":np.asarray(outgoing,dtype=float),
            "_atlas_flux":np.asarray(relative_flux,dtype=float),
        })

        if not np.allclose(
            spec["_atlas_Z"].to_numpy(float),
            np.round(spec["_atlas_Z"].to_numpy(float)),
            rtol=0.0,
            atol=1e-12,
        ):
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_ATOMIC_NUMBER",
                "Canonical PSSD atomic_number contains non-integer values"
            )
        spec["_atlas_Z"]=np.round(spec["_atlas_Z"]).astype(int)

        # One element symbol must map to one atomic number and vice versa.
        element_z=spec[["_atlas_element","_atlas_Z"]].drop_duplicates()
        if (
            int(element_z.groupby("_atlas_element")["_atlas_Z"].nunique().max())!=1
            or int(element_z.groupby("_atlas_Z")["_atlas_element"].nunique().max())!=1
        ):
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_ELEMENT_IDENTITY",
                "Canonical PSSD element/atomic_number mapping is not one-to-one"
            )

        global_counts={
            "elements":int(spec["_atlas_element"].nunique()),
            "atomic_numbers":int(spec["_atlas_Z"].nunique()),
            "incident_energies":int(spec["_atlas_incident"].nunique()),
            "depths":int(spec["_atlas_depth"].nunique()),
            "outgoing_energies":int(spec["_atlas_outgoing"].nunique()),
        }
        expected_counts={
            "elements":expected_element_count,
            "atomic_numbers":expected_element_count,
            "incident_energies":expected_incident_count,
            "depths":expected_depth_count,
            "outgoing_energies":expected_energy_count,
        }
        if global_counts!=expected_counts:
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_CARDINALITY",
                f"Canonical PSSD global cardinalities {global_counts} do not match "
                f"expected {expected_counts}"
            )

        # Scientific trajectory identity remains element + incident energy.
        # Internally include canonical atomic number so deterministic subsampling
        # is ordered across the physical Z domain rather than alphabetically by
        # element symbol.
        key_columns=["_atlas_Z","_atlas_element","_atlas_incident"]
        coordinate_columns=key_columns+["_atlas_depth","_atlas_outgoing"]

        duplicate_count=int(spec.duplicated(coordinate_columns,keep=False).sum())
        if duplicate_count:
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_DUPLICATE_COORDINATES",
                f"Canonical PSSD contains {duplicate_count} row(s) participating "
                "in duplicate spectral coordinates"
            )

        trajectory_stats=(
            spec.groupby(key_columns,as_index=False,sort=True)
            .agg(
                n_depth=("_atlas_depth","nunique"),
                n_outgoing=("_atlas_outgoing","nunique"),
                n_rows=("_atlas_flux","size"),
            )
            .sort_values(
                ["_atlas_Z","_atlas_incident"],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(trajectory_stats)!=expected_trajectory_count:
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_TRAJECTORY_COUNT",
                f"Canonical PSSD has {len(trajectory_stats)} trajectories; "
                f"expected {expected_trajectory_count}"
            )

        complete_mask=(
            trajectory_stats["n_depth"].eq(expected_depth_count)
            & trajectory_stats["n_outgoing"].eq(expected_energy_count)
            & trajectory_stats["n_rows"].eq(expected_depth_count*expected_energy_count)
        )
        incomplete_count=int((~complete_mask).sum())
        if incomplete_count:
            preview=trajectory_stats.loc[
                ~complete_mask,
                ["_atlas_Z","_atlas_element","_atlas_incident","n_depth","n_outgoing","n_rows"],
            ].head(10).to_dict("records")
            raise AtlasStructuralFail(
                "PSSD_CANONICAL_INCOMPLETE_TRAJECTORY",
                f"{incomplete_count} canonical PSSD trajectory/trajectories are not "
                f"exactly {expected_depth_count} depths × {expected_energy_count} "
                f"outgoing-energy bins; first failures={preview}"
            )

        # Choose a deterministic, domain-spanning candidate set before matrix
        # construction, preserving v11's speed-dependent trajectory budget.
        n_candidates=min(candidate_trajectory_count,len(trajectory_stats))
        if len(trajectory_stats)>n_candidates:
            ii=np.unique(
                np.linspace(
                    0,
                    len(trajectory_stats)-1,
                    n_candidates,
                ).round().astype(int)
            )
            selected_stats=trajectory_stats.iloc[ii].copy()
        else:
            selected_stats=trajectory_stats.copy()

        selected_keys=[
            (int(z),str(el),float(ein))
            for z,el,ein in selected_stats[key_columns].itertuples(
                index=False,
                name=None,
            )
        ]

        selected_index=pd.MultiIndex.from_tuples(
            selected_keys,
            names=key_columns,
        )
        row_index=pd.MultiIndex.from_arrays(
            [
                spec["_atlas_Z"],
                spec["_atlas_element"],
                spec["_atlas_incident"],
            ],
            names=key_columns,
        )
        selected_spec=spec.loc[row_index.isin(selected_index)].copy()
        selected_rows=int(len(selected_spec))

        complete_states=[]
        rejected_incomplete_tensor_count=0

        for key,g in selected_spec.groupby(key_columns,sort=True,dropna=False):
            gg=g.sort_values(
                ["_atlas_depth","_atlas_outgoing"],
                kind="mergesort",
            )
            depths=np.sort(gg["_atlas_depth"].unique().astype(float))
            energies=np.sort(gg["_atlas_outgoing"].unique().astype(float))

            if (
                len(depths)!=expected_depth_count
                or len(energies)!=expected_energy_count
                or len(gg)!=(expected_depth_count*expected_energy_count)
            ):
                rejected_incomplete_tensor_count += 1
                continue

            pivot=(
                gg.pivot(
                    index="_atlas_depth",
                    columns="_atlas_outgoing",
                    values="_atlas_flux",
                )
                .reindex(index=depths,columns=energies)
            )
            M=pivot.to_numpy(float)

            if (
                M.shape!=(expected_depth_count,expected_energy_count)
                or not np.all(np.isfinite(M))
            ):
                rejected_incomplete_tensor_count += 1
                continue

            complete_states.append((key,depths,energies,M))

        if not complete_states:
            raise AtlasStructuralFail(
                "PSSD_NO_COMPLETE_TRAJ_SPEC",
                "No complete canonical P005_SIM.TRAJ_SPEC tensor survived construction"
            )

        complete_states=sorted(
            complete_states,
            key=lambda z:(int(z[0][0]),float(z[0][2])),
        )
        if len(complete_states)>target_trajectory_count:
            ii=np.unique(
                np.linspace(
                    0,
                    len(complete_states)-1,
                    target_trajectory_count,
                ).round().astype(int)
            )
            complete_states=[complete_states[i] for i in ii]

        explicit_state_groups={}
        projection_cap={"full":64,"balanced":32,"fast":16,"ultra":8}[ATLAS_SPEED]
        if ATLAS_CURVE_CAP is not None:
            projection_cap=max(
                3,
                min(
                    projection_cap,
                    ATLAS_CURVE_CAP//max(1,len(complete_states)),
                ),
            )

        for key,depths,energies,M in complete_states:
            atomic_z,element_key,incident_energy=key

            # Preserve the public group-label shape; atomic number controls only
            # identity validation and physical-domain ordering.
            group=(
                f"P005_TRAJ_SPEC__element={atlas_safe_name(element_key,40)}"
                f"__Ein={float(incident_energy):.8g}MeV"
            )
            labels=[f"Eout={float(e):.10g}MeV" for e in energies]
            explicit_state_groups[group]={
                "coordinate":depths,
                "matrix":M,
                "labels":labels,
            }

            # Scalar channel curves are projections of the already-constructed
            # complete spectral state. The matrix above remains authoritative.
            nproj=min(projection_cap,len(energies))
            jj=np.unique(
                np.linspace(
                    0,
                    len(energies)-1,
                    nproj,
                ).round().astype(int)
            )
            for j in jj:
                y=M[:,j]
                curves.append({
                    "label":f"{group}__{labels[j]}",
                    "observable":"relative_flux",
                    "ensemble_group":group,
                    "coordinate":depths.copy(),
                    "values":y.copy(),
                })

        if not explicit_state_groups:
            raise AtlasStructuralFail(
                "PSSD_NO_COMPLETE_TRAJ_SPEC",
                "No complete P005_SIM.TRAJ_SPEC state group survived construction"
            )
        if not any(
            sum(c.get("ensemble_group")==g for c in curves)>=2
            for g in explicit_state_groups
        ):
            raise AtlasStructuralFail(
                "PSSD_INSUFFICIENT_STATE_PROJECTIONS",
                "Complete spectral trajectories exist but fewer than two "
                "nondegenerate spectral-channel projections survived"
            )

        pssd_diagnostics={
            "source_mode":"canonical_spectra",
            "selected_source_path":str(source_path),
            "source_trajectory_count":int(len(trajectory_stats)),
            "selected_candidate_trajectory_count":int(len(selected_stats)),
            "constructed_trajectory_count":int(len(explicit_state_groups)),
            "trajectory_key_columns":["element","incident_energy_MeV"],
            "trajectory_identity_column":"atomic_number",
            "trajectory_selection_order":["atomic_number","incident_energy_MeV"],
            "trajectory_coordinate_column":"depth_MFP",
            "state_axis_column":"outgoing_energy_MeV",
            "state_value_column":"relative_flux",
            "state_semantics":"complete_outgoing_energy_spectrum_ordered_by_depth_MFP",
            "canonical_expected_depth_count":int(expected_depth_count),
            "canonical_expected_outgoing_energy_count":int(expected_energy_count),
            "scalar_curve_role":"spectral_channel_projections_of_TRAJ_SPEC",
            "projection_cap_per_trajectory":int(projection_cap),
            "rejected_incomplete_tensor_count":int(rejected_incomplete_tensor_count),
            "raw_rows_seen":int(raw_rows_seen),
            "selected_rows":int(selected_rows),
        }
    elif dataset_id.startswith("PVAL_"):
        names={"PVAL_6MV":"canonical_pdd_40x40.csv","PVAL_10MV":"canonical_pdd_40x40.csv","PVAL_15MV":"canonical_pdd_40x40.csv","PVAL_16MV":"canonical_pdd_40x40.csv","PVAL_18MV":"canonical_pdd_40x40_clinical.csv"}
        token=dataset_id.split("_")[1].lower(); source_path=atlas_resolve_by_basename(names[dataset_id],preferred_contains=(token,))
        if source_path is None:
            candidates=list(PROJECT_ROOT.rglob(names[dataset_id])); source_path=next((p for p in candidates if token in str(p).lower()),None)
        if source_path is None: raise FileNotFoundError(f"{dataset_id} PDD file")
        df=atlas_read_table(source_path); native="water_depth_cm"; dc=atlas_find_column(df,exact=("depth_cm",),contains=("depth",)); vc=atlas_find_column(df,exact=("pdd_percent",),contains=("pdd",),exclude=("normalization",)); x,y=atlas_clean_curve(df[dc],df[vc]); curves.append({"label":"PDD","observable":"pdd_percent","ensemble_group":"PDD","coordinate":x,"values":y})

    elif dataset_id=="NVAL_MATHEW":
        source_path=atlas_resolve_by_basename("canonical_measured_neutron_spectra.csv")
        if source_path is None: raise FileNotFoundError("Mathew measured neutron spectra")
        df=atlas_read_table(source_path); native="neutron_energy_MeV"; ec=atlas_find_column(df,exact=("energy_center_MeV",),contains=("energy_center",)); vc=atlas_find_column(df,exact=("fluence_rate_n_cm2_s_per_nominal_0p2dec_bin",),contains=("fluence_rate",)); gc=atlas_find_column(df,exact=("spectrometer",),contains=("spectrometer",))
        for label,g in df.groupby(gc,sort=True):
            x,y=atlas_clean_curve(g[ec],g[vc]); curves.append({"label":str(label),"observable":"neutron_fluence_rate","ensemble_group":"spectrometers","coordinate":x,"values":y})

    elif dataset_id=="NVAL_ZAMORANO":
        source_path=atlas_resolve_by_basename("canonical_measured_thermal_neutron_fluence.csv")
        if source_path is None: raise FileNotFoundError("Zamorano thermal neutron fluence")
        df=atlas_read_table(source_path); native="measurement_position_index"; pc=atlas_find_column(df,exact=("position_id",),contains=("position",)); vc=atlas_find_column(df,exact=("thermal_neutron_fluence_n_cm2_Gy",),contains=("thermal_neutron_fluence",)); gc=atlas_find_column(df,exact=("machine",),contains=("machine",)); pos=pd.to_numeric(df[pc].astype(str).str.extract(r"(\d+)")[0],errors="coerce"); df=df.assign(_position_numeric=pos)
        for label,g in df.groupby(gc,sort=True):
            try:x,y=atlas_clean_curve(g["_position_numeric"],g[vc])
            except AtlasStructuralFail: continue
            curves.append({"label":str(label),"observable":"thermal_neutron_fluence","ensemble_group":"machines","coordinate":x,"values":y})

    elif dataset_id=="C001":
        source_path=atlas_resolve_by_basename("C001_H1_capture_gamma.csv")
        if source_path is None: raise FileNotFoundError("C001_H1_capture_gamma.csv")
        df=atlas_read_table(source_path); native="gamma_energy_keV"; x,y=atlas_clean_curve(df["gamma_energy_keV"],df["partial_gamma_production_cross_section_b"]); curves.append({"label":"capture_gamma_yield","observable":"partial_gamma_production_cross_section_b","ensemble_group":"capture_lines","coordinate":x,"values":y})

    elif dataset_id=="PN001":
        source_path=atlas_resolve_by_basename("PN001_tungsten_photoneutron_yield.csv")
        if source_path is None: raise FileNotFoundError("PN001_tungsten_photoneutron_yield.csv")
        df=atlas_read_table(source_path); native="incident_electron_energy_MeV"; x,y=atlas_clean_curve(df["incident_electron_energy_MeV"],df["neutron_yield_per_incident_electron"]); curves.append({"label":"photoneutron_yield","observable":"neutron_yield_per_incident_electron","ensemble_group":"yield_curve","coordinate":x,"values":y})

    else:
        raise NotImplementedError(dataset_id)

    bundle={"dataset_id":dataset_id,"object_id":object_id,"coordinate_native_name":native,"curves":curves,"source_path":str(source_path),"registry":ATLAS_DATASET_REGISTRY.get(dataset_id,{})}
    if dataset_id=="P005_SIM" and "pssd_diagnostics" in locals():
        bundle.update(pssd_diagnostics)
        bundle["explicit_state_groups"]=explicit_state_groups
    bundle=atlas_finalize_bundle(bundle)
    bundle=atlas_build_default_state_groups(bundle)
    ATLAS_BUNDLE_CACHE[cache_key]=bundle
    return bundle


def atlas_structural_summary(bundle):
    lengths=[len(c["coordinate"]) for c in bundle["curves"]]
    return len(lengths),int(max(lengths) if lengths else 0)


def atlas_write_dataset_manifest(bundle):
    out=atlas_dataset_dir(bundle["dataset_id"])/"dataset_atlas_manifest.json"; ncurves,npts=atlas_structural_summary(bundle)
    payload={
        "dataset_id":bundle["dataset_id"],"object_id":bundle["object_id"],
        "coordinate_native":bundle["coordinate_native_name"],"coordinate_analysis":bundle["coordinate_analysis_name"],
        "coordinate_transform":bundle["coordinate_transform"],"coordinate_reference":bundle["coordinate_reference"],"coordinate_unit":bundle["coordinate_unit"],
        "source_path":public_project_path(bundle["source_path"]),"n_curves_loaded":ncurves,"max_curve_points":npts,
        "curve_cap_applied":bool(bundle["curve_cap_applied"]),"atlas_speed":ATLAS_SPEED,
        "ensemble_groups":{k:len(v) for k,v in atlas_group_curves(bundle).items()},"registry":bundle["registry"],
        "state_group_shapes":{
            str(k):{
                "n_coordinate_points":int(np.asarray(v.get("matrix",[])).shape[0]) if np.asarray(v.get("matrix",[])).ndim==2 else 0,
                "state_dimension":int(np.asarray(v.get("matrix",[])).shape[1]) if np.asarray(v.get("matrix",[])).ndim==2 else 0,
            }
            for k,v in bundle.get("state_groups",{}).items()
        },
    }
    for key in (
        "source_trajectory_count","selected_candidate_trajectory_count","constructed_trajectory_count",
        "trajectory_key_columns","trajectory_coordinate_column","state_axis_column","state_value_column",
        "state_semantics","scalar_curve_role","projection_cap_per_trajectory",
        "rejected_incomplete_tensor_count","raw_rows_seen","selected_rows"
    ):
        if key in bundle:
            payload[key]=bundle[key]
    out.write_text(json.dumps(payload,indent=2,default=str),encoding="utf-8")


def atlas_record_exception(dataset_id,cell,ent,exc,bundle=None):
    obj=ent.get("object_id")
    if isinstance(exc,AtlasStructuralFail): status="conditional_fail"; code=exc.code
    elif isinstance(exc,AtlasDependencyFail): status="dependency_missing"; code=exc.code
    elif isinstance(exc,(FileNotFoundError,KeyError)): status="data_unavailable"; code=type(exc).__name__.upper()
    else: status="error"; code=ent.get("reason_code") or "UNEXPECTED_RUNTIME_ERROR"
    ncurves,npts=(atlas_structural_summary(bundle) if bundle else (np.nan,np.nan))
    atlas_register_ledger(dataset_id,cell,obj,ent.get("status"),status,code,f"{type(exc).__name__}: {exc}",ncurves,npts,bundle=bundle)


# ------------------------------------------------------------
# Clean only atlas-produced material. Keep N003 reference outputs and remove obsolete root-level cell_* files.
# ------------------------------------------------------------
if ATLAS_CLEAN_RESULTS:
    for did in ATLAS_DATASETS:
        d=PHASE2_RESULTS_ROOT/did
        if d.exists(): shutil.rmtree(d)
    for p in PHASE2_RESULTS_ROOT.iterdir():
        if p.is_file() and (p.name.startswith("cell_") or p.name in {
            "phase2_applicability_results.csv","phase2_feature_registry.csv","phase2_cross_dataset_summary.csv",
            "phase2_result_file_QA.csv","phase2_run_complete.json"
        }):
            p.unlink()

print("Cross-dataset infrastructure ready.")
print("Reference dataset retained from Cells 0–28:",REFERENCE_DATASET)
print("Atlas datasets:",len(ATLAS_DATASETS))
print("ATLAS_SPEED:",ATLAS_SPEED)
print("Results root:",PHASE2_RESULTS_ROOT)


In [ ]:
# ============================================================
# INGEST VALIDATED N003 REFERENCE OUTPUTS INTO THE GLOBAL LEDGERS
# ============================================================

def atlas_reference_bundle_n003():
    try:
        return atlas_load_bundle("N003", atlas_app_entry(4,"N003").get("object_id"))
    except Exception:
        return {
            "coordinate_native_name":"neutron_energy_eV",
            "coordinate_analysis_name":"log(neutron_energy_eV/E_ref)",
            "coordinate_transform":"ln(x / dataset_min_positive_energy)",
            "coordinate_reference":np.nan,
            "coordinate_unit":"dimensionless_log_energy",
            "curves":[],
        }


def atlas_ingest_reference_n003():
    did="N003"; out=atlas_dataset_dir(did); bundle=atlas_reference_bundle_n003(); atlas_write_dataset_manifest(bundle) if bundle.get("curves") else None
    ncurves,npts=atlas_structural_summary(bundle) if bundle.get("curves") else (np.nan,np.nan)

    # Cells 4/6 — promote only
    # embedding dimensions that pass the cell-specific local-step reliability QC.
    p=out/"cell_04_lyapunov_spectrum_results_expert_metrics.csv"
    rawp=out/"cell_04_lyapunov_spectrum_results.csv"
    if p.exists() or rawp.exists():
        if p.exists():
            d=pd.read_csv(p)
        else:
            d=pd.read_csv(rawp)
            qcp=out/"cell_04_lyapunov_spectrum_results_quality_control.csv"
            if not qcp.exists():
                raise FileNotFoundError("N003 Lyapunov QC table required: "+str(qcp))
            qc=pd.read_csv(qcp)
            keys=[c for c in ("file","channel","embedding_dimension") if c in d.columns and c in qc.columns]
            if len(keys)<2:
                raise KeyError("N003 Lyapunov QC table lacks merge keys")
            d=d.merge(
                qc[keys+["reliable_steps_flag"]].drop_duplicates(keys),
                on=keys,how="left",validate="many_to_one"
            )
        ok=d[d.get("status","ok").astype(str).eq("ok")].copy() if "status" in d else d.copy()
        expcols=[c for c in ok.columns if re.fullmatch(r"Exp\d+",str(c))]
        if len(ok) and expcols:
            ok["LLE"]=ok[expcols].max(axis=1,skipna=True)
            if "reliable_steps_flag" not in ok.columns or ok["reliable_steps_flag"].isna().any():
                raise ValueError("N003 reliable_steps_flag missing for one or more successful Cell-4 rows")
            ok["_reliable_steps"]=ok["reliable_steps_flag"].astype(str).str.lower().isin({"true","1","yes"})
            for (obs,m),g in ok.groupby(["observable","embedding_dimension"]):
                relfrac=float(g["_reliable_steps"].mean())
                q="validated_reference" if len(g)>=5 and relfrac>=0.8 else "unreliable_embedding_support"
                atlas_register_feature(
                    did,4,"lyapunov_local_linear",atlas_app_entry(4,did).get("object_id"),str(obs),bundle,
                    f"LLE_median_d{int(m)}",g.LLE.median(),"per_analysis_coordinate",q,
                    n_curves=g["sample_id"].nunique() if "sample_id" in g else len(g),
                    n_points=float(g.get("n_samples_used",pd.Series([np.nan])).median()),
                    notes=f"source reliable_steps fraction={relfrac:.3f}"
                )
            for obs,g in ok.groupby("observable"):
                grel=g[g["_reliable_steps"]].copy()
                meds=grel.groupby("embedding_dimension").LLE.median()
                if len(meds):
                    q="validated_reference" if len(meds)>=3 else "low_support"
                    atlas_register_feature(
                        did,6,"lyapunov_summary",atlas_app_entry(6,did).get("object_id"),str(obs),bundle,
                        "LLE_grand_median",meds.median(),"per_analysis_coordinate",q,
                        n_curves=grel["sample_id"].nunique() if "sample_id" in grel else np.nan,
                        n_points=float(grel.get("n_samples_used",pd.Series([np.nan])).median()),
                        notes="summary restricted to embedding dimensions passing source reliable_steps_flag"
                    )
                    atlas_register_feature(
                        did,6,"lyapunov_summary",atlas_app_entry(6,did).get("object_id"),str(obs),bundle,
                        "embedding_sensitivity_LLE_range",meds.max()-meds.min(),
                        "per_analysis_coordinate","diagnostic_only",
                        notes="range across reliable embedding dimensions only"
                    )
            atlas_register_ledger(did,4,atlas_app_entry(4,did).get("object_id"),atlas_app_entry(4,did).get("status"),"valid","","N003 Cell 4 valid; model eligibility respects reliability QC",ncurves,npts,bundle)
            atlas_register_ledger(did,6,atlas_app_entry(6,did).get("object_id"),atlas_app_entry(6,did).get("status"),"valid","","summary restricted to source-reliable embedding dimensions",ncurves,npts,bundle)

    # Cell 9 — use the per-oscillator relative-phase implementation: per-oscillator relative phase
    # is averaged through time, so drive lock is not algebraically identical to r.
    grids=sorted(out.glob("cell_09_arnold_tongue_kuramoto_*_results.csv"))
    grids=[p for p in grids if "characteristic" not in p.name]
    if grids:
        g=pd.read_csv(grids[-1]); stem=grids[-1].name[:-4]; fp=out/f"{stem}_characteristic_frequencies.csv"
        if fp.exists():
            f=pd.read_csv(fp); atlas_register_feature(did,9,"kuramoto_reduced_phase",atlas_app_entry(9,did).get("object_id"),"transmission_samples",bundle,"characteristic_frequency_cv",f.spectral_centroid_frequency.std(ddof=0)/f.spectral_centroid_frequency.mean(),"dimensionless","validated_reference",ensemble_group="transmission_samples",n_curves=len(f))
        for col,feat,q in [("drive_lock_gain_vs_no_drive","max_drive_lock_gain_vs_no_drive","scan_grid_dependent"),("sync_gain_vs_no_drive","max_sync_gain_vs_no_drive","scan_grid_dependent"),("frequency_error_improvement_vs_no_drive","max_frequency_error_improvement_vs_no_drive","scan_grid_dependent"),("drive_lock","max_drive_lock_raw","diagnostic_only")]:
            if col in g: atlas_register_feature(did,9,"kuramoto_reduced_phase",atlas_app_entry(9,did).get("object_id"),"transmission_samples",bundle,feat,g[col].max(),"",q,ensemble_group="transmission_samples")
        atlas_register_ledger(did,9,atlas_app_entry(9,did).get("object_id"),atlas_app_entry(9,did).get("status"),"valid","","N003 Cell 9 valid",ncurves,npts,bundle)

    # Cell 13 — raw locking is retained as model-dominated; phase resultant is data-derived.
    p=out/"cell_13_circle_map_fixed_point_locking_results.csv"; npz=out/"cell_13_circle_map_fixed_point_locking_results.npz"
    if p.exists():
        d=pd.read_csv(p); atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),"transmission_samples",bundle,"mean_fixed_point_locking",d.proportion_fixed_point_locked.mean(),"","model_baseline_dominated",ensemble_group="transmission_samples"); atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),"transmission_samples",bundle,"max_fixed_point_locking",d.proportion_fixed_point_locked.max(),"","model_baseline_dominated",ensemble_group="transmission_samples")
        if npz.exists():
            z=np.load(npz,allow_pickle=True); R=np.asarray(z["phase_resultant_length_full"],float); atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),"transmission_samples",bundle,"mean_phase_resultant",np.nanmean(R),"","validated_reference",ensemble_group="transmission_samples")
        atlas_register_ledger(did,13,atlas_app_entry(13,did).get("object_id"),atlas_app_entry(13,did).get("status"),"valid","","N003 Cell 13 valid; raw locking flagged model-dominated",ncurves,npts,bundle)

    # Cell 17 — recompute primary summaries from unreduced rotation increments.
    npz=out/"cell_17_circle_map_rotation_numbers_results.npz"
    if npz.exists():
        z=np.load(npz,allow_pickle=True); raw=np.asarray(z["rotation_numbers_unwrapped_all"],float); rawmean=np.nanmean(raw,axis=0); rawstd=np.nanstd(raw,axis=0)
        lo=int(np.floor(np.nanmin(rawmean)))-1; hi=int(np.ceil(np.nanmax(rawmean)))+1; vals=sorted(set(float(Fraction(p,q)) for q in range(1,9) for p in range(lo*q,hi*q+1))); rv=np.asarray(vals,float); dist=np.min(np.abs(rawmean[None,:,:]-rv[:,None,None]),axis=0)
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),"transmission_samples",bundle,"mean_unwrapped_rotation",np.nanmean(rawmean),"cycles_per_iterate","model_baseline_dominated",ensemble_group="transmission_samples",notes="raw global circle-map summary is model-baseline-dominated; use the baseline-referenced cross-dataset feature for model eligibility")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),"transmission_samples",bundle,"mean_cross_curve_unwrapped_rotation_std",np.nanmean(rawstd),"cycles_per_iterate","model_baseline_dominated",ensemble_group="transmission_samples",notes="raw global circle-map summary is model-baseline-dominated; use the baseline-referenced cross-dataset feature for model eligibility")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),"transmission_samples",bundle,"fraction_near_low_order_rational_unwrapped",np.nanmean(dist<=0.01),"","model_baseline_dominated",ensemble_group="transmission_samples",notes="raw global circle-map rational-lock fraction is dominated by the common Ω/K scan; use the baseline-referenced cross-dataset feature for model eligibility")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),"transmission_samples",bundle,"mean_rotation_mod1",np.nanmean(np.mod(rawmean,1.0)),"","diagnostic_only",ensemble_group="transmission_samples")
        atlas_register_ledger(did,17,atlas_app_entry(17,did).get("object_id"),atlas_app_entry(17,did).get("status"),"valid","","unwrapped rotation is the primary estimator; raw global summaries are model-baseline-dominated and baseline contrasts are used for model eligibility",ncurves,npts,bundle)

    # Cell 19 — candidate status depends on actual finite-time stabilization.
    npz=out/"cell_19_circle_map_phase_pushforward_results.npz"
    if npz.exists():
        z=np.load(npz,allow_pickle=True); tv=np.asarray(z["agg_stabilization_tv"],float); conc=np.asarray(z["agg_concentration"],float); mean_tv=float(np.nanmean(tv)); p90_tv=float(np.nanpercentile(tv,90)); bad_frac=float(np.nanmean(tv>ATLAS_TV_STABLE_THRESHOLD))
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),"transmission_samples",bundle,"mean_distribution_concentration",np.nanmean(conc),"","reference_only_noncomparable",ensemble_group="transmission_samples",notes="raw cell concentration lacks the atlas uniform-phase baseline; excluded from cross-dataset model pool")
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),"transmission_samples",bundle,"mean_stabilization_TV",mean_tv,"TV","diagnostic_only",ensemble_group="transmission_samples",notes=f"p90 TV={p90_tv:.4g}; fraction above threshold={bad_frac:.4g}")
        atlas_register_ledger(did,19,atlas_app_entry(19,did).get("object_id"),atlas_app_entry(19,did).get("status"),"valid","","N003 Cell 19 valid with stability gate",ncurves,npts,bundle)

    # Cell 21 — raw entropy is usable for N003 because adjacency domination is low.
    p=out/"cell_21_nearest_neighbor_distance_entropy_results.csv"
    if p.exists():
        d=pd.read_csv(p); ok=d[d.status.astype(str).eq("ok")].copy(); adj=float(ok.adjacent_neighbor_fraction.mean())
        atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),"all_observables",bundle,"NN_entropy_grand_median",ok.nn_distance_entropy_bits.median(),"bits","reference_only_noncomparable",n_curves=ok[["sample_id","observable"]].drop_duplicates().shape[0],n_points=float(ok.n_points_used.median()),notes="raw-NN estimator combines transmission and Sigma_R and is not cross-dataset comparable; use the observable-specific Theiler-excluded features for cross-dataset comparison")
        atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),"all_observables",bundle,"adjacent_neighbor_fraction_mean",adj,"","diagnostic_only")
        atlas_register_ledger(did,21,atlas_app_entry(21,did).get("object_id"),atlas_app_entry(21,did).get("status"),"valid","","N003 Cell 21 valid with adjacency gate",ncurves,npts,bundle)

    # Cell 25 — the method implementation already selects each lag in log-energy units.
    p=out/"cell_25_pyragas_delayed_feedback_results.csv"; qfile=out/"cell_25_pyragas_delayed_feedback_results_mi_delays.csv"
    if p.exists():
        d=pd.read_csv(p); delay_col="delay_log_energy" if "delay_log_energy" in d else "delay_coordinate"; atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),"all_observables",bundle,"median_MI_delay_coordinate",d[delay_col].median(),"dimensionless_log_energy","reference_only_noncomparable",notes="combined transmission+Sigma_R summary excluded; use observable-specific features for cross-dataset comparison"); atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),"all_observables",bundle,"median_relative_feedback_rms",d.relative_feedback_rms.median(),"","reference_only_noncomparable")
        if qfile.exists():
            qd=pd.read_csv(qfile); frac=qd.selection_method.astype(str).str.contains("first local minimum",case=False).mean(); atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),"all_observables",bundle,"fraction_MI_delays_with_local_minimum",frac,"","diagnostic_only")
        atlas_register_ledger(did,25,atlas_app_entry(25,did).get("object_id"),atlas_app_entry(25,did).get("status"),"valid","","N003 Cell 25 valid",ncurves,npts,bundle)

    # Cell 28 — fixed-point residual/inside_F are theorem/optimizer diagnostics, not predictive features.
    p=out/"cell_28_kakutani_fixed_point_results.csv"
    if p.exists():
        d=pd.read_csv(p).iloc[0]; atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),"multichannel_state",bundle,"PCA_first_two_explained_fraction",d.pca_first_two_explained_fraction,"","reference_only_noncomparable",notes="PCA/train fit is diagnostic only; use blocked-holdout features for cross-dataset comparison"); atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),"multichannel_state",bundle,"affine_one_step_R2_train",d.affine_r2_joint,"","diagnostic_only"); atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),"multichannel_state",bundle,"epsilon_median_residual",d.epsilon_median_one_step_residual,"PCA_distance","diagnostic_only"); atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),"multichannel_state",bundle,"fixed_inclusion_residual",d.fixed_inclusion_residual,"PCA_distance","theorem_diagnostic")
        atlas_register_ledger(did,28,atlas_app_entry(28,did).get("object_id"),atlas_app_entry(28,did).get("status"),"valid","","N003 Cell 28 valid; fixed-point residual excluded from predictive pool",ncurves,npts,bundle)

    atlas_flush_ledgers()

atlas_ingest_reference_n003()
print("Validated N003 outputs ingested into global ledgers.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELLS 4 AND 6
# Local-linear spectrum + robust embedding summary
# ============================================================

def atlas_delay_embed(x,m,tau=1):
    x=np.asarray(x,float).reshape(-1); n=len(x)-(m-1)*tau
    if n<8: raise AtlasStructuralFail("EMBEDDING_TOO_SHORT","trajectory too short for requested embedding")
    return np.column_stack([x[i:i+n] for i in range(0,m*tau,tau)])


def atlas_lyap_spectrum(X,dt,k=10,theiler=5,max_steps=180):
    X=np.asarray(X,float); n,d=X.shape
    if n<max(50,10*d,k+10): raise AtlasStructuralFail("INSUFFICIENT_EMBEDDING_SUPPORT",f"n={n}, d={d}")
    X0,X1=X[:-1],X[1:]; tree=cKDTree(X0); anchors=np.linspace(0,len(X0)-2,min(max_steps,len(X0)-1)).round().astype(int); Q=np.eye(d); sums=np.zeros(d); used=0
    qk=min(len(X0),max(k*5+2*theiler+5,d+25))
    for i in anchors:
        try: _,idx=tree.query(X0[i],k=qk,workers=-1)
        except TypeError: _,idx=tree.query(X0[i],k=qk)
        idx=np.atleast_1d(idx); idx=idx[(idx>=0)&(idx<len(X0)-1)&(np.abs(idx-i)>theiler)][:k]
        if len(idx)<max(d+1,6): continue
        A=X0[idx]-X0[i]; B=X1[idx]-X1[i]
        if np.linalg.matrix_rank(A)<min(d,A.shape[1]): continue
        try: JT,*_=np.linalg.lstsq(A,B,rcond=None)
        except np.linalg.LinAlgError: continue
        Z=JT.T@Q; Q,R=np.linalg.qr(Z); diag=np.abs(np.diag(R)); diag=np.maximum(diag,np.finfo(float).tiny); sums+=np.log(diag); used+=1
    if used<max(20,3*d): raise AtlasStructuralFail("INSUFFICIENT_LOCAL_JACOBIANS",f"only {used} usable local-linear steps")
    return sums/(used*dt),used


def atlas_kaplan_yorke(exps):
    lam=np.sort(np.asarray(exps,float)[np.isfinite(exps)])[::-1]
    if not len(lam): return np.nan
    cs=np.cumsum(lam); non=np.where(cs>=0)[0]
    if not len(non): return 0.0
    j=non[-1]
    if j>=len(lam)-1: return float(len(lam))
    return float(j+1+cs[j]/(abs(lam[j+1])+1e-12))


def atlas_trimmed_mean(v,trim=0.1):
    v=np.sort(np.asarray(v,float)[np.isfinite(v)])
    if not len(v): return np.nan
    k=int(np.floor(trim*len(v)))
    if 2*k>=len(v): return float(np.mean(v))
    return float(np.mean(v[k:len(v)-k]))

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(4,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(4,dataset_id):
        atlas_register_ledger(dataset_id,4,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or "");
        e6=atlas_app_entry(6,dataset_id); atlas_register_ledger(dataset_id,6,e6.get("object_id"),e6.get("status"),"not_applicable",e6.get("reason_code") or "BANK_NA",e6.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); atlas_write_dataset_manifest(bundle); out=atlas_dataset_dir(dataset_id); rows=[]
        for curve in bundle["curves"]:
            try: x,y,dx,prepmeta=atlas_prepare_curve(bundle,curve,min_points=45,max_points=ATLAS_POINT_CAP)
            except AtlasStructuralFail: continue
            ys=atlas_robust_standardize(y)
            for m in range(2,11):
                try:
                    X=atlas_delay_embed(ys,m,1); n_eff=len(X)
                    # Explicit support gate: dimensions are not forced onto short curves.
                    if n_eff<max(55,10*m+20): raise AtlasStructuralFail("DIMENSION_UNDER_SUPPORTED",f"m={m}, n_eff={n_eff}")
                    exps,used=atlas_lyap_spectrum(X,dx,k=max(8,m+2),theiler=max(3,m),max_steps={"full":500,"balanced":320,"fast":180,"ultra":100}[ATLAS_SPEED])
                    rec={"curve":curve["label"],"observable":curve["observable"],"ensemble_group":curve.get("ensemble_group",""),"embedding_dimension":m,"n_points_used":n_eff,"steps_used":used,"coordinate_step":dx,"threshold_trimmed":prepmeta["threshold_trimmed"],"quality_status":"valid"}
                    for j,v in enumerate(exps,1): rec[f"Exp{j}"]=float(v)
                    rec.update({"LLE":float(np.max(exps)),"sum_exponents":float(np.sum(exps)),"positive_count":int(np.sum(exps>1e-3)),"KS_entropy_proxy":float(np.sum(exps[exps>0])),"Kaplan_Yorke_dim":atlas_kaplan_yorke(exps)})
                    rows.append(rec)
                except AtlasStructuralFail:
                    continue
        if not rows: raise AtlasStructuralFail("NO_VALID_LYAPUNOV_RESULTS","No curve/embedding combination passed support and local-Jacobian gates")
        df=pd.DataFrame(rows); stem="cell_04_lyapunov_spectrum_results"; df.to_csv(out/f"{stem}.csv",index=False)
        # Robust within-observable summaries; do not average unlike physics channels together.
        summ=[]
        for (obs,m),g in df.groupby(["observable","embedding_dimension"]):
            vals=g.LLE.to_numpy(float); med=float(np.median(vals)); mad=float(1.4826*np.median(np.abs(vals-med))) if len(vals)>1 else np.nan
            summ.append({"observable":obs,"embedding_dimension":m,"n":len(g),"LLE_median":med,"LLE_trimmed_mean":atlas_trimmed_mean(vals),"LLE_MAD":mad,"KS_proxy_median":float(np.median(g.KS_entropy_proxy)),"KY_median":float(np.median(g.Kaplan_Yorke_dim)),"positive_count_median":float(np.median(g.positive_count))})
        summary=pd.DataFrame(summ); summary.to_csv(out/"cell_06_lyapunov_summary_results.csv",index=False)
        fig,ax=plt.subplots(figsize=(10,6))
        for _,g in df.groupby("curve"):
            ax.plot(g.embedding_dimension,g.LLE,color=RED,alpha=.15,lw=1)
        agg=df.groupby("embedding_dimension").LLE.median(); ax.plot(agg.index,agg.values,color=WHITE,marker="o",lw=2.5,label="median across valid curves")
        support_by_dim=df.groupby("embedding_dimension").curve.nunique().sort_index()
        max_curve_support=int(support_by_dim.max()) if len(support_by_dim) else 0
        low_support_dims=support_by_dim[(support_by_dim<3) | (support_by_dim<0.75*max_curve_support)].index
        if len(low_support_dims):
            qc_y=agg.reindex(low_support_dims).dropna()
            ax.scatter(qc_y.index,qc_y.values,marker="x",s=90,linewidths=2.0,color=RED,label="QC-low/changing curve support")
        atlas_style_ax(ax,f"{dataset_id} — local-separation rate vs embedding dimension","Embedding dimension",f"Largest fitted exponent per {bundle['coordinate_analysis_name']}"); ax.legend(); atlas_save_fig(fig,out/f"{stem}_plots_01_lle_vs_embedding_dimension.png")
        # Outlier-safe features by physical observable.
        for obs,g in summary.groupby("observable"):
            obsdf=df[df.observable==obs]
            nobs=int(obsdf.curve.nunique())
            max_support=int(g["n"].max()) if len(g) else 0
            stable_rows=g[g["n"]>=max(3,int(np.ceil(0.75*max_support)))].copy()
            for _,r in g.iterrows():
                n_dim=int(r["n"])
                med_points=float(obsdf[obsdf.embedding_dimension==r.embedding_dimension].n_points_used.median())
                support_fraction=(n_dim/max_support) if max_support else 0.0
                quality=("low_support" if (n_dim<3 or med_points<55) else ("changing_curve_population" if support_fraction<0.75 else "valid"))
                atlas_register_feature(
                    dataset_id,4,"lyapunov_local_linear",obj,str(obs),bundle,
                    f"LLE_median_d{int(r.embedding_dimension)}",r.LLE_median,
                    "per_analysis_coordinate",quality,n_curves=n_dim,n_points=med_points,
                    notes=f"dimension-specific curve support={n_dim}/{max_support}"
                )
            if len(stable_rows):
                grand_q="valid" if len(stable_rows)>=3 else "low_support"
                atlas_register_feature(
                    dataset_id,6,"lyapunov_summary",atlas_app_entry(6,dataset_id).get("object_id"),
                    str(obs),bundle,"LLE_grand_median",stable_rows.LLE_median.median(),
                    "per_analysis_coordinate",grand_q,n_curves=max_support,
                    notes="summary restricted to dimensions retaining >=75% of maximum curve support"
                )
                atlas_register_feature(
                    dataset_id,6,"lyapunov_summary",atlas_app_entry(6,dataset_id).get("object_id"),
                    str(obs),bundle,"embedding_sensitivity_LLE_range",
                    stable_rows.LLE_median.max()-stable_rows.LLE_median.min(),
                    "per_analysis_coordinate","diagnostic_only",
                    notes="range across support-stable embedding dimensions only"
                )
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,4,obj,ent.get("status"),"valid","","robust observable-specific summaries; unsupported dimensions omitted",ncurves,npts,bundle); e6=atlas_app_entry(6,dataset_id); atlas_register_ledger(dataset_id,6,e6.get("object_id"),e6.get("status"),"valid","","summary from valid Cell-4 fits",ncurves,npts,bundle)
    except Exception as exc:
        atlas_record_exception(dataset_id,4,ent,exc,bundle); e6=atlas_app_entry(6,dataset_id); atlas_register_ledger(dataset_id,6,e6.get("object_id"),e6.get("status"),"skipped_dependency","CELL4_UNAVAILABLE",f"Cell 4 unavailable: {type(exc).__name__}: {exc}",bundle=bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cells 4/6 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 9
# Radiation-informed reduced Kuramoto Arnold tongue
# ============================================================

def atlas_phase_feature(y,dx,observable=""):
    y=np.asarray(y,float); obs=str(observable).lower()
    base=-np.log(np.clip(y,1e-12,None)) if "transmission" in obs and np.all(y>0) else atlas_robust_standardize(y)
    n=len(base); win=max(7,int(round(.03*n))); win += (win%2==0)
    if win>=n: win=n-1 if (n-1)%2 else n-2
    sm=savgol_filter(base,window_length=win,polyorder=min(3,win-2),mode="interp") if win>=5 else base
    slope=np.gradient(sm,dx); z=detrend(slope,type="linear"); z-=np.mean(z); sd=np.std(z)
    if not np.isfinite(sd) or sd<=1e-12: raise AtlasStructuralFail("DEGENERATE_PHASE_FEATURE","phase feature is degenerate")
    return z/sd


def atlas_characteristic_frequency(y,dx,observable=""):
    z=atlas_phase_feature(y,dx,observable); n=len(z); F=np.fft.rfft(z*np.hanning(n)); f=np.fft.rfftfreq(n,d=dx); p=np.abs(F)**2; df=f[1]-f[0]; nyq=f[-1]; low=max(3*df,.01*nyq); high=.90*nyq; mask=(f>=low)&(f<=high)&np.isfinite(p)
    if mask.sum()<5 or np.sum(p[mask])<=0: raise AtlasStructuralFail("INSUFFICIENT_SPECTRAL_SUPPORT","not enough Fourier support")
    idx=np.where(mask)[0]; j=int(idx[np.argmax(p[idx])]); peak=f[j]
    if 0<j<len(f)-1:
        a,b,c=np.log(p[j-1]+1e-300),np.log(p[j]+1e-300),np.log(p[j+1]+1e-300); den=a-2*b+c
        if np.isfinite(den) and abs(den)>1e-14: peak=float(f[j]+np.clip(.5*(a-c)/den,-.5,.5)*df)
    w=p[idx]+1e-300; centroid=float(np.sum(f[idx]*w)/np.sum(w)); spread=float(np.sqrt(np.sum((f[idx]-centroid)**2*w)/np.sum(w)))
    return centroid,float(peak),spread


def atlas_phase_ensemble_groups(bundle,min_points=48,min_curves=2):
    out=[]
    for group,curves in atlas_group_curves(bundle,minimum=min_curves).items():
        prepared=[]
        for c in curves:
            try:
                x,y,dx,_=atlas_prepare_curve(bundle,c,min_points=min_points,max_points=ATLAS_POINT_CAP); feat=atlas_phase_feature(y,dx,c["observable"]); phase=np.angle(hilbert(feat))/(2*np.pi)%1.0; prepared.append((c,x,phase,y,dx))
            except AtlasStructuralFail: pass
        if len(prepared)<min_curves: continue
        lo=max(p[1].min() for p in prepared); hi=min(p[1].max() for p in prepared); n=min(len(p[1]) for p in prepared)
        if hi<=lo or n<min_points: continue
        grid=np.linspace(lo,hi,n); ph=[]; labels=[]
        for c,x,p,_,_ in prepared:
            z=np.exp(1j*2*np.pi*p); zr=np.interp(grid,x,z.real); zi=np.interp(grid,x,z.imag); ph.append(np.angle(zr+1j*zi)/(2*np.pi)%1.0); labels.append(c["label"])
        out.append({"group":group,"grid":grid,"phase_matrix":np.column_stack(ph),"labels":labels,"prepared":prepared})
    return out


def atlas_kuramoto_rhs(theta,omega,K,a,b,t):
    z=np.mean(np.exp(1j*theta)); r=np.abs(z); psi=np.angle(z)
    return omega + K*r*np.sin(psi-theta) + a*np.sin(2*np.pi*b*t-theta)


def atlas_kuramoto_sim(omega,K,a,b,theta0,dt=.05,t_end=20.0):
    th=np.array(theta0,float); steps=max(80,int(t_end/dt)); start=steps//2; theta_ss=[]; r_ss=[]; t_ss=[]
    for s in range(steps):
        t=s*dt; k1=atlas_kuramoto_rhs(th,omega,K,a,b,t); k2=atlas_kuramoto_rhs(th+.5*dt*k1,omega,K,a,b,t+.5*dt); th=th+dt*k2
        if s>=start: theta_ss.append(th.copy()); r_ss.append(abs(np.mean(np.exp(1j*th)))); t_ss.append(t)
    theta_ss=np.asarray(theta_ss).T; t_ss=np.asarray(t_ss); drive_phase=2*np.pi*b*t_ss
    relative=np.exp(1j*(theta_ss-drive_phase[None,:])); per_osc=np.abs(np.mean(relative,axis=1)); drive_lock=float(np.mean(per_osc))
    z=np.mean(np.exp(1j*theta_ss),axis=0); collective=float(abs(np.mean(np.exp(1j*(np.angle(z)-drive_phase)))))
    unwrapped=np.unwrap(theta_ss,axis=1); obs_omega=(unwrapped[:,-1]-unwrapped[:,0])/(t_ss[-1]-t_ss[0]); ferr=float(np.mean(np.abs(obs_omega-2*np.pi*b)))
    return float(np.mean(r_ss)),float(np.std(r_ss)),drive_lock,collective,ferr

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(9,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(9,dataset_id): atlas_register_ledger(dataset_id,9,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); atlas_write_dataset_manifest(bundle); out=atlas_dataset_dir(dataset_id); groups=atlas_phase_ensemble_groups(bundle,min_points=48,min_curves=3)
        if not groups: raise AtlasStructuralFail("INSUFFICIENT_PHASE_ENSEMBLE","need >=3 comparable, nondegenerate phase-bearing curves with >=48 common points")
        succeeded=0
        for gp in groups:
            group=gp["group"]; freqrows=[]
            for c,x,phase,y,dx in gp["prepared"]:
                try: cen,peak,spread=atlas_characteristic_frequency(y,dx,c["observable"]); freqrows.append({"curve":c["label"],"observable":c["observable"],"spectral_centroid_frequency":cen,"peak_frequency":peak,"spectral_spread":spread})
                except AtlasStructuralFail: pass
            if len(freqrows)<3: continue
            fdf=pd.DataFrame(freqrows); median_f=float(np.median(fdf.spectral_centroid_frequency)); norm=fdf.spectral_centroid_frequency.to_numpy()/median_f; cv=float(np.std(norm)/np.mean(norm))
            if cv<.01: continue
            omega=2*np.pi*norm; spread=float(np.std(omega)); K=max(.05,.75*spread); center=float(np.median(norm)); half=max(.35,2.5*np.std(norm)); ngrid={"full":72,"balanced":50,"fast":34,"ultra":22}[ATLAS_SPEED]; aval=np.linspace(0,max(.5,4*spread),ngrid); bval=np.linspace(max(.02,center-half),center+half,ngrid)
            ninit={"full":5,"balanced":3,"fast":2,"ultra":1}[ATLAS_SPEED]; rng=np.random.default_rng(ATLAS_RANDOM_SEED); initials=[rng.uniform(0,2*np.pi,len(omega)) for _ in range(ninit)]
            sync=np.zeros((ngrid,ngrid)); syncsd=np.zeros_like(sync); lock=np.zeros_like(sync); collective=np.zeros_like(sync); ferr=np.zeros_like(sync)
            for ia,a in enumerate(aval):
                for ib,b in enumerate(bval):
                    vals=[atlas_kuramoto_sim(omega,K,float(a),float(b),th0,t_end={"full":35,"balanced":28,"fast":20,"ultra":14}[ATLAS_SPEED]) for th0 in initials]
                    sync[ia,ib]=np.mean([v[0] for v in vals]); syncsd[ia,ib]=np.mean([v[1] for v in vals]); lock[ia,ib]=np.mean([v[2] for v in vals]); collective[ia,ib]=np.mean([v[3] for v in vals]); ferr[ia,ib]=np.mean([v[4] for v in vals])
            # a=0 baseline for every drive-frequency column.
            sync_gain=sync-sync[0:1,:]; lock_gain=lock-lock[0:1,:]; ferr_improve=ferr[0:1,:]-ferr
            if np.nanmax(np.abs(lock-sync))<1e-8: raise RuntimeError("logic check failed: drive-lock matrix collapsed algebraically to synchronization")
            suffix=atlas_safe_name(group); stem=f"cell_09_arnold_tongue_kuramoto_{ATLAS_SPEED}_results__{suffix}"; fdf.assign(normalized_frequency=norm,omega_rad_per_model_unit=omega).to_csv(out/f"{stem}_characteristic_frequencies.csv",index=False)
            np.savez_compressed(out/f"{stem}.npz",sync=sync,sync_std=syncsd,drive_lock=lock,collective_drive_lock=collective,frequency_error=ferr,sync_gain=sync_gain,drive_lock_gain=lock_gain,frequency_error_improvement=ferr_improve,a_values=aval,b_values=bval,omega=omega,K=K)
            rows=[{"a":a,"b":b,"mean_sync":sync[ia,ib],"std_sync":syncsd[ia,ib],"drive_lock":lock[ia,ib],"collective_drive_lock":collective[ia,ib],"frequency_error":ferr[ia,ib],"sync_gain_vs_no_drive":sync_gain[ia,ib],"drive_lock_gain_vs_no_drive":lock_gain[ia,ib],"frequency_error_improvement_vs_no_drive":ferr_improve[ia,ib]} for ia,a in enumerate(aval) for ib,b in enumerate(bval)]; pd.DataFrame(rows).to_csv(out/f"{stem}.csv",index=False)
            for name,M,title in [("drive_lock_gain",lock_gain,"Drive-lock gain vs no drive"),("sync_gain",sync_gain,"Synchronization gain vs no drive"),("frequency_error_improvement",ferr_improve,"Frequency-error improvement vs no drive")]:
                lim=np.nanpercentile(np.abs(M),98) if np.isfinite(M).any() else 1; lim=max(float(lim),1e-12); fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(M,origin="lower",aspect="auto",extent=[bval.min(),bval.max(),aval.min(),aval.max()],cmap=ATLAS_DIV_CMAP,vmin=-lim,vmax=lim); atlas_style_ax(ax,f"{dataset_id} / {group} — {title}","Drive frequency b (cycles/model-unit)","Drive amplitude a"); cb=fig.colorbar(im,ax=ax); cb.ax.tick_params(colors=RED); atlas_save_fig(fig,out/f"{stem}_plots_{name}.png")
            q="valid" if len(freqrows)>=4 else "low_support"; atlas_register_feature(dataset_id,9,"kuramoto_reduced_phase",obj,group,bundle,"characteristic_frequency_cv",cv,"dimensionless",q,ensemble_group=group,n_curves=len(freqrows)); atlas_register_feature(dataset_id,9,"kuramoto_reduced_phase",obj,group,bundle,"max_drive_lock_gain_vs_no_drive",np.nanmax(lock_gain),"",q,ensemble_group=group); atlas_register_feature(dataset_id,9,"kuramoto_reduced_phase",obj,group,bundle,"max_sync_gain_vs_no_drive",np.nanmax(sync_gain),"",q,ensemble_group=group); atlas_register_feature(dataset_id,9,"kuramoto_reduced_phase",obj,group,bundle,"max_frequency_error_improvement_vs_no_drive",np.nanmax(ferr_improve),"rad/model-unit",q,ensemble_group=group); atlas_register_feature(dataset_id,9,"kuramoto_reduced_phase",obj,group,bundle,"max_drive_lock_raw",np.nanmax(lock),"","diagnostic_only",ensemble_group=group); succeeded+=1
        if not succeeded: raise AtlasStructuralFail("NO_IDENTIFIABLE_KURAMOTO_GROUP","all comparable groups were too small or frequency-degenerate")
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,9,obj,ent.get("status"),"valid","",f"{succeeded} comparable ensemble group(s)",ncurves,npts,bundle,successful_groups=succeeded)
    except Exception as exc: atlas_record_exception(dataset_id,9,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 9 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 13
# Fixed-point locking with a uniform-phase baseline contrast
# ============================================================

def atlas_circle_map(theta,Omega,K):
    return (theta+Omega-(K/(2*np.pi))*np.sin(2*np.pi*theta))%1.0

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(13,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(13,dataset_id): atlas_register_ledger(dataset_id,13,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); groups=atlas_phase_ensemble_groups(bundle,min_points=40,min_curves=2)
        if not groups: raise AtlasStructuralFail("INSUFFICIENT_PHASE_ENSEMBLE","need >=2 comparable phase-bearing curves and >=40 common coordinate points")
        succeeded=0
        for gp in groups:
            group=gp["group"]; grid=gp["grid"]; phase_mat=gp["phase_matrix"]; mean_z=np.mean(np.exp(1j*2*np.pi*phase_mat),axis=1); R=np.abs(mean_z); phase=(np.angle(mean_z)/(2*np.pi))%1.0
            if len(phase)>1200: phase=phase[np.linspace(0,len(phase)-1,1200).astype(int)]
            uniform=np.linspace(0,1,len(phase),endpoint=False); ngrid={"full":90,"balanced":64,"fast":44,"ultra":30}[ATLAS_SPEED]; O=np.linspace(0,1,ngrid); Ks=np.linspace(0,4*np.pi,ngrid); its={"full":24,"balanced":20,"fast":16,"ultra":12}[ATLAS_SPEED]; tol=1e-6
            rad=np.zeros((ngrid,ngrid)); base=np.zeros_like(rad)
            for i,K in enumerate(Ks):
                for j,Om in enumerate(O):
                    th=phase.copy(); ub=uniform.copy()
                    for _ in range(its): th=atlas_circle_map(th,Om,K); ub=atlas_circle_map(ub,Om,K)
                    nxt=atlas_circle_map(th,Om,K); nxtb=atlas_circle_map(ub,Om,K); d=np.minimum((nxt-th)%1,(th-nxt)%1); db=np.minimum((nxtb-ub)%1,(ub-nxtb)%1); rad[i,j]=np.mean(d<tol); base[i,j]=np.mean(db<tol)
            excess=rad-base; suffix=atlas_safe_name(group); stem=f"cell_13_circle_map_fixed_point_locking_results__{suffix}"; np.savez_compressed(out/f"{stem}.npz",radiation_locked=rad,uniform_baseline_locked=base,locking_excess=excess,omegas=O,K_values=Ks,phase=phase,resultant=R)
            pd.DataFrame([{"Omega":Om,"K":K,"radiation_locking":rad[i,j],"uniform_baseline_locking":base[i,j],"locking_excess":excess[i,j]} for i,K in enumerate(Ks) for j,Om in enumerate(O)]).to_csv(out/f"{stem}.csv",index=False)
            fig,ax=plt.subplots(figsize=(8,6)); lim=max(float(np.nanpercentile(np.abs(excess),98)),1e-12); im=ax.imshow(excess,origin="lower",aspect="auto",extent=[0,1,Ks.min(),Ks.max()],cmap=ATLAS_DIV_CMAP,vmin=-lim,vmax=lim); atlas_style_ax(ax,f"{dataset_id} / {group} — locking excess vs uniform initial phases","Ω","K"); cb=fig.colorbar(im,ax=ax); cb.ax.tick_params(colors=RED); atlas_save_fig(fig,out/f"{stem}_plots_01_locking_excess_vs_uniform.png")
            fig,ax=plt.subplots(figsize=(9,4)); ax.plot(grid,R,color=RED); atlas_style_ax(ax,f"{dataset_id} / {group} — cross-curve phase resultant",bundle["coordinate_analysis_name"],"Resultant length"); ax.set_ylim(0,1.02); atlas_save_fig(fig,out/f"{stem}_plots_02_phase_resultant_length.png")
            q="valid" if phase_mat.shape[1]>=3 else "low_support"; atlas_register_feature(dataset_id,13,"circle_map_fixed_point",obj,group,bundle,"mean_abs_locking_excess_vs_uniform",np.mean(np.abs(excess)),"",q,ensemble_group=group,n_curves=phase_mat.shape[1],n_points=phase_mat.shape[0]); atlas_register_feature(dataset_id,13,"circle_map_fixed_point",obj,group,bundle,"max_abs_locking_excess_vs_uniform",np.max(np.abs(excess)),"","scan_grid_dependent",ensemble_group=group,notes="maximum depends on Ω/K grid extent and resolution; retained as diagnostic"); atlas_register_feature(dataset_id,13,"circle_map_fixed_point",obj,group,bundle,"mean_phase_resultant",np.mean(R),"",q,ensemble_group=group); atlas_register_feature(dataset_id,13,"circle_map_fixed_point",obj,group,bundle,"mean_fixed_point_locking_raw",np.mean(rad),"","model_baseline_dominated",ensemble_group=group,notes="raw map locking is dominated by imposed circle-map geometry"); succeeded+=1
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,13,obj,ent.get("status"),"valid","",f"{succeeded} group(s); raw locking marked model-dominated",ncurves,npts,bundle,successful_groups=succeeded)
    except Exception as exc: atlas_record_exception(dataset_id,13,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 13 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 17
# Lifted circle-map rotation numbers: UNWRAPPED primary quantity
# ============================================================

def atlas_circle_lift(theta,Omega,K):
    return theta+Omega-(K/(2*np.pi))*np.sin(2*np.pi*theta)


def atlas_rotation_unwrapped(Omegas,K,theta0,iterations=150,transient=30):
    th=np.full_like(Omegas,float(theta0),dtype=float)
    for _ in range(transient): th=atlas_circle_lift(th,Omegas,K)
    start=th.copy()
    for _ in range(iterations): th=atlas_circle_lift(th,Omegas,K)
    return (th-start)/iterations


def atlas_rational_distance_unwrapped(rho,max_denominator=8):
    rho=np.asarray(rho,float); lo=int(np.floor(np.nanmin(rho)))-1; hi=int(np.ceil(np.nanmax(rho)))+1; vals=sorted(set(float(Fraction(p,q)) for q in range(1,max_denominator+1) for p in range(lo*q,hi*q+1))); arr=np.asarray(vals,float); return np.min(np.abs(rho[None,...]-arr[(slice(None),)+(None,)*rho.ndim]),axis=0)

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(17,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(17,dataset_id): atlas_register_ledger(dataset_id,17,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); groups=atlas_phase_ensemble_groups(bundle,min_points=40,min_curves=2)
        if not groups: raise AtlasStructuralFail("INSUFFICIENT_PHASE_ENSEMBLE","need >=2 comparable phase-bearing curves")
        succeeded=0
        for gp in groups:
            group=gp["group"]; P=gp["phase_matrix"]; theta0=(np.angle(np.mean(np.exp(1j*2*np.pi*P),axis=0))/(2*np.pi))%1.0; ngrid={"full":110,"balanced":76,"fast":52,"ultra":34}[ATLAS_SPEED]; O=np.linspace(0,1,ngrid); Ks=np.linspace(0,2*np.pi,ngrid); its={"full":300,"balanced":220,"fast":150,"ultra":90}[ATLAS_SPEED]; trans=max(20,its//5); allraw=[]
            for t0 in theta0:
                R=np.zeros((ngrid,ngrid))
                for i,K in enumerate(Ks): R[i]=atlas_rotation_unwrapped(O,K,t0,its,trans)
                allraw.append(R)
            allraw=np.stack(allraw); meanraw=np.mean(allraw,axis=0); stdraw=np.std(allraw,axis=0); modview=np.mod(meanraw,1.0); dist=atlas_rational_distance_unwrapped(meanraw,8)
            uniform_theta0=np.linspace(0,1,len(theta0),endpoint=False); baseall=[]
            for t0 in uniform_theta0:
                R=np.zeros((ngrid,ngrid))
                for i,K in enumerate(Ks): R[i]=atlas_rotation_unwrapped(O,K,t0,its,trans)
                baseall.append(R)
            baseall=np.stack(baseall); meanbase=np.mean(baseall,axis=0); stdbase=np.std(baseall,axis=0); basedist=atlas_rational_distance_unwrapped(meanbase,8)
            rotation_excess=meanraw-meanbase; std_excess=stdraw-stdbase
            raw_lock_fraction=float(np.mean(dist<=.01)); baseline_lock_fraction=float(np.mean(basedist<=.01)); rational_lock_excess=raw_lock_fraction-baseline_lock_fraction
            suffix=atlas_safe_name(group); stem=f"cell_17_circle_map_rotation_numbers_results__{suffix}"; np.savez_compressed(out/f"{stem}.npz",rotation_numbers_unwrapped_all=allraw,rotation_unwrapped_mean=meanraw,rotation_unwrapped_std=stdraw,rotation_mod1_view=modview,rational_distance_unwrapped=dist,uniform_baseline_rotation_numbers_unwrapped_all=baseall,uniform_baseline_rotation_unwrapped_mean=meanbase,uniform_baseline_rotation_unwrapped_std=stdbase,uniform_baseline_rational_distance_unwrapped=basedist,rotation_excess_vs_uniform=rotation_excess,rotation_std_excess_vs_uniform=std_excess,omegas=O,K_values=Ks,theta0=theta0,uniform_theta0=uniform_theta0)
            pd.DataFrame({"curve":gp["labels"],"theta0_cycles":theta0,"uniform_baseline_theta0_cycles":uniform_theta0}).to_csv(out/f"{stem}.csv",index=False)
            for name,M,title in [("unwrapped_mean",meanraw,"Unwrapped mean rotation increment"),("unwrapped_std",stdraw,"Cross-curve unwrapped rotation variability"),("rational_distance_unwrapped",dist,"Distance to low-order rational (unwrapped)"),("rotation_excess_vs_uniform",rotation_excess,"Unwrapped rotation excess vs uniform-phase baseline")]:
                fig,ax=plt.subplots(figsize=(8,6))
                if name=="rotation_excess_vs_uniform":
                    lim=float(np.nanmax(np.abs(M))) if np.isfinite(M).any() else 1.0
                    if not np.isfinite(lim) or lim<=0: lim=1.0
                    im=ax.imshow(M,origin="lower",aspect="auto",extent=[0,1,Ks.min(),Ks.max()],cmap=ATLAS_DIV_CMAP,vmin=-lim,vmax=lim)
                else:
                    im=ax.imshow(M,origin="lower",aspect="auto",extent=[0,1,Ks.min(),Ks.max()],cmap=ATLAS_CMAP)
                atlas_style_ax(ax,f"{dataset_id} / {group} — {title}","Ω","K"); cb=fig.colorbar(im,ax=ax); cb.ax.tick_params(colors=RED); atlas_save_fig(fig,out/f"{stem}_plots_{name}.png")
            q="valid" if len(theta0)>=3 else "low_support"
            raw_note="unwrapped estimator is valid, but the global summary is dominated by the common Ω/K scan unless baseline-referenced"
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"mean_unwrapped_rotation",np.mean(meanraw),"cycles_per_iterate","model_baseline_dominated",ensemble_group=group,n_curves=len(theta0),notes=raw_note)
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"mean_cross_curve_unwrapped_rotation_std",np.mean(stdraw),"cycles_per_iterate","model_baseline_dominated",ensemble_group=group,notes=raw_note)
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"fraction_near_low_order_rational_unwrapped",raw_lock_fraction,"","model_baseline_dominated",ensemble_group=group,notes=raw_note)
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"mean_abs_unwrapped_rotation_excess_vs_uniform",np.mean(np.abs(rotation_excess)),"cycles_per_iterate",q,ensemble_group=group,n_curves=len(theta0),notes="uniform-initial-phase baseline removes the common circle-map scan contribution")
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"mean_abs_cross_curve_rotation_std_excess_vs_uniform",np.mean(np.abs(std_excess)),"cycles_per_iterate",q,ensemble_group=group,n_curves=len(theta0),notes="uniform-initial-phase baseline contrast")
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"fraction_near_low_order_rational_excess_vs_uniform",rational_lock_excess,"",q,ensemble_group=group,n_curves=len(theta0),notes=f"raw fraction={raw_lock_fraction:.6g}; uniform baseline={baseline_lock_fraction:.6g}")
            atlas_register_feature(dataset_id,17,"circle_map_rotation",obj,group,bundle,"mean_rotation_mod1",np.mean(modview),"","diagnostic_only",ensemble_group=group,notes="modulo-one representation retained only for circular visualization"); succeeded+=1
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,17,obj,ent.get("status"),"valid","",f"{succeeded} group(s); unwrapped rotation remains primary and model features use a uniform-phase baseline contrast",ncurves,npts,bundle,successful_groups=succeeded)
    except Exception as exc: atlas_record_exception(dataset_id,17,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 17 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 19
# Finite-time phase pushforward with stabilization + uniform baseline
# ============================================================

def atlas_norm_concentration(prob):
    p=np.asarray(prob,float); n=p.shape[-1]; s=np.sum(p*p,axis=-1); return (s-1/n)/(1-1/n) if n>1 else np.zeros(p.shape[:-1])

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(19,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(19,dataset_id): atlas_register_ledger(dataset_id,19,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); groups=atlas_phase_ensemble_groups(bundle,min_points=32,min_curves=1)
        if not groups: raise AtlasStructuralFail("NO_PHASE_DISTRIBUTION","no phase-bearing curve with >=32 points")
        succeeded=0
        for gp in groups:
            group=gp["group"]; P=gp["phase_matrix"]; Omega=1/3; nk={"full":150,"balanced":96,"fast":62,"ultra":40}[ATLAS_SPEED]; Ks=np.linspace(0,4*np.pi,nk); nbins=40; edges=np.linspace(0,1,nbins+1); its={"full":60,"balanced":45,"fast":32,"ultra":24}[ATLAS_SPEED]; extra=max(8,its//3); allconc=[]; alltvs=[]; excessconc=[]; uniform_init=np.linspace(0,1,min(1200,P.shape[0]),endpoint=False)
            baseline_conc=np.zeros(nk)
            for i,K in enumerate(Ks):
                u=uniform_init.copy()
                for _ in range(its): u=atlas_circle_map(u,Omega,K)
                up=np.histogram(u,bins=edges)[0].astype(float); up/=up.sum()+1e-12; baseline_conc[i]=atlas_norm_concentration(up)
            meanhist=np.zeros((nk,nbins));
            for j in range(P.shape[1]):
                init=P[:,j];
                if len(init)>1200: init=init[np.linspace(0,len(init)-1,1200).round().astype(int)]
                conc=np.zeros(nk); tv=np.zeros(nk); hists=np.zeros((nk,nbins))
                for i,K in enumerate(Ks):
                    th=init.copy()
                    for _ in range(its): th=atlas_circle_map(th,Omega,K)
                    p=np.histogram(th,bins=edges)[0].astype(float); p/=p.sum()+1e-12; hists[i]=p; conc[i]=atlas_norm_concentration(p); th2=th.copy()
                    for _ in range(extra): th2=atlas_circle_map(th2,Omega,K)
                    q=np.histogram(th2,bins=edges)[0].astype(float); q/=q.sum()+1e-12; tv[i]=.5*np.sum(np.abs(p-q))
                meanhist+=hists/P.shape[1]; allconc.append(conc); alltvs.append(tv); excessconc.append(conc-baseline_conc)
            allconc=np.asarray(allconc); alltvs=np.asarray(alltvs); excessconc=np.asarray(excessconc); meanconc=np.mean(allconc,axis=0); meanTV=np.mean(alltvs,axis=0); meanex=np.mean(excessconc,axis=0); suffix=atlas_safe_name(group); stem=f"cell_19_circle_map_phase_pushforward_results__{suffix}"; np.savez_compressed(out/f"{stem}.npz",avg_hist_prob=meanhist,K_values=Ks,concentration=allconc,stabilization_tv=alltvs,uniform_baseline_concentration=baseline_conc,concentration_excess_vs_uniform=excessconc)
            pd.DataFrame({"K":Ks,"mean_concentration":meanconc,"uniform_baseline_concentration":baseline_conc,"mean_concentration_excess":meanex,"mean_stabilization_TV":meanTV}).to_csv(out/f"{stem}.csv",index=False)
            fig,ax=plt.subplots(figsize=(9,5)); ax.plot(Ks,meanTV,color=RED); ax.axhline(ATLAS_TV_STABLE_THRESHOLD,color=WHITE,ls="--"); atlas_style_ax(ax,f"{dataset_id} / {group} — finite-time stabilization","K","TV distance after extra iterations"); atlas_save_fig(fig,out/f"{stem}_plots_01_stabilization_tv.png")
            fig,ax=plt.subplots(figsize=(9,5)); ax.plot(Ks,meanex,color=RED); atlas_style_ax(ax,f"{dataset_id} / {group} — concentration excess vs uniform baseline","K","Concentration excess"); atlas_save_fig(fig,out/f"{stem}_plots_02_concentration_excess_vs_uniform.png")
            mtv=float(np.mean(meanTV)); p90_tv=float(np.percentile(meanTV,90)); bad_frac=float(np.mean(meanTV>ATLAS_TV_STABLE_THRESHOLD))
            q="stable" if (mtv<=ATLAS_TV_STABLE_THRESHOLD and p90_tv<=ATLAS_TV_P90_THRESHOLD and bad_frac<=ATLAS_TV_BAD_FRACTION_MAX) else "unstable_pushforward"
            note=f"mean TV={mtv:.4g}; p90 TV={p90_tv:.4g}; fraction TV>{ATLAS_TV_STABLE_THRESHOLD:.3g}={bad_frac:.3f}"
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"mean_concentration_excess_vs_uniform",np.mean(meanex),"",q,ensemble_group=group,n_curves=P.shape[1],n_points=P.shape[0],notes=note)
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"mean_abs_concentration_excess_vs_uniform",np.mean(np.abs(meanex)),"",q,ensemble_group=group,n_curves=P.shape[1],n_points=P.shape[0],notes=note)
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"mean_stabilization_TV",mtv,"TV","diagnostic_only",ensemble_group=group)
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"p90_stabilization_TV",p90_tv,"TV","diagnostic_only",ensemble_group=group)
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"fraction_K_above_TV_threshold",bad_frac,"","diagnostic_only",ensemble_group=group)
            atlas_register_feature(dataset_id,19,"circle_map_phase_pushforward",obj,group,bundle,"mean_distribution_concentration_raw",np.mean(meanconc),"","model_baseline_dominated",ensemble_group=group)
            succeeded+=1
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,19,obj,ent.get("status"),"valid","",f"{succeeded} group(s); unstable groups retained but excluded from model",ncurves,npts,bundle,successful_groups=succeeded)
    except Exception as exc: atlas_record_exception(dataset_id,19,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 19 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 21
# NN-distance entropy with sampling-geometry diagnostic and Theiler exclusion
# ============================================================

def atlas_nn_metrics(X,bins=70,theiler=0):
    X=np.asarray(X,float); n=len(X); tree=cKDTree(X); k=min(n,max(8,2*theiler+8));
    try:d,idx=tree.query(X,k=k,workers=-1)
    except TypeError:d,idx=tree.query(X,k=k)
    if k==1: d=d[:,None]; idx=idx[:,None]
    nn=[]; gaps=[]
    for i in range(n):
        chosen=None
        for jj,dd in zip(np.atleast_1d(idx[i])[1:],np.atleast_1d(d[i])[1:]):
            if np.isfinite(dd) and dd>0 and abs(int(jj)-i)>theiler:
                chosen=(float(dd),abs(int(jj)-i)); break
        if chosen is not None: nn.append(chosen[0]); gaps.append(chosen[1])
    nn=np.asarray(nn,float); gaps=np.asarray(gaps,int)
    if len(nn)<max(20,.25*n): raise AtlasStructuralFail("TOO_FEW_NONLOCAL_NEIGHBORS",f"only {len(nn)} acceptable neighbors out of {n}")
    counts,_=np.histogram(nn,bins=bins); p=counts[counts>0]/np.sum(counts); H=float(-np.sum(p*np.log2(p)))
    return H,nn,gaps

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(21,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(21,dataset_id): atlas_register_ledger(dataset_id,21,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); rows=[]
        for c in bundle["curves"]:
            try:x,y,dx,_=atlas_prepare_curve(bundle,c,min_points=18,max_points=ATLAS_POINT_CAP)
            except AtlasStructuralFail: continue
            ys=atlas_robust_standardize(y)
            for m in range(2,11):
                try:
                    X=atlas_delay_embed(ys,m,1); n=len(X)
                    if n<max(35,6*m+15): continue
                    tw=max(2,m)
                    Hraw,nnraw,graw=atlas_nn_metrics(X,70,theiler=0)
                    Hth,nnth,gth=atlas_nn_metrics(X,70,theiler=tw)
                    try:
                        Hth2,nnth2,gth2=atlas_nn_metrics(X,70,theiler=2*tw)
                        sensitivity=abs(Hth2-Hth)
                    except AtlasStructuralFail:
                        Hth2=np.nan; sensitivity=np.nan
                    adj=float(np.mean(graw==1)); q="sampling_geometry_dominated" if adj>ATLAS_ADJACENCY_DOMINANCE_THRESHOLD else "valid"
                    rows.append({"curve":c["label"],"observable":c["observable"],"ensemble_group":c.get("ensemble_group",""),"embedding_dimension":m,"nn_distance_entropy_raw_bits":Hraw,"nn_distance_entropy_theiler_bits":Hth,"nn_distance_entropy_theiler2_bits":Hth2,"theiler_sensitivity_bits":sensitivity,"theiler_window":tw,"nearest_distance_mean_theiler":float(np.mean(nnth)),"nearest_distance_cv_theiler":float(np.std(nnth)/(np.mean(nnth)+1e-12)),"adjacent_neighbor_fraction_raw":adj,"n_points":n,"raw_quality_status":q})
                except AtlasStructuralFail: continue
        if not rows: raise AtlasStructuralFail("NO_VALID_NN_ENTROPY_RESULTS","no embeddings passed neighbor-support gates")
        df=pd.DataFrame(rows); stem="cell_21_nearest_neighbor_distance_entropy_results"; df.to_csv(out/f"{stem}.csv",index=False); summ=df.groupby("embedding_dimension").agg(theiler_entropy_median=("nn_distance_entropy_theiler_bits","median"),raw_entropy_median=("nn_distance_entropy_raw_bits","median"),adjacent_fraction_median=("adjacent_neighbor_fraction_raw","median"),n=("curve","count")).reset_index(); summ.to_csv(out/f"{stem}_dimension_summary.csv",index=False)
        fig,ax=plt.subplots(figsize=(9,6)); ax.plot(summ.embedding_dimension,summ.theiler_entropy_median,color=RED,marker="o",label="Theiler-excluded"); ax.plot(summ.embedding_dimension,summ.raw_entropy_median,color=WHITE,marker="o",alpha=.8,label="raw NN"); atlas_style_ax(ax,f"{dataset_id} — nearest-neighbor distance entropy","Embedding dimension","Entropy (bits)"); ax.legend(); atlas_save_fig(fig,out/f"{stem}_plots_01_entropy_by_dimension.png")
        fig,ax=plt.subplots(figsize=(9,5)); ax.plot(summ.embedding_dimension,summ.adjacent_fraction_median,color=RED,marker="o"); ax.axhline(ATLAS_ADJACENCY_DOMINANCE_THRESHOLD,color=WHITE,ls="--"); atlas_style_ax(ax,f"{dataset_id} — raw nearest-neighbor adjacency diagnostic","Embedding dimension","Fraction |Δindex|=1"); ax.set_ylim(0,1); atlas_save_fig(fig,out/f"{stem}_plots_02_adjacent_neighbor_fraction.png")
        # Theiler-excluded entropy is the candidate feature; raw entropy is diagnostic if dominated by adjacent sampling geometry.
        for obs,g in df.groupby("observable"):
            adj=float(g.adjacent_neighbor_fraction_raw.mean())
            nobs=int(g.curve.nunique())
            nmed=float(g.n_points.median())
            sens=float(g.theiler_sensitivity_bits.dropna().median()) if g.theiler_sensitivity_bits.notna().any() else np.inf
            q="theiler_excluded" if (
                nobs>=ATLAS_MIN_MODEL_CURVES
                and nmed>=ATLAS_MIN_MODEL_POINTS
                and sens<=ATLAS_MAX_THEILER_ENTROPY_SENSITIVITY_BITS
            ) else "exploratory_nn_entropy"
            note=f"raw adjacency={adj:.4g}; curves={nobs}; median points={nmed:.0f}; median |H(2W)-H(W)|={sens:.4g} bits"
            atlas_register_feature(dataset_id,21,"nearest_neighbor_distance_entropy",obj,str(obs),bundle,"NN_entropy_theiler_grand_median",g.nn_distance_entropy_theiler_bits.median(),"bits",q,n_curves=nobs,n_points=nmed,notes=note)
            atlas_register_feature(dataset_id,21,"nearest_neighbor_distance_entropy",obj,str(obs),bundle,"NN_entropy_raw_grand_median",g.nn_distance_entropy_raw_bits.median(),"bits","sampling_geometry_dominated" if adj>ATLAS_ADJACENCY_DOMINANCE_THRESHOLD else "diagnostic_only")
            atlas_register_feature(dataset_id,21,"nearest_neighbor_distance_entropy",obj,str(obs),bundle,"adjacent_neighbor_fraction_raw",adj,"","diagnostic_only")
            if np.isfinite(sens):
                atlas_register_feature(dataset_id,21,"nearest_neighbor_distance_entropy",obj,str(obs),bundle,"median_theiler_window_sensitivity_bits",sens,"bits","diagnostic_only")
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,21,obj,ent.get("status"),"valid","","Theiler-excluded entropy used as candidate; raw adjacency recorded",ncurves,npts,bundle)
    except Exception as exc: atlas_record_exception(dataset_id,21,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 21 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 25
# MI coordinate lag + Pyragas-style delayed difference
# ============================================================

def atlas_mutual_information_for_lags(signal,lags):
    y=atlas_robust_standardize(signal); out=[]
    for lag in np.asarray(lags,int):
        if lag<=0 or lag>=len(y)-2: out.append(np.nan); continue
        out.append(mutual_info_regression(y[:-lag].reshape(-1,1),y[lag:],discrete_features=False,random_state=ATLAS_RANDOM_SEED)[0])
    return np.asarray(out,float)


def atlas_select_mi_delay(signal,dx,min_fraction=.005,max_fraction=.20,max_samples=3000):
    y=atlas_robust_standardize(signal); n=len(y); dec=max(1,int(np.ceil(n/max_samples))); yy=y[::dec]; n2=len(yy); lo=max(1,int(round(min_fraction*n2))); hi=min(n2//3,max(lo+2,int(round(max_fraction*n2))))
    if hi<=lo+1: raise AtlasStructuralFail("MI_LAG_RANGE_TOO_SMALL","not enough lag candidates")
    nc={"full":160,"balanced":120,"fast":80,"ultra":50}[ATLAS_SPEED]; coarse=np.unique(np.round(np.linspace(lo,hi,min(nc,hi-lo+1))).astype(int)); cmi=atlas_mutual_information_for_lags(yy,coarse); valid=np.isfinite(cmi)
    if valid.sum()<3: raise AtlasStructuralFail("MI_ESTIMATION_FAILED","too few finite MI values")
    cv=coarse[valid]; mv=cmi[valid]; mins=np.where((mv[1:-1]<mv[:-2])&(mv[1:-1]<mv[2:]))[0]+1
    if len(mins): best=int(mins[0]); method="first local minimum"
    else: best=int(np.argmin(mv)); method="global minimum in allowed range"
    chosen=int(cv[best]); spacing=int(max(1,np.median(np.diff(cv)))) if len(cv)>1 else 1; rr=max(6,spacing); refine=np.arange(max(lo,chosen-rr),min(hi,chosen+rr)+1,dtype=int); rmi=atlas_mutual_information_for_lags(yy,refine); rv=np.isfinite(rmi)
    if rv.sum()>=3:
        xr=refine[rv]; mr=rmi[rv]; rmins=np.where((mr[1:-1]<mr[:-2])&(mr[1:-1]<mr[2:]))[0]+1
        if len(rmins): chosen=int(xr[int(rmins[0])]); method="first local minimum"
        else: chosen=int(xr[int(np.argmin(mr))]); method="global minimum in allowed range"
    lag_samples=int(chosen*dec); lag_coord=float(lag_samples*dx); lags=np.unique(np.r_[coarse,refine])*dec; mi=atlas_mutual_information_for_lags(y,lags)
    return {"delay_samples":lag_samples,"delay_coordinate":lag_coord,"lags_samples":lags,"lags_coordinate":lags*dx,"mi":mi,"selection_method":method,"decimation_factor":dec}


def atlas_pyragas(y,delay,gain=.1):
    y=np.asarray(y,float); delayed=np.empty_like(y); delayed[:delay]=y[0]; delayed[delay:]=y[:-delay]; fb=gain*(delayed-y); return y+fb,fb

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(25,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(25,dataset_id):
        atlas_register_ledger(dataset_id,25,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or "")
        continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); prepared=[]; micurves=[]
        for c in bundle["curves"]:
            try:
                x,y,dx,_=atlas_prepare_curve(bundle,c,min_points=24,max_points=ATLAS_POINT_CAP)
                mi=atlas_select_mi_delay(y,dx)
                grp=c.get("ensemble_group") or c.get("observable") or "default"
                prepared.append((c,x,y,dx,mi,grp))
                micurves.extend([
                    {"curve":c["label"],"observable":c["observable"],"ensemble_group":grp,
                     "lag_samples":int(l),"lag_coordinate":float(lc),"mutual_information":float(v)}
                    for l,lc,v in zip(mi["lags_samples"],mi["lags_coordinate"],mi["mi"]) if np.isfinite(v)
                ])
            except AtlasStructuralFail:
                pass
        if not prepared:
            raise AtlasStructuralFail("NO_MI_CAPABLE_CURVES","no curves long enough/nondegenerate for MI delay estimation")

        # Select the common lag independently within each physical ensemble group.
        # This prevents algebraically related or physically unlike observables from
        # double-weighting a single dataset-wide lag.
        group_common={}
        for grp in sorted(set(p[5] for p in prepared)):
            vals=[p[4]["delay_coordinate"] for p in prepared if p[5]==grp]
            group_common[grp]=float(np.median(vals))

        rows=[]
        for c,x,y,dx,mi,grp in prepared:
            common_coord=group_common[grp]
            d=int(np.clip(round(common_coord/dx),1,len(y)-2))
            ctrl,fb=atlas_pyragas(y,d,.1)
            rough0=np.sqrt(np.mean(np.gradient(y,dx)**2))
            rough1=np.sqrt(np.mean(np.gradient(ctrl,dx)**2))
            rms0=np.sqrt(np.mean(y*y)); rmsfb=np.sqrt(np.mean(fb*fb))
            rows.append({
                "curve":c["label"],"observable":c["observable"],"ensemble_group":grp,
                "individual_MI_delay_samples":mi["delay_samples"],
                "individual_MI_delay_coordinate":mi["delay_coordinate"],
                "selection_method":mi["selection_method"],
                "common_delay_coordinate":common_coord,
                "applied_delay_samples":d,"applied_delay_coordinate":d*dx,
                "relative_feedback_rms":rmsfb/(rms0+1e-12),
                "roughness_ratio":rough1/(rough0+1e-12),
                "variance_ratio":np.var(ctrl)/(np.var(y)+1e-12),
                "correlation_original_controlled":np.corrcoef(y,ctrl)[0,1]
            })

        df=pd.DataFrame(rows); stem="cell_25_pyragas_delayed_feedback_results"
        df.to_csv(out/f"{stem}.csv",index=False)
        pd.DataFrame(micurves).to_csv(out/f"{stem}_mi_curves.csv",index=False)

        # Group-specific vertical lag markers show the selected delay for each
        # physical ensemble group.
        fig,ax=plt.subplots(figsize=(10,6)); mdf=pd.DataFrame(micurves)
        for label,g in mdf.groupby("curve"):
            ax.plot(g.lag_coordinate,g.mutual_information,color=RED,alpha=.18,lw=1)
        for k,(grp,lag) in enumerate(group_common.items()):
            ax.axvline(lag,color=WHITE,ls="--",alpha=max(.35,1.0-.08*k),
                       label=f"{grp} median lag")
        atlas_style_ax(ax,f"{dataset_id} — mutual-information coordinate lag",
                       f"Lag in {bundle['coordinate_analysis_name']}","Mutual information")
        if len(group_common)<=8: ax.legend()
        atlas_save_fig(fig,out/f"{stem}_plots_01_mutual_information_lags.png")

        plot=df.sort_values("relative_feedback_rms").copy()
        n_all=len(plot)
        if n_all>30:
            plot=pd.concat([plot.head(15),plot.tail(15)]).drop_duplicates(subset=["curve"])
            plot=plot.sort_values("relative_feedback_rms")
            plot_title=f"{dataset_id} — delayed-difference magnitude (30 extremes; all {n_all} curves in CSV)"
        else:
            plot_title=f"{dataset_id} — delayed-difference magnitude"

        scalar_plot_path=out/f"{stem}_plots_02_relative_feedback.png"
        if len(plot)>=2:
            fig_h=max(4.5,min(12.0,0.28*len(plot)+2.2))
            fig,ax=plt.subplots(figsize=(9,fig_h))
            ax.barh(plot.curve.astype(str),plot.relative_feedback_rms,color=RED)
            atlas_style_ax(ax,plot_title,"Relative feedback RMS","Curve")
            atlas_save_fig(fig,scalar_plot_path)
        elif scalar_plot_path.exists():
            scalar_plot_path.unlink()

        # Registry is observable-specific; no all_observables feature is promoted.
        for obs,g in df.groupby("observable"):
            nobs=int(g.curve.nunique())
            local_frac=float(g.selection_method.str.contains("first local minimum",case=False).mean())
            q="valid" if nobs>=ATLAS_MIN_MODEL_CURVES and local_frac>=.5 else "weak_or_low_support_MI_delay"
            grp=str(g.ensemble_group.iloc[0]) if "ensemble_group" in g else ""
            atlas_register_feature(dataset_id,25,"pyragas_coordinate_transform",obj,str(obs),bundle,
                                   "median_MI_delay_coordinate",g.individual_MI_delay_coordinate.median(),
                                   bundle["coordinate_unit"],q,ensemble_group=grp,n_curves=nobs,
                                   notes=f"observable-specific; local-minimum fraction={local_frac:.3f}")
            atlas_register_feature(dataset_id,25,"pyragas_coordinate_transform",obj,str(obs),bundle,
                                   "median_relative_feedback_rms",g.relative_feedback_rms.median(),"",q,
                                   ensemble_group=grp,n_curves=nobs)
            atlas_register_feature(dataset_id,25,"pyragas_coordinate_transform",obj,str(obs),bundle,
                                   "fraction_MI_delays_with_local_minimum",local_frac,"","diagnostic_only",
                                   ensemble_group=grp,n_curves=nobs)
            atlas_register_feature(dataset_id,25,"pyragas_coordinate_transform",obj,str(obs),bundle,
                                   "median_roughness_ratio",g.roughness_ratio.median(),"","diagnostic_only",
                                   ensemble_group=grp,n_curves=nobs)

        ncurves,npts=atlas_structural_summary(bundle)
        atlas_register_ledger(dataset_id,25,obj,ent.get("status"),"valid","",
                              "common lag selected in analysis-coordinate units within physical ensemble groups",
                              ncurves,npts,bundle)
    except Exception as exc:
        atlas_record_exception(dataset_id,25,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 25 complete.")


In [ ]:
# ============================================================
# CROSS-DATASET EXECUTION — CELL 28
# Kakutani/Brouwer construction + blocked held-out affine validation
# ============================================================

def atlas_r2(y,yhat):
    y=np.asarray(y,float); yhat=np.asarray(yhat,float); ss=np.sum((y-yhat)**2); st=np.sum((y-np.mean(y,axis=0,keepdims=True))**2); return float(1-ss/(st+1e-12))

for dataset_id in ATLAS_DATASETS:
    ent=atlas_app_entry(28,dataset_id); obj=ent.get("object_id"); bundle=None
    if not atlas_bank_allows(28,dataset_id): atlas_register_ledger(dataset_id,28,obj,ent.get("status"),"not_applicable",ent.get("reason_code") or "BANK_NA",ent.get("note") or ""); continue
    try:
        bundle=atlas_load_bundle(dataset_id,obj); out=atlas_dataset_dir(dataset_id); state_groups=bundle.get("state_groups",{})
        # Strict anti-overfit gate: P003-like 15-state and 2-channel examples no longer
        # receive near-perfect but meaningless affine scores.
        candidates=[(g,s) for g,s in state_groups.items() if s["matrix"].shape[0]>=30 and s["matrix"].shape[1]>=3]
        if not candidates: raise AtlasStructuralFail("INSUFFICIENT_MULTICHANNEL_STATE","need >=30 ordered states and >=3 nondegenerate channels in a comparable state group")
        succeeded=0
        for group,state in candidates:
            coord=np.asarray(state["coordinate"],float); X=np.asarray(state["matrix"],float); labels=np.asarray(state["labels"],dtype=object); good=np.all(np.isfinite(X),axis=1); coord,X=coord[good],X[good]; n=len(X); split=max(20,int(.70*n)); split=min(split,n-10)
            if split<20 or n-split<10: continue
            mu=X[:split].mean(axis=0); sd=X[:split].std(axis=0); keep=sd>1e-10; X=X[:,keep]; labels=labels[keep]
            if X.shape[1]<3: continue
            Xs=(X-mu[keep])/(sd[keep]+1e-12); U,s,Vt=np.linalg.svd(Xs[:split],full_matrices=False); comps=Vt[:2].T; Z=Xs@comps; explained=(s*s)/(np.sum(s*s)+1e-12); train0=Z[:split-1]; train1=Z[1:split]; test0=Z[split:-1]; test1=Z[split+1:]
            B=np.linalg.lstsq(np.c_[train0,np.ones(len(train0))],train1,rcond=None)[0]; A=B[:2].T; b=B[2]; lo=Z[:split].min(axis=0); hi=Z[:split].max(axis=0)
            def g(z): return np.clip(A@np.asarray(z)+b,lo,hi)
            pred_train=np.array([g(z) for z in train0]); pred_test=np.array([g(z) for z in test0]); resid_train=np.linalg.norm(train1-pred_train,axis=1); resid_test=np.linalg.norm(test1-pred_test,axis=1); epsilon=float(np.median(resid_train)); train_r2=atlas_r2(train1,pred_train); test_r2=atlas_r2(test1,pred_test); train_rmse=float(np.sqrt(np.mean(np.sum((train1-pred_train)**2,axis=1)))); test_rmse=float(np.sqrt(np.mean(np.sum((test1-pred_test)**2,axis=1))))
            seeds=np.vstack([Z[:split].mean(axis=0),Z[np.linspace(0,split-1,min(8,split)).astype(int)]]); best=None
            for seed in seeds:
                fit=least_squares(lambda z:z-g(z),np.clip(seed,lo,hi),bounds=(lo,hi),max_nfev=500); rr=float(np.linalg.norm(fit.x-g(fit.x))); best=(rr,fit.x,fit) if best is None or rr<best[0] else best
            fixed_resid,zstar,fit=best; near=float(np.min(np.linalg.norm(Z-zstar,axis=1))); suffix=atlas_safe_name(group); stem=f"cell_28_kakutani_fixed_point_validation_results__{suffix}"; row={"state_group":group,"n_states":n,"n_channels":X.shape[1],"train_states":split,"test_states":n-split,"pca_pc1_train_explained_fraction":explained[0],"pca_pc2_train_explained_fraction":explained[1],"pca_first_two_train_explained_fraction":explained[:2].sum(),"affine_r2_train":train_r2,"affine_r2_heldout":test_r2,"affine_rmse_train":train_rmse,"affine_rmse_heldout":test_rmse,"epsilon_median_train_residual":epsilon,"heldout_residual_median":float(np.median(resid_test)),"fixed_inclusion_residual":fixed_resid,"inside_F_train_radius":bool(fixed_resid<=epsilon+1e-10),"nearest_observed_state_distance":near}; pd.DataFrame([row]).to_csv(out/f"{stem}.csv",index=False); np.savez_compressed(out/f"{stem}.npz",coordinate=coord,Z=Z,A=A,b=b,epsilon=epsilon,z_star=zstar,residual_train=resid_train,residual_heldout=resid_test,channel_labels=labels,split_index=split)
            fig,ax=plt.subplots(figsize=(8,7)); ax.scatter(Z[:split,0],Z[:split,1],s=8,color=RED,alpha=.3,label="train"); ax.scatter(Z[split:,0],Z[split:,1],s=10,color=WHITE,alpha=.6,label="held-out"); ax.scatter([zstar[0]],[zstar[1]],marker="x",s=120,color=WHITE); atlas_style_ax(ax,f"{dataset_id} / {group} — PCA state plane with blocked holdout","PC1","PC2"); ax.legend(); atlas_save_fig(fig,out/f"{stem}_plots_01_pca_train_holdout.png")
            fig,ax=plt.subplots(figsize=(8,5)); ax.hist(resid_train,bins=35,color=RED,alpha=.7,label="train"); ax.hist(resid_test,bins=35,histtype="step",color=WHITE,label="held-out"); atlas_style_ax(ax,f"{dataset_id} / {group} — one-step residuals","Residual norm","Count"); ax.legend(); atlas_save_fig(fig,out/f"{stem}_plots_02_train_vs_heldout_residuals.png")
            gap=train_r2-test_r2
            if not (np.isfinite(test_r2) and X.shape[1]>=4 and n-split>=15):
                q="low_support"
            elif test_r2<=0:
                q="poor_generalization"
            elif gap>ATLAS_MAX_GENERALIZATION_GAP:
                q="large_generalization_gap"
            else:
                q="valid"
            note=f"train R2={train_r2:.4g}; heldout R2={test_r2:.4g}; gap={gap:.4g}; channels={X.shape[1]}; heldout states={n-split}"
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"affine_one_step_R2_heldout",test_r2,"",q,ensemble_group=group,n_curves=X.shape[1],n_points=n,notes=note)
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"affine_one_step_R2_train",train_r2,"","diagnostic_only",ensemble_group=group)
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"affine_R2_generalization_gap",gap,"","diagnostic_only",ensemble_group=group)
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"epsilon_median_train_residual",epsilon,"PCA_distance",q,ensemble_group=group,notes=note)
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"PCA_first_two_train_explained_fraction",explained[:2].sum(),"",q,ensemble_group=group,notes=note)
            atlas_register_feature(dataset_id,28,"kakutani_fixed_point",obj,group,bundle,"fixed_inclusion_residual",fixed_resid,"PCA_distance","theorem_diagnostic",ensemble_group=group,notes="optimizer/theorem diagnostic; excluded from predictive feature pool")
            succeeded+=1
        if not succeeded: raise AtlasStructuralFail("NO_VALID_MULTICHANNEL_STATE","no state group passed blocked-validation gates")
        ncurves,npts=atlas_structural_summary(bundle); atlas_register_ledger(dataset_id,28,obj,ent.get("status"),"valid","",f"{succeeded} state group(s) with blocked held-out validation",ncurves,npts,bundle,successful_groups=succeeded)
    except Exception as exc: atlas_record_exception(dataset_id,28,ent,exc,bundle)
    atlas_flush_ledgers()
print("Cross-dataset Cell 28 complete.")


In [ ]:
# ============================================================
# N003 CROSS-DATASET FEATURE ALIGNMENT
# Preserves all validated cell mathematics and plots.
# Adds estimator-identical atlas features plus separately named companion outputs for cross-dataset comparability.
# ============================================================

def atlas_demote_feature_rows(dataset_id, cell, feature_names=None, quality="reference_only_noncomparable"):
    names=set(feature_names or [])
    for row in ATLAS_FEATURE_ROWS:
        if row.get("dataset_id")==dataset_id and int(row.get("cell",-1))==int(cell):
            if (not names) or row.get("feature") in names:
                row["quality_status"]=quality
                row["eligible_for_model"]=False


def atlas_align_n003_features():
    did="N003"
    bundle=atlas_load_bundle(did, atlas_app_entry(4,did).get("object_id"))
    out=atlas_dataset_dir(did)

    # Remove obsolete companion artifacts for the two regenerated output families.
    _keep_companion={
        "cell_21_nearest_neighbor_distance_entropy_theiler_atlas.csv",
        "cell_21_nearest_neighbor_distance_entropy_plots_13_theiler_vs_raw_by_dimension.png",
        "cell_25_pyragas_delayed_feedback_observable_specific_atlas.csv",
        "cell_25_pyragas_delayed_feedback_plots_09_observable_specific_mi_delay.png",
    }
    for _pattern in (
        "cell_21_nearest_neighbor_distance_entropy_*atlas.csv",
        "cell_21_nearest_neighbor_distance_entropy_*plots_13_theiler_vs_raw_by_dimension.png",
        "cell_25_pyragas_delayed_feedback_*atlas.csv",
        "cell_25_pyragas_delayed_feedback_*plots_09_observable_specific_mi_delay.png",
    ):
        for _artifact in out.glob(_pattern):
            if _artifact.name not in _keep_companion:
                _artifact.unlink()


    # Cell 9 maxima depend on scan density/range and are therefore diagnostic.
    atlas_demote_feature_rows(
        did,9,
        {"max_drive_lock_gain_vs_no_drive","max_sync_gain_vs_no_drive",
         "max_frequency_error_improvement_vs_no_drive","max_drive_lock_raw"},
        quality="scan_grid_dependent"
    )
    for row in ATLAS_FEATURE_ROWS:
        if int(row.get("cell",-1))==9 and str(row.get("feature","")).startswith("max_"):
            row["quality_status"]="scan_grid_dependent"
            row["eligible_for_model"]=False

    # ---- Cell 13: uniform-phase baseline contrast for model eligibility.
    groups=atlas_phase_ensemble_groups(bundle,min_points=40,min_curves=2)
    for gp in groups:
        group=gp["group"]; grid=gp["grid"]; phase_mat=gp["phase_matrix"]
        mean_z=np.mean(np.exp(1j*2*np.pi*phase_mat),axis=1)
        R=np.abs(mean_z)
        phase=(np.angle(mean_z)/(2*np.pi))%1.0
        if len(phase)>1200:
            phase=phase[np.linspace(0,len(phase)-1,1200).astype(int)]
        uniform=np.linspace(0,1,len(phase),endpoint=False)
        ngrid={"full":90,"balanced":64,"fast":44,"ultra":30}[ATLAS_SPEED]
        O=np.linspace(0,1,ngrid); Ks=np.linspace(0,4*np.pi,ngrid)
        its={"full":24,"balanced":20,"fast":16,"ultra":12}[ATLAS_SPEED]; tol=1e-6
        rad=np.zeros((ngrid,ngrid)); base=np.zeros_like(rad)
        for i,K in enumerate(Ks):
            for j,Om in enumerate(O):
                th=phase.copy(); ub=uniform.copy()
                for _ in range(its):
                    th=atlas_circle_map(th,Om,K); ub=atlas_circle_map(ub,Om,K)
                nxt=atlas_circle_map(th,Om,K); nxtb=atlas_circle_map(ub,Om,K)
                d=np.minimum((nxt-th)%1,(th-nxt)%1)
                db=np.minimum((nxtb-ub)%1,(ub-nxtb)%1)
                rad[i,j]=np.mean(d<tol); base[i,j]=np.mean(db<tol)
        excess=rad-base
        q="valid" if phase_mat.shape[1]>=ATLAS_MIN_MODEL_CURVES else "low_support"
        atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),
                               group,bundle,"mean_abs_locking_excess_vs_uniform",np.mean(np.abs(excess)),
                               "",q,ensemble_group=group,n_curves=phase_mat.shape[1],n_points=phase_mat.shape[0],
                               notes="N003 uses the cross-dataset estimator")
        atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),
                               group,bundle,"max_abs_locking_excess_vs_uniform",np.max(np.abs(excess)),
                               "","scan_grid_dependent",ensemble_group=group,n_curves=phase_mat.shape[1])
        atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),
                               group,bundle,"mean_phase_resultant",np.mean(R),"",q,ensemble_group=group,
                               n_curves=phase_mat.shape[1],n_points=phase_mat.shape[0])
        atlas_register_feature(did,13,"circle_map_fixed_point",atlas_app_entry(13,did).get("object_id"),
                               group,bundle,"mean_fixed_point_locking_raw",np.mean(rad),"",
                               "model_baseline_dominated",ensemble_group=group)

    # ---- Cell 17: lifted/unwrapped estimator with a uniform-initial-phase
    # baseline contrast for model-eligible global summaries.
    atlas_demote_feature_rows(
        did,17,{"mean_unwrapped_rotation","mean_cross_curve_unwrapped_rotation_std",
                "fraction_near_low_order_rational_unwrapped"},
        quality="model_baseline_dominated"
    )
    groups=atlas_phase_ensemble_groups(bundle,min_points=40,min_curves=2)
    for gp in groups:
        group=gp["group"]; P=gp["phase_matrix"]
        theta0=(np.angle(np.mean(np.exp(1j*2*np.pi*P),axis=0))/(2*np.pi))%1.0
        uniform_theta0=np.linspace(0,1,len(theta0),endpoint=False)
        ngrid={"full":110,"balanced":76,"fast":52,"ultra":34}[ATLAS_SPEED]
        O=np.linspace(0,1,ngrid); Ks=np.linspace(0,2*np.pi,ngrid)
        its={"full":300,"balanced":220,"fast":150,"ultra":90}[ATLAS_SPEED]; trans=max(20,its//5)
        allraw=[]; baseall=[]
        for starts,target in ((theta0,allraw),(uniform_theta0,baseall)):
            for t0 in starts:
                Rg=np.zeros((ngrid,ngrid))
                for i,K in enumerate(Ks):
                    Rg[i]=atlas_rotation_unwrapped(O,K,t0,its,trans)
                target.append(Rg)
        allraw=np.stack(allraw); baseall=np.stack(baseall)
        meanraw=np.mean(allraw,axis=0); stdraw=np.std(allraw,axis=0)
        meanbase=np.mean(baseall,axis=0); stdbase=np.std(baseall,axis=0)
        dist=atlas_rational_distance_unwrapped(meanraw,8); basedist=atlas_rational_distance_unwrapped(meanbase,8)
        rotation_excess=meanraw-meanbase; std_excess=stdraw-stdbase
        raw_frac=float(np.mean(dist<=.01)); base_frac=float(np.mean(basedist<=.01)); frac_excess=raw_frac-base_frac
        q="valid" if len(theta0)>=ATLAS_MIN_MODEL_CURVES else "low_support"
        raw_note="unwrapped estimator is valid; global Ω/K summary is excluded from model use without a baseline reference"
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"mean_unwrapped_rotation",np.mean(meanraw),"cycles_per_iterate",
                               "model_baseline_dominated",ensemble_group=group,n_curves=len(theta0),notes=raw_note)
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"mean_cross_curve_unwrapped_rotation_std",np.mean(stdraw),
                               "cycles_per_iterate","model_baseline_dominated",ensemble_group=group,n_curves=len(theta0),notes=raw_note)
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"fraction_near_low_order_rational_unwrapped",raw_frac,
                               "","model_baseline_dominated",ensemble_group=group,n_curves=len(theta0),notes=raw_note)
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"mean_abs_unwrapped_rotation_excess_vs_uniform",np.mean(np.abs(rotation_excess)),
                               "cycles_per_iterate",q,ensemble_group=group,n_curves=len(theta0),
                               notes="uniform-initial-phase baseline contrast")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"mean_abs_cross_curve_rotation_std_excess_vs_uniform",np.mean(np.abs(std_excess)),
                               "cycles_per_iterate",q,ensemble_group=group,n_curves=len(theta0),notes="uniform-initial-phase baseline contrast")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"fraction_near_low_order_rational_excess_vs_uniform",frac_excess,
                               "",q,ensemble_group=group,n_curves=len(theta0),notes=f"raw fraction={raw_frac:.6g}; uniform baseline={base_frac:.6g}")
        atlas_register_feature(did,17,"circle_map_rotation",atlas_app_entry(17,did).get("object_id"),
                               group,bundle,"mean_rotation_mod1",np.mean(np.mod(meanraw,1.0)),"",
                               "diagnostic_only",ensemble_group=group)

    # ---- Cell 19: uniform baseline with the stabilization gate.
    groups=atlas_phase_ensemble_groups(bundle,min_points=32,min_curves=1)
    for gp in groups:
        group=gp["group"]; P=gp["phase_matrix"]; Omega=1/3
        nk={"full":150,"balanced":96,"fast":62,"ultra":40}[ATLAS_SPEED]
        Ks=np.linspace(0,4*np.pi,nk); nbins=40; edges=np.linspace(0,1,nbins+1)
        its={"full":60,"balanced":45,"fast":32,"ultra":24}[ATLAS_SPEED]; extra=max(8,its//3)
        uniform_init=np.linspace(0,1,min(1200,P.shape[0]),endpoint=False)
        baseline_conc=np.zeros(nk)
        for i,K in enumerate(Ks):
            u=uniform_init.copy()
            for _ in range(its): u=atlas_circle_map(u,Omega,K)
            up=np.histogram(u,bins=edges)[0].astype(float); up/=up.sum()+1e-12
            baseline_conc[i]=atlas_norm_concentration(up)
        allconc=[]; alltvs=[]; excessconc=[]
        for j in range(P.shape[1]):
            init=P[:,j]
            if len(init)>1200:
                init=init[np.linspace(0,len(init)-1,1200).round().astype(int)]
            conc=np.zeros(nk); tv=np.zeros(nk)
            for i,K in enumerate(Ks):
                th=init.copy()
                for _ in range(its): th=atlas_circle_map(th,Omega,K)
                pp=np.histogram(th,bins=edges)[0].astype(float); pp/=pp.sum()+1e-12
                conc[i]=atlas_norm_concentration(pp)
                th2=th.copy()
                for _ in range(extra): th2=atlas_circle_map(th2,Omega,K)
                qq=np.histogram(th2,bins=edges)[0].astype(float); qq/=qq.sum()+1e-12
                tv[i]=.5*np.sum(np.abs(pp-qq))
            allconc.append(conc); alltvs.append(tv); excessconc.append(conc-baseline_conc)
        allconc=np.asarray(allconc); alltvs=np.asarray(alltvs); excessconc=np.asarray(excessconc)
        meanTV=np.mean(alltvs,axis=0); meanex=np.mean(excessconc,axis=0)
        mtv=float(np.mean(meanTV)); p90_tv=float(np.percentile(meanTV,90))
        bad_frac=float(np.mean(meanTV>ATLAS_TV_STABLE_THRESHOLD))
        q="stable" if (mtv<=ATLAS_TV_STABLE_THRESHOLD and p90_tv<=ATLAS_TV_P90_THRESHOLD
                       and bad_frac<=ATLAS_TV_BAD_FRACTION_MAX) else "unstable_pushforward"
        note=f"N003 cross-dataset; mean TV={mtv:.4g}; p90 TV={p90_tv:.4g}; bad fraction={bad_frac:.3f}"
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),
                               group,bundle,"mean_concentration_excess_vs_uniform",np.mean(meanex),"",q,
                               ensemble_group=group,n_curves=P.shape[1],n_points=P.shape[0],notes=note)
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),
                               group,bundle,"mean_abs_concentration_excess_vs_uniform",np.mean(np.abs(meanex)),
                               "",q,ensemble_group=group,n_curves=P.shape[1],n_points=P.shape[0],notes=note)
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),
                               group,bundle,"mean_stabilization_TV",mtv,"TV","diagnostic_only",ensemble_group=group)
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),
                               group,bundle,"p90_stabilization_TV",p90_tv,"TV","diagnostic_only",ensemble_group=group)
        atlas_register_feature(did,19,"circle_map_phase_pushforward",atlas_app_entry(19,did).get("object_id"),
                               group,bundle,"fraction_K_above_TV_threshold",bad_frac,"","diagnostic_only",
                               ensemble_group=group)

    # ---- Cell 21: Theiler-excluded estimator evaluated separately by observable.
    rows=[]
    for c in bundle["curves"]:
        try:
            x,y,dx,_=atlas_prepare_curve(bundle,c,min_points=18,max_points=ATLAS_POINT_CAP)
        except AtlasStructuralFail:
            continue
        ys=atlas_robust_standardize(y)
        for m in range(2,11):
            try:
                X=atlas_delay_embed(ys,m,1); n=len(X)
                if n<max(35,6*m+15): continue
                tw=max(2,m)
                Hraw,nnraw,graw=atlas_nn_metrics(X,70,theiler=0)
                Hth,nnth,gth=atlas_nn_metrics(X,70,theiler=tw)
                try:
                    Hth2,_,_=atlas_nn_metrics(X,70,theiler=2*tw)
                    sensitivity=abs(Hth2-Hth)
                except AtlasStructuralFail:
                    sensitivity=np.nan
                rows.append({"curve":c["label"],"observable":c["observable"],"embedding_dimension":m,
                             "Hraw":Hraw,"Hth":Hth,"sensitivity":sensitivity,
                             "adj":float(np.mean(graw==1)),"n_points":n})
            except AtlasStructuralFail:
                continue
    if rows:
        rdf=pd.DataFrame(rows)
        rdf.to_csv(out/"cell_21_nearest_neighbor_distance_entropy_theiler_atlas.csv",index=False)
        fig,ax=plt.subplots(figsize=(10,6))
        observable_styles=[(RED,WHITE),(ORANGE,YELLOW)]
        for k,(obs,gplot) in enumerate(rdf.groupby("observable")):
            med=gplot.groupby("embedding_dimension")[["Hth","Hraw"]].median().sort_index()
            c_main,c_raw=observable_styles[k % len(observable_styles)]
            ax.plot(med.index,med.Hth,marker="o",lw=2.0,color=c_main,label=f"{obs} — Theiler-excluded")
            ax.plot(med.index,med.Hraw,linestyle="--",lw=1.5,alpha=.85,color=c_raw,label=f"{obs} — Raw NN")
        atlas_style_ax(ax,"N003 — NN entropy estimator comparison","Embedding dimension","NN distance entropy [bits]")
        ax.legend()
        atlas_save_fig(fig,out/"cell_21_nearest_neighbor_distance_entropy_plots_13_theiler_vs_raw_by_dimension.png")
        for obs,g in rdf.groupby("observable"):
            nobs=int(g.curve.nunique()); nmed=float(g.n_points.median())
            sens=float(g.sensitivity.dropna().median()) if g.sensitivity.notna().any() else np.inf
            adj=float(g.adj.mean())
            q="theiler_excluded" if (
                nobs>=ATLAS_MIN_MODEL_CURVES and nmed>=ATLAS_MIN_MODEL_POINTS
                and sens<=ATLAS_MAX_THEILER_ENTROPY_SENSITIVITY_BITS
            ) else "exploratory_nn_entropy"
            note=f"N003 cross-dataset; raw adjacency={adj:.4g}; curves={nobs}; median points={nmed:.0f}; sensitivity={sens:.4g}"
            atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),
                                   str(obs),bundle,"NN_entropy_theiler_grand_median",g.Hth.median(),"bits",q,
                                   n_curves=nobs,n_points=nmed,notes=note)
            atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),
                                   str(obs),bundle,"NN_entropy_raw_grand_median",g.Hraw.median(),"bits",
                                   "diagnostic_only" if adj<=ATLAS_ADJACENCY_DOMINANCE_THRESHOLD else "sampling_geometry_dominated")
            atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),
                                   str(obs),bundle,"adjacent_neighbor_fraction_raw",adj,"","diagnostic_only")
            if np.isfinite(sens):
                atlas_register_feature(did,21,"nearest_neighbor_distance_entropy",atlas_app_entry(21,did).get("object_id"),
                                       str(obs),bundle,"median_theiler_window_sensitivity_bits",sens,"bits","diagnostic_only")

    # ---- Cell 25: observable- and group-specific mutual-information features.
    prep=[]
    for c in bundle["curves"]:
        try:
            x,y,dx,_=atlas_prepare_curve(bundle,c,min_points=24,max_points=ATLAS_POINT_CAP)
            mi=atlas_select_mi_delay(y,dx)
            grp=c.get("ensemble_group") or c.get("observable") or "default"
            ctrl,fb=atlas_pyragas(y,max(1,min(mi["delay_samples"],len(y)-2)),.1)
            rms0=np.sqrt(np.mean(y*y)); rmsfb=np.sqrt(np.mean(fb*fb))
            rough0=np.sqrt(np.mean(np.gradient(y,dx)**2)); rough1=np.sqrt(np.mean(np.gradient(ctrl,dx)**2))
            prep.append({"curve":c["label"],"observable":c["observable"],"group":grp,
                         "delay":mi["delay_coordinate"],"method":mi["selection_method"],
                         "relative_feedback_rms":rmsfb/(rms0+1e-12),"roughness_ratio":rough1/(rough0+1e-12)})
        except AtlasStructuralFail:
            pass
    if prep:
        pdf=pd.DataFrame(prep)
        pdf.to_csv(out/"cell_25_pyragas_delayed_feedback_observable_specific_atlas.csv",index=False)
        delay_summary=pdf.groupby("observable").delay.median().sort_values()
        fig,ax=plt.subplots(figsize=(9,5))
        delay_colors=[RED,ORANGE,YELLOW,WHITE][:len(delay_summary)]
        ax.barh(delay_summary.index.astype(str),delay_summary.values,color=delay_colors,edgecolor=WHITE,linewidth=.7)
        atlas_style_ax(ax,"N003 — observable-specific MI delay",f"Median MI-selected delay [{bundle['coordinate_unit']}]","Observable")
        atlas_save_fig(fig,out/"cell_25_pyragas_delayed_feedback_plots_09_observable_specific_mi_delay.png")
        for obs,g in pdf.groupby("observable"):
            nobs=int(g.curve.nunique())
            local_frac=float(g.method.astype(str).str.contains("first local minimum",case=False).mean())
            q="valid" if nobs>=ATLAS_MIN_MODEL_CURVES and local_frac>=.5 else "weak_or_low_support_MI_delay"
            grp=str(g.group.iloc[0])
            atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),
                                   str(obs),bundle,"median_MI_delay_coordinate",g.delay.median(),bundle["coordinate_unit"],
                                   q,ensemble_group=grp,n_curves=nobs,
                                   notes=f"N003 cross-dataset; local-minimum fraction={local_frac:.3f}")
            atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),
                                   str(obs),bundle,"median_relative_feedback_rms",g.relative_feedback_rms.median(),"",
                                   q,ensemble_group=grp,n_curves=nobs)
            atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),
                                   str(obs),bundle,"fraction_MI_delays_with_local_minimum",local_frac,"",
                                   "diagnostic_only",ensemble_group=grp,n_curves=nobs)
            atlas_register_feature(did,25,"pyragas_coordinate_transform",atlas_app_entry(25,did).get("object_id"),
                                   str(obs),bundle,"median_roughness_ratio",g.roughness_ratio.median(),"",
                                   "diagnostic_only",ensemble_group=grp,n_curves=nobs)

    # ---- Cell 28: blocked holdout estimator for feature registration.
    for group,state in bundle.get("state_groups",{}).items():
        coord=np.asarray(state["coordinate"],float); X=np.asarray(state["matrix"],float)
        labels=np.asarray(state["labels"],dtype=object)
        if X.shape[0]<30 or X.shape[1]<3: continue
        good=np.all(np.isfinite(X),axis=1); coord,X=coord[good],X[good]; n=len(X)
        split=max(20,int(.70*n)); split=min(split,n-10)
        if split<20 or n-split<10: continue
        mu=X[:split].mean(axis=0); sd=X[:split].std(axis=0); keep=sd>1e-10
        X=X[:,keep]; labels=labels[keep]
        if X.shape[1]<3: continue
        Xs=(X-mu[keep])/(sd[keep]+1e-12)
        U,sv,Vt=np.linalg.svd(Xs[:split],full_matrices=False)
        comps=Vt[:2].T; Z=Xs@comps; explained=(sv*sv)/(np.sum(sv*sv)+1e-12)
        train0=Z[:split-1]; train1=Z[1:split]; test0=Z[split:-1]; test1=Z[split+1:]
        B=np.linalg.lstsq(np.c_[train0,np.ones(len(train0))],train1,rcond=None)[0]
        A=B[:2].T; b=B[2]; lo=Z[:split].min(axis=0); hi=Z[:split].max(axis=0)
        def gmap(z): return np.clip(A@np.asarray(z)+b,lo,hi)
        pred_train=np.array([gmap(z) for z in train0]); pred_test=np.array([gmap(z) for z in test0])
        resid_train=np.linalg.norm(train1-pred_train,axis=1)
        epsilon=float(np.median(resid_train))
        train_r2=atlas_r2(train1,pred_train); test_r2=atlas_r2(test1,pred_test); gap=train_r2-test_r2
        if not (np.isfinite(test_r2) and X.shape[1]>=4 and n-split>=15):
            q="low_support"
        elif test_r2<=0:
            q="poor_generalization"
        elif gap>ATLAS_MAX_GENERALIZATION_GAP:
            q="large_generalization_gap"
        else:
            q="valid"
        note=f"N003 cross-dataset; train R2={train_r2:.4g}; heldout R2={test_r2:.4g}; gap={gap:.4g}"
        atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),
                               group,bundle,"affine_one_step_R2_heldout",test_r2,"",q,
                               ensemble_group=group,n_curves=X.shape[1],n_points=n,notes=note)
        atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),
                               group,bundle,"affine_one_step_R2_train",train_r2,"","diagnostic_only",
                               ensemble_group=group)
        atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),
                               group,bundle,"affine_R2_generalization_gap",gap,"","diagnostic_only",
                               ensemble_group=group)
        atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),
                               group,bundle,"epsilon_median_train_residual",epsilon,"PCA_distance",q,
                               ensemble_group=group,notes=note)
        atlas_register_feature(did,28,"kakutani_fixed_point",atlas_app_entry(28,did).get("object_id"),
                               group,bundle,"PCA_first_two_train_explained_fraction",explained[:2].sum(),"",q,
                               ensemble_group=group,notes=note)

    atlas_flush_ledgers()


atlas_align_n003_features()
print("N003 cross-dataset harmonization complete; validated existing plots/results were not overwritten and companion cross-dataset outputs use distinct filenames.")


In [ ]:
# ============================================================
# FINAL CROSS-DATASET LEDGERS / QA SUMMARY
# ============================================================

# Ensure the feature-only N003 feature alignment is present before finalizing.
# This never overwrites validated N003 cell result files or plots.
required_n003_companion = {
    (13, "mean_abs_locking_excess_vs_uniform"),
    (17, "fraction_near_low_order_rational_excess_vs_uniform"),
    (19, "mean_abs_concentration_excess_vs_uniform"),
    (21, "NN_entropy_theiler_grand_median"),
    (25, "median_MI_delay_coordinate"),
    (28, "affine_one_step_R2_heldout"),
}
present_n003 = {
    (int(r.get("cell",-1)), str(r.get("feature","")))
    for r in ATLAS_FEATURE_ROWS
    if r.get("dataset_id")=="N003"
}
if not required_n003_companion.issubset(present_n003):
    atlas_align_n003_features()

atlas_flush_ledgers()
ledger=pd.DataFrame(ATLAS_LEDGER_ROWS)
features=pd.DataFrame(ATLAS_FEATURE_ROWS)

# Every dataset × source computation should have one explicit ledger outcome.
expected={(did,cell) for did in PRIMARY_DATASETS for cell in sorted(ATLAS_CELL_TO_CODE)}
seen=set(zip(ledger.dataset_id,ledger.cell)) if len(ledger) else set()
missing_before_fill=sorted(expected-seen)

# Record missing outcomes explicitly so an exported result package can be
# distinguished from a complete in-kernel execution without guessing.
for did,cell in missing_before_fill:
    e=atlas_app_entry(cell,did)
    atlas_register_ledger(
        did,cell,e.get("object_id"),e.get("status"),
        "not_executed_in_current_kernel","EXECUTION_INCOMPLETE",
        "No ledger outcome was present when final QA executed."
    )

atlas_flush_ledgers()
ledger=pd.read_csv(PHASE2_RESULTS_ROOT/"phase2_applicability_results.csv")
features=pd.read_csv(PHASE2_RESULTS_ROOT/"phase2_feature_registry.csv") if (PHASE2_RESULTS_ROOT/"phase2_feature_registry.csv").exists() else pd.DataFrame()

summary=[]
for did in PRIMARY_DATASETS:
    sub=ledger[ledger.dataset_id==did]
    f=features[features.dataset_id==did] if len(features) else pd.DataFrame()
    summary.append({
        "dataset_id":did,
        "dataset_name":ATLAS_DATASET_REGISTRY.get(did,{}).get("name"),
        "ledger_outcomes":int(len(sub)),
        "valid_method_cells":int(np.sum(sub.execution_status=="valid")),
        "not_applicable":int(np.sum(sub.execution_status=="not_applicable")),
        "conditional_fail":int(np.sum(sub.execution_status=="conditional_fail")),
        "skipped_dependency":int(np.sum(sub.execution_status=="skipped_dependency")),
        "dependency_missing":int(np.sum(sub.execution_status=="dependency_missing")),
        "data_unavailable":int(np.sum(sub.execution_status=="data_unavailable")),
        "errors":int(np.sum(sub.execution_status=="error")),
        "not_executed_in_current_kernel":int(np.sum(sub.execution_status=="not_executed_in_current_kernel")),
        "feature_count":int(len(f)),
        "model_eligible_feature_count":int(f.eligible_for_model.fillna(False).astype(bool).sum()) if len(f) else 0,
    })
summary_df=pd.DataFrame(summary)
status_summary_cols=[
    "valid_method_cells","not_applicable","conditional_fail","skipped_dependency",
    "dependency_missing","data_unavailable","errors","not_executed_in_current_kernel"
]
summary_df["status_count_total"]=summary_df[status_summary_cols].sum(axis=1).astype(int)
if not (summary_df["status_count_total"]==summary_df["ledger_outcomes"]).all():
    raise AssertionError("Cross-dataset summary status counts do not equal ledger outcomes")
summary_df.to_csv(PHASE2_RESULTS_ROOT/"phase2_cross_dataset_summary.csv",index=False)

# Defensive result-file QA: validate the saved CSV, PNG, NPZ, JSON and TXT
# artifacts rather than implying that a CSV-only check covered the whole package.
qa=[]
qa_columns=["dataset_id","file","file_type","readable","rows","columns","problem"]
for did in PRIMARY_DATASETS:
    d=PHASE2_RESULTS_ROOT/did
    if not d.exists():
        continue
    candidates=[p for p in d.rglob("cell_*") if p.is_file()]
    manifest=d/"dataset_atlas_manifest.json"
    if manifest.is_file():
        candidates.append(manifest)
    for p in sorted(set(candidates)):
        ext=p.suffix.lower()
        if ext not in {".csv",".png",".npz",".json",".txt"}:
            continue
        row={"dataset_id":did,"file":str(p.relative_to(d)),"file_type":ext.lstrip("."),
             "readable":True,"rows":np.nan,"columns":np.nan,"problem":""}
        try:
            if ext==".csv":
                t=pd.read_csv(p)
                row["rows"]=len(t); row["columns"]=len(t.columns)
                if len(t)==0 or len(t.columns)==0:
                    row["problem"]="empty_table"
            elif ext==".png":
                img=plt.imread(p)
                if np.asarray(img).size==0 or np.asarray(img).ndim<2:
                    row["problem"]="invalid_or_empty_image"
            elif ext==".npz":
                with np.load(p,allow_pickle=True) as z:
                    if not z.files:
                        row["problem"]="empty_npz"
                    else:
                        for key in z.files:
                            arr=np.asarray(z[key])
                            if np.issubdtype(arr.dtype,np.number) and arr.size and not np.all(np.isfinite(arr)):
                                row["problem"]=f"nonfinite_numeric_array:{key}"
                                break
            elif ext==".json":
                obj=json.loads(p.read_text(encoding="utf-8"))
                if obj is None or obj=={} or obj==[]:
                    row["problem"]="empty_json"
            elif ext==".txt":
                if not p.read_text(encoding="utf-8",errors="replace").strip():
                    row["problem"]="empty_text"
        except Exception as exc:
            row["readable"]=False
            row["problem"]=f"{type(exc).__name__}: {exc}"
        qa.append(row)
# Root ledgers are part of the result package and use the table QA contract.
# The QA table and completion JSON are written after this check and validated
# by construction and serialization.
for p in [
    PHASE2_RESULTS_ROOT/"phase2_applicability_results.csv",
    PHASE2_RESULTS_ROOT/"phase2_feature_registry.csv",
    PHASE2_RESULTS_ROOT/"phase2_cross_dataset_summary.csv",
]:
    if not p.is_file():
        continue
    row={"dataset_id":"__root__","file":p.name,"file_type":"csv",
         "readable":True,"rows":np.nan,"columns":np.nan,"problem":""}
    try:
        t=pd.read_csv(p)
        row["rows"]=len(t); row["columns"]=len(t.columns)
        if len(t)==0 or len(t.columns)==0:
            row["problem"]="empty_table"
    except Exception as exc:
        row["readable"]=False
        row["problem"]=f"{type(exc).__name__}: {exc}"
    qa.append(row)

qa_df=pd.DataFrame(qa,columns=qa_columns)
qa_df.to_csv(PHASE2_RESULTS_ROOT/"phase2_result_file_QA.csv",index=False)

# Sanity assertions for confirmed problems only.
if len(features):
    bad17=features[
        (features.cell==17)
        & features.feature.astype(str).str.contains("mod1")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(bad17):
        raise AssertionError("Modulo-one Cell-17 features incorrectly marked model-eligible")

    bad28=features[
        (features.cell==28)
        & features.feature.eq("fixed_inclusion_residual")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(bad28):
        raise AssertionError("Theorem/optimizer fixed-point residual incorrectly marked model-eligible")

    badmax=features[
        (features.cell==9)
        & features.feature.astype(str).str.startswith("max_")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(badmax):
        raise AssertionError("Scan-grid-dependent Cell-9 maxima incorrectly marked model-eligible")

    bad13max=features[
        (features.cell==13)
        & features.feature.eq("max_abs_locking_excess_vs_uniform")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(bad13max):
        raise AssertionError("Scan-grid-dependent Cell-13 locking maximum incorrectly marked model-eligible")

    raw17=features[
        (features.cell==17)
        & features.feature.isin([
            "mean_unwrapped_rotation","mean_cross_curve_unwrapped_rotation_std",
            "fraction_near_low_order_rational_unwrapped"
        ])
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(raw17):
        raise AssertionError("Raw global Cell-17 circle-map summaries are model-eligible without baseline check")

    badall=features[
        (features.dataset_id=="N003")
        & (features.cell.isin([21,25]))
        & (features.observable=="all_observables")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(badall):
        raise AssertionError("Combined N003 transmission+Sigma_R features incorrectly model-eligible")

    n003_high=features[
        (features.dataset_id=="N003")
        & (features.cell==4)
        & features.feature.astype(str).str.match(r"LLE_median_d(?:8|9|10)$")
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(n003_high):
        raise AssertionError("N003 d8–d10 Lyapunov features are model-eligible despite failed source QC")

    excluded_statuses={
        "low_support","unreliable_embedding_support","changing_curve_population",
        "unstable_pushforward","exploratory_nn_entropy","weak_or_low_support_MI_delay",
        "poor_generalization","large_generalization_gap","reference_only_noncomparable",
        "scan_grid_dependent","model_baseline_dominated","sampling_geometry_dominated",
        "diagnostic_only","theorem_diagnostic"
    }
    badq=features[
        features.quality_status.astype(str).isin(excluded_statuses)
        & features.eligible_for_model.fillna(False).astype(bool)
    ]
    if len(badq):
        raise AssertionError(f"{len(badq)} explicitly non-model quality rows are marked eligible")

    dup=features.duplicated(
        subset=["dataset_id","cell","method","object_id","observable","ensemble_group","feature"],
        keep=False
    )
    if dup.any():
        raise AssertionError("Duplicate feature-registry keys remain after feature alignment")

# P005_SIM must be represented as TRAJ_SPEC: depth is the ordered coordinate
# and each state is a multichannel outgoing-energy spectrum.
p005_manifest=PHASE2_RESULTS_ROOT/"P005_SIM"/"dataset_atlas_manifest.json"
if p005_manifest.exists():
    manifest=json.loads(p005_manifest.read_text(encoding="utf-8"))
    if str(manifest.get("object_id",""))=="P005_SIM.TRAJ_SPEC":
        if "depth" not in str(manifest.get("trajectory_coordinate_column",manifest.get("coordinate_native",""))).lower():
            raise AssertionError("P005_SIM.TRAJ_SPEC is not ordered by shielding depth")
        if str(manifest.get("state_semantics",""))!="complete_outgoing_energy_spectrum_ordered_by_depth_MFP":
            raise AssertionError("P005_SIM.TRAJ_SPEC manifest does not identify the complete spectrum as the state")
        shapes=manifest.get("state_group_shapes",{})
        if not any(int(v.get("n_coordinate_points",0))>=3 and int(v.get("state_dimension",0))>=3 for v in shapes.values()):
            raise AssertionError("P005_SIM.TRAJ_SPEC has no complete multichannel spectral trajectory state group")

qa_problem_count=int((qa_df.problem.astype(str)!="").sum()) if len(qa_df) else 0
status_counts=ledger.execution_status.value_counts(dropna=False).to_dict() if len(ledger) else {}
run_complete={
    "execution_complete": bool(len(missing_before_fill)==0 and qa_problem_count==0),
    "completion_semantics": (
        "execution_complete means every expected dataset×cell received a ledger outcome and saved result files passed QA; "
        "it does not mean every applicable method produced a valid scientific result"
    ),
    "expected_dataset_cell_outcomes": int(len(expected)),
    "ledger_rows": int(len(ledger)),
    "missing_before_final_QA": [
        {"dataset_id":did,"cell":int(cell)} for did,cell in missing_before_fill
    ],
    "execution_status_counts": {str(k):int(v) for k,v in status_counts.items()},
    "feature_rows": int(len(features)),
    "result_file_QA_problems": int(qa_problem_count),
}
(PHASE2_RESULTS_ROOT/"phase2_run_complete.json").write_text(
    json.dumps(run_complete,indent=2,default=str),
    encoding="utf-8",
)

print("Phase-II atlas final QA complete.")
display(summary_df)
print("\nFeature quality counts:")
if len(features):
    display(features.groupby(["quality_status","eligible_for_model"]).size().rename("count").reset_index())
print("\nResult-file QA problems:",qa_problem_count)
print("Missing ledger outcomes before final QA:",len(missing_before_fill))
print("Execution-completion record:",PHASE2_RESULTS_ROOT/"phase2_run_complete.json")
print("Root ledgers:")
for name in [
    "phase2_applicability_results.csv",
    "phase2_feature_registry.csv",
    "phase2_cross_dataset_summary.csv",
    "phase2_result_file_QA.csv",
    "phase2_run_complete.json",
]:
    print(" ",PHASE2_RESULTS_ROOT/name)


## Additional physically ordered objects

This section adds analyses for scientifically useful objects that are not represented by the primary nine-cell applicability grid. Existing Phase-II results and ledgers remain unchanged.

- **N001/N002 spectral sequences:** each measured transmitted spectrum is analyzed as an energy-ordered object at its own benchmark condition. Historical shielding configurations are not reinterpreted as a controlled thickness sweep.
- **N001 controlled field:** when the fixed-geometry `controlled_R_E_t.npz` product is present, its spectra are analyzed across the controlled concrete-thickness coordinate.
- **P003 buildup curves:** the 15-point buildup trajectories are retained for lower-order shape and response descriptors rather than forcing estimators whose support requirements are not met.
- **Sparse reference datasets:** their intended reference role is recorded explicitly without inventing pseudo-samples.

All supplemental outputs are written under `results/phase2/supplemental_ordered_objects/`.


In [ ]:
# ============================================================
# ADDITIONAL PHYSICALLY ORDERED OBJECTS
# Existing primary atlas outputs are not modified by this section.
# ============================================================

SUPPLEMENTAL_ROOT = PHASE2_RESULTS_ROOT / "supplemental_ordered_objects"
SUPPLEMENTAL_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENTAL_FEATURE_ROWS = []

WARM_SEQUENCE = [RED, "#FF6A00", "#FFB000", "#FFD84D", WHITE, "#FFB3B3", "#FF8A65", "#FFE082", "#FFF3E0"]


def _supp_trapezoid(y, x):
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


def _supp_add_feature(dataset_id, object_id, family, condition, feature_name, value, unit="", quality="descriptive", model_eligible=False, notes=""):
    if value is None or not np.isfinite(float(value)):
        return
    SUPPLEMENTAL_FEATURE_ROWS.append({
        "dataset_id": str(dataset_id),
        "object_id": str(object_id),
        "analysis_family": str(family),
        "condition": str(condition),
        "feature_name": str(feature_name),
        "value": float(value),
        "unit": str(unit),
        "quality_status": str(quality),
        "model_eligible": bool(model_eligible),
        "notes": str(notes),
    })


def _supp_clean_energy_sequence(energy, values, min_points=12):
    x = np.asarray(energy, float).reshape(-1)
    y = np.asarray(values, float).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x, y = x[good], y[good]
    if len(x) < min_points:
        raise AtlasStructuralFail("SUPPLEMENTAL_SEQUENCE_TOO_SHORT", f"need >= {min_points} finite positive-energy samples")
    order = np.argsort(x)
    x, y = x[order], y[order]
    ux = np.unique(x)
    if len(ux) != len(x):
        yy = np.array([np.mean(y[x == e]) for e in ux], float)
        x, y = ux, yy
    if len(x) < min_points:
        raise AtlasStructuralFail("SUPPLEMENTAL_SEQUENCE_TOO_SHORT", f"need >= {min_points} unique energy samples")
    return x, y


def _supp_spectral_metrics(energy, values, energy_lower=None, energy_upper=None, response_sigma=None):
    """Descriptors for an energy-ordered spectrum.

    N001/N002 responses are lethargy-flux densities.  When physical bin edges
    are supplied, integrated response and energy moments therefore use
    du = ln(E_hi/E_lo), not dE.  Sequence-only diagnostics are still evaluated
    on a uniform log-energy coordinate so irregular physical bin widths do not
    create an artificial lag structure.
    """
    Eseq, yseq = _supp_clean_energy_sequence(energy, values, min_points=12)
    ypseq = np.clip(yseq, 0.0, None)

    # Physical binned descriptors.  For a lethargy density R=dPhi/du:
    #   Phi_i      = R_i * ln(E_hi/E_lo)
    #   int E dPhi = R_i * (E_hi-E_lo)
    #   int E^2 dPhi = 0.5 * R_i * (E_hi^2-E_lo^2)
    use_edges = energy_lower is not None and energy_upper is not None
    median_rel_sigma = np.nan
    p90_rel_sigma = np.nan
    positive_fraction = np.nan
    covered_lethargy = np.nan

    if use_edges:
        lo0 = np.asarray(energy_lower, float).reshape(-1)
        hi0 = np.asarray(energy_upper, float).reshape(-1)
        y0 = np.asarray(values, float).reshape(-1)
        if not (len(lo0) == len(hi0) == len(y0)):
            raise AtlasStructuralFail("SUPPLEMENTAL_BIN_SHAPE_MISMATCH", "energy edges and response must have equal lengths")
        good = np.isfinite(lo0) & np.isfinite(hi0) & np.isfinite(y0) & (lo0 > 0) & (hi0 > lo0)
        if int(good.sum()) < 12:
            raise AtlasStructuralFail("SUPPLEMENTAL_SEQUENCE_TOO_SHORT", "need >= 12 finite positive-energy bins with valid edges")
        lo, hi, yb = lo0[good], hi0[good], np.clip(y0[good], 0.0, None)
        centers = np.sqrt(lo * hi)
        order = np.argsort(centers)
        lo, hi, yb, centers = lo[order], hi[order], yb[order], centers[order]
        du = np.log(hi / lo)
        mass = yb * du
        area = float(np.sum(mass))
        if not np.isfinite(area) or area <= 0:
            raise AtlasStructuralFail("SUPPLEMENTAL_NONPOSITIVE_SPECTRUM", "positive lethargy-integrated response is required")
        centroid = float(np.sum(yb * (hi - lo)) / area)
        second_moment = float(np.sum(0.5 * yb * (hi ** 2 - lo ** 2)) / area)
        variance = max(second_moment - centroid ** 2, 0.0)
        spread = float(np.sqrt(variance))
        peak_energy = float((0.5 * (lo + hi))[int(np.nanargmax(yb))])
        p_mass = mass / area
        positive = p_mass > 0
        entropy_bits = float(-np.sum(p_mass[positive] * np.log2(p_mass[positive])))
        effective_bins = float(2.0 ** entropy_bits)
        positive_fraction = float(np.mean(yb > 0))
        covered_lethargy = float(np.sum(du))

        if response_sigma is not None:
            s0 = np.asarray(response_sigma, float).reshape(-1)
            if len(s0) == len(y0):
                sb = s0[good][order]
                rel_good = (yb > 0) & np.isfinite(sb) & (sb >= 0)
                if np.any(rel_good):
                    rel = sb[rel_good] / yb[rel_good]
                    rel = rel[np.isfinite(rel)]
                    if len(rel):
                        median_rel_sigma = float(np.median(rel))
                        p90_rel_sigma = float(np.quantile(rel, 0.90))
    else:
        area = _supp_trapezoid(ypseq, Eseq)
        if not np.isfinite(area) or area <= 0:
            raise AtlasStructuralFail("SUPPLEMENTAL_NONPOSITIVE_SPECTRUM", "positive spectral integral is required")
        centroid = _supp_trapezoid(Eseq * ypseq, Eseq) / area
        variance = _supp_trapezoid(((Eseq - centroid) ** 2) * ypseq, Eseq) / area
        spread = float(np.sqrt(max(variance, 0.0)))
        peak_energy = float(Eseq[int(np.nanargmax(ypseq))])
        p_mass = ypseq / (np.sum(ypseq) + 1e-300)
        positive = p_mass > 0
        entropy_bits = float(-np.sum(p_mass[positive] * np.log2(p_mass[positive])))
        effective_bins = float(2.0 ** entropy_bits)
        positive_fraction = float(np.mean(ypseq > 0))

    # Sequence-shape diagnostics use a uniform log-energy grid.
    logE = np.log(Eseq)
    grid = np.linspace(float(logE.min()), float(logE.max()), len(Eseq))
    yu = np.interp(grid, logE, ypseq)
    tv_norm = float(np.sum(np.abs(np.diff(yu))) / (np.sum(np.abs(yu)) + 1e-300))

    nn_entropy = np.nan
    if len(yu) >= 36:
        try:
            z = atlas_robust_standardize(yu)
            X = atlas_delay_embed(z, 2, 1)
            nn_entropy, _, _ = atlas_nn_metrics(X, bins=min(40, max(16, len(X) // 2)), theiler=2)
        except Exception:
            nn_entropy = np.nan

    mi_delay_logE = np.nan
    mi_delay_samples = np.nan
    if len(yu) >= 24:
        try:
            dx = float(np.median(np.diff(grid)))
            mi = atlas_select_mi_delay(yu, dx, min_fraction=.02, max_fraction=.25, max_samples=3000)
            mi_delay_logE = float(mi["delay_coordinate"])
            mi_delay_samples = float(mi["delay_samples"])
        except Exception:
            pass

    return {
        "n_energy": float(len(Eseq)),
        "spectral_integral": float(area),
        "response_weighted_mean_energy_MeV": float(centroid),
        "response_weighted_energy_spread_MeV": float(spread),
        "peak_energy_MeV": peak_energy,
        "shape_entropy_bits": entropy_bits,
        "effective_shape_bins": effective_bins,
        "normalized_shape_total_variation": tv_norm,
        "positive_response_bin_fraction": positive_fraction,
        "covered_lethargy_width": covered_lethargy,
        "median_relative_bin_mc_sigma": median_rel_sigma,
        "p90_relative_bin_mc_sigma": p90_rel_sigma,
        "theiler_nn_entropy_bits_d2": float(nn_entropy) if np.isfinite(nn_entropy) else np.nan,
        "mi_delay_log_energy": float(mi_delay_logE) if np.isfinite(mi_delay_logE) else np.nan,
        "mi_delay_samples": float(mi_delay_samples) if np.isfinite(mi_delay_samples) else np.nan,
    }


def _supp_write_spectrum_features(dataset_id, object_id, condition, energy, values, notes, quality="descriptive", model_eligible=False, energy_lower=None, energy_upper=None, response_sigma=None):
    metrics = _supp_spectral_metrics(
        energy,
        values,
        energy_lower=energy_lower,
        energy_upper=energy_upper,
        response_sigma=response_sigma,
    )
    units = {
        "n_energy": "count",
        "spectral_integral": "n_cm-2_uC-1",
        "response_weighted_mean_energy_MeV": "MeV",
        "response_weighted_energy_spread_MeV": "MeV",
        "peak_energy_MeV": "MeV",
        "shape_entropy_bits": "bit",
        "effective_shape_bins": "count",
        "normalized_shape_total_variation": "",
        "positive_response_bin_fraction": "fraction",
        "covered_lethargy_width": "lethargy",
        "median_relative_bin_mc_sigma": "fraction",
        "p90_relative_bin_mc_sigma": "fraction",
        "theiler_nn_entropy_bits_d2": "bit",
        "mi_delay_log_energy": "log_energy",
        "mi_delay_samples": "samples",
    }
    families = {
        "theiler_nn_entropy_bits_d2": "nearest_neighbor_entropy",
        "mi_delay_log_energy": "mutual_information_delay",
        "mi_delay_samples": "mutual_information_delay",
        "median_relative_bin_mc_sigma": "statistical_support",
        "p90_relative_bin_mc_sigma": "statistical_support",
        "positive_response_bin_fraction": "statistical_support",
        "covered_lethargy_width": "spectral_support",
    }
    for name, value in metrics.items():
        _supp_add_feature(dataset_id, object_id, families.get(name, "spectral_shape"), condition, name, value, units.get(name, ""), quality, model_eligible, notes)
    return metrics


def _supp_plot_spectra(dataset_id, object_id, title, energy, series, outpath):
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (label, values) in enumerate(series):
        E, y = _supp_clean_energy_sequence(energy, values, min_points=12)
        scale = float(np.nanmax(np.abs(y)))
        yn = y / scale if np.isfinite(scale) and scale > 0 else y
        ax.plot(E, yn, lw=1.8, color=WARM_SEQUENCE[i % len(WARM_SEQUENCE)], label=str(label))
    if np.all(np.asarray(energy, float) > 0):
        ax.set_xscale("log")
    atlas_style_ax(ax, title, "Energy [MeV]", "Normalized response")
    ax.legend(fontsize=8, ncol=2)
    atlas_save_fig(fig, outpath)


# ------------------------------------------------------------
# N001 / N002: measured spectra as energy-ordered objects.
# ------------------------------------------------------------
for dataset_id, mev in (("N001", "43MEV"), ("N002", "68MEV")):
    basename = f"{dataset_id}_JAERI_TIARA_{mev}_R_E_t_x_reference.npz"
    source = PROJECT_ROOT / "results/phase1/ordered_response_fields" / basename
    if not source.is_file():
        source = atlas_resolve_by_basename(basename)
    if source is None or not Path(source).is_file():
        continue

    d = np.load(source, allow_pickle=False)
    energy_lo = np.asarray(d["energy_lower_MeV"], float)
    energy_hi = np.asarray(d["energy_upper_MeV"], float)
    E = 0.5 * (energy_lo + energy_hi)
    thickness = np.asarray(d["thickness_cm"], float)
    off_axis = np.asarray(d["off_axis_cm"], float)
    response = np.asarray(d["response"], float)
    obj = f"{dataset_id}.SEQ_SPEC"
    notes = "Each spectrum is analyzed at its own measured benchmark condition; historical configurations are not interpreted as a controlled thickness-only trajectory."

    for it, t in enumerate(thickness):
        for io, o in enumerate(off_axis):
            try:
                condition = f"thickness={float(t):.8g}cm;off_axis={float(o):.8g}cm"
                _supp_write_spectrum_features(
                    dataset_id, obj, condition, E, response[:, it, io], notes,
                    quality="descriptive", model_eligible=False,
                    energy_lower=energy_lo, energy_upper=energy_hi,
                )
            except Exception:
                continue

    io0 = int(np.argmin(np.abs(off_axis)))
    vis = [(f"{float(t):g} cm", response[:, it, io0]) for it, t in enumerate(thickness)]
    outdir = SUPPLEMENTAL_ROOT / dataset_id
    outdir.mkdir(parents=True, exist_ok=True)
    _supp_plot_spectra(
        dataset_id,
        obj,
        f"{dataset_id} — transmitted-spectrum shape by benchmark condition",
        E,
        vis,
        outdir / f"{dataset_id}_SEQ_SPEC_central_axis_spectra.png",
    )


# ------------------------------------------------------------
# N001: controlled fixed-geometry thickness sweep, when present.
# ------------------------------------------------------------
controlled = PROJECT_ROOT / "results/phase1/ordered_response_fields/controlled_R_E_t.npz"
if not controlled.is_file():
    candidates = [p for p in PROJECT_ROOT.rglob("controlled_R_E_t.npz") if not any(str(part).lower() in {"archive", "archives", "backup", "backups", ".ipynb_checkpoints"} for part in p.parts)]
    controlled = sorted(candidates, key=lambda p: (len(p.parts), len(str(p)), str(p)))[0] if candidates else None

if controlled is not None and Path(controlled).is_file():
    d = np.load(controlled, allow_pickle=False)
    energy_lo = np.asarray(d["energy_lower_MeV"], float)
    energy_hi = np.asarray(d["energy_upper_MeV"], float)
    E = 0.5 * (energy_lo + energy_hi)
    thickness = np.asarray(d["thickness_cm"], float)
    M = np.asarray(d["response"], float)
    S = np.asarray(d["response_sigma"], float) if "response_sigma" in d.files else None
    if M.shape == (len(thickness), len(E)):
        M = M.T
    if S is not None and S.shape == (len(thickness), len(E)):
        S = S.T
    if S is not None and S.shape != M.shape:
        S = None
    if M.shape == (len(E), len(thickness)):
        obj = "N001.TRAJ_DEPTH_CONTROLLED"
        notes = "Fixed geometry, source normalization, source energy and energy grid; only concrete thickness varies."
        rows = []
        for j, t in enumerate(thickness):
            try:
                metrics = _supp_write_spectrum_features(
                    "N001", obj, f"thickness={float(t):.8g}cm", E, M[:, j], notes,
                    quality="valid_low_state_count", model_eligible=False,
                    energy_lower=energy_lo, energy_upper=energy_hi,
                    response_sigma=(S[:, j] if S is not None else None),
                )
                rows.append({"thickness_cm": float(t), **metrics})
            except Exception:
                continue
        outdir = SUPPLEMENTAL_ROOT / "N001_CONTROLLED"
        outdir.mkdir(parents=True, exist_ok=True)
        if rows:
            cdf = pd.DataFrame(rows).sort_values("thickness_cm")
            cdf.to_csv(outdir / "N001_controlled_spectral_descriptors_by_thickness.csv", index=False)
            vis = [(f"{float(t):g} cm", M[:, j]) for j, t in enumerate(thickness)]
            _supp_plot_spectra("N001", obj, "N001 controlled field — spectral shape across concrete thickness", E, vis, outdir / "N001_controlled_spectra_by_thickness.png")

            for col, ylabel, filename in [
                ("response_weighted_mean_energy_MeV", "Response-weighted mean energy [MeV]", "N001_controlled_mean_energy_vs_thickness.png"),
                ("shape_entropy_bits", "Spectral shape entropy [bit]", "N001_controlled_shape_entropy_vs_thickness.png"),
                ("spectral_integral", r"Integrated response [n cm$^{-2}$ $\mu$C$^{-1}$]", "N001_controlled_integrated_response_vs_thickness.png"),
            ]:
                if col in cdf and cdf[col].notna().sum() >= 3:
                    fig, ax = plt.subplots(figsize=(8, 5))
                    ax.plot(cdf["thickness_cm"], cdf[col], color=RED, marker="o", markerfacecolor=BLACK, markeredgecolor=RED)
                    if col == "spectral_integral" and np.all(cdf[col].to_numpy(float) > 0):
                        ax.set_yscale("log")
                    atlas_style_ax(ax, f"N001 controlled field — {ylabel.split(' [')[0].lower()}", "Concrete thickness [cm]", ylabel)
                    atlas_save_fig(fig, outdir / filename)

            if {"median_relative_bin_mc_sigma", "p90_relative_bin_mc_sigma"}.issubset(cdf.columns):
                q = cdf[["thickness_cm", "median_relative_bin_mc_sigma", "p90_relative_bin_mc_sigma"]].dropna()
                if len(q) >= 3:
                    fig, ax = plt.subplots(figsize=(8, 5))
                    ax.plot(q["thickness_cm"], q["median_relative_bin_mc_sigma"], color=RED, marker="o", label="Median bin MC relative sigma")
                    ax.plot(q["thickness_cm"], q["p90_relative_bin_mc_sigma"], color=WHITE, marker="o", label="90th percentile bin MC relative sigma")
                    atlas_style_ax(ax, "N001 controlled field — Monte Carlo spectral support", "Concrete thickness [cm]", "Relative MC sigma")
                    ax.legend(fontsize=8)
                    atlas_save_fig(fig, outdir / "N001_controlled_mc_relative_uncertainty_vs_thickness.png")

            if len(cdf) >= 3:
                x = cdf["thickness_cm"].to_numpy(float)
                for col, unit in (("response_weighted_mean_energy_MeV", "MeV_per_cm"), ("shape_entropy_bits", "bit_per_cm")):
                    y = cdf[col].to_numpy(float)
                    good = np.isfinite(x) & np.isfinite(y)
                    if good.sum() >= 3:
                        slope = float(np.polyfit(x[good], y[good], 1)[0])
                        trend_notes = notes + " Trend is retained as a descriptive controlled-field feature; model eligibility requires an explicit statistical-support criterion."
                        _supp_add_feature("N001", obj, "controlled_depth_trend", "all_controlled_thicknesses", f"linear_slope_{col}", slope, unit, "descriptive_controlled_trend", False, trend_notes)
                integral = cdf["spectral_integral"].to_numpy(float)
                good = np.isfinite(x) & np.isfinite(integral) & (integral > 0)
                if good.sum() >= 3:
                    attenuation = float(-np.polyfit(x[good], np.log(integral[good]), 1)[0])
                    trend_notes = notes + " Trend is retained as a descriptive controlled-field feature; model eligibility requires an explicit statistical-support criterion."
                    _supp_add_feature("N001", obj, "controlled_depth_trend", "all_controlled_thicknesses", "effective_integrated_response_attenuation_per_cm", attenuation, "per_cm", "descriptive_controlled_trend", False, trend_notes)


# ------------------------------------------------------------
# P003: lower-order descriptors for 15-point buildup trajectories.
# ------------------------------------------------------------
p003 = atlas_resolve_by_basename("P003_concrete_exposure_buildup_FLUKA.csv")
if p003 is not None and Path(p003).is_file():
    df = atlas_read_table(p003)
    outdir = SUPPLEMENTAL_ROOT / "P003"
    outdir.mkdir(parents=True, exist_ok=True)
    curve_rows = []
    plotted = []
    for energy, g in df.groupby("photon_energy_MeV", sort=True):
        x, y = atlas_clean_curve(g["penetration_mfp"], g["exposure_buildup_factor"], min_points=8)
        order = np.argsort(x)
        x, y = np.asarray(x)[order], np.asarray(y)[order]
        area = _supp_trapezoid(y, x)
        terminal = float(y[-1])
        monotone_fraction = float(np.mean(np.diff(y) >= 0)) if len(y) > 1 else np.nan
        log_growth = float(np.polyfit(x, np.log(np.clip(y, 1e-300, None)), 1)[0]) if len(y) >= 3 else np.nan
        dy = np.gradient(y, x)
        d2 = np.gradient(dy, x)
        curvature_rms = float(np.sqrt(np.mean(d2 ** 2)))
        rec = {
            "photon_energy_MeV": float(energy),
            "n_points": int(len(x)),
            "area_under_buildup_curve_MFP": area,
            "terminal_buildup_factor": terminal,
            "monotone_step_fraction": monotone_fraction,
            "log_buildup_growth_per_MFP": log_growth,
            "curvature_rms_per_MFP2": curvature_rms,
        }
        curve_rows.append(rec)
        plotted.append((f"{float(energy):g} MeV", x, y))
        for name, value, unit in [
            ("area_under_buildup_curve_MFP", area, "MFP"),
            ("terminal_buildup_factor", terminal, ""),
            ("monotone_step_fraction", monotone_fraction, ""),
            ("log_buildup_growth_per_MFP", log_growth, "per_MFP"),
            ("curvature_rms_per_MFP2", curvature_rms, "per_MFP2"),
        ]:
            _supp_add_feature("P003", "P003.TRAJ_BUILD", "buildup_shape", f"photon_energy={float(energy):.8g}MeV", name, value, unit, "valid_short_curve", False, "Lower-order descriptors are used because the 15-point trajectories do not meet the support requirements of the higher-order estimators.")

    if curve_rows:
        pdf = pd.DataFrame(curve_rows).sort_values("photon_energy_MeV")
        pdf.to_csv(outdir / "P003_buildup_lower_order_descriptors.csv", index=False)
        fig, ax = plt.subplots(figsize=(9, 6))
        for i, (label, x, y) in enumerate(plotted):
            ax.plot(x, y, color=WARM_SEQUENCE[i % len(WARM_SEQUENCE)], lw=2, marker="o", ms=3, label=label)
        atlas_style_ax(ax, "P003 — concrete exposure buildup trajectories", "Penetration [MFP]", "Exposure buildup factor")
        ax.legend()
        atlas_save_fig(fig, outdir / "P003_buildup_trajectories.png")

        for col, ylabel, filename in [
            ("area_under_buildup_curve_MFP", "Area under buildup curve [MFP]", "P003_buildup_area_vs_energy.png"),
            ("log_buildup_growth_per_MFP", "Log-buildup growth [per MFP]", "P003_log_growth_vs_energy.png"),
            ("curvature_rms_per_MFP2", "Curvature RMS [per MFP²]", "P003_curvature_rms_vs_energy.png"),
        ]:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(pdf["photon_energy_MeV"], pdf[col], color=RED, marker="o", markerfacecolor=BLACK, markeredgecolor=RED)
            atlas_style_ax(ax, f"P003 — {ylabel.split(' [')[0].lower()}", "Photon energy [MeV]", ylabel)
            atlas_save_fig(fig, outdir / filename)


# ------------------------------------------------------------
# Sparse/reference roles remain explicit without pseudo-samples.
# ------------------------------------------------------------
sparse_roles = pd.DataFrame([
    {"dataset_id": "P002", "supplemental_role": "engineering_reference", "recommended_use": "TVL trend and reference constraints", "high_order_sequence_status": "insufficient ordered support"},
    {"dataset_id": "C001", "supplemental_role": "nuclear_data_anchor", "recommended_use": "capture-gamma line/reference constraint", "high_order_sequence_status": "single-line reference object"},
    {"dataset_id": "PN001", "supplemental_role": "benchmark_anchor", "recommended_use": "three-energy photoneutron yield trend", "high_order_sequence_status": "insufficient ordered support"},
    {"dataset_id": "P004", "supplemental_role": "material_response_field", "recommended_use": "material-by-energy response/regression with explicit material descriptors", "high_order_sequence_status": "five energy samples per material"},
    {"dataset_id": "NVAL_ZAMORANO", "supplemental_role": "spatial_validation", "recommended_use": "five-position measured validation", "high_order_sequence_status": "insufficient spatial samples"},
])
sparse_roles.to_csv(SUPPLEMENTAL_ROOT / "phase2_sparse_reference_roles.csv", index=False)


supp_features = pd.DataFrame(SUPPLEMENTAL_FEATURE_ROWS)
if len(supp_features):
    supp_features = supp_features.sort_values(["dataset_id", "object_id", "analysis_family", "condition", "feature_name"]).reset_index(drop=True)
    supp_features.to_csv(SUPPLEMENTAL_ROOT / "phase2_supplemental_feature_registry.csv", index=False)

supp_summary = pd.DataFrame([
    {"dataset_id": did, "feature_rows": int(len(g)), "object_count": int(g["object_id"].nunique()), "model_eligible_rows": int(g["model_eligible"].sum())}
    for did, g in supp_features.groupby("dataset_id", sort=True)
]) if len(supp_features) else pd.DataFrame(columns=["dataset_id", "feature_rows", "object_count", "model_eligible_rows"])
supp_summary.to_csv(SUPPLEMENTAL_ROOT / "phase2_supplemental_summary.csv", index=False)

display(supp_summary)
print("Additional ordered-object analyses complete:", SUPPLEMENTAL_ROOT)


In [ ]:
# ============================================================
# SUPPLEMENTAL OUTPUT QA
# This is intentionally separate from the established primary Phase-2 QA.
# ============================================================

SUPPLEMENTAL_QA_PATH = SUPPLEMENTAL_ROOT / "phase2_supplemental_result_QA.csv"
supp_qa_rows = []

for p in sorted(SUPPLEMENTAL_ROOT.rglob("*")):
    if not p.is_file() or p == SUPPLEMENTAL_QA_PATH:
        continue
    rel = str(p.relative_to(SUPPLEMENTAL_ROOT))
    suffix = p.suffix.lower()
    status = "ok"
    detail = ""
    try:
        if suffix == ".csv":
            qdf = pd.read_csv(p)
            if qdf.empty:
                raise ValueError("CSV has no data rows")
            num = qdf.select_dtypes(include=[np.number])
            if len(num.columns) and np.isinf(num.to_numpy(float, na_value=np.nan)).any():
                raise ValueError("numeric CSV content contains infinity")
            detail = f"rows={len(qdf)};cols={len(qdf.columns)}"
        elif suffix == ".png":
            img = plt.imread(p)
            if img.size == 0 or img.ndim < 2:
                raise ValueError("PNG has no readable pixel array")
            detail = f"shape={tuple(img.shape)}"
        elif suffix == ".json":
            json.loads(p.read_text(encoding="utf-8"))
            detail = "JSON parsed"
        elif suffix == ".txt":
            p.read_text(encoding="utf-8")
            detail = "text decoded"
        elif suffix == ".npz":
            with np.load(p, allow_pickle=True) as z:
                for key in z.files:
                    arr = np.asarray(z[key])
                    if np.issubdtype(arr.dtype, np.number) and not np.isfinite(arr).all():
                        raise ValueError(f"NPZ numeric array {key} contains nonfinite values")
            detail = "NPZ arrays loaded"
        else:
            detail = "file present"
    except Exception as exc:
        status = "problem"
        detail = f"{type(exc).__name__}: {exc}"
    supp_qa_rows.append({
        "relative_path": rel,
        "file_type": suffix.lstrip(".") or "none",
        "size_bytes": int(p.stat().st_size),
        "status": status,
        "detail": detail,
    })

supplemental_result_qa = pd.DataFrame(supp_qa_rows)
supplemental_result_qa.to_csv(SUPPLEMENTAL_QA_PATH, index=False)
if len(supplemental_result_qa) and not supplemental_result_qa["status"].eq("ok").all():
    bad = supplemental_result_qa.loc[~supplemental_result_qa["status"].eq("ok")]
    display(bad)
    raise RuntimeError("Supplemental output QA found one or more invalid artifacts")

display(supplemental_result_qa)
print("Supplemental output QA complete:", SUPPLEMENTAL_QA_PATH)
